# Alpha Auditor — Executable Calculation & Verification Notebook

**Experiment Title:** SPY -2.0% Daily Shock Rebound (1d Hold)  
**Original Hypothesis:** `I think the S&P 500 tends to recover the day after falling more than 2%.`  
**Audit Run ID:** `audit-284e1b9742` (Schema `2.0`)  
**Market Data As-Of Date:** `2025-12-31` | **Chronological Split Date:** `2021-12-30`  
**Spec SHA-256:** `2d2c474c5c58019e6bd51b25cf47459a99663dcef4e242d501990971508f2e10`  
**Aligned Data Slice SHA-256:** `58d5519a29358e420245ef62b9921bbf1d009a495c6c9d5e07a8dab0bf0c47d6`

---
This notebook is self-contained and executes end-to-end offline without external network access or API keys. Every signal, trade fill, fee deduction, high-water mark drawdown, econometric test, adversarial stress test, and scorecard verdict is recomputed from the aligned daily OHLCV dataset and verified against the AuditReport.

In [1]:
# Cell 1: Environment Setup, Library Versions & Deterministic Seeds
import hashlib
import json
import math
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
from scipy import stats
import statsmodels
import statsmodels.api as sm

REPO_ROOT = Path('/Users/simbashakib/Desktop/Alpha Audit')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from engine.schemas import StrategySpec
from engine.backtester import (
    compute_causal_signals,
    compute_pair_leg_weights,
    compute_parameter_sensitivity_surface,
    compute_standard_bracket_profiles,
    simulate_strategy_core,
    _compute_period_metrics,
)
from engine.econometrics import (
    TRADING_DAYS_PER_YEAR,
    compute_pairs_econometrics,
    deflated_sharpe_ratio_bailey_lopez,
    exact_sign_test_p_value,
    lo_mertens_sharpe_se_and_psr,
    newey_west_factor_regression,
    stationary_bootstrap_cis,
)
from engine.stress_tests import (
    build_hypothesis_scorecard_and_ledger,
    run_adversarial_stress_suite,
)

BOOTSTRAP_SEED = 20261107
MONTE_CARLO_SEED = 42
print(f'Python: {sys.version.split()[0]} | NumPy: {np.__version__} | pandas: {pd.__version__} | SciPy: {scipy.__version__} | statsmodels: {statsmodels.__version__}')
print(f'Audit Run ID: audit-284e1b9742 | Spec SHA-256: 2d2c474c5c58019e...')

Python: 3.14.6 | NumPy: 2.4.6 | pandas: 3.0.3 | SciPy: 1.18.0 | statsmodels: 0.14.6
Audit Run ID: audit-284e1b9742 | Spec SHA-256: 2d2c474c5c58019e...


## Cell 2 — Confirmed `StrategySpec` & Explicit vs. Software-Default Assumptions
Separates parameters explicitly stated in the user's natural-language prompt from literature defaults filled by the compiler.

In [2]:
# Cell 2: Reconstruct Confirmed StrategySpec & Inspect Assumptions
SPEC_JSON = '{\n  "hypothesis_text": "I think the S&P 500 tends to recover the day after falling more than 2%.",\n  "title": "SPY -2.0% Daily Shock Rebound (1d Hold)",\n  "strategy_family": "single_asset",\n  "template_id": "return_threshold",\n  "signal_variant": "cum_return_drop",\n  "asset_a": "SPY",\n  "asset_b": null,\n  "benchmark_ticker": "SPY",\n  "start_date": "2018-01-01",\n  "end_date": "2025-12-31",\n  "threshold_mode": "user_specified",\n  "parameters": {\n    "lookback_window": 1,\n    "secondary_window": 50,\n    "streak_length": 3,\n    "entry_threshold": 2.0,\n    "exit_threshold": 0.0,\n    "stop_z_threshold": 4.0,\n    "holding_period": 1,\n    "exit_mode": "fixed_hold",\n    "direction": "long"\n  },\n  "execution": {\n    "initial_capital": 100000.0,\n    "execution_timing": "next_open",\n    "commission_bps": 5.0,\n    "slippage_bps": 5.0,\n    "train_split_ratio": 0.5\n  },\n  "risk_controls": {\n    "max_drawdown_circuit_breaker_pct": 25.0,\n    "per_trade_stop_loss_pct": 15.0,\n    "max_holding_days": 20\n  },\n  "selected_diagnostics": [\n    "stationary_bootstrap",\n    "newey_west_alpha",\n    "subperiod_regimes",\n    "cost_sensitivity",\n    "parameter_sensitivity_grid"\n  ],\n  "human_confirmed": true,\n  "user_specified_fields": [\n    "asset_a",\n    "parameters.entry_threshold",\n    "parameters.holding_period"\n  ],\n  "software_inferred_defaults": {\n    "parameters.lookback_window": "1d",\n    "execution.execution_timing": "next_open (t+1)",\n    "execution.cost_bps": "10 bps (5 comm + 5 slip)"\n  },\n  "evaluation_stage": "confirmatory",\n  "prior_holdout_touches": 0\n}'
EXPECTED_SNAPSHOT = json.loads('{\n  "run_id": "audit-284e1b9742",\n  "schema_version": "2.0",\n  "spec_sha256": "2d2c474c5c58019e6bd51b25cf47459a99663dcef4e242d501990971508f2e10",\n  "split_date": "2021-12-30",\n  "market_as_of_date": "2025-12-31",\n  "aligned_slice_sha256": "58d5519a29358e420245ef62b9921bbf1d009a495c6c9d5e07a8dab0bf0c47d6",\n  "trade_count": 76,\n  "full_sample": {\n    "period_label": "Full Sample",\n    "start_date": "2018-01-02",\n    "end_date": "2025-12-31",\n    "trading_days": 2011,\n    "trade_count": 76,\n    "exposure_pct": 3.78,\n    "total_gross_return_pct": 9.74,\n    "total_net_return_pct": -5.73,\n    "cagr_net_pct": -0.74,\n    "benchmark_return_pct": 187.48,\n    "benchmark_cagr_pct": 14.15,\n    "excess_return_pct": -193.21,\n    "annualized_volatility_pct": 7.25,\n    "benchmark_volatility_pct": 19.45,\n    "max_drawdown_pct": 14.64,\n    "benchmark_max_drawdown_pct": 33.72,\n    "sharpe_ratio": -0.066,\n    "benchmark_sharpe_ratio": 0.778,\n    "sharpe_se_lo2002": 0.352,\n    "probabilistic_sharpe_psr_pct": 42.62,\n    "sortino_ratio": -0.085,\n    "calmar_ratio": -0.05,\n    "win_rate_pct": 53.95,\n    "profit_factor": 0.926,\n    "mean_trade_net_pct": -0.057,\n    "median_trade_net_pct": 0.221,\n    "std_trade_net_pct": 2.029,\n    "best_trade_net_pct": 3.823,\n    "worst_trade_net_pct": -6.784,\n    "skewness_daily": -2.89,\n    "kurtosis_daily": 50.0,\n    "t_stat_mean_return": -0.185,\n    "p_value_two_sided": 0.8532,\n    "sign_test_p_value": 0.5666,\n    "circuit_breaker_triggered": false,\n    "circuit_breaker_date": null,\n    "is_zero_exposure": false,\n    "halted_prior_to_period": false,\n    "status_note": null\n  },\n  "in_sample": {\n    "period_label": "In-Sample Formation",\n    "start_date": "2018-01-02",\n    "end_date": "2021-12-29",\n    "trading_days": 1006,\n    "trade_count": 42,\n    "exposure_pct": 4.17,\n    "total_gross_return_pct": 5.1,\n    "total_net_return_pct": -3.37,\n    "cagr_net_pct": -0.85,\n    "benchmark_return_pct": 90.41,\n    "benchmark_cagr_pct": 17.51,\n    "excess_return_pct": -93.77,\n    "annualized_volatility_pct": 9.02,\n    "benchmark_volatility_pct": 20.82,\n    "max_drawdown_pct": 14.64,\n    "benchmark_max_drawdown_pct": 33.72,\n    "sharpe_ratio": -0.05,\n    "benchmark_sharpe_ratio": 0.88,\n    "sharpe_se_lo2002": 0.499,\n    "probabilistic_sharpe_psr_pct": 46.05,\n    "sortino_ratio": -0.064,\n    "calmar_ratio": -0.058,\n    "win_rate_pct": 57.14,\n    "profit_factor": 0.925,\n    "mean_trade_net_pct": -0.06,\n    "median_trade_net_pct": 0.292,\n    "std_trade_net_pct": 2.072,\n    "best_trade_net_pct": 3.094,\n    "worst_trade_net_pct": -6.289,\n    "skewness_daily": -2.714,\n    "kurtosis_daily": 50.0,\n    "t_stat_mean_return": -0.099,\n    "p_value_two_sided": 0.9212,\n    "sign_test_p_value": 0.4408,\n    "circuit_breaker_triggered": false,\n    "circuit_breaker_date": null,\n    "is_zero_exposure": false,\n    "halted_prior_to_period": false,\n    "status_note": null\n  },\n  "out_of_sample": {\n    "period_label": "Out-of-Sample (Held-Out)",\n    "start_date": "2021-12-30",\n    "end_date": "2025-12-31",\n    "trading_days": 1005,\n    "trade_count": 34,\n    "exposure_pct": 3.38,\n    "total_gross_return_pct": 4.42,\n    "total_net_return_pct": -2.45,\n    "cagr_net_pct": -0.62,\n    "benchmark_return_pct": 50.98,\n    "benchmark_cagr_pct": 10.88,\n    "excess_return_pct": -53.43,\n    "annualized_volatility_pct": 4.86,\n    "benchmark_volatility_pct": 17.99,\n    "max_drawdown_pct": 9.23,\n    "benchmark_max_drawdown_pct": 24.5,\n    "sharpe_ratio": -0.103,\n    "benchmark_sharpe_ratio": 0.664,\n    "sharpe_se_lo2002": 0.498,\n    "probabilistic_sharpe_psr_pct": 41.77,\n    "sortino_ratio": -0.14,\n    "calmar_ratio": -0.067,\n    "win_rate_pct": 50.0,\n    "profit_factor": 0.927,\n    "mean_trade_net_pct": -0.053,\n    "median_trade_net_pct": 0.091,\n    "std_trade_net_pct": 2.006,\n    "best_trade_net_pct": 3.823,\n    "worst_trade_net_pct": -6.784,\n    "skewness_daily": -1.805,\n    "kurtosis_daily": 50.0,\n    "t_stat_mean_return": -0.207,\n    "p_value_two_sided": 0.8363,\n    "sign_test_p_value": 1.0,\n    "circuit_breaker_triggered": false,\n    "circuit_breaker_date": null,\n    "is_zero_exposure": false,\n    "halted_prior_to_period": false,\n    "status_note": null\n  },\n  "factor_regression": {\n    "annualized_alpha_pct": -2.86,\n    "alpha_t_stat_hac": -1.441,\n    "alpha_p_value_hac": 0.1495,\n    "beta": 0.157,\n    "beta_t_stat_hac": 3.79,\n    "r_squared": 0.1786,\n    "information_ratio": -0.884,\n    "newey_west_lags": 7\n  },\n  "bootstrap_cis": [\n    {\n      "metric_name": "Annualized Return (Mean)",\n      "point_estimate": -0.47,\n      "ci_lower_95": -4.11,\n      "ci_upper_95": 2.91,\n      "bootstrap_std_error": 1.76,\n      "block_length": 13,\n      "replications": 1000,\n      "unit": "%"\n    },\n    {\n      "metric_name": "Annualized Sharpe Ratio",\n      "point_estimate": -0.066,\n      "ci_lower_95": -0.578,\n      "ci_upper_95": 0.423,\n      "bootstrap_std_error": 0.255,\n      "block_length": 13,\n      "replications": 1000,\n      "unit": "ratio"\n    },\n    {\n      "metric_name": "Annualized Excess Return vs. Benchmark",\n      "point_estimate": -15.61,\n      "ci_lower_95": -25.66,\n      "ci_upper_95": -5.25,\n      "bootstrap_std_error": 5.2,\n      "block_length": 13,\n      "replications": 1000,\n      "unit": "%"\n    },\n    {\n      "metric_name": "Mean Net Return per Trade",\n      "point_estimate": -0.057,\n      "ci_lower_95": -0.505,\n      "ci_upper_95": 0.371,\n      "bootstrap_std_error": 0.224,\n      "block_length": 1,\n      "replications": 1000,\n      "unit": "%"\n    }\n  ],\n  "stress_test_outcomes": [\n    {\n      "test_id": "outlier_removal",\n      "outcome_badge": "HIGH_VULNERABILITY",\n      "baseline_value": "-5.73% (N=76)",\n      "stressed_value": "-15.07% (N=73)"\n    },\n    {\n      "test_id": "execution_latency_shock",\n      "outcome_badge": "HIGH_VULNERABILITY",\n      "baseline_value": "-5.73% (SR -0.07)",\n      "stressed_value": "-16.49% (SR -0.35)"\n    },\n    {\n      "test_id": "regime_vulnerability",\n      "outcome_badge": "HIGH_VULNERABILITY",\n      "baseline_value": "1 / 5 Regimes",\n      "stressed_value": "0.03 vs 0.61"\n    },\n    {\n      "test_id": "monte_carlo_random_entry",\n      "outcome_badge": "MODERATE_SENSITIVITY",\n      "baseline_value": "-5.73% (71th pct)",\n      "stressed_value": "-10.99% (p=0.290\\u00b10.020)"\n    },\n    {\n      "test_id": "parameter_cliff_edge",\n      "outcome_badge": "MODERATE_SENSITIVITY",\n      "baseline_value": "-0.07",\n      "stressed_value": "-0.14 (10/25 pos)"\n    }\n  ],\n  "pairs_diagnostics": null,\n  "verdict_label": "CONTRADICTED BY HISTORICAL DATA",\n  "supported_checks_count": 0,\n  "evaluated_checks_count": 5,\n  "dimension_statuses": {\n    "post_signal_direction": "MIXED",\n    "statistical_significance": "CONTRADICTED",\n    "out_of_sample_persistence": "CONTRADICTED",\n    "transaction_cost_survival": "CONTRADICTED",\n    "stress_test_resilience": "CONTRADICTED"\n  }\n}')
spec = StrategySpec.model_validate_json(SPEC_JSON)
computed_spec_sha256 = hashlib.sha256(spec.model_dump_json().encode('utf-8')).hexdigest()
assert computed_spec_sha256 == EXPECTED_SNAPSHOT['spec_sha256'], 'Spec SHA-256 mismatch!'

assumptions_rows = [
    ('Strategy Family / Template', f'{spec.strategy_family} / {spec.template_id} ({spec.signal_variant})', 'Core Mapping'),
    ('Target Asset(s)', f'{spec.asset_a}' + (f' / {spec.asset_b}' if spec.asset_b else ''), 'User Stated' if 'asset_a' in spec.user_specified_fields else 'Default'),
    ('Benchmark Ticker', spec.benchmark_ticker, 'Default'),
    ('Primary Lookback Window', f'{spec.parameters.lookback_window} trading days', 'User Stated' if 'parameters.lookback_window' in spec.user_specified_fields else 'Software Default'),
    ('Entry Threshold', f'{spec.parameters.entry_threshold}', 'User Stated' if 'parameters.entry_threshold' in spec.user_specified_fields else 'Software Default'),
    ('Exit Threshold / Mode', f'{spec.parameters.exit_threshold} ({spec.parameters.exit_mode})', 'Software Default'),
    ('Holding Period / Max Hold', f'{spec.parameters.holding_period}d (Max {spec.risk_controls.max_holding_days}d)', 'User Stated' if 'parameters.holding_period' in spec.user_specified_fields else 'Software Default'),
    ('Execution Timing & Costs', f'{spec.execution.execution_timing}, {spec.execution.commission_bps} bps comm + {spec.execution.slippage_bps} bps slip', 'Software Default'),
    ('Risk Controls', f'Circuit Breaker: {spec.risk_controls.max_drawdown_circuit_breaker_pct}%, Stop-Loss: {spec.risk_controls.per_trade_stop_loss_pct}%', 'Software Default'),
]
assumptions_df = pd.DataFrame(assumptions_rows, columns=['Parameter', 'Configured Value', 'Provenance'])
display(assumptions_df)

,Parameter,Configured Value,Provenance
0,Strategy Family / Template,single_asset / return_threshold (cum_return_drop),Core Mapping
1,Target Asset(s),SPY,User Stated
2,Benchmark Ticker,SPY,Default
3,Primary Lookback Window,1 trading days,Software Default
4,Entry Threshold,2.0,User Stated
5,Exit Threshold / Mode,0.0 (fixed_hold),Software Default
6,Holding Period / Max Hold,1d (Max 20d),User Stated
7,Execution Timing & Costs,"next_open, 5.0 bps comm + 5.0 bps slip",Software Default
8,Risk Controls,"Circuit Breaker: 25.0%, Stop-Loss: 15.0%",Software Default


## Cell 3 — Immutable Aligned Market Data Snapshot & Integrity Verification
Loads the exact inner-joined daily OHLCV market slice embedded at audit time and verifies its cryptographic SHA-256 digest, date monotonicity, and chronological In-Sample / Out-of-Sample split boundary.

In [3]:
# Cell 3: Load Embedded Aligned Market Data & Verify Integrity
DF_RECORDS = json.loads('[{"date":"2018-01-02","open_a":234.55539,"high_a":235.404849,"low_a":234.170067,"close_a":235.369812,"volume_a":86655700,"open_bench":234.55539,"close_bench":235.369812},{"date":"2018-01-03","open_a":235.536252,"high_a":237.007498,"low_a":235.536252,"close_a":236.858612,"volume_a":90070400,"open_bench":235.536252,"close_bench":236.858612},{"date":"2018-01-04","open_a":237.497832,"high_a":238.338525,"low_a":236.919847,"close_a":237.856857,"volume_a":80636400,"open_bench":237.497832,"close_bench":237.856857},{"date":"2018-01-05","open_a":238.645069,"high_a":239.564574,"low_a":238.154663,"close_a":239.441986,"volume_a":83524000,"open_bench":238.645069,"close_bench":239.441986},{"date":"2018-01-08","open_a":239.345582,"high_a":240.037416,"low_a":239.056604,"close_a":239.879791,"volume_a":57319200,"open_bench":239.345582,"close_bench":239.879791},{"date":"2018-01-09","open_a":240.300206,"high_a":241.044582,"low_a":240.019966,"close_a":240.422821,"volume_a":57254000,"open_bench":240.300206,"close_bench":240.422821},{"date":"2018-01-10","open_a":239.669639,"high_a":240.317697,"low_a":239.004103,"close_a":240.054962,"volume_a":69574300,"open_bench":239.669639,"close_bench":240.054962},{"date":"2018-01-11","open_a":240.606667,"high_a":241.806412,"low_a":240.440276,"close_a":241.806412,"volume_a":62361500,"open_bench":240.606667,"close_bench":241.806412},{"date":"2018-01-12","open_a":242.069233,"high_a":243.549192,"low_a":241.771461,"close_a":243.382828,"volume_a":90816100,"open_bench":242.069233,"close_bench":243.382828},{"date":"2018-01-16","open_a":244.635068,"high_a":245.283099,"low_a":241.858993,"close_a":242.550827,"volume_a":106555100,"open_bench":244.635068,"close_bench":242.550827},{"date":"2018-01-17","open_a":243.479076,"high_a":245.24804,"low_a":242.550805,"close_a":244.862717,"volume_a":113258800,"open_bench":243.479076,"close_bench":244.862717},{"date":"2018-01-18","open_a":244.748901,"high_a":245.169235,"low_a":243.960723,"close_a":244.451157,"volume_a":100728000,"open_bench":244.748901,"close_bench":244.451157},{"date":"2018-01-19","open_a":245.029115,"high_a":245.563324,"low_a":244.451157,"close_a":245.563324,"volume_a":140920100,"open_bench":245.029115,"close_bench":245.563324},{"date":"2018-01-22","open_a":245.353093,"high_a":247.559921,"low_a":245.300525,"close_a":247.559921,"volume_a":91322400,"open_bench":245.353093,"close_bench":247.559921},{"date":"2018-01-23","open_a":247.603797,"high_a":248.374444,"low_a":247.279781,"close_a":248.085464,"volume_a":97084700,"open_bench":247.603797,"close_bench":248.085464},{"date":"2018-01-24","open_a":248.72473,"high_a":249.320247,"low_a":246.815646,"close_a":247.98912,"volume_a":134816100,"open_bench":248.72473,"close_bench":247.98912},{"date":"2018-01-25","open_a":248.84738,"high_a":248.943697,"low_a":247.306087,"close_a":248.094238,"volume_a":84587300,"open_bench":248.84738,"close_bench":248.094238},{"date":"2018-01-26","open_a":248.92613,"high_a":251.010371,"low_a":248.672161,"close_a":250.966568,"volume_a":107743100,"open_bench":248.92613,"close_bench":250.966568},{"date":"2018-01-29","open_a":250.397351,"high_a":250.835215,"low_a":249.145064,"close_a":249.302689,"volume_a":90118300,"open_bench":250.397351,"close_bench":249.302689},{"date":"2018-01-30","open_a":247.481215,"high_a":249.355263,"low_a":246.272704,"close_a":246.745605,"volume_a":131796400,"open_bench":247.481215,"close_bench":246.745605},{"date":"2018-01-31","open_a":247.595035,"high_a":248.09418,"low_a":245.799773,"close_a":246.868164,"volume_a":108364800,"open_bench":247.595035,"close_bench":246.868164},{"date":"2018-02-01","open_a":246.141256,"high_a":247.883949,"low_a":245.799709,"close_a":246.58786,"volume_a":90102500,"open_bench":246.141256,"close_bench":246.58786},{"date":"2018-02-02","open_a":245.274363,"high_a":245.405744,"low_a":241.18472,"close_a":241.219757,"volume_a":173174800,"open_bench":245.274363,"close_bench":241.219757},{"date":"2018-02-05","open_a":239.468296,"high_a":241.570042,"low_a":230.588384,"close_a":231.131332,"volume_a":294681800,"open_bench":239.468296,"close_bench":231.131332},{"date":"2018-02-06","open_a":227.637162,"high_a":236.184292,"low_a":226.551266,"close_a":235.68512,"volume_a":355026800,"open_bench":227.637162,"close_bench":235.68512},{"date":"2018-02-07","open_a":235.13343,"high_a":238.513733,"low_a":234.327747,"close_a":234.406586,"volume_a":167376100,"open_bench":235.13343,"close_bench":234.406586},{"date":"2018-02-08","open_a":234.704257,"high_a":234.844377,"low_a":225.579145,"close_a":225.614182,"volume_a":246449500,"open_bench":234.704257,"close_bench":225.614182},{"date":"2018-02-09","open_a":228.390274,"high_a":230.851073,"low_a":221.489534,"close_a":229.003296,"volume_a":283565300,"open_bench":228.390274,"close_bench":229.003296},{"date":"2018-02-12","open_a":231.043698,"high_a":233.828538,"low_a":229.14338,"close_a":232.366058,"volume_a":143736000,"open_bench":231.043698,"close_bench":232.366058},{"date":"2018-02-13","open_a":231.166301,"high_a":233.486979,"low_a":230.588316,"close_a":232.944031,"volume_a":81223600,"open_bench":231.166301,"close_bench":232.944031},{"date":"2018-02-14","open_a":231.46413,"high_a":236.447035,"low_a":231.455364,"close_a":236.087982,"volume_a":120735700,"open_bench":231.46413,"close_bench":236.087982},{"date":"2018-02-15","open_a":237.821845,"high_a":239.109169,"low_a":235.369787,"close_a":239.100403,"volume_a":111200300,"open_bench":237.821845,"close_bench":239.100403},{"date":"2018-02-16","open_a":238.478664,"high_a":241.105853,"low_a":238.434862,"close_a":239.170471,"volume_a":160420100,"open_bench":238.478664,"close_bench":239.170471},{"date":"2018-02-20","open_a":238.224687,"high_a":239.660897,"low_a":236.884822,"close_a":237.672974,"volume_a":86369700,"open_bench":238.224687,"close_bench":237.672974},{"date":"2018-02-21","open_a":238.11095,"high_a":240.580515,"low_a":236.394527,"close_a":236.490845,"volume_a":98883700,"open_bench":238.11095,"close_bench":236.490845},{"date":"2018-02-22","open_a":237.410308,"high_a":239.117965,"low_a":236.131751,"close_a":236.797287,"volume_a":110511300,"open_bench":237.410308,"close_bench":236.797287},{"date":"2018-02-23","open_a":238.01454,"high_a":240.571655,"low_a":237.541639,"close_a":240.571655,"volume_a":92766400,"open_bench":238.01454,"close_bench":240.571655},{"date":"2018-02-26","open_a":241.640061,"high_a":243.461594,"low_a":241.053337,"close_a":243.36525,"volume_a":86491400,"open_bench":241.640061,"close_bench":243.36525},{"date":"2018-02-27","open_a":243.549187,"high_a":244.258552,"low_a":240.2652,"close_a":240.326508,"volume_a":99099200,"open_bench":243.549187,"close_bench":240.326508},{"date":"2018-02-28","open_a":241.421126,"high_a":241.867757,"low_a":237.576687,"close_a":237.891937,"volume_a":121907800,"open_bench":241.421126,"close_bench":237.891937},{"date":"2018-03-01","open_a":237.681749,"high_a":239.223042,"low_a":232.944049,"close_a":234.4328,"volume_a":176855100,"open_bench":237.681749,"close_bench":234.4328},{"date":"2018-03-02","open_a":232.76892,"high_a":236.201791,"low_a":231.910722,"close_a":235.641312,"volume_a":139083200,"open_bench":232.76892,"close_bench":235.641312},{"date":"2018-03-05","open_a":234.459061,"high_a":238.977828,"low_a":234.353951,"close_a":238.364807,"volume_a":97307400,"open_bench":234.459061,"close_bench":238.364807},{"date":"2018-03-06","open_a":239.336847,"high_a":239.415686,"low_a":237.480304,"close_a":238.969055,"volume_a":79213200,"open_bench":239.336847,"close_bench":238.969055},{"date":"2018-03-07","open_a":236.814776,"high_a":239.231771,"low_a":236.622115,"close_a":238.881485,"volume_a":87063500,"open_bench":236.814776,"close_bench":238.881485},{"date":"2018-03-08","open_a":239.555778,"high_a":240.160033,"low_a":238.566226,"close_a":240.037445,"volume_a":66901200,"open_bench":239.555778,"close_bench":240.037445},{"date":"2018-03-09","open_a":241.438613,"high_a":244.214661,"low_a":241.123337,"close_a":244.214661,"volume_a":113625300,"open_bench":241.438613,"close_bench":244.214661},{"date":"2018-03-12","open_a":244.503643,"high_a":245.125403,"low_a":243.522804,"close_a":243.908127,"volume_a":71924800,"open_bench":244.503643,"close_bench":243.908127},{"date":"2018-03-13","open_a":245.064166,"high_a":245.563338,"low_a":241.727638,"close_a":242.331894,"volume_a":91968900,"open_bench":245.064166,"close_bench":242.331894},{"date":"2018-03-14","open_a":243.286407,"high_a":243.470303,"low_a":240.536631,"close_a":241.088318,"volume_a":105895100,"open_bench":243.286407,"close_bench":241.088318},{"date":"2018-03-15","open_a":241.596299,"high_a":242.235565,"low_a":240.326481,"close_a":240.825653,"volume_a":83433000,"open_bench":241.596299,"close_bench":240.825653},{"date":"2018-03-16","open_a":241.350537,"high_a":242.133071,"low_a":241.034025,"close_a":241.086777,"volume_a":100343700,"open_bench":241.350537,"close_bench":241.086777},{"date":"2018-03-19","open_a":240.33943,"high_a":241.262618,"low_a":236.18063,"close_a":237.824799,"volume_a":109208400,"open_bench":240.33943,"close_bench":237.824799},{"date":"2018-03-20","open_a":238.220493,"high_a":238.862346,"low_a":237.552264,"close_a":238.229294,"volume_a":59757300,"open_bench":238.220493,"close_bench":238.229294},{"date":"2018-03-21","open_a":238.185383,"high_a":240.269171,"low_a":237.561132,"close_a":237.772141,"volume_a":78709600,"open_bench":238.185383,"close_bench":237.772141},{"date":"2018-03-22","open_a":235.556371,"high_a":236.400431,"low_a":231.555827,"close_a":231.828415,"volume_a":148785900,"open_bench":235.556371,"close_bench":231.828415},{"date":"2018-03-23","open_a":232.267975,"high_a":232.593288,"low_a":226.693591,"close_a":226.887024,"volume_a":183534800,"open_bench":232.267975,"close_bench":226.887024},{"date":"2018-03-26","open_a":230.474328,"high_a":233.375799,"low_a":228.082803,"close_a":233.094437,"volume_a":141956100,"open_bench":230.474328,"close_bench":233.094437},{"date":"2018-03-27","open_a":234.026491,"high_a":234.554012,"low_a":227.58167,"close_a":229.129135,"volume_a":129941400,"open_bench":234.026491,"close_bench":229.129135},{"date":"2018-03-28","open_a":229.260997,"high_a":230.922768,"low_a":227.353041,"close_a":228.452087,"volume_a":146452300,"open_bench":229.260997,"close_bench":228.452087},{"date":"2018-03-29","open_a":229.58629,"high_a":233.226343,"low_a":228.460868,"close_a":231.37114,"volume_a":111601600,"open_bench":229.58629,"close_bench":231.37114},{"date":"2018-04-02","open_a":230.843617,"high_a":231.353589,"low_a":223.91524,"close_a":226.377106,"volume_a":186286300,"open_bench":230.843617,"close_bench":226.377106},{"date":"2018-04-03","open_a":227.60802,"high_a":229.75336,"low_a":225.82317,"close_a":229.278564,"volume_a":119956900,"open_bench":227.60802,"close_bench":229.278564},{"date":"2018-04-04","open_a":225.744103,"high_a":232.435084,"low_a":225.612223,"close_a":231.731705,"volume_a":123715300,"open_bench":225.744103,"close_bench":231.731705},{"date":"2018-04-05","open_a":233.481331,"high_a":234.439723,"low_a":232.399887,"close_a":233.560486,"volume_a":82652600,"open_bench":233.481331,"close_bench":233.560486},{"date":"2018-04-06","open_a":231.608595,"high_a":233.094481,"low_a":226.843119,"close_a":228.355408,"volume_a":179521200,"open_bench":231.608595,"close_bench":228.355408},{"date":"2018-04-09","open_a":229.806164,"high_a":232.857117,"low_a":228.548861,"close_a":229.48085,"volume_a":105442900,"open_bench":229.806164,"close_bench":229.48085},{"date":"2018-04-10","open_a":232.355936,"high_a":233.912203,"low_a":231.22174,"close_a":233.129669,"volume_a":103529000,"open_bench":232.355936,"close_bench":233.129669},{"date":"2018-04-11","open_a":231.65253,"high_a":233.560486,"low_a":231.582203,"close_a":231.907516,"volume_a":91140200,"open_bench":231.65253,"close_bench":231.907516},{"date":"2018-04-12","open_a":233.226447,"high_a":234.756311,"low_a":233.050589,"close_a":233.815521,"volume_a":68890500,"open_bench":233.226447,"close_bench":233.815521},{"date":"2018-04-13","open_a":235.116753,"high_a":235.231058,"low_a":232.127353,"close_a":233.129669,"volume_a":85079200,"open_bench":235.116753,"close_bench":233.129669},{"date":"2018-04-16","open_a":234.756235,"high_a":235.81133,"low_a":233.938551,"close_a":235.046371,"volume_a":63405300,"open_bench":234.756235,"close_bench":235.046371},{"date":"2018-04-17","open_a":236.804817,"high_a":238.158849,"low_a":236.294871,"close_a":237.560974,"volume_a":64682000,"open_bench":236.804817,"close_bench":237.560974},{"date":"2018-04-18","open_a":238.000668,"high_a":238.53699,"low_a":237.279688,"close_a":237.736908,"volume_a":57303900,"open_bench":238.000668,"close_bench":237.736908},{"date":"2018-04-19","open_a":237.086293,"high_a":237.288527,"low_a":235.389372,"close_a":236.418091,"volume_a":77655900,"open_bench":237.086293,"close_bench":236.418091},{"date":"2018-04-20","open_a":236.347616,"high_a":236.567425,"low_a":233.534048,"close_a":234.413284,"volume_a":99953100,"open_bench":236.347616,"close_bench":234.413284},{"date":"2018-04-23","open_a":234.98488,"high_a":235.538804,"low_a":233.305534,"close_a":234.378204,"volume_a":65558000,"open_bench":234.98488,"close_bench":234.378204},{"date":"2018-04-24","open_a":235.398116,"high_a":235.617925,"low_a":229.727027,"close_a":231.221741,"volume_a":112885500,"open_bench":235.398116,"close_bench":231.221741},{"date":"2018-04-25","open_a":231.160146,"high_a":232.232816,"low_a":229.34892,"close_a":231.793198,"volume_a":103840900,"open_bench":231.160146,"close_bench":231.793198},{"date":"2018-04-26","open_a":232.813106,"high_a":234.976021,"low_a":232.373488,"close_a":234.149536,"volume_a":67731900,"open_bench":232.813106,"close_bench":234.149536},{"date":"2018-04-27","open_a":234.756159,"high_a":235.055097,"low_a":233.437305,"close_a":234.369293,"volume_a":57053600,"open_bench":234.756159,"close_bench":234.369293},{"date":"2018-04-30","open_a":234.984919,"high_a":235.538842,"low_a":232.496663,"close_a":232.567017,"volume_a":82182300,"open_bench":234.984919,"close_bench":232.567017},{"date":"2018-05-01","open_a":232.004212,"high_a":233.085683,"low_a":230.456747,"close_a":232.980179,"volume_a":74203400,"open_bench":232.004212,"close_bench":232.980179},{"date":"2018-05-02","open_a":232.786799,"high_a":233.595682,"low_a":231.028325,"close_a":231.415192,"volume_a":86368900,"open_bench":232.786799,"close_bench":231.415192},{"date":"2018-05-03","open_a":230.588654,"high_a":231.555793,"low_a":227.766285,"close_a":230.905167,"volume_a":136311500,"open_bench":230.588654,"close_bench":230.905167},{"date":"2018-05-04","open_a":229.938026,"high_a":234.571622,"low_a":229.612713,"close_a":233.894592,"volume_a":91222100,"open_bench":229.938026,"close_bench":233.894592},{"date":"2018-05-07","open_a":234.659522,"high_a":235.653037,"low_a":233.973691,"close_a":234.685898,"volume_a":55304900,"open_bench":234.659522,"close_bench":234.685898},{"date":"2018-05-08","open_a":234.316607,"high_a":235.046361,"low_a":233.129632,"close_a":234.685898,"volume_a":67499200,"open_bench":234.316607,"close_bench":234.685898},{"date":"2018-05-09","open_a":235.354098,"high_a":237.279629,"low_a":234.835351,"close_a":236.954315,"volume_a":59666100,"open_bench":235.354098,"close_bench":236.954315},{"date":"2018-05-10","open_a":237.692841,"high_a":239.495293,"low_a":237.587337,"close_a":239.169952,"volume_a":72063900,"open_bench":237.692841,"close_bench":239.169952},{"date":"2018-05-11","open_a":239.293089,"high_a":240.163525,"low_a":238.783117,"close_a":239.899765,"volume_a":59871500,"open_bench":239.293089,"close_bench":239.899765},{"date":"2018-05-14","open_a":240.330566,"high_a":240.981192,"low_a":239.468904,"close_a":240.014053,"volume_a":54790600,"open_bench":240.330566,"close_bench":240.014053},{"date":"2018-05-15","open_a":238.791978,"high_a":238.809553,"low_a":237.42037,"close_a":238.36116,"volume_a":87036100,"open_bench":238.791978,"close_bench":238.36116},{"date":"2018-05-16","open_a":238.396371,"high_a":239.820731,"low_a":238.369968,"close_a":239.36351,"volume_a":53942600,"open_bench":238.396371,"close_bench":239.36351},{"date":"2018-05-17","open_a":239.099671,"high_a":240.233894,"low_a":238.387492,"close_a":239.161224,"volume_a":56536400,"open_bench":239.099671,"close_bench":239.161224},{"date":"2018-05-18","open_a":238.818371,"high_a":239.178862,"low_a":238.211696,"close_a":238.563385,"volume_a":64368000,"open_bench":238.818371,"close_bench":238.563385},{"date":"2018-05-21","open_a":240.040482,"high_a":240.893343,"low_a":239.653616,"close_a":240.356995,"volume_a":58025900,"open_bench":240.040482,"close_bench":240.356995},{"date":"2018-05-22","open_a":240.875834,"high_a":241.13082,"low_a":239.363545,"close_a":239.688858,"volume_a":52966400,"open_bench":240.875834,"close_bench":239.688858},{"date":"2018-05-23","open_a":238.422656,"high_a":240.374563,"low_a":238.264373,"close_a":240.34816,"volume_a":64694200,"open_bench":238.422656,"close_bench":240.34816},{"date":"2018-05-24","open_a":239.952565,"high_a":240.225126,"low_a":238.079786,"close_a":239.855835,"volume_a":76043800,"open_bench":239.952565,"close_bench":239.855835},{"date":"2018-05-25","open_a":239.284271,"high_a":239.908522,"low_a":238.7831,"close_a":239.284271,"volume_a":56374800,"open_bench":239.284271,"close_bench":239.284271},{"date":"2018-05-29","open_a":237.666557,"high_a":238.422714,"low_a":235.424513,"close_a":236.532333,"volume_a":115908600,"open_bench":237.666557,"close_bench":236.532333},{"date":"2018-05-30","open_a":237.83368,"high_a":240.128477,"low_a":237.763353,"close_a":239.688858,"volume_a":69678400,"open_bench":237.83368,"close_bench":239.688858},{"date":"2018-05-31","open_a":239.284282,"high_a":239.583219,"low_a":237.622539,"close_a":238.220413,"volume_a":93519900,"open_bench":239.284282,"close_bench":238.220413},{"date":"2018-06-01","open_a":239.512925,"high_a":240.858157,"low_a":239.442571,"close_a":240.559219,"volume_a":71258400,"open_bench":239.512925,"close_bench":240.559219},{"date":"2018-06-04","open_a":241.376896,"high_a":241.957196,"low_a":241.139512,"close_a":241.702209,"volume_a":45385200,"open_bench":241.376896,"close_bench":241.702209},{"date":"2018-06-05","open_a":241.83412,"high_a":242.256163,"low_a":241.069188,"close_a":241.878098,"volume_a":51135000,"open_bench":241.83412,"close_bench":241.878098},{"date":"2018-06-06","open_a":242.484741,"high_a":244.005803,"low_a":241.869264,"close_a":243.900299,"volume_a":62732200,"open_bench":242.484741,"close_bench":243.900299},{"date":"2018-06-07","open_a":244.383874,"high_a":244.67401,"low_a":242.968289,"close_a":243.873901,"volume_a":72969400,"open_bench":244.383874,"close_bench":243.873901},{"date":"2018-06-08","open_a":243.416735,"high_a":244.647662,"low_a":243.249678,"close_a":244.59491,"volume_a":72139700,"open_bench":243.416735,"close_bench":244.59491},{"date":"2018-06-11","open_a":244.814738,"high_a":245.632422,"low_a":244.700433,"close_a":244.920242,"volume_a":58892500,"open_bench":244.814738,"close_bench":244.920242},{"date":"2018-06-12","open_a":245.333413,"high_a":245.597173,"low_a":244.594857,"close_a":245.23671,"volume_a":72329000,"open_bench":245.333413,"close_bench":245.23671},{"date":"2018-06-13","open_a":245.474126,"high_a":245.729112,"low_a":244.251974,"close_a":244.454208,"volume_a":79070600,"open_bench":245.474126,"close_bench":244.454208},{"date":"2018-06-14","open_a":245.315796,"high_a":245.597131,"low_a":244.48051,"close_a":245.069611,"volume_a":77097600,"open_bench":245.315796,"close_bench":245.069611},{"date":"2018-06-15","open_a":244.288932,"high_a":245.092634,"low_a":243.184951,"close_a":244.757019,"volume_a":120041600,"open_bench":244.288932,"close_bench":244.757019},{"date":"2018-06-18","open_a":243.308602,"high_a":244.377275,"low_a":242.831702,"close_a":244.253616,"volume_a":52917600,"open_bench":243.308602,"close_bench":244.253616},{"date":"2018-06-19","open_a":241.992666,"high_a":243.53824,"low_a":241.577568,"close_a":243.317444,"volume_a":97531500,"open_bench":241.992666,"close_bench":243.317444},{"date":"2018-06-20","open_a":243.99755,"high_a":244.394994,"low_a":243.39699,"close_a":243.732605,"volume_a":53785500,"open_bench":243.99755,"close_bench":243.732605},{"date":"2018-06-21","open_a":243.723776,"high_a":243.741457,"low_a":241.710115,"close_a":242.204697,"volume_a":71061400,"open_bench":243.723776,"close_bench":242.204697},{"date":"2018-06-22","open_a":243.458782,"high_a":243.573601,"low_a":242.425444,"close_a":242.64624,"volume_a":54898500,"open_bench":243.458782,"close_bench":242.64624},{"date":"2018-06-25","open_a":241.498129,"high_a":241.657095,"low_a":237.665109,"close_a":239.343155,"volume_a":137854200,"open_bench":241.498129,"close_bench":239.343155},{"date":"2018-06-26","open_a":239.908382,"high_a":240.720897,"low_a":239.157669,"close_a":239.873047,"volume_a":68547400,"open_bench":239.908382,"close_bench":239.873047},{"date":"2018-06-27","open_a":240.455951,"high_a":241.877866,"low_a":237.735726,"close_a":237.88588,"volume_a":105110700,"open_bench":240.455951,"close_bench":237.88588},{"date":"2018-06-28","open_a":237.832916,"high_a":240.005544,"low_a":237.126352,"close_a":239.246017,"volume_a":76650500,"open_bench":237.832916,"close_bench":239.246017},{"date":"2018-06-29","open_a":240.332326,"high_a":241.692439,"low_a":239.475636,"close_a":239.590454,"volume_a":97592500,"open_bench":240.332326,"close_bench":239.590454},{"date":"2018-07-02","open_a":238.027168,"high_a":240.261624,"low_a":237.788691,"close_a":240.102631,"volume_a":63554800,"open_bench":238.027168,"close_bench":240.102631},{"date":"2018-07-03","open_a":240.994682,"high_a":241.091846,"low_a":238.830895,"close_a":239.254807,"volume_a":42187100,"open_bench":240.994682,"close_bench":239.254807},{"date":"2018-07-05","open_a":240.376496,"high_a":241.268494,"low_a":239.307823,"close_a":241.206665,"volume_a":56925900,"open_bench":240.376496,"close_bench":241.206665},{"date":"2018-07-06","open_a":241.233165,"high_a":243.617749,"low_a":240.853375,"close_a":243.246826,"volume_a":66493700,"open_bench":241.233165,"close_bench":243.246826},{"date":"2018-07-09","open_a":244.244874,"high_a":245.490169,"low_a":244.200726,"close_a":245.43718,"volume_a":50550400,"open_bench":244.244874,"close_bench":245.43718},{"date":"2018-07-10","open_a":245.88756,"high_a":246.417477,"low_a":245.596094,"close_a":246.320312,"volume_a":51966800,"open_bench":245.88756,"close_bench":246.320312},{"date":"2018-07-11","open_a":244.774655,"high_a":245.560703,"low_a":244.218245,"close_a":244.518524,"volume_a":77054700,"open_bench":244.774655,"close_bench":244.518524},{"date":"2018-07-12","open_a":245.772683,"high_a":246.78834,"low_a":245.172124,"close_a":246.735352,"volume_a":60124700,"open_bench":245.772683,"close_bench":246.735352},{"date":"2018-07-13","open_a":246.55875,"high_a":247.229952,"low_a":246.108317,"close_a":246.929672,"volume_a":48216000,"open_bench":246.55875,"close_bench":246.929672},{"date":"2018-07-16","open_a":246.973864,"high_a":247.11515,"low_a":246.2673,"close_a":246.708893,"volume_a":48201000,"open_bench":246.973864,"close_bench":246.708893},{"date":"2018-07-17","open_a":245.940494,"high_a":248.095467,"low_a":245.887505,"close_a":247.706863,"volume_a":52315500,"open_bench":245.940494,"close_bench":247.706863},{"date":"2018-07-18","open_a":247.786359,"high_a":248.333929,"low_a":247.344766,"close_a":248.227951,"volume_a":44593500,"open_bench":247.786359,"close_bench":248.227951},{"date":"2018-07-19","open_a":247.565594,"high_a":247.945357,"low_a":246.814882,"close_a":247.291809,"volume_a":61412100,"open_bench":247.565594,"close_bench":247.291809},{"date":"2018-07-20","open_a":247.088623,"high_a":247.715704,"low_a":246.850173,"close_a":247.00914,"volume_a":82337700,"open_bench":247.088623,"close_bench":247.00914},{"date":"2018-07-23","open_a":246.806071,"high_a":247.671575,"low_a":246.461616,"close_a":247.46846,"volume_a":47047600,"open_bench":246.806071,"close_bench":247.46846},{"date":"2018-07-24","open_a":248.872678,"high_a":249.552721,"low_a":247.84818,"close_a":248.713684,"volume_a":68026900,"open_bench":248.872678,"close_bench":248.713684},{"date":"2018-07-25","open_a":248.466403,"high_a":251.151292,"low_a":248.422254,"close_a":250.833359,"volume_a":78882900,"open_bench":248.466403,"close_bench":250.833359},{"date":"2018-07-26","open_a":250.117991,"high_a":250.921666,"low_a":250.020827,"close_a":250.241623,"volume_a":57919500,"open_bench":250.117991,"close_bench":250.241623},{"date":"2018-07-27","open_a":250.568372,"high_a":250.665537,"low_a":247.627379,"close_a":248.545898,"volume_a":76768700,"open_bench":250.568372,"close_bench":248.545898},{"date":"2018-07-30","open_a":248.625478,"high_a":248.784445,"low_a":246.726608,"close_a":247.247711,"volume_a":63742500,"open_bench":248.625478,"close_bench":247.247711},{"date":"2018-07-31","open_a":248.007215,"high_a":249.075861,"low_a":247.627451,"close_a":248.466461,"volume_a":68570500,"open_bench":248.007215,"close_bench":248.466461},{"date":"2018-08-01","open_a":248.669556,"high_a":249.172977,"low_a":247.406608,"close_a":248.051315,"volume_a":53853300,"open_bench":248.669556,"close_bench":248.051315},{"date":"2018-08-02","open_a":246.753079,"high_a":249.570414,"low_a":246.549936,"close_a":249.402634,"volume_a":63426400,"open_bench":246.753079,"close_bench":249.402634},{"date":"2018-08-03","open_a":249.526238,"high_a":250.524241,"low_a":249.34959,"close_a":250.471252,"volume_a":53935400,"open_bench":249.526238,"close_bench":250.471252},{"date":"2018-08-06","open_a":250.506586,"high_a":251.698864,"low_a":250.117982,"close_a":251.389771,"volume_a":39400900,"open_bench":250.506586,"close_bench":251.389771},{"date":"2018-08-07","open_a":252.05219,"high_a":252.599761,"low_a":251.919691,"close_a":252.219971,"volume_a":43196600,"open_bench":252.05219,"close_bench":252.219971},{"date":"2018-08-08","open_a":252.052165,"high_a":252.511412,"low_a":251.654721,"close_a":252.113968,"volume_a":42114600,"open_bench":252.052165,"close_bench":252.113968},{"date":"2018-08-09","open_a":252.175819,"high_a":252.564423,"low_a":251.637089,"close_a":251.769562,"volume_a":35717000,"open_bench":252.175819,"close_bench":251.769562},{"date":"2018-08-10","open_a":250.33888,"high_a":250.87761,"low_a":249.376184,"close_a":250.082748,"volume_a":77076000,"open_bench":250.33888,"close_bench":250.082748},{"date":"2018-08-13","open_a":250.35652,"high_a":250.96592,"low_a":248.855094,"close_a":249.146561,"volume_a":65732900,"open_bench":250.35652,"close_bench":249.146561},{"date":"2018-08-14","open_a":249.870748,"high_a":250.974729,"low_a":249.482144,"close_a":250.736252,"volume_a":43842000,"open_bench":249.870748,"close_bench":250.736252},{"date":"2018-08-15","open_a":249.393777,"high_a":249.53509,"low_a":247.433106,"close_a":248.863861,"volume_a":102925400,"open_bench":249.393777,"close_bench":248.863861},{"date":"2018-08-16","open_a":250.294612,"high_a":251.743049,"low_a":250.259277,"close_a":250.877518,"volume_a":69967900,"open_bench":250.294612,"close_bench":250.877518},{"date":"2018-08-17","open_a":250.674445,"high_a":252.202365,"low_a":250.268188,"close_a":251.760773,"volume_a":65618500,"open_bench":250.674445,"close_bench":251.760773},{"date":"2018-08-20","open_a":252.211145,"high_a":252.564414,"low_a":251.760712,"close_a":252.299469,"volume_a":39807500,"open_bench":252.211145,"close_bench":252.299469},{"date":"2018-08-21","open_a":252.811706,"high_a":253.74788,"low_a":252.334779,"close_a":252.89119,"volume_a":67272000,"open_bench":252.811706,"close_bench":252.89119},{"date":"2018-08-22","open_a":252.484897,"high_a":253.262104,"low_a":252.219925,"close_a":252.741028,"volume_a":44993300,"open_bench":252.484897,"close_bench":252.741028},{"date":"2018-08-23","open_a":252.564454,"high_a":253.421145,"low_a":252.087526,"close_a":252.405487,"volume_a":49204900,"open_bench":252.564454,"close_bench":252.405487},{"date":"2018-08-24","open_a":252.9795,"high_a":254.065828,"low_a":252.926511,"close_a":253.924515,"volume_a":57487400,"open_bench":252.9795,"close_bench":253.924515},{"date":"2018-08-27","open_a":255.11684,"high_a":256.03536,"low_a":254.957873,"close_a":255.929382,"volume_a":57072400,"open_bench":255.11684,"close_bench":255.929382},{"date":"2018-08-28","open_a":256.388597,"high_a":256.494602,"low_a":255.593736,"close_a":256.053009,"volume_a":46943500,"open_bench":256.388597,"close_bench":256.053009},{"date":"2018-08-29","open_a":256.264945,"high_a":257.660365,"low_a":256.026495,"close_a":257.430756,"volume_a":61485500,"open_bench":256.264945,"close_bench":257.430756},{"date":"2018-08-30","open_a":256.953893,"high_a":257.324815,"low_a":255.796922,"close_a":256.388641,"volume_a":61229500,"open_bench":256.953893,"close_bench":256.388641},{"date":"2018-08-31","open_a":255.982363,"high_a":256.839053,"low_a":255.496622,"close_a":256.397461,"volume_a":66140800,"open_bench":255.982363,"close_bench":256.397461},{"date":"2018-09-04","open_a":255.982366,"high_a":256.30914,"low_a":254.957868,"close_a":255.955872,"volume_a":57594400,"open_bench":255.982366,"close_bench":255.955872},{"date":"2018-09-05","open_a":255.602582,"high_a":255.805725,"low_a":254.260151,"close_a":255.266968,"volume_a":72452400,"open_bench":255.602582,"close_bench":255.266968},{"date":"2018-09-06","open_a":255.37291,"high_a":255.67319,"low_a":253.474068,"close_a":254.498566,"volume_a":65909900,"open_bench":255.37291,"close_bench":254.498566},{"date":"2018-09-07","open_a":253.456504,"high_a":254.975583,"low_a":253.218026,"close_a":254.004074,"volume_a":73524800,"open_bench":253.456504,"close_bench":254.004074},{"date":"2018-09-10","open_a":255.010903,"high_a":255.275875,"low_a":254.251377,"close_a":254.445679,"volume_a":50210900,"open_bench":255.010903,"close_bench":254.445679},{"date":"2018-09-11","open_a":253.800847,"high_a":255.726184,"low_a":253.456419,"close_a":255.284592,"volume_a":50530500,"open_bench":253.800847,"close_bench":255.284592},{"date":"2018-09-12","open_a":255.293401,"high_a":255.946949,"low_a":254.560369,"close_a":255.34639,"volume_a":59810800,"open_bench":255.293401,"close_bench":255.34639},{"date":"2018-09-13","open_a":256.406222,"high_a":257.042117,"low_a":256.123597,"close_a":256.856628,"volume_a":51034200,"open_bench":256.406222,"close_bench":256.856628},{"date":"2018-09-14","open_a":257.059815,"high_a":257.245277,"low_a":256.123641,"close_a":256.900848,"volume_a":55079900,"open_bench":257.059815,"close_bench":256.900848},{"date":"2018-09-17","open_a":256.84788,"high_a":256.883188,"low_a":255.266971,"close_a":255.540756,"volume_a":68244000,"open_bench":256.84788,"close_bench":255.540756},{"date":"2018-09-18","open_a":255.752717,"high_a":257.519087,"low_a":255.726222,"close_a":256.927368,"volume_a":61930400,"open_bench":255.752717,"close_bench":256.927368},{"date":"2018-09-19","open_a":256.980376,"high_a":257.61627,"low_a":256.856717,"close_a":257.201172,"volume_a":49080600,"open_bench":256.980376,"close_bench":257.201172},{"date":"2018-09-20","open_a":258.455292,"high_a":259.603422,"low_a":257.218812,"close_a":259.285461,"volume_a":100360600,"open_bench":258.455292,"close_bench":259.285461},{"date":"2018-09-21","open_a":260.02443,"high_a":260.139768,"low_a":258.888837,"close_a":259.048523,"volume_a":105479700,"open_bench":260.02443,"close_bench":259.048523},{"date":"2018-09-24","open_a":258.471802,"high_a":258.613754,"low_a":257.611233,"close_a":258.187897,"volume_a":53409600,"open_bench":258.471802,"close_bench":258.187897},{"date":"2018-09-25","open_a":258.640549,"high_a":258.747007,"low_a":257.709018,"close_a":257.948547,"volume_a":44370000,"open_bench":258.640549,"close_bench":257.948547},{"date":"2018-09-26","open_a":258.090433,"high_a":259.270375,"low_a":256.759659,"close_a":257.176636,"volume_a":79739700,"open_bench":258.090433,"close_bench":257.176636},{"date":"2018-09-27","open_a":257.646853,"high_a":258.977628,"low_a":257.371829,"close_a":257.895264,"volume_a":59249500,"open_bench":257.646853,"close_bench":257.895264},{"date":"2018-09-28","open_a":257.274191,"high_a":258.418665,"low_a":257.238723,"close_a":257.921844,"volume_a":70091400,"open_bench":257.274191,"close_bench":257.921844},{"date":"2018-10-01","open_a":259.154917,"high_a":259.882413,"low_a":258.152423,"close_a":258.81781,"volume_a":62078900,"open_bench":259.154917,"close_bench":258.81781},{"date":"2018-10-02","open_a":258.667023,"high_a":259.376758,"low_a":258.294421,"close_a":258.667023,"volume_a":47258200,"open_bench":258.667023,"close_bench":258.667023},{"date":"2018-10-03","open_a":259.713877,"high_a":260.130854,"low_a":258.454092,"close_a":258.80896,"volume_a":64694600,"open_bench":259.713877,"close_bench":258.80896},{"date":"2018-10-04","open_a":258.329945,"high_a":258.383173,"low_a":255.20707,"close_a":256.786255,"volume_a":111545900,"open_bench":258.329945,"close_bench":256.786255},{"date":"2018-10-05","open_a":257.008057,"high_a":257.522612,"low_a":253.929531,"close_a":255.34903,"volume_a":105951700,"open_bench":257.008057,"close_bench":255.34903},{"date":"2018-10-08","open_a":254.665881,"high_a":255.703897,"low_a":253.290758,"close_a":255.34903,"volume_a":87742200,"open_bench":254.665881,"close_bench":255.34903},{"date":"2018-10-09","open_a":254.967541,"high_a":256.271674,"low_a":254.417465,"close_a":254.976395,"volume_a":74339000,"open_bench":254.967541,"close_bench":254.976395},{"date":"2018-10-10","open_a":254.470732,"high_a":254.541722,"low_a":246.530459,"close_a":246.903061,"volume_a":214731000,"open_bench":254.470732,"close_bench":246.903061},{"date":"2018-10-11","open_a":245.82066,"high_a":247.435339,"low_a":239.858789,"close_a":241.464615,"volume_a":274840500,"open_bench":245.82066,"close_bench":241.464615},{"date":"2018-10-12","open_a":245.545692,"high_a":245.829597,"low_a":241.642091,"close_a":244.818222,"volume_a":183186500,"open_bench":245.545692,"close_bench":244.818222},{"date":"2018-10-15","open_a":244.463263,"high_a":245.785185,"low_a":243.354285,"close_a":243.443008,"volume_a":102263700,"open_bench":244.463263,"close_bench":243.443008},{"date":"2018-10-16","open_a":245.394783,"high_a":249.138696,"low_a":244.924578,"close_a":248.766068,"volume_a":118255800,"open_bench":245.394783,"close_bench":248.766068},{"date":"2018-10-17","open_a":248.801575,"high_a":249.431467,"low_a":246.246484,"close_a":248.810455,"volume_a":110626000,"open_bench":248.801575,"close_bench":248.810455},{"date":"2018-10-18","open_a":247.878895,"high_a":248.473319,"low_a":243.948681,"close_a":245.217346,"volume_a":134557500,"open_bench":247.878895,"close_bench":245.217346},{"date":"2018-10-19","open_a":245.865122,"high_a":247.790295,"low_a":244.392394,"close_a":245.084396,"volume_a":139901600,"open_bench":245.865122,"close_bench":245.084396},{"date":"2018-10-22","open_a":245.749723,"high_a":246.069096,"low_a":243.451922,"close_a":243.984238,"volume_a":82415800,"open_bench":245.749723,"close_bench":243.984238},{"date":"2018-10-23","open_a":240.382228,"high_a":243.85997,"low_a":238.306197,"close_a":242.742111,"volume_a":146352700,"open_bench":240.382228,"close_bench":242.742111},{"date":"2018-10-24","open_a":242.493692,"high_a":242.875201,"low_a":234.837326,"close_a":235.387375,"volume_a":177806700,"open_bench":242.493692,"close_bench":235.387375},{"date":"2018-10-25","open_a":237.215004,"high_a":241.145218,"low_a":236.194749,"close_a":239.610382,"volume_a":138061500,"open_bench":237.215004,"close_bench":239.610382},{"date":"2018-10-26","open_a":235.919702,"high_a":240.426579,"low_a":232.699224,"close_a":235.39624,"volume_a":201574600,"open_bench":235.919702,"close_bench":235.39624},{"date":"2018-10-29","open_a":238.474804,"high_a":239.76123,"low_a":230.534532,"close_a":234.092117,"volume_a":160749100,"open_bench":238.474804,"close_bench":234.092117},{"date":"2018-10-30","open_a":233.923561,"high_a":237.871509,"low_a":233.435594,"close_a":237.560989,"volume_a":157116000,"open_bench":233.923561,"close_bench":237.560989},{"date":"2018-10-31","open_a":240.116062,"high_a":242.405009,"low_a":239.645856,"close_a":240.098328,"volume_a":128296300,"open_bench":240.116062,"close_bench":240.098328},{"date":"2018-11-01","open_a":240.958904,"high_a":242.848608,"low_a":239.87654,"close_a":242.653427,"volume_a":99495000,"open_bench":240.958904,"close_bench":242.653427},{"date":"2018-11-02","open_a":243.753563,"high_a":244.179421,"low_a":239.175695,"close_a":241.216232,"volume_a":122634100,"open_bench":243.753563,"close_bench":241.216232},{"date":"2018-11-05","open_a":241.704132,"high_a":243.097016,"low_a":240.737106,"close_a":242.546967,"volume_a":65622500,"open_bench":241.704132,"close_bench":242.546967},{"date":"2018-11-06","open_a":242.484899,"high_a":244.241504,"low_a":242.42279,"close_a":244.081818,"volume_a":60085900,"open_bench":242.484899,"close_bench":244.081818},{"date":"2018-11-07","open_a":246.246595,"high_a":249.387232,"low_a":245.820737,"close_a":249.307388,"volume_a":102752100,"open_bench":246.246595,"close_bench":249.307388},{"date":"2018-11-08","open_a":248.508768,"high_a":249.493555,"low_a":247.719189,"close_a":248.854782,"volume_a":65584900,"open_bench":248.508768,"close_bench":248.854782},{"date":"2018-11-09","open_a":247.55077,"high_a":247.737071,"low_a":245.022292,"close_a":246.424057,"volume_a":98812600,"open_bench":247.55077,"close_bench":246.424057},{"date":"2018-11-12","open_a":245.918239,"high_a":246.157768,"low_a":241.304877,"close_a":241.819458,"volume_a":99673600,"open_bench":245.918239,"close_bench":241.819458},{"date":"2018-11-13","open_a":242.280786,"high_a":244.268068,"low_a":240.648373,"close_a":241.366989,"volume_a":98176600,"open_bench":242.280786,"close_bench":241.366989},{"date":"2018-11-14","open_a":243.230141,"high_a":243.629358,"low_a":238.164333,"close_a":239.716904,"volume_a":125335900,"open_bench":243.230141,"close_bench":239.716904},{"date":"2018-11-15","open_a":238.456952,"high_a":242.67995,"low_a":236.886648,"close_a":242.218597,"volume_a":135101400,"open_bench":238.456952,"close_bench":242.218597},{"date":"2018-11-16","open_a":241.12745,"high_a":243.753504,"low_a":240.612869,"close_a":242.848587,"volume_a":126668000,"open_bench":241.12745,"close_bench":242.848587},{"date":"2018-11-19","open_a":242.245339,"high_a":242.538125,"low_a":237.827185,"close_a":238.740982,"volume_a":103061700,"open_bench":242.245339,"close_bench":238.740982},{"date":"2018-11-20","open_a":235.422913,"high_a":236.877907,"low_a":233.462246,"close_a":234.322815,"volume_a":136021300,"open_bench":235.422913,"close_bench":234.322815},{"date":"2018-11-21","open_a":235.8664,"high_a":237.010874,"low_a":235.112317,"close_a":235.12117,"volume_a":75563700,"open_bench":235.8664,"close_bench":235.12117},{"date":"2018-11-23","open_a":233.488825,"high_a":234.943818,"low_a":233.391247,"close_a":233.550934,"volume_a":42807900,"open_bench":233.488825,"close_bench":233.550934},{"date":"2018-11-26","open_a":235.795486,"high_a":237.543237,"low_a":235.405123,"close_a":237.321442,"volume_a":79981400,"open_bench":235.795486,"close_bench":237.321442},{"date":"2018-11-27","open_a":236.292294,"high_a":238.119888,"low_a":235.689016,"close_a":238.119888,"volume_a":75502400,"open_bench":236.292294,"close_bench":238.119888},{"date":"2018-11-28","open_a":239.184568,"high_a":243.602722,"low_a":238.057828,"close_a":243.602722,"volume_a":127629600,"open_bench":239.184568,"close_bench":243.602722},{"date":"2018-11-29","open_a":242.830832,"high_a":244.463245,"low_a":241.695239,"close_a":243.070389,"volume_a":82346400,"open_bench":242.830832,"close_bench":243.070389},{"date":"2018-11-30","open_a":242.919588,"high_a":245.110932,"low_a":242.600215,"close_a":244.552002,"volume_a":98204200,"open_bench":242.919588,"close_bench":244.552002},{"date":"2018-12-03","open_a":248.659625,"high_a":248.766083,"low_a":246.202138,"close_a":247.790176,"volume_a":103176300,"open_bench":248.659625,"close_bench":247.790176},{"date":"2018-12-04","open_a":246.965153,"high_a":247.39101,"low_a":239.450711,"close_a":239.76123,"volume_a":177986000,"open_bench":246.965153,"close_bench":239.76123},{"date":"2018-12-06","open_a":235.919719,"high_a":239.512799,"low_a":232.832312,"close_a":239.397461,"volume_a":204185400,"open_bench":235.919719,"close_bench":239.397461},{"date":"2018-12-07","open_a":239.060348,"high_a":240.621799,"low_a":233.0009,"close_a":233.834854,"volume_a":161018900,"open_bench":239.060348,"close_bench":233.834854},{"date":"2018-12-10","open_a":233.657358,"high_a":235.245423,"low_a":229.443239,"close_a":234.278397,"volume_a":151445900,"open_bench":233.657358,"close_bench":234.278397},{"date":"2018-12-11","open_a":237.463389,"high_a":237.64969,"low_a":232.867788,"close_a":234.331635,"volume_a":121504400,"open_bench":237.463389,"close_bench":234.331635},{"date":"2018-12-12","open_a":237.294797,"high_a":238.652186,"low_a":235.431708,"close_a":235.511551,"volume_a":97976700,"open_bench":237.294797,"close_bench":235.511551},{"date":"2018-12-13","open_a":236.452033,"high_a":237.312602,"low_a":234.322799,"close_a":235.431778,"volume_a":96662700,"open_bench":236.452033,"close_bench":235.431778},{"date":"2018-12-14","open_a":233.29361,"high_a":234.242903,"low_a":230.534485,"close_a":231.084534,"volume_a":116961100,"open_bench":233.29361,"close_bench":231.084534},{"date":"2018-12-17","open_a":230.135269,"high_a":231.244248,"low_a":224.927509,"close_a":226.551056,"volume_a":165492300,"open_bench":230.135269,"close_bench":226.551056},{"date":"2018-12-18","open_a":228.183465,"high_a":228.848852,"low_a":224.705696,"close_a":226.302628,"volume_a":134515100,"open_bench":228.183465,"close_bench":226.302628},{"date":"2018-12-19","open_a":226.382509,"high_a":230.13529,"low_a":221.219111,"close_a":222.91362,"volume_a":214992800,"open_bench":226.382509,"close_bench":222.91362},{"date":"2018-12-20","open_a":221.671559,"high_a":223.232996,"low_a":217.049329,"close_a":219.285034,"volume_a":252053400,"open_bench":221.671559,"close_bench":219.285034},{"date":"2018-12-21","open_a":220.181862,"high_a":222.832184,"low_a":214.149474,"close_a":214.791977,"volume_a":255345600,"open_bench":220.181862,"close_bench":214.791977},{"date":"2018-12-24","open_a":213.310623,"high_a":214.91688,"low_a":209.054058,"close_a":209.116516,"volume_a":147311600,"open_bench":213.310623,"close_bench":209.116516},{"date":"2018-12-26","open_a":210.571102,"high_a":219.682129,"low_a":208.598971,"close_a":219.682129,"volume_a":218485400,"open_bench":210.571102,"close_bench":219.682129},{"date":"2018-12-27","open_a":216.460697,"high_a":221.565006,"low_a":213.239263,"close_a":221.368698,"volume_a":186267300,"open_bench":216.460697,"close_bench":221.368698},{"date":"2018-12-28","open_a":222.716173,"high_a":224.340268,"low_a":219.923069,"close_a":221.083145,"volume_a":153100200,"open_bench":222.716173,"close_bench":221.083145},{"date":"2018-12-31","open_a":222.698341,"high_a":223.260534,"low_a":220.833303,"close_a":223.019592,"volume_a":144299400,"open_bench":222.698341,"close_bench":223.019592},{"date":"2019-01-02","open_a":219.503661,"high_a":224.170734,"low_a":219.476892,"close_a":223.251587,"volume_a":126925200,"open_bench":219.503661,"close_bench":223.251587},{"date":"2019-01-03","open_a":221.511472,"high_a":221.814886,"low_a":217.442295,"close_a":217.924179,"volume_a":144140700,"open_bench":221.511472,"close_bench":217.924179},{"date":"2019-01-04","open_a":220.94036,"high_a":225.866212,"low_a":220.565569,"close_a":225.223709,"volume_a":142628800,"open_bench":220.94036,"close_bench":225.223709},{"date":"2019-01-07","open_a":225.491384,"high_a":228.400485,"low_a":224.599021,"close_a":226.999481,"volume_a":103139100,"open_bench":225.491384,"close_bench":226.999481},{"date":"2019-01-08","open_a":229.176928,"high_a":229.614178,"low_a":226.660455,"close_a":229.132294,"volume_a":102512600,"open_bench":229.176928,"close_bench":229.132294},{"date":"2019-01-09","open_a":229.837222,"high_a":231.041919,"low_a":228.614688,"close_a":230.203094,"volume_a":95006600,"open_bench":229.837222,"close_bench":230.203094},{"date":"2019-01-10","open_a":228.677239,"high_a":231.265089,"low_a":227.999033,"close_a":231.015228,"volume_a":96823900,"open_bench":228.677239,"close_bench":231.015228},{"date":"2019-01-11","open_a":229.944342,"high_a":231.131201,"low_a":229.364311,"close_a":231.104431,"volume_a":73858100,"open_bench":229.944342,"close_bench":231.104431},{"date":"2019-01-14","open_a":229.212559,"high_a":230.497565,"low_a":228.811011,"close_a":229.694443,"volume_a":70908200,"open_bench":229.212559,"close_bench":229.694443},{"date":"2019-01-15","open_a":230.069195,"high_a":232.639206,"low_a":230.060263,"close_a":232.326874,"volume_a":85208300,"open_bench":230.069195,"close_bench":232.326874},{"date":"2019-01-16","open_a":232.755223,"high_a":233.772531,"low_a":232.549997,"close_a":232.889099,"volume_a":77636700,"open_bench":232.755223,"close_bench":232.889099},{"date":"2019-01-17","open_a":232.023549,"high_a":235.512695,"low_a":231.978914,"close_a":234.656006,"volume_a":96118400,"open_bench":232.023549,"close_bench":234.656006},{"date":"2019-01-18","open_a":236.458555,"high_a":238.243282,"low_a":234.691665,"close_a":237.779236,"volume_a":127900300,"open_bench":236.458555,"close_bench":237.779236},{"date":"2019-01-22","open_a":236.315779,"high_a":236.529938,"low_a":232.960483,"close_a":234.566727,"volume_a":115531200,"open_bench":236.315779,"close_bench":234.566727},{"date":"2019-01-23","open_a":235.593026,"high_a":236.289069,"low_a":232.603601,"close_a":235.057602,"volume_a":86030300,"open_bench":235.593026,"close_bench":235.057602},{"date":"2019-01-24","open_a":234.879095,"high_a":235.762553,"low_a":233.870719,"close_a":235.182495,"volume_a":59204100,"open_bench":234.879095,"close_bench":235.182495},{"date":"2019-01-25","open_a":237.020756,"high_a":237.993457,"low_a":235.280663,"close_a":237.17247,"volume_a":96883400,"open_bench":237.020756,"close_bench":237.17247},{"date":"2019-01-28","open_a":235.039733,"high_a":235.432348,"low_a":233.611946,"close_a":235.369904,"volume_a":85613700,"open_bench":235.039733,"close_bench":235.369904},{"date":"2019-01-29","open_a":235.512716,"high_a":236.074883,"low_a":234.22771,"close_a":235.057602,"volume_a":66136300,"open_bench":235.512716,"close_bench":235.057602},{"date":"2019-01-30","open_a":236.565596,"high_a":239.617465,"low_a":235.807082,"close_a":238.778641,"volume_a":92473700,"open_bench":236.565596,"close_bench":238.778641},{"date":"2019-01-31","open_a":238.716258,"high_a":241.357647,"low_a":238.502072,"close_a":240.875763,"volume_a":104012100,"open_bench":238.716258,"close_bench":240.875763},{"date":"2019-02-01","open_a":241.072054,"high_a":242.009053,"low_a":240.206461,"close_a":240.991745,"volume_a":85782500,"open_bench":241.072054,"close_bench":240.991745},{"date":"2019-02-04","open_a":241.036393,"high_a":242.749743,"low_a":240.36712,"close_a":242.687271,"volume_a":60744800,"open_bench":241.036393,"close_bench":242.687271},{"date":"2019-02-05","open_a":243.115566,"high_a":244.007929,"low_a":242.615844,"close_a":243.704529,"volume_a":79552800,"open_bench":243.115566,"close_bench":243.704529},{"date":"2019-02-06","open_a":243.427951,"high_a":243.91874,"low_a":242.651598,"close_a":243.383316,"volume_a":58347800,"open_bench":243.427951,"close_bench":243.383316},{"date":"2019-02-07","open_a":241.777082,"high_a":242.321411,"low_a":239.412323,"close_a":241.063202,"volume_a":95482000,"open_bench":241.777082,"close_bench":241.063202},{"date":"2019-02-08","open_a":239.822769,"high_a":241.455782,"low_a":239.001782,"close_a":241.357635,"volume_a":75788900,"open_bench":239.822769,"close_bench":241.357635},{"date":"2019-02-11","open_a":242.009148,"high_a":242.267914,"low_a":240.96507,"close_a":241.491562,"volume_a":68021400,"open_bench":242.009148,"close_bench":241.491562},{"date":"2019-02-12","open_a":243.097714,"high_a":244.971656,"low_a":243.026309,"close_a":244.596878,"volume_a":72270200,"open_bench":243.097714,"close_bench":244.596878},{"date":"2019-02-13","open_a":245.426892,"high_a":246.230014,"low_a":245.00748,"close_a":245.39119,"volume_a":65277200,"open_bench":245.426892,"close_bench":245.39119},{"date":"2019-02-14","open_a":244.311408,"high_a":245.97122,"low_a":243.499354,"close_a":244.846832,"volume_a":83234400,"open_bench":244.311408,"close_bench":244.846832},{"date":"2019-02-15","open_a":246.613627,"high_a":247.550625,"low_a":246.408401,"close_a":247.514923,"volume_a":97088700,"open_bench":246.613627,"close_bench":247.514923},{"date":"2019-02-19","open_a":246.720825,"high_a":248.594768,"low_a":246.711893,"close_a":247.943359,"volume_a":59120800,"open_bench":246.720825,"close_bench":247.943359},{"date":"2019-02-20","open_a":247.907569,"high_a":248.898107,"low_a":247.407848,"close_a":248.442993,"volume_a":76610800,"open_bench":247.907569,"close_bench":248.442993},{"date":"2019-02-21","open_a":247.809324,"high_a":248.166264,"low_a":246.604628,"close_a":247.559464,"volume_a":64214700,"open_bench":247.809324,"close_bench":247.559464},{"date":"2019-02-22","open_a":248.175291,"high_a":249.290746,"low_a":247.54172,"close_a":249.094452,"volume_a":78114600,"open_bench":248.175291,"close_bench":249.094452},{"date":"2019-02-25","open_a":250.513313,"high_a":251.030872,"low_a":249.353223,"close_a":249.433533,"volume_a":69030700,"open_bench":250.513313,"close_bench":249.433533},{"date":"2019-02-26","open_a":249.085515,"high_a":250.129565,"low_a":248.880261,"close_a":249.255066,"volume_a":56844100,"open_bench":249.085515,"close_bench":249.255066},{"date":"2019-02-27","open_a":248.541121,"high_a":249.495957,"low_a":247.613083,"close_a":249.147949,"volume_a":56921600,"open_bench":248.541121,"close_bench":249.147949},{"date":"2019-02-28","open_a":248.933836,"high_a":249.371113,"low_a":248.362737,"close_a":248.683975,"volume_a":69268300,"open_bench":248.933836,"close_bench":248.683975},{"date":"2019-03-01","open_a":250.254517,"high_a":250.647159,"low_a":248.808891,"close_a":250.236679,"volume_a":78880500,"open_bench":250.254517,"close_bench":250.236679},{"date":"2019-03-04","open_a":251.289627,"high_a":251.530555,"low_a":247.041967,"close_a":249.326416,"volume_a":106494600,"open_bench":251.289627,"close_bench":249.326416},{"date":"2019-03-05","open_a":249.451351,"high_a":249.647672,"low_a":248.442976,"close_a":248.987305,"volume_a":59114600,"open_bench":249.451351,"close_bench":248.987305},{"date":"2019-03-06","open_a":249.103295,"high_a":249.112228,"low_a":247.157949,"close_a":247.479187,"volume_a":75039800,"open_bench":249.103295,"close_bench":247.479187},{"date":"2019-03-07","open_a":247.033093,"high_a":247.175875,"low_a":244.570188,"close_a":245.409012,"volume_a":94885100,"open_bench":247.033093,"close_bench":245.409012},{"date":"2019-03-08","open_a":243.561799,"high_a":245.087734,"low_a":243.09778,"close_a":244.918182,"volume_a":85795800,"open_bench":243.561799,"close_bench":244.918182},{"date":"2019-03-11","open_a":245.632032,"high_a":248.630361,"low_a":245.605262,"close_a":248.469742,"volume_a":65098900,"open_bench":245.632032,"close_bench":248.469742},{"date":"2019-03-12","open_a":249.023028,"high_a":249.924325,"low_a":248.83564,"close_a":249.406738,"volume_a":79667500,"open_bench":249.023028,"close_bench":249.406738},{"date":"2019-03-13","open_a":250.290243,"high_a":251.985729,"low_a":250.129597,"close_a":251.057663,"volume_a":80639200,"open_bench":250.290243,"close_bench":251.057663},{"date":"2019-03-14","open_a":251.084422,"high_a":251.503835,"low_a":250.459784,"close_a":250.897034,"volume_a":67518400,"open_bench":251.084422,"close_bench":250.897034},{"date":"2019-03-15","open_a":251.446399,"high_a":252.943195,"low_a":251.258157,"close_a":252.136536,"volume_a":81309000,"open_bench":251.446399,"close_bench":252.136536},{"date":"2019-03-18","open_a":252.351718,"high_a":253.34662,"low_a":252.127645,"close_a":253.050827,"volume_a":62199800,"open_bench":252.351718,"close_bench":253.050827},{"date":"2019-03-19","open_a":254.108472,"high_a":254.870301,"low_a":252.226248,"close_a":253.113571,"volume_a":90268100,"open_bench":254.108472,"close_bench":253.113571},{"date":"2019-03-20","open_a":252.898438,"high_a":254.09947,"low_a":251.24926,"close_a":252.351685,"volume_a":84609200,"open_bench":252.898438,"close_bench":252.351685},{"date":"2019-03-21","open_a":251.536094,"high_a":255.605251,"low_a":251.491263,"close_a":255.201935,"volume_a":79550400,"open_bench":251.536094,"close_bench":255.201935},{"date":"2019-03-22","open_a":253.848481,"high_a":254.36832,"low_a":250.227444,"close_a":250.290192,"volume_a":122659300,"open_bench":253.848481,"close_bench":250.290192},{"date":"2019-03-25","open_a":249.949638,"high_a":251.132753,"low_a":248.847213,"close_a":250.10202,"volume_a":85575200,"open_bench":249.949638,"close_bench":250.10202},{"date":"2019-03-26","open_a":251.849771,"high_a":252.916363,"low_a":250.568076,"close_a":251.966293,"volume_a":68125900,"open_bench":251.849771,"close_bench":251.966293},{"date":"2019-03-27","open_a":251.957325,"high_a":252.539938,"low_a":249.107115,"close_a":250.648743,"volume_a":72224700,"open_bench":251.957325,"close_bench":250.648743},{"date":"2019-03-28","open_a":251.276148,"high_a":252.046948,"low_a":250.128892,"close_a":251.598801,"volume_a":56238500,"open_bench":251.276148,"close_bench":251.598801},{"date":"2019-03-29","open_a":253.104578,"high_a":253.507895,"low_a":251.984211,"close_a":253.185242,"volume_a":82186800,"open_bench":253.104578,"close_bench":253.185242},{"date":"2019-04-01","open_a":255.175016,"high_a":256.483598,"low_a":254.906111,"close_a":256.187805,"volume_a":77617900,"open_bench":255.175016,"close_bench":256.187805},{"date":"2019-04-02","open_a":256.376041,"high_a":256.546339,"low_a":255.52455,"close_a":256.313293,"volume_a":40070400,"open_bench":256.376041,"close_bench":256.313293},{"date":"2019-04-03","open_a":257.523273,"high_a":257.917645,"low_a":256.116085,"close_a":256.716614,"volume_a":68243200,"open_bench":257.523273,"close_bench":256.716614},{"date":"2019-04-04","open_a":257.039284,"high_a":257.648758,"low_a":256.349148,"close_a":257.397797,"volume_a":48997500,"open_bench":257.039284,"close_bench":257.397797},{"date":"2019-04-05","open_a":258.06106,"high_a":258.697422,"low_a":257.77424,"close_a":258.643646,"volume_a":58621700,"open_bench":258.06106,"close_bench":258.643646},{"date":"2019-04-08","open_a":258.222466,"high_a":258.948463,"low_a":257.568162,"close_a":258.840912,"volume_a":53566300,"open_bench":258.222466,"close_bench":258.840912},{"date":"2019-04-09","open_a":257.881766,"high_a":258.204419,"low_a":256.967556,"close_a":257.514282,"volume_a":66142300,"open_bench":257.881766,"close_bench":257.514282},{"date":"2019-04-10","open_a":257.926731,"high_a":258.482457,"low_a":257.514444,"close_a":258.392822,"volume_a":52601500,"open_bench":257.926731,"close_bench":258.392822},{"date":"2019-04-11","open_a":258.876713,"high_a":258.885684,"low_a":257.756345,"close_a":258.321014,"volume_a":55093100,"open_bench":258.876713,"close_bench":258.321014},{"date":"2019-04-12","open_a":259.925285,"high_a":260.346544,"low_a":258.365742,"close_a":260.068695,"volume_a":69727800,"open_bench":259.925285,"close_bench":260.068695},{"date":"2019-04-15","open_a":260.140458,"high_a":260.239064,"low_a":259.100753,"close_a":259.898468,"volume_a":49596700,"open_bench":260.140458,"close_bench":259.898468},{"date":"2019-04-16","open_a":260.776775,"high_a":260.830551,"low_a":259.477138,"close_a":260.068695,"volume_a":52153200,"open_bench":260.776775,"close_bench":260.068695},{"date":"2019-04-17","open_a":261.1801,"high_a":261.206987,"low_a":259.020028,"close_a":259.432343,"volume_a":58268300,"open_bench":261.1801,"close_bench":259.432343},{"date":"2019-04-18","open_a":260.015048,"high_a":260.212234,"low_a":258.724381,"close_a":259.943329,"volume_a":68708500,"open_bench":260.015048,"close_bench":259.943329},{"date":"2019-04-22","open_a":259.181487,"high_a":260.319771,"low_a":259.091852,"close_a":260.167389,"volume_a":40160100,"open_bench":259.181487,"close_bench":260.167389},{"date":"2019-04-23","open_a":260.534855,"high_a":262.739759,"low_a":260.301837,"close_a":262.506714,"volume_a":52246600,"open_bench":260.534855,"close_bench":262.506714},{"date":"2019-04-24","open_a":262.426087,"high_a":262.757711,"low_a":261.780753,"close_a":261.924164,"volume_a":50392900,"open_bench":262.426087,"close_bench":261.924164},{"date":"2019-04-25","open_a":261.825504,"high_a":262.417061,"low_a":260.579669,"close_a":261.762756,"volume_a":57770900,"open_bench":261.825504,"close_bench":261.762756},{"date":"2019-04-26","open_a":261.807637,"high_a":263.053473,"low_a":261.03681,"close_a":262.981781,"volume_a":50916400,"open_bench":261.807637,"close_bench":262.981781},{"date":"2019-04-29","open_a":263.071299,"high_a":263.913817,"low_a":262.981664,"close_a":263.393951,"volume_a":57197700,"open_bench":263.071299,"close_bench":263.393951},{"date":"2019-04-30","open_a":263.053468,"high_a":263.815324,"low_a":261.646307,"close_a":263.528503,"volume_a":81111700,"open_bench":263.053468,"close_bench":263.528503},{"date":"2019-05-01","open_a":264.155765,"high_a":264.361923,"low_a":261.538575,"close_a":261.547546,"volume_a":71671900,"open_bench":264.155765,"close_bench":261.547546},{"date":"2019-05-02","open_a":261.431149,"high_a":262.345387,"low_a":259.49515,"close_a":260.983002,"volume_a":65030200,"open_bench":261.431149,"close_bench":260.983002},{"date":"2019-05-03","open_a":262.452967,"high_a":263.815325,"low_a":261.090582,"close_a":263.537476,"volume_a":56543700,"open_bench":262.452967,"close_bench":263.537476},{"date":"2019-05-06","open_a":259.253164,"high_a":262.892117,"low_a":258.939456,"close_a":262.452942,"volume_a":107198100,"open_bench":259.253164,"close_bench":262.452942},{"date":"2019-05-07","open_a":260.059843,"high_a":260.651401,"low_a":256.169928,"close_a":258.070068,"volume_a":144729900,"open_bench":260.059843,"close_bench":258.070068},{"date":"2019-05-08","open_a":257.711609,"high_a":259.414563,"low_a":257.120051,"close_a":257.711609,"volume_a":91568300,"open_bench":257.711609,"close_bench":257.711609},{"date":"2019-05-09","open_a":255.650099,"high_a":257.532295,"low_a":253.920229,"close_a":256.931793,"volume_a":103471100,"open_bench":255.650099,"close_bench":256.931793},{"date":"2019-05-10","open_a":255.999592,"high_a":258.975296,"low_a":253.023888,"close_a":258.222412,"volume_a":112429300,"open_bench":255.999592,"close_bench":258.222412},{"date":"2019-05-13","open_a":253.131414,"high_a":254.090428,"low_a":250.899623,"close_a":251.73317,"volume_a":127290500,"open_bench":253.131414,"close_bench":251.73317},{"date":"2019-05-14","open_a":252.746079,"high_a":255.533569,"low_a":252.620612,"close_a":254.009857,"volume_a":77003200,"open_bench":252.746079,"close_bench":254.009857},{"date":"2019-05-15","open_a":252.387508,"high_a":256.134011,"low_a":252.18135,"close_a":255.49765,"volume_a":73956400,"open_bench":252.387508,"close_bench":255.49765},{"date":"2019-05-16","open_a":256.1968,"high_a":259.217308,"low_a":256.125109,"close_a":257.863922,"volume_a":76749600,"open_bench":256.1968,"close_bench":257.863922},{"date":"2019-05-17","open_a":255.569327,"high_a":258.670497,"low_a":255.551383,"close_a":256.196716,"volume_a":100353000,"open_bench":255.569327,"close_bench":256.196716},{"date":"2019-05-20","open_a":254.601372,"high_a":256.304325,"low_a":253.758853,"close_a":254.502792,"volume_a":62877600,"open_bench":254.601372,"close_bench":254.502792},{"date":"2019-05-21","open_a":256.187832,"high_a":257.173761,"low_a":255.93687,"close_a":256.797333,"volume_a":46847100,"open_bench":256.187832,"close_bench":256.797333},{"date":"2019-05-22","open_a":255.84728,"high_a":256.958676,"low_a":255.533571,"close_a":256.008606,"volume_a":49482500,"open_bench":255.84728,"close_bench":256.008606},{"date":"2019-05-23","open_a":253.794734,"high_a":253.839538,"low_a":251.473335,"close_a":252.880524,"volume_a":98733800,"open_bench":253.794734,"close_bench":252.880524},{"date":"2019-05-24","open_a":254.314582,"high_a":254.726898,"low_a":252.835702,"close_a":253.454147,"volume_a":55268100,"open_bench":254.314582,"close_bench":253.454147},{"date":"2019-05-28","open_a":253.73197,"high_a":254.68204,"low_a":251.078947,"close_a":251.096863,"volume_a":70029400,"open_bench":253.73197,"close_bench":251.096863},{"date":"2019-05-29","open_a":249.985537,"high_a":250.388854,"low_a":248.013678,"close_a":249.411896,"volume_a":104972900,"open_bench":249.985537,"close_bench":249.411896},{"date":"2019-05-30","open_a":250.164755,"high_a":250.99833,"low_a":248.999583,"close_a":250.093063,"volume_a":62523800,"open_bench":250.164755,"close_bench":250.093063},{"date":"2019-05-31","open_a":247.556536,"high_a":248.381112,"low_a":246.696074,"close_a":246.722961,"volume_a":86862800,"open_bench":247.556536,"close_bench":246.722961},{"date":"2019-06-03","open_a":246.758784,"high_a":247.870181,"low_a":244.76901,"close_a":246.095535,"volume_a":96428000,"open_bench":246.758784,"close_bench":246.095535},{"date":"2019-06-04","open_a":248.381117,"high_a":251.571923,"low_a":247.93297,"close_a":251.437485,"volume_a":77231900,"open_bench":248.381117,"close_bench":251.437485},{"date":"2019-06-05","open_a":253.050763,"high_a":253.64232,"low_a":251.24923,"close_a":253.615433,"volume_a":71169700,"open_bench":253.050763,"close_bench":253.615433},{"date":"2019-06-06","open_a":253.911217,"high_a":255.936823,"low_a":253.265884,"close_a":255.264603,"volume_a":69430400,"open_bench":253.911217,"close_bench":255.264603},{"date":"2019-06-07","open_a":256.277495,"high_a":258.894687,"low_a":256.107196,"close_a":257.819122,"volume_a":74272200,"open_bench":256.277495,"close_bench":257.819122},{"date":"2019-06-10","open_a":259.36074,"high_a":260.660378,"low_a":258.912593,"close_a":259.002228,"volume_a":60799100,"open_bench":259.36074,"close_bench":259.002228},{"date":"2019-06-11","open_a":260.812767,"high_a":261.180251,"low_a":258.294181,"close_a":258.939514,"volume_a":58641300,"open_bench":260.812767,"close_bench":258.939514},{"date":"2019-06-12","open_a":258.706465,"high_a":259.262163,"low_a":257.971497,"close_a":258.482391,"volume_a":47096300,"open_bench":258.706465,"close_bench":258.482391},{"date":"2019-06-13","open_a":259.387563,"high_a":259.907429,"low_a":258.688454,"close_a":259.548889,"volume_a":48945200,"open_bench":259.387563,"close_bench":259.548889},{"date":"2019-06-14","open_a":259.262207,"high_a":259.862709,"low_a":258.500351,"close_a":259.262207,"volume_a":52324700,"open_bench":259.262207,"close_bench":259.262207},{"date":"2019-06-17","open_a":259.495126,"high_a":260.122543,"low_a":259.190389,"close_a":259.360687,"volume_a":39205700,"open_bench":259.495126,"close_bench":259.360687},{"date":"2019-06-18","open_a":261.171299,"high_a":263.125215,"low_a":260.812759,"close_a":262.076538,"volume_a":85434800,"open_bench":261.171299,"close_bench":262.076538},{"date":"2019-06-19","open_a":262.210972,"high_a":263.196901,"low_a":261.242986,"close_a":262.668091,"volume_a":78674400,"open_bench":262.210972,"close_bench":262.668091},{"date":"2019-06-20","open_a":265.338966,"high_a":265.580956,"low_a":262.730746,"close_a":265.177612,"volume_a":116570000,"open_bench":265.338966,"close_bench":265.177612},{"date":"2019-06-21","open_a":264.909326,"high_a":266.16122,"low_a":264.576088,"close_a":264.792236,"volume_a":83309500,"open_bench":264.909326,"close_bench":264.792236},{"date":"2019-06-24","open_a":264.999431,"high_a":265.314638,"low_a":264.314925,"close_a":264.468048,"volume_a":47582700,"open_bench":264.999431,"close_bench":264.468048},{"date":"2019-06-25","open_a":264.521978,"high_a":264.548996,"low_a":261.76598,"close_a":261.874054,"volume_a":82028700,"open_bench":264.521978,"close_bench":261.874054},{"date":"2019-06-26","open_a":262.765659,"high_a":263.270023,"low_a":261.504749,"close_a":261.612823,"volume_a":51584900,"open_bench":262.765659,"close_bench":261.612823},{"date":"2019-06-27","open_a":262.369431,"high_a":263.044922,"low_a":261.991172,"close_a":262.540558,"volume_a":40355200,"open_bench":262.369431,"close_bench":262.540558},{"date":"2019-06-28","open_a":263.513223,"high_a":264.386859,"low_a":262.999872,"close_a":263.89151,"volume_a":59350900,"open_bench":263.513223,"close_bench":263.89151},{"date":"2019-07-01","open_a":267.20598,"high_a":267.422156,"low_a":265.089438,"close_a":266.287323,"volume_a":78705600,"open_bench":267.20598,"close_bench":266.287323},{"date":"2019-07-02","open_a":266.242201,"high_a":267.03478,"low_a":265.4046,"close_a":266.980743,"volume_a":61504500,"open_bench":266.242201,"close_bench":266.980743},{"date":"2019-07-03","open_a":267.656332,"high_a":269.133418,"low_a":267.512224,"close_a":269.115387,"volume_a":40898900,"open_bench":267.656332,"close_bench":269.115387},{"date":"2019-07-05","open_a":267.890425,"high_a":268.971221,"low_a":266.602497,"close_a":268.809082,"volume_a":51677300,"open_bench":267.890425,"close_bench":268.809082},{"date":"2019-07-08","open_a":267.503158,"high_a":268.628975,"low_a":266.791634,"close_a":267.332031,"volume_a":45841800,"open_bench":267.503158,"close_bench":267.332031},{"date":"2019-07-09","open_a":266.179271,"high_a":267.962548,"low_a":266.125234,"close_a":267.665344,"volume_a":41101300,"open_bench":266.179271,"close_bench":267.665344},{"date":"2019-07-10","open_a":268.727974,"high_a":269.889825,"low_a":268.196592,"close_a":268.944122,"volume_a":58448500,"open_bench":268.727974,"close_bench":268.944122},{"date":"2019-07-11","open_a":269.583722,"high_a":269.817874,"low_a":268.574994,"close_a":269.574707,"volume_a":50826100,"open_bench":269.583722,"close_bench":269.574707},{"date":"2019-07-12","open_a":270.060982,"high_a":270.853562,"low_a":269.754763,"close_a":270.781494,"volume_a":40326000,"open_bench":270.060982,"close_bench":270.781494},{"date":"2019-07-15","open_a":271.213896,"high_a":271.213896,"low_a":270.367279,"close_a":270.871643,"volume_a":33900000,"open_bench":271.213896,"close_bench":270.871643},{"date":"2019-07-16","open_a":270.781458,"high_a":270.988619,"low_a":269.691675,"close_a":269.997894,"volume_a":36650100,"open_bench":270.781458,"close_bench":269.997894},{"date":"2019-07-17","open_a":269.970937,"high_a":270.133048,"low_a":268.160614,"close_a":268.160614,"volume_a":36036300,"open_bench":269.970937,"close_bench":268.160614},{"date":"2019-07-18","open_a":267.665246,"high_a":269.52059,"low_a":267.223934,"close_a":269.142303,"volume_a":51392600,"open_bench":267.665246,"close_bench":269.142303},{"date":"2019-07-19","open_a":270.23215,"high_a":270.259168,"low_a":267.476151,"close_a":267.647278,"volume_a":58678600,"open_bench":270.23215,"close_bench":267.647278},{"date":"2019-07-22","open_a":268.043582,"high_a":268.845177,"low_a":267.53023,"close_a":268.304779,"volume_a":43638100,"open_bench":268.043582,"close_bench":268.304779},{"date":"2019-07-23","open_a":269.421607,"high_a":270.223175,"low_a":268.592994,"close_a":270.223175,"volume_a":44564500,"open_bench":269.421607,"close_bench":270.223175},{"date":"2019-07-24","open_a":269.466602,"high_a":271.493073,"low_a":269.376531,"close_a":271.493073,"volume_a":47213200,"open_bench":269.466602,"close_bench":271.493073},{"date":"2019-07-25","open_a":271.042723,"high_a":271.09676,"low_a":269.394511,"close_a":270.196106,"volume_a":55394100,"open_bench":271.042723,"close_bench":270.196106},{"date":"2019-07-26","open_a":270.880561,"high_a":272.204523,"low_a":270.754457,"close_a":272.006378,"volume_a":45084100,"open_bench":270.880561,"close_bench":272.006378},{"date":"2019-07-29","open_a":271.889395,"high_a":272.006485,"low_a":270.961723,"close_a":271.511108,"volume_a":38126500,"open_bench":271.889395,"close_bench":271.511108},{"date":"2019-07-30","open_a":270.114986,"high_a":271.249818,"low_a":269.736699,"close_a":270.844513,"volume_a":45849000,"open_bench":270.114986,"close_bench":270.844513},{"date":"2019-07-31","open_a":271.087733,"high_a":271.276891,"low_a":265.872969,"close_a":267.881409,"volume_a":104245200,"open_bench":271.087733,"close_bench":267.881409},{"date":"2019-08-01","open_a":268.03458,"high_a":270.979708,"low_a":264.756187,"close_a":265.548767,"volume_a":142646600,"open_bench":268.03458,"close_bench":265.548767},{"date":"2019-08-02","open_a":264.657069,"high_a":264.900236,"low_a":262.00013,"close_a":263.549255,"volume_a":116749700,"open_bench":264.657069,"close_bench":263.549255},{"date":"2019-08-05","open_a":259.469378,"high_a":259.577453,"low_a":253.732218,"close_a":255.623596,"volume_a":178745400,"open_bench":259.469378,"close_bench":255.623596},{"date":"2019-08-06","open_a":257.50603,"high_a":259.424428,"low_a":256.037959,"close_a":259.208252,"volume_a":120711700,"open_bench":257.50603,"close_bench":259.208252},{"date":"2019-08-07","open_a":256.145927,"high_a":260.126828,"low_a":254.020397,"close_a":259.361267,"volume_a":140572300,"open_bench":256.145927,"close_bench":259.361267},{"date":"2019-08-08","open_a":260.847336,"high_a":264.449951,"low_a":260.297951,"close_a":264.449951,"volume_a":87713900,"open_bench":260.847336,"close_bench":264.449951},{"date":"2019-08-09","open_a":263.51324,"high_a":264.107675,"low_a":260.874332,"close_a":262.648621,"volume_a":93730000,"open_bench":263.51324,"close_bench":262.648621},{"date":"2019-08-12","open_a":261.153515,"high_a":262.639588,"low_a":258.505591,"close_a":259.451294,"volume_a":65527600,"open_bench":261.153515,"close_bench":259.451294},{"date":"2019-08-13","open_a":259.154154,"high_a":264.927349,"low_a":258.811901,"close_a":263.486298,"volume_a":94299800,"open_bench":259.154154,"close_bench":263.486298},{"date":"2019-08-14","open_a":259.451325,"high_a":260.054748,"low_a":255.56951,"close_a":255.695587,"volume_a":135622100,"open_bench":259.451325,"close_bench":255.695587},{"date":"2019-08-15","open_a":256.578254,"high_a":257.26276,"low_a":254.335635,"close_a":256.371094,"volume_a":99556600,"open_bench":256.578254,"close_bench":256.371094},{"date":"2019-08-16","open_a":258.019234,"high_a":260.586074,"low_a":256.425059,"close_a":260.153778,"volume_a":83018300,"open_bench":258.019234,"close_bench":260.153778},{"date":"2019-08-19","open_a":263.162009,"high_a":263.963576,"low_a":262.486518,"close_a":263.288086,"volume_a":53571800,"open_bench":263.162009,"close_bench":263.288086},{"date":"2019-08-20","open_a":262.783691,"high_a":263.315073,"low_a":261.144522,"close_a":261.270599,"volume_a":51596400,"open_bench":262.783691,"close_bench":261.270599},{"date":"2019-08-21","open_a":263.423228,"high_a":263.765454,"low_a":262.738723,"close_a":263.39621,"volume_a":49524700,"open_bench":263.423228,"close_bench":263.39621},{"date":"2019-08-22","open_a":264.098686,"high_a":264.729127,"low_a":261.54982,"close_a":263.315094,"volume_a":51666400,"open_bench":264.098686,"close_bench":263.315094},{"date":"2019-08-23","open_a":262.018152,"high_a":263.675351,"low_a":255.308272,"close_a":256.551178,"volume_a":149161500,"open_bench":262.018152,"close_bench":256.551178},{"date":"2019-08-26","open_a":258.730818,"high_a":259.388306,"low_a":257.208711,"close_a":259.388306,"volume_a":72423800,"open_bench":258.730818,"close_bench":259.388306},{"date":"2019-08-27","open_a":260.775333,"high_a":261.144604,"low_a":257.614029,"close_a":258.370575,"volume_a":66668900,"open_bench":260.775333,"close_bench":258.370575},{"date":"2019-08-28","open_a":257.713113,"high_a":260.352022,"low_a":256.911518,"close_a":260.189911,"volume_a":59696700,"open_bench":257.713113,"close_bench":260.189911},{"date":"2019-08-29","open_a":262.738787,"high_a":264.035731,"low_a":261.739047,"close_a":263.513336,"volume_a":57899400,"open_bench":262.738787,"close_bench":263.513336},{"date":"2019-08-30","open_a":264.990357,"high_a":265.00836,"low_a":262.468537,"close_a":263.39621,"volume_a":62901200,"open_bench":264.990357,"close_bench":263.39621},{"date":"2019-09-03","open_a":261.703014,"high_a":262.612656,"low_a":260.532147,"close_a":261.85611,"volume_a":69101400,"open_bench":261.703014,"close_bench":261.85611},{"date":"2019-09-04","open_a":264.017573,"high_a":264.846159,"low_a":263.270015,"close_a":264.828156,"volume_a":46887300,"open_bench":264.017573,"close_bench":264.828156},{"date":"2019-09-05","open_a":267.305024,"high_a":269.142338,"low_a":264.792193,"close_a":268.232697,"volume_a":83258100,"open_bench":267.305024,"close_bench":268.232697},{"date":"2019-09-06","open_a":268.54786,"high_a":269.079242,"low_a":267.87237,"close_a":268.439758,"volume_a":49584300,"open_bench":268.54786,"close_bench":268.439758},{"date":"2019-09-09","open_a":269.421568,"high_a":269.511611,"low_a":267.638264,"close_a":268.574951,"volume_a":51260300,"open_bench":269.421568,"close_bench":268.574951},{"date":"2019-09-10","open_a":267.81832,"high_a":268.574893,"low_a":266.566426,"close_a":268.511841,"volume_a":57947100,"open_bench":267.81832,"close_bench":268.511841},{"date":"2019-09-11","open_a":268.818163,"high_a":270.502381,"low_a":268.169691,"close_a":270.421326,"volume_a":68821100,"open_bench":268.818163,"close_bench":270.421326},{"date":"2019-09-12","open_a":271.321998,"high_a":272.411782,"low_a":270.565452,"close_a":271.358032,"volume_a":72908700,"open_bench":271.321998,"close_bench":271.358032},{"date":"2019-09-13","open_a":271.799279,"high_a":272.150547,"low_a":270.808555,"close_a":271.177826,"volume_a":62104800,"open_bench":271.799279,"close_bench":271.177826},{"date":"2019-09-16","open_a":270.051964,"high_a":271.22283,"low_a":269.700723,"close_a":270.340179,"volume_a":58191200,"open_bench":270.051964,"close_bench":270.340179},{"date":"2019-09-17","open_a":270.142099,"high_a":271.114793,"low_a":269.970973,"close_a":271.02475,"volume_a":41475500,"open_bench":270.142099,"close_bench":271.02475},{"date":"2019-09-18","open_a":270.637385,"high_a":271.294872,"low_a":268.610915,"close_a":271.186798,"volume_a":73375800,"open_bench":270.637385,"close_bench":271.186798},{"date":"2019-09-19","open_a":271.574098,"high_a":272.564822,"low_a":270.835555,"close_a":271.168793,"volume_a":76560500,"open_bench":271.574098,"close_bench":271.168793},{"date":"2019-09-20","open_a":271.769603,"high_a":272.05012,"low_a":269.100421,"close_a":269.887604,"volume_a":89565000,"open_bench":271.769603,"close_bench":269.887604},{"date":"2019-09-23","open_a":269.227069,"high_a":270.539059,"low_a":268.973723,"close_a":269.824249,"volume_a":43476800,"open_bench":269.227069,"close_bench":269.824249},{"date":"2019-09-24","open_a":270.910077,"high_a":271.29914,"low_a":266.747932,"close_a":267.707031,"volume_a":94869400,"open_bench":270.910077,"close_bench":267.707031},{"date":"2019-09-25","open_a":267.788443,"high_a":269.733785,"low_a":266.313593,"close_a":269.290436,"volume_a":71854000,"open_bench":267.788443,"close_bench":269.290436},{"date":"2019-09-26","open_a":269.299468,"high_a":269.507557,"low_a":267.326982,"close_a":268.729431,"volume_a":56179700,"open_bench":269.299468,"close_bench":268.729431},{"date":"2019-09-27","open_a":269.480396,"high_a":269.588997,"low_a":265.734486,"close_a":267.281708,"volume_a":84746600,"open_bench":269.480396,"close_bench":267.281708},{"date":"2019-09-30","open_a":267.797492,"high_a":269.227085,"low_a":267.752263,"close_a":268.521332,"volume_a":51662400,"open_bench":267.797492,"close_bench":268.521332},{"date":"2019-10-01","open_a":269.399021,"high_a":270.050487,"low_a":265.110216,"close_a":265.327362,"volume_a":88242400,"open_bench":269.399021,"close_bench":265.327362},{"date":"2019-10-02","open_a":263.752908,"high_a":263.761965,"low_a":259.355532,"close_a":260.64035,"volume_a":122539500,"open_bench":263.752908,"close_bench":260.64035},{"date":"2019-10-03","open_a":260.414181,"high_a":262.8029,"low_a":257.708799,"close_a":262.775757,"volume_a":85278800,"open_bench":260.414181,"close_bench":262.775757},{"date":"2019-10-04","open_a":263.427193,"high_a":266.584981,"low_a":263.372879,"close_a":266.331635,"volume_a":65091200,"open_bench":263.427193,"close_bench":266.331635},{"date":"2019-10-07","open_a":265.535416,"high_a":267.155039,"low_a":264.902036,"close_a":265.182526,"volume_a":59610500,"open_bench":265.535416,"close_bench":265.182526},{"date":"2019-10-08","open_a":263.336763,"high_a":264.069659,"low_a":261.029474,"close_a":261.065674,"volume_a":95708100,"open_bench":263.336763,"close_bench":261.065674},{"date":"2019-10-09","open_a":263.074399,"high_a":264.476848,"low_a":262.450075,"close_a":263.544891,"volume_a":62359400,"open_bench":263.074399,"close_bench":263.544891},{"date":"2019-10-10","open_a":263.463449,"high_a":266.205032,"low_a":263.30059,"close_a":265.327362,"volume_a":55296300,"open_bench":263.463449,"close_bench":265.327362},{"date":"2019-10-11","open_a":268.069007,"high_a":270.303897,"low_a":267.951404,"close_a":268.078064,"volume_a":98720400,"open_bench":268.069007,"close_bench":268.078064},{"date":"2019-10-14","open_a":267.761336,"high_a":268.430916,"low_a":267.435616,"close_a":267.779449,"volume_a":40394800,"open_bench":267.761336,"close_bench":267.779449},{"date":"2019-10-15","open_a":268.820006,"high_a":271.172526,"low_a":268.702376,"close_a":270.430573,"volume_a":46754500,"open_bench":268.820006,"close_bench":270.430573},{"date":"2019-10-16","open_a":269.969042,"high_a":270.683852,"low_a":269.561893,"close_a":269.996185,"volume_a":48928200,"open_bench":269.969042,"close_bench":269.996185},{"date":"2019-10-17","open_a":271.154278,"high_a":271.660971,"low_a":270.104692,"close_a":270.792358,"volume_a":45736600,"open_bench":271.154278,"close_bench":270.792358},{"date":"2019-10-18","open_a":270.258522,"high_a":270.900931,"low_a":268.756529,"close_a":269.607056,"volume_a":64304000,"open_bench":270.258522,"close_bench":269.607056},{"date":"2019-10-21","open_a":270.919123,"high_a":271.633905,"low_a":270.484802,"close_a":271.434845,"volume_a":39048600,"open_bench":270.919123,"close_bench":271.434845},{"date":"2019-10-22","open_a":271.968662,"high_a":272.258208,"low_a":270.457639,"close_a":270.548126,"volume_a":48594700,"open_bench":271.968662,"close_bench":270.548126},{"date":"2019-10-23","open_a":270.294828,"high_a":271.389644,"low_a":270.086711,"close_a":271.335358,"volume_a":34352200,"open_bench":270.294828,"close_bench":271.335358},{"date":"2019-10-24","open_a":272.267263,"high_a":272.412036,"low_a":270.955273,"close_a":271.778656,"volume_a":35453100,"open_bench":272.267263,"close_bench":271.778656},{"date":"2019-10-25","open_a":271.208665,"high_a":273.434525,"low_a":271.154379,"close_a":272.891632,"volume_a":45205400,"open_bench":271.208665,"close_bench":272.891632},{"date":"2019-10-28","open_a":274.10409,"high_a":274.927473,"low_a":274.076947,"close_a":274.42981,"volume_a":42147000,"open_bench":274.10409,"close_bench":274.42981},{"date":"2019-10-29","open_a":274.158294,"high_a":275.271224,"low_a":274.031607,"close_a":274.348297,"volume_a":44284900,"open_bench":274.158294,"close_bench":274.348297},{"date":"2019-10-30","open_a":274.547382,"high_a":275.560768,"low_a":273.244449,"close_a":275.189819,"volume_a":49643900,"open_bench":274.547382,"close_bench":275.189819},{"date":"2019-10-31","open_a":275.180715,"high_a":275.180715,"low_a":273.00917,"close_a":274.456848,"volume_a":69053800,"open_bench":275.180715,"close_bench":274.456848},{"date":"2019-11-01","open_a":275.895578,"high_a":277.04468,"low_a":275.73269,"close_a":276.999451,"volume_a":71141500,"open_bench":275.895578,"close_bench":276.999451},{"date":"2019-11-04","open_a":278.546594,"high_a":278.682311,"low_a":277.741298,"close_a":278.112274,"volume_a":60606900,"open_bench":278.546594,"close_bench":278.112274},{"date":"2019-11-05","open_a":278.311503,"high_a":278.610107,"low_a":277.515263,"close_a":277.80481,"volume_a":42933200,"open_bench":278.311503,"close_bench":277.80481},{"date":"2019-11-06","open_a":277.804698,"high_a":278.139475,"low_a":276.927029,"close_a":277.868042,"volume_a":46487100,"open_bench":277.804698,"close_bench":277.868042},{"date":"2019-11-07","open_a":279.198105,"high_a":280.175291,"low_a":278.374722,"close_a":278.845215,"volume_a":54272300,"open_bench":279.198105,"close_bench":278.845215},{"date":"2019-11-08","open_a":278.501368,"high_a":279.587154,"low_a":277.804672,"close_a":279.532867,"volume_a":49032100,"open_bench":278.501368,"close_bench":279.532867},{"date":"2019-11-11","open_a":278.157585,"high_a":279.170971,"low_a":278.021841,"close_a":278.999054,"volume_a":35797300,"open_bench":278.157585,"close_bench":278.999054},{"date":"2019-11-12","open_a":279.361016,"high_a":280.482975,"low_a":278.818123,"close_a":279.587219,"volume_a":46484600,"open_bench":279.361016,"close_bench":279.587219},{"date":"2019-11-13","open_a":278.600883,"high_a":280.075732,"low_a":278.37468,"close_a":279.677612,"volume_a":53917700,"open_bench":278.600883,"close_bench":279.677612},{"date":"2019-11-14","open_a":279.3972,"high_a":280.166296,"low_a":278.76382,"close_a":280.084839,"volume_a":51219900,"open_bench":279.3972,"close_bench":280.084839},{"date":"2019-11-15","open_a":281.414971,"high_a":282.156924,"low_a":280.727331,"close_a":282.111694,"volume_a":62023600,"open_bench":281.414971,"close_bench":282.111694},{"date":"2019-11-18","open_a":281.876444,"high_a":282.555054,"low_a":281.424038,"close_a":282.319794,"volume_a":49228000,"open_bench":281.876444,"close_bench":282.319794},{"date":"2019-11-19","open_a":282.916952,"high_a":282.926009,"low_a":281.595933,"close_a":282.238342,"volume_a":67804700,"open_bench":282.916952,"close_bench":282.238342},{"date":"2019-11-20","open_a":281.650154,"high_a":282.165904,"low_a":279.641468,"close_a":281.18869,"volume_a":79406200,"open_bench":281.650154,"close_bench":281.18869},{"date":"2019-11-21","open_a":281.297305,"high_a":281.405878,"low_a":279.940085,"close_a":280.736298,"volume_a":54664700,"open_bench":281.297305,"close_bench":280.736298},{"date":"2019-11-22","open_a":281.478195,"high_a":281.613912,"low_a":280.356236,"close_a":281.360565,"volume_a":44850200,"open_bench":281.478195,"close_bench":281.360565},{"date":"2019-11-25","open_a":282.283614,"high_a":283.54129,"low_a":282.283614,"close_a":283.54129,"volume_a":48647200,"open_bench":282.283614,"close_bench":283.54129},{"date":"2019-11-26","open_a":283.577385,"high_a":284.364567,"low_a":283.260695,"close_a":284.183594,"volume_a":37569000,"open_bench":283.577385,"close_bench":284.183594},{"date":"2019-11-27","open_a":284.663168,"high_a":285.450378,"low_a":284.446022,"close_a":285.450378,"volume_a":44444600,"open_bench":284.663168,"close_bench":285.450378},{"date":"2019-11-29","open_a":284.88939,"high_a":285.133707,"low_a":284.165551,"close_a":284.391754,"volume_a":36592700,"open_bench":284.88939,"close_bench":284.391754},{"date":"2019-12-02","open_a":284.645165,"high_a":284.708508,"low_a":281.550719,"close_a":281.975983,"volume_a":75767800,"open_bench":284.645165,"close_bench":281.975983},{"date":"2019-12-03","open_a":279.270513,"high_a":280.166296,"low_a":277.895207,"close_a":280.084839,"volume_a":73941700,"open_bench":279.270513,"close_bench":280.084839},{"date":"2019-12-04","open_a":281.12541,"high_a":282.410229,"low_a":280.781577,"close_a":281.813049,"volume_a":49080000,"open_bench":281.12541,"close_bench":281.813049},{"date":"2019-12-05","open_a":282.509824,"high_a":282.527911,"low_a":281.016861,"close_a":282.319794,"volume_a":40709000,"open_bench":282.509824,"close_bench":282.319794},{"date":"2019-12-06","open_a":284.219889,"high_a":285.296619,"low_a":284.210832,"close_a":284.898499,"volume_a":48927000,"open_bench":284.219889,"close_bench":284.898499},{"date":"2019-12-09","open_a":284.50944,"high_a":285.178993,"low_a":283.930346,"close_a":284.002747,"volume_a":34838500,"open_bench":284.50944,"close_bench":284.002747},{"date":"2019-12-10","open_a":283.948439,"high_a":284.608935,"low_a":283.034569,"close_a":283.686035,"volume_a":52649800,"open_bench":283.948439,"close_bench":283.686035},{"date":"2019-12-11","open_a":284.138443,"high_a":284.74468,"low_a":283.604606,"close_a":284.491333,"volume_a":53429100,"open_bench":284.138443,"close_bench":284.491333},{"date":"2019-12-12","open_a":284.500262,"high_a":287.721394,"low_a":284.26503,"close_a":286.943268,"volume_a":96389600,"open_bench":284.500262,"close_bench":286.943268},{"date":"2019-12-13","open_a":286.708149,"high_a":288.336829,"low_a":285.939052,"close_a":287.115326,"volume_a":81503900,"open_bench":286.708149,"close_bench":287.115326},{"date":"2019-12-16","open_a":288.834483,"high_a":289.675953,"low_a":287.052,"close_a":289.08783,"volume_a":82749700,"open_bench":288.834483,"close_bench":289.08783},{"date":"2019-12-17","open_a":289.467813,"high_a":289.766389,"low_a":289.069693,"close_a":289.151123,"volume_a":61097700,"open_bench":289.467813,"close_bench":289.151123},{"date":"2019-12-18","open_a":289.540197,"high_a":289.7664,"low_a":289.114933,"close_a":289.16922,"volume_a":48133000,"open_bench":289.540197,"close_bench":289.16922},{"date":"2019-12-19","open_a":289.359192,"high_a":290.426893,"low_a":289.105846,"close_a":290.354492,"volume_a":85310500,"open_bench":289.359192,"close_bench":290.354492},{"date":"2019-12-20","open_a":291.381954,"high_a":292.754947,"low_a":290.409065,"close_a":291.627472,"volume_a":147142100,"open_bench":291.381954,"close_bench":291.627472},{"date":"2019-12-23","open_a":292.409329,"high_a":292.463882,"low_a":291.927421,"close_a":292.072906,"volume_a":52990000,"open_bench":292.409329,"close_bench":292.072906},{"date":"2019-12-24","open_a":292.300306,"high_a":292.345758,"low_a":291.78202,"close_a":292.082092,"volume_a":20270000,"open_bench":292.300306,"close_bench":292.082092},{"date":"2019-12-26","open_a":292.463886,"high_a":293.645942,"low_a":292.454812,"close_a":293.636841,"volume_a":30911200,"open_bench":292.463886,"close_bench":293.636841},{"date":"2019-12-27","open_a":294.364333,"high_a":294.418887,"low_a":293.036819,"close_a":293.564178,"volume_a":42528800,"open_bench":294.364333,"close_bench":293.564178},{"date":"2019-12-30","open_a":293.646021,"high_a":293.782405,"low_a":291.463771,"close_a":291.945679,"volume_a":49729100,"open_bench":293.646021,"close_bench":291.945679},{"date":"2019-12-31","open_a":291.445571,"high_a":292.900395,"low_a":291.100047,"close_a":292.654877,"volume_a":57077300,"open_bench":291.445571,"close_bench":292.654877},{"date":"2020-01-02","open_a":294.182418,"high_a":295.409927,"low_a":293.264055,"close_a":295.391724,"volume_a":59151200,"open_bench":294.182418,"close_bench":295.391724},{"date":"2020-01-03","open_a":292.018422,"high_a":294.2734,"low_a":291.963868,"close_a":293.154999,"volume_a":77709700,"open_bench":292.018422,"close_bench":293.154999},{"date":"2020-01-06","open_a":291.409241,"high_a":294.355268,"low_a":291.291033,"close_a":294.273438,"volume_a":55653900,"open_bench":291.409241,"close_bench":294.273438},{"date":"2020-01-07","open_a":293.70968,"high_a":294.182514,"low_a":293.000457,"close_a":293.446014,"volume_a":40496400,"open_bench":293.70968,"close_bench":293.446014},{"date":"2020-01-08","open_a":293.636924,"high_a":296.219224,"low_a":293.391434,"close_a":295.009918,"volume_a":68296000,"open_bench":293.636924,"close_bench":295.009918},{"date":"2020-01-09","open_a":296.564847,"high_a":297.083132,"low_a":295.982906,"close_a":297.010376,"volume_a":48473300,"open_bench":296.564847,"close_bench":297.010376},{"date":"2020-01-10","open_a":297.592197,"high_a":297.746756,"low_a":295.691844,"close_a":296.155548,"volume_a":53029300,"open_bench":297.592197,"close_bench":296.155548},{"date":"2020-01-13","open_a":296.773937,"high_a":298.201457,"low_a":296.346583,"close_a":298.192383,"volume_a":47086800,"open_bench":296.773937,"close_bench":298.192383},{"date":"2020-01-14","open_a":297.755846,"high_a":298.801491,"low_a":297.183007,"close_a":297.737671,"volume_a":62832800,"open_bench":297.755846,"close_bench":297.737671},{"date":"2020-01-15","open_a":297.646777,"high_a":299.165228,"low_a":297.564946,"close_a":298.410553,"volume_a":72056600,"open_bench":297.646777,"close_bench":298.410553},{"date":"2020-01-16","open_a":299.783522,"high_a":300.892822,"low_a":299.556207,"close_a":300.892822,"volume_a":54050300,"open_bench":299.783522,"close_bench":300.892822},{"date":"2020-01-17","open_a":301.602061,"high_a":302.038489,"low_a":300.829183,"close_a":301.829376,"volume_a":95846000,"open_bench":301.602061,"close_bench":301.829376},{"date":"2020-01-21","open_a":300.874582,"high_a":302.038436,"low_a":300.801854,"close_a":301.238281,"volume_a":77742400,"open_bench":300.874582,"close_bench":301.238281},{"date":"2020-01-22","open_a":302.093049,"high_a":302.738645,"low_a":301.12016,"close_a":301.274719,"volume_a":48914900,"open_bench":302.093049,"close_bench":301.274719},{"date":"2020-01-23","open_a":300.629148,"high_a":302.029418,"low_a":299.519847,"close_a":301.620239,"volume_a":51963000,"open_bench":300.629148,"close_bench":301.620239},{"date":"2020-01-24","open_a":302.274806,"high_a":302.356636,"low_a":297.655744,"close_a":298.937805,"volume_a":87578400,"open_bench":302.274806,"close_bench":298.937805},{"date":"2020-01-27","open_a":293.718734,"high_a":295.619087,"low_a":293.382311,"close_a":294.146088,"volume_a":84062500,"open_bench":293.718734,"close_bench":294.146088},{"date":"2020-01-28","open_a":295.564491,"high_a":298.101338,"low_a":294.236977,"close_a":297.228455,"volume_a":63834000,"open_bench":295.564491,"close_bench":297.228455},{"date":"2020-01-29","open_a":298.58328,"high_a":298.810596,"low_a":296.782933,"close_a":296.982971,"volume_a":53888900,"open_bench":298.58328,"close_bench":296.982971},{"date":"2020-01-30","open_a":294.928053,"high_a":298.155948,"low_a":294.18248,"close_a":297.946808,"volume_a":75491800,"open_bench":294.928053,"close_bench":297.946808},{"date":"2020-01-31","open_a":297.328511,"high_a":297.483097,"low_a":291.627451,"close_a":292.536713,"volume_a":113845600,"open_bench":297.328511,"close_bench":292.536713},{"date":"2020-02-03","open_a":294.009717,"high_a":296.56474,"low_a":293.891509,"close_a":294.709839,"volume_a":69083000,"open_bench":294.009717,"close_bench":294.709839},{"date":"2020-02-04","open_a":298.30147,"high_a":300.065439,"low_a":297.983222,"close_a":299.20163,"volume_a":62573200,"open_bench":298.30147,"close_bench":299.20163},{"date":"2020-02-05","open_a":302.120338,"high_a":302.865939,"low_a":300.665542,"close_a":302.656799,"volume_a":65951100,"open_bench":302.120338,"close_bench":302.656799},{"date":"2020-02-06","open_a":303.611485,"high_a":303.866078,"low_a":302.602191,"close_a":303.67514,"volume_a":50359700,"open_bench":303.611485,"close_bench":303.67514},{"date":"2020-02-07","open_a":302.620439,"high_a":303.68426,"low_a":301.511139,"close_a":302.056702,"volume_a":64139400,"open_bench":302.620439,"close_bench":302.056702},{"date":"2020-02-10","open_a":301.17471,"high_a":304.3753,"low_a":301.138332,"close_a":304.311646,"volume_a":42070000,"open_bench":301.17471,"close_bench":304.311646},{"date":"2020-02-11","open_a":305.657319,"high_a":306.43927,"low_a":304.311602,"close_a":304.838989,"volume_a":54864500,"open_bench":305.657319,"close_bench":304.838989},{"date":"2020-02-12","open_a":306.26652,"high_a":307.012121,"low_a":305.902821,"close_a":306.803009,"volume_a":43992700,"open_bench":306.26652,"close_bench":306.803009},{"date":"2020-02-13","open_a":305.384461,"high_a":307.439401,"low_a":305.111694,"close_a":306.475586,"volume_a":54501900,"open_bench":305.384461,"close_bench":306.475586},{"date":"2020-02-14","open_a":306.884814,"high_a":307.084853,"low_a":305.693684,"close_a":306.966644,"volume_a":64582200,"open_bench":306.884814,"close_bench":306.966644},{"date":"2020-02-18","open_a":305.975621,"high_a":307.030367,"low_a":304.793564,"close_a":306.175659,"volume_a":57226200,"open_bench":305.975621,"close_bench":306.175659},{"date":"2020-02-19","open_a":307.139383,"high_a":308.31231,"low_a":306.857514,"close_a":307.639465,"volume_a":48814700,"open_bench":307.139383,"close_bench":307.639465},{"date":"2020-02-20","open_a":307.093999,"high_a":307.912356,"low_a":303.402399,"close_a":306.375702,"volume_a":74163400,"open_bench":307.093999,"close_bench":306.375702},{"date":"2020-02-21","open_a":305.029941,"high_a":305.339087,"low_a":302.402163,"close_a":303.22052,"volume_a":113788200,"open_bench":305.029941,"close_bench":303.22052},{"date":"2020-02-24","open_a":293.818762,"high_a":303.293251,"low_a":292.091143,"close_a":293.164093,"volume_a":161088400,"open_bench":293.818762,"close_bench":293.164093},{"date":"2020-02-25","open_a":294.546148,"high_a":295.155337,"low_a":283.407695,"close_a":284.280579,"volume_a":218913200,"open_bench":294.546148,"close_bench":284.280579},{"date":"2020-02-26","open_a":285.671769,"high_a":289.24516,"low_a":282.507557,"close_a":283.234955,"volume_a":194773800,"open_bench":285.671769,"close_bench":283.234955},{"date":"2020-02-27","open_a":277.742985,"high_a":283.289486,"low_a":270.514374,"close_a":270.514374,"volume_a":284353500,"open_bench":277.742985,"close_bench":270.514374},{"date":"2020-02-28","open_a":262.503793,"high_a":270.859908,"low_a":259.630524,"close_a":269.377808,"volume_a":384975800,"open_bench":262.503793,"close_bench":269.377808},{"date":"2020-03-02","open_a":271.150841,"high_a":281.107265,"low_a":267.741111,"close_a":281.04361,"volume_a":238703600,"open_bench":271.150841,"close_bench":281.04361},{"date":"2020-03-03","open_a":281.416381,"high_a":285.362572,"low_a":270.5689,"close_a":272.996613,"volume_a":300139100,"open_bench":281.416381,"close_bench":272.996613},{"date":"2020-03-04","open_a":278.343114,"high_a":284.689769,"low_a":275.806266,"close_a":284.471527,"volume_a":176613400,"open_bench":278.343114,"close_bench":284.471527},{"date":"2020-03-05","open_a":277.306523,"high_a":280.479837,"low_a":272.787494,"close_a":275.015167,"volume_a":186366800,"open_bench":277.306523,"close_bench":275.015167},{"date":"2020-03-06","open_a":266.549988,"high_a":271.669134,"low_a":263.89496,"close_a":270.468903,"volume_a":228667200,"open_bench":266.549988,"close_bench":270.468903},{"date":"2020-03-09","open_a":250.319736,"high_a":258.403086,"low_a":248.637625,"close_a":249.346848,"volume_a":309417300,"open_bench":250.319736,"close_bench":249.346848},{"date":"2020-03-10","open_a":258.812139,"high_a":262.340049,"low_a":248.682955,"close_a":262.249146,"volume_a":276444100,"open_bench":258.812139,"close_bench":262.249146},{"date":"2020-03-11","open_a":255.229708,"high_a":256.357183,"low_a":246.300754,"close_a":249.464966,"volume_a":255316300,"open_bench":255.229708,"close_bench":249.464966},{"date":"2020-03-12","open_a":232.770919,"high_a":242.463649,"low_a":225.205858,"close_a":225.596848,"volume_a":392220700,"open_bench":232.770919,"close_bench":225.596848},{"date":"2020-03-13","open_a":239.2176,"high_a":246.846317,"low_a":225.969667,"close_a":244.882309,"volume_a":329566100,"open_bench":239.2176,"close_bench":244.882309},{"date":"2020-03-16","open_a":219.295686,"high_a":233.589278,"low_a":215.822314,"close_a":218.08638,"volume_a":297240000,"open_bench":219.295686,"close_bench":218.08638},{"date":"2020-03-17","open_a":222.805435,"high_a":232.925534,"low_a":215.558633,"close_a":229.861313,"volume_a":262070500,"open_bench":222.805435,"close_bench":229.861313},{"date":"2020-03-18","open_a":214.813047,"high_a":225.833293,"low_a":207.329829,"close_a":218.222778,"volume_a":327597100,"open_bench":214.813047,"close_bench":218.222778},{"date":"2020-03-19","open_a":217.540813,"high_a":224.933114,"low_a":211.148706,"close_a":218.686478,"volume_a":289322000,"open_bench":217.540813,"close_bench":218.686478},{"date":"2020-03-20","open_a":221.81991,"high_a":223.594252,"low_a":208.987959,"close_a":209.262344,"volume_a":347158800,"open_bench":221.81991,"close_bench":209.262344},{"date":"2020-03-23","open_a":208.704432,"high_a":210.067189,"low_a":199.622366,"close_a":203.91188,"volume_a":326025200,"open_bench":208.704432,"close_bench":203.91188},{"date":"2020-03-24","open_a":214.402452,"high_a":223.255867,"low_a":213.8354,"close_a":222.386978,"volume_a":235494500,"open_bench":214.402452,"close_bench":222.386978},{"date":"2020-03-25","open_a":223.960049,"high_a":234.459759,"low_a":219.277261,"close_a":225.716095,"volume_a":299430300,"open_bench":223.960049,"close_bench":225.716095},{"date":"2020-03-26","open_a":228.213059,"high_a":240.359042,"low_a":227.783192,"close_a":238.895691,"volume_a":257632800,"open_bench":228.213059,"close_bench":238.895691},{"date":"2020-03-27","open_a":231.642844,"high_a":238.538984,"low_a":229.612412,"close_a":231.780029,"volume_a":224341200,"open_bench":231.642844,"close_bench":231.780029},{"date":"2020-03-30","open_a":233.865305,"high_a":240.020614,"low_a":231.880607,"close_a":239.30722,"volume_a":171369500,"open_bench":233.865305,"close_bench":239.30722},{"date":"2020-03-31","open_a":238.310296,"high_a":240.843751,"low_a":234.3409,"close_a":235.74025,"volume_a":194881100,"open_bench":238.310296,"close_bench":235.74025},{"date":"2020-04-01","open_a":226.804533,"high_a":235.657947,"low_a":223.07293,"close_a":225.130798,"volume_a":189554600,"open_bench":226.804533,"close_bench":225.130798},{"date":"2020-04-02","open_a":224.252778,"high_a":231.103184,"low_a":223.704007,"close_a":230.325775,"volume_a":177660400,"open_bench":224.252778,"close_bench":230.325775},{"date":"2020-04-03","open_a":229.347148,"high_a":231.688557,"low_a":224.280225,"close_a":226.996613,"volume_a":135561200,"open_bench":229.347148,"close_bench":226.996613},{"date":"2020-04-06","open_a":235.822564,"high_a":244.200378,"low_a":226.978305,"close_a":242.243103,"volume_a":188061200,"open_bench":235.822564,"close_bench":242.243103},{"date":"2020-04-07","open_a":250.794759,"high_a":251.544745,"low_a":242.27063,"close_a":242.490128,"volume_a":201427200,"open_bench":250.794759,"close_bench":242.490128},{"date":"2020-04-08","open_a":245.078403,"high_a":252.43186,"low_a":242.599822,"close_a":250.630081,"volume_a":153774500,"open_bench":245.078403,"close_bench":250.630081},{"date":"2020-04-09","open_a":253.876935,"high_a":257.18784,"low_a":251.947125,"close_a":254.444016,"volume_a":189999200,"open_bench":253.876935,"close_bench":254.444016},{"date":"2020-04-13","open_a":253.474556,"high_a":253.812957,"low_a":248.233841,"close_a":252.120926,"volume_a":114839100,"open_bench":253.474556,"close_bench":252.120926},{"date":"2020-04-14","open_a":256.986624,"high_a":260.571873,"low_a":251.983716,"close_a":259.556671,"volume_a":134143400,"open_bench":256.986624,"close_bench":259.556671},{"date":"2020-04-15","open_a":253.867757,"high_a":259.693805,"low_a":251.937919,"close_a":254.041534,"volume_a":121775000,"open_bench":253.867757,"close_bench":254.041534},{"date":"2020-04-16","open_a":255.312855,"high_a":256.117714,"low_a":252.212348,"close_a":255.267136,"volume_a":131798300,"open_bench":255.312855,"close_bench":255.267136},{"date":"2020-04-17","open_a":261.010945,"high_a":262.766978,"low_a":258.285402,"close_a":262.163361,"volume_a":146684800,"open_bench":261.010945,"close_bench":262.163361},{"date":"2020-04-20","open_a":258.477385,"high_a":262.300468,"low_a":257.324998,"close_a":257.544495,"volume_a":100109300,"open_bench":258.477385,"close_bench":257.544495},{"date":"2020-04-21","open_a":253.099547,"high_a":254.297681,"low_a":248.791722,"close_a":249.72464,"volume_a":126385700,"open_bench":253.099547,"close_bench":249.72464},{"date":"2020-04-22","open_a":254.58118,"high_a":257.004885,"low_a":253.264142,"close_a":255.267136,"volume_a":92951600,"open_bench":254.58118,"close_bench":255.267136},{"date":"2020-04-23","open_a":256.538441,"high_a":259.693851,"low_a":254.947032,"close_a":255.24884,"volume_a":104709700,"open_bench":256.538441,"close_bench":255.24884},{"date":"2020-04-24","open_a":256.757988,"high_a":259.474376,"low_a":254.718402,"close_a":258.806702,"volume_a":85063200,"open_bench":256.757988,"close_bench":258.806702},{"date":"2020-04-27","open_a":260.772991,"high_a":263.654,"low_a":260.315687,"close_a":262.538177,"volume_a":77896600,"open_bench":260.772991,"close_bench":262.538177},{"date":"2020-04-28","open_a":266.169282,"high_a":266.516837,"low_a":261.029188,"close_a":261.331024,"volume_a":105270000,"open_bench":266.169282,"close_bench":261.331024},{"date":"2020-04-29","open_a":266.635706,"high_a":269.699649,"low_a":265.61135,"close_a":268.172241,"volume_a":118745600,"open_bench":266.635706,"close_bench":268.172241},{"date":"2020-04-30","open_a":266.800304,"high_a":268.272837,"low_a":263.946731,"close_a":265.675354,"volume_a":122901700,"open_bench":266.800304,"close_bench":265.675354},{"date":"2020-05-01","open_a":260.946924,"high_a":265.840084,"low_a":257.48055,"close_a":258.64212,"volume_a":125180000,"open_bench":260.946924,"close_bench":258.64212},{"date":"2020-05-04","open_a":256.767082,"high_a":259.657247,"low_a":255.294576,"close_a":259.355438,"volume_a":80873200,"open_bench":256.767082,"close_bench":259.355438},{"date":"2020-05-05","open_a":262.163386,"high_a":264.5505,"low_a":259.483561,"close_a":261.751801,"volume_a":79569900,"open_bench":262.163386,"close_bench":261.751801},{"date":"2020-05-06","open_a":263.443759,"high_a":263.827879,"low_a":259.547519,"close_a":259.977386,"volume_a":73632600,"open_bench":263.443759,"close_bench":259.977386},{"date":"2020-05-07","open_a":263.178501,"high_a":265.035154,"low_a":262.611448,"close_a":263.114471,"volume_a":75250400,"open_bench":263.178501,"close_bench":263.114471},{"date":"2020-05-08","open_a":266.233291,"high_a":267.934477,"low_a":265.108313,"close_a":267.468018,"volume_a":76452400,"open_bench":266.233291,"close_bench":267.468018},{"date":"2020-05-11","open_a":265.547422,"high_a":268.894892,"low_a":265.12671,"close_a":267.52298,"volume_a":79514200,"open_bench":265.547422,"close_bench":267.52298},{"date":"2020-05-12","open_a":268.702866,"high_a":269.114423,"low_a":262.053643,"close_a":262.190857,"volume_a":95870800,"open_bench":268.702866,"close_bench":262.190857},{"date":"2020-05-13","open_a":261.632763,"high_a":262.666275,"low_a":255.139041,"close_a":257.553619,"volume_a":144721100,"open_bench":261.632763,"close_bench":257.553619},{"date":"2020-05-14","open_a":255.130023,"high_a":260.763986,"low_a":249.678937,"close_a":260.635956,"volume_a":121977900,"open_bench":255.130023,"close_bench":260.635956},{"date":"2020-05-15","open_a":258.257864,"high_a":261.879704,"low_a":257.315818,"close_a":261.833984,"volume_a":111146300,"open_bench":258.257864,"close_bench":261.833984},{"date":"2020-05-18","open_a":268.025951,"high_a":271.410013,"low_a":267.70586,"close_a":269.809448,"volume_a":119703800,"open_bench":268.025951,"close_bench":269.809448},{"date":"2020-05-19","open_a":269.214857,"high_a":270.916015,"low_a":267.019804,"close_a":267.038086,"volume_a":95189300,"open_bench":269.214857,"close_bench":267.038086},{"date":"2020-05-20","open_a":270.559383,"high_a":272.434319,"low_a":270.330731,"close_a":271.574585,"volume_a":85861700,"open_bench":270.559383,"close_bench":271.574585},{"date":"2020-05-21","open_a":271.446551,"high_a":272.25141,"low_a":268.61126,"close_a":269.699646,"volume_a":78293900,"open_bench":271.446551,"close_bench":269.699646},{"date":"2020-05-22","open_a":269.416118,"high_a":270.3856,"low_a":268.181391,"close_a":270.211823,"volume_a":63958200,"open_bench":269.416118,"close_bench":270.211823},{"date":"2020-05-26","open_a":276.147716,"high_a":276.385523,"low_a":270.230199,"close_a":273.541077,"volume_a":88951400,"open_bench":276.147716,"close_bench":273.541077},{"date":"2020-05-27","open_a":276.321422,"high_a":277.647615,"low_a":271.519729,"close_a":277.611023,"volume_a":104817400,"open_bench":276.321422,"close_bench":277.611023},{"date":"2020-05-28","open_a":278.635474,"high_a":280.638468,"low_a":276.431264,"close_a":277.098938,"volume_a":90405200,"open_bench":278.635474,"close_bench":277.098938},{"date":"2020-05-29","open_a":276.632371,"high_a":278.918892,"low_a":273.897702,"close_a":278.333557,"volume_a":119090800,"open_bench":276.632371,"close_bench":278.333557},{"date":"2020-06-01","open_a":277.693339,"high_a":280.062171,"low_a":277.181161,"close_a":279.458527,"volume_a":55758300,"open_bench":277.693339,"close_bench":279.458527},{"date":"2020-06-02","open_a":280.373142,"high_a":281.818239,"low_a":279.046977,"close_a":281.772491,"volume_a":74267200,"open_bench":280.373142,"close_bench":281.772491},{"date":"2020-06-03","open_a":283.748118,"high_a":286.473662,"low_a":283.473747,"close_a":285.522461,"volume_a":92567600,"open_bench":283.748118,"close_bench":285.522461},{"date":"2020-06-04","open_a":284.543748,"high_a":286.272371,"low_a":282.687094,"close_a":284.7724,"volume_a":75794400,"open_bench":284.543748,"close_bench":284.7724},{"date":"2020-06-05","open_a":290.141204,"high_a":293.836202,"low_a":290.077175,"close_a":292.071014,"volume_a":150524700,"open_bench":290.141204,"close_bench":292.071014},{"date":"2020-06-08","open_a":292.875807,"high_a":295.793409,"low_a":292.336191,"close_a":295.601349,"volume_a":73641200,"open_bench":292.875807,"close_bench":295.601349},{"date":"2020-06-09","open_a":292.94904,"high_a":295.674583,"low_a":292.089306,"close_a":293.397217,"volume_a":77479200,"open_bench":292.94904,"close_bench":293.397217},{"date":"2020-06-10","open_a":293.973374,"high_a":294.860545,"low_a":291.046617,"close_a":291.76001,"volume_a":95000800,"open_bench":293.973374,"close_bench":291.76001},{"date":"2020-06-11","open_a":284.86381,"high_a":285.494892,"low_a":274.391565,"close_a":274.940308,"volume_a":209243600,"open_bench":284.86381,"close_bench":274.940308},{"date":"2020-06-12","open_a":281.91881,"high_a":282.687078,"low_a":273.102002,"close_a":278.232941,"volume_a":194678900,"open_bench":281.91881,"close_bench":278.232941},{"date":"2020-06-15","open_a":272.571563,"high_a":281.955453,"low_a":271.400865,"close_a":280.830475,"volume_a":135782700,"open_bench":272.571563,"close_bench":280.830475},{"date":"2020-06-16","open_a":288.54064,"high_a":288.686981,"low_a":281.397551,"close_a":286.235809,"volume_a":137627500,"open_bench":288.54064,"close_bench":286.235809},{"date":"2020-06-17","open_a":287.250992,"high_a":287.543674,"low_a":284.31508,"close_a":285.046783,"volume_a":83398900,"open_bench":287.250992,"close_bench":285.046783},{"date":"2020-06-18","open_a":283.537587,"high_a":285.632019,"low_a":283.080283,"close_a":285.156433,"volume_a":80828700,"open_bench":283.537587,"close_bench":285.156433},{"date":"2020-06-19","open_a":288.606956,"high_a":288.799861,"low_a":281.588586,"close_a":283.526917,"volume_a":135549600,"open_bench":288.606956,"close_bench":283.526917},{"date":"2020-06-22","open_a":282.929785,"high_a":285.7408,"low_a":281.790689,"close_a":285.345795,"volume_a":74649400,"open_bench":282.929785,"close_bench":285.345795},{"date":"2020-06-23","open_a":287.982196,"high_a":288.910024,"low_a":286.255162,"close_a":286.659363,"volume_a":68471200,"open_bench":287.982196,"close_bench":286.659363},{"date":"2020-06-24","open_a":284.629247,"high_a":285.244743,"low_a":277.519036,"close_a":279.347107,"volume_a":132813500,"open_bench":284.629247,"close_bench":279.347107},{"date":"2020-06-25","open_a":278.777557,"high_a":282.608269,"low_a":276.765749,"close_a":282.341858,"volume_a":89468000,"open_bench":278.777557,"close_bench":282.341858},{"date":"2020-06-26","open_a":281.248603,"high_a":281.459898,"low_a":275.057027,"close_a":275.635742,"volume_a":127961000,"open_bench":281.248603,"close_bench":275.635742},{"date":"2020-06-29","open_a":276.885162,"high_a":279.824771,"low_a":274.606942,"close_a":279.686981,"volume_a":79773300,"open_bench":276.885162,"close_bench":279.686981},{"date":"2020-06-30","open_a":279.255232,"high_a":284.959963,"low_a":279.09908,"close_a":283.269653,"volume_a":113394800,"open_bench":279.255232,"close_bench":283.269653},{"date":"2020-07-01","open_a":284.381222,"high_a":286.512457,"low_a":283.921905,"close_a":285.253906,"volume_a":72396500,"open_bench":284.381222,"close_bench":285.253906},{"date":"2020-07-02","open_a":288.671293,"high_a":290.012518,"low_a":286.163443,"close_a":286.82486,"volume_a":69344200,"open_bench":288.671293,"close_bench":286.82486},{"date":"2020-07-06","open_a":290.627991,"high_a":291.831398,"low_a":289.8839,"close_a":291.252655,"volume_a":61713800,"open_bench":290.627991,"close_bench":291.252655},{"date":"2020-07-07","open_a":289.718537,"high_a":291.684397,"low_a":287.872075,"close_a":288.248718,"volume_a":82910000,"open_bench":289.718537,"close_bench":288.248718},{"date":"2020-07-08","open_a":289.011109,"high_a":290.563601,"low_a":287.256544,"close_a":290.453369,"volume_a":54638600,"open_bench":289.011109,"close_bench":290.453369},{"date":"2020-07-09","open_a":291.059664,"high_a":291.298518,"low_a":285.400882,"close_a":288.799835,"volume_a":83354200,"open_bench":291.059664,"close_bench":288.799835},{"date":"2020-07-10","open_a":288.735572,"high_a":292.015099,"low_a":287.311702,"close_a":291.748688,"volume_a":57550400,"open_bench":288.735572,"close_bench":291.748688},{"date":"2020-07-13","open_a":294.081989,"high_a":296.452049,"low_a":288.570191,"close_a":289.222412,"volume_a":102997500,"open_bench":294.081989,"close_bench":289.222412},{"date":"2020-07-14","open_a":287.807748,"high_a":293.742138,"low_a":286.613536,"close_a":292.97049,"volume_a":93657000,"open_bench":287.807748,"close_bench":292.97049},{"date":"2020-07-15","open_a":296.176419,"high_a":296.755163,"low_a":293.291899,"close_a":295.661987,"volume_a":87196500,"open_bench":296.176419,"close_bench":295.661987},{"date":"2020-07-16","open_a":293.769691,"high_a":295.138445,"low_a":293.126636,"close_a":294.688324,"volume_a":54622500,"open_bench":293.769691,"close_bench":294.688324},{"date":"2020-07-17","open_a":295.689618,"high_a":296.323477,"low_a":293.723729,"close_a":295.542633,"volume_a":62774900,"open_bench":295.689618,"close_bench":295.542633},{"date":"2020-07-20","open_a":295.276168,"high_a":298.675121,"low_a":294.532077,"close_a":297.93103,"volume_a":56308800,"open_bench":295.276168,"close_bench":297.93103},{"date":"2020-07-21","open_a":299.887714,"high_a":300.32864,"low_a":297.581936,"close_a":298.56488,"volume_a":57292200,"open_bench":299.887714,"close_bench":298.56488},{"date":"2020-07-22","open_a":298.206736,"high_a":300.576825,"low_a":298.096504,"close_a":300.264465,"volume_a":57792900,"open_bench":298.206736,"close_bench":300.264465},{"date":"2020-07-23","open_a":299.906143,"high_a":300.604313,"low_a":295.322173,"close_a":296.681732,"volume_a":75738000,"open_bench":299.906143,"close_bench":296.681732},{"date":"2020-07-24","open_a":294.835246,"high_a":295.790604,"low_a":293.273559,"close_a":294.770935,"volume_a":73766600,"open_bench":294.835246,"close_bench":294.770935},{"date":"2020-07-27","open_a":295.459909,"high_a":297.095075,"low_a":294.66987,"close_a":296.920532,"volume_a":48293000,"open_bench":295.459909,"close_bench":296.920532},{"date":"2020-07-28","open_a":296.194904,"high_a":297.30647,"low_a":294.743476,"close_a":295.037445,"volume_a":57495000,"open_bench":296.194904,"close_bench":295.037445},{"date":"2020-07-29","open_a":295.910086,"high_a":299.226366,"low_a":295.873333,"close_a":298.665985,"volume_a":48454200,"open_bench":295.910086,"close_bench":298.665985},{"date":"2020-07-30","open_a":295.70793,"high_a":298.013707,"low_a":293.631838,"close_a":297.600311,"volume_a":61861700,"open_bench":295.70793,"close_bench":297.600311},{"date":"2020-07-31","open_a":299.382448,"high_a":300.05306,"low_a":295.184289,"close_a":299.951996,"volume_a":84986800,"open_bench":299.382448,"close_bench":299.951996},{"date":"2020-08-03","open_a":301.605595,"high_a":302.799807,"low_a":301.063605,"close_a":302.037354,"volume_a":53077900,"open_bench":301.605595,"close_bench":302.037354},{"date":"2020-08-04","open_a":301.183097,"high_a":303.204102,"low_a":301.183097,"close_a":303.204102,"volume_a":41917900,"open_bench":301.183097,"close_bench":303.204102},{"date":"2020-08-05","open_a":304.499309,"high_a":305.344463,"low_a":304.232897,"close_a":305.087219,"volume_a":42697700,"open_bench":304.499309,"close_bench":305.087219},{"date":"2020-08-06","open_a":304.508414,"high_a":307.245922,"low_a":304.186887,"close_a":307.126495,"volume_a":43679400,"open_bench":304.508414,"close_bench":307.126495},{"date":"2020-08-07","open_a":306.161971,"high_a":307.631789,"low_a":305.261701,"close_a":307.347015,"volume_a":57308300,"open_bench":306.161971,"close_bench":307.347015},{"date":"2020-08-10","open_a":307.797235,"high_a":308.449457,"low_a":305.8681,"close_a":308.265747,"volume_a":44282100,"open_bench":307.797235,"close_bench":308.265747},{"date":"2020-08-11","open_a":309.441488,"high_a":310.075347,"low_a":304.995309,"close_a":305.721008,"volume_a":69601100,"open_bench":309.441488,"close_bench":305.721008},{"date":"2020-08-12","open_a":308.146224,"high_a":310.755138,"low_a":308.118666,"close_a":309.98349,"volume_a":53774400,"open_bench":308.146224,"close_bench":309.98349},{"date":"2020-08-13","open_a":309.221087,"high_a":310.727659,"low_a":308.504554,"close_a":309.423187,"volume_a":41816100,"open_bench":309.221087,"close_bench":309.423187},{"date":"2020-08-14","open_a":309.037246,"high_a":309.965074,"low_a":308.311518,"close_a":309.432251,"volume_a":47260400,"open_bench":309.037246,"close_bench":309.432251},{"date":"2020-08-17","open_a":310.442933,"high_a":310.81038,"low_a":309.441626,"close_a":310.415375,"volume_a":35481000,"open_bench":310.442933,"close_bench":310.415375},{"date":"2020-08-18","open_a":310.81027,"high_a":311.50844,"low_a":309.221025,"close_a":311.085876,"volume_a":38733900,"open_bench":310.81027,"close_bench":311.085876},{"date":"2020-08-19","open_a":311.46254,"high_a":311.976973,"low_a":309.230269,"close_a":309.790649,"volume_a":68054200,"open_bench":311.46254,"close_bench":309.790649},{"date":"2020-08-20","open_a":308.072676,"high_a":311.232775,"low_a":307.944082,"close_a":310.755096,"volume_a":42207800,"open_bench":308.072676,"close_bench":310.755096},{"date":"2020-08-21","open_a":310.424479,"high_a":312.078007,"low_a":310.084561,"close_a":311.857544,"volume_a":55106600,"open_bench":310.424479,"close_bench":311.857544},{"date":"2020-08-24","open_a":314.282746,"high_a":315.091148,"low_a":311.830012,"close_a":315.01767,"volume_a":48588700,"open_bench":314.282746,"close_bench":315.01767},{"date":"2020-08-25","open_a":315.577975,"high_a":316.202638,"low_a":314.420488,"close_a":316.119965,"volume_a":38463400,"open_bench":315.577975,"close_bench":316.119965},{"date":"2020-08-26","open_a":316.70792,"high_a":319.55566,"low_a":316.16593,"close_a":319.289276,"volume_a":50790200,"open_bench":316.70792,"close_bench":319.289276},{"date":"2020-08-27","open_a":320.152771,"high_a":321.429657,"low_a":318.333868,"close_a":319.987396,"volume_a":58034100,"open_bench":320.152771,"close_bench":319.987396},{"date":"2020-08-28","open_a":321.007125,"high_a":322.182974,"low_a":319.82208,"close_a":322.054352,"volume_a":48588900,"open_bench":321.007125,"close_bench":322.054352},{"date":"2020-08-31","open_a":321.843081,"high_a":322.715766,"low_a":320.658037,"close_a":320.887695,"volume_a":66099200,"open_bench":321.843081,"close_bench":320.887695},{"date":"2020-09-01","open_a":321.714397,"high_a":324.010979,"low_a":320.823322,"close_a":323.909943,"volume_a":54908700,"open_bench":321.714397,"close_bench":323.909943},{"date":"2020-09-02","open_a":325.811514,"high_a":329.559524,"low_a":324.672391,"close_a":328.594971,"volume_a":69540000,"open_bench":325.811514,"close_bench":328.594971},{"date":"2020-09-03","open_a":326.913939,"high_a":327.38245,"low_a":314.714493,"close_a":317.286682,"volume_a":148011100,"open_bench":326.913939,"close_bench":317.286682},{"date":"2020-09-04","open_a":317.966375,"high_a":319.528034,"low_a":307.622561,"close_a":314.696045,"volume_a":139156300,"open_bench":317.966375,"close_bench":314.696045},{"date":"2020-09-08","open_a":309.312933,"high_a":314.760448,"low_a":305.794581,"close_a":306.097717,"volume_a":114465300,"open_bench":309.312933,"close_bench":306.097717},{"date":"2020-09-09","open_a":310.084516,"high_a":314.595007,"low_a":309.220999,"close_a":312.142273,"volume_a":91462300,"open_bench":310.084516,"close_bench":312.142273},{"date":"2020-09-10","open_a":314.007134,"high_a":314.659356,"low_a":305.766995,"close_a":306.722382,"volume_a":90569500,"open_bench":314.007134,"close_bench":306.722382},{"date":"2020-09-11","open_a":308.495282,"high_a":309.551704,"low_a":304.067465,"close_a":306.878479,"volume_a":84680200,"open_bench":308.495282,"close_bench":306.878479},{"date":"2020-09-14","open_a":310.029427,"high_a":312.684289,"low_a":307.025507,"close_a":310.920502,"volume_a":65605700,"open_bench":310.029427,"close_bench":310.920502},{"date":"2020-09-15","open_a":313.364079,"high_a":314.190843,"low_a":310.929707,"close_a":312.491394,"volume_a":52920900,"open_bench":313.364079,"close_bench":312.491394},{"date":"2020-09-16","open_a":313.722346,"high_a":315.146215,"low_a":310.975614,"close_a":311.251221,"volume_a":82096000,"open_bench":313.722346,"close_bench":311.251221},{"date":"2020-09-17","open_a":306.419251,"high_a":310.222405,"low_a":305.895623,"close_a":308.513733,"volume_a":91523300,"open_bench":306.419251,"close_bench":308.513733},{"date":"2020-09-18","open_a":309.31522,"high_a":309.425893,"low_a":302.490129,"close_a":304.961914,"volume_a":105877900,"open_bench":309.31522,"close_bench":304.961914},{"date":"2020-09-21","open_a":300.396516,"high_a":301.715414,"low_a":296.734943,"close_a":301.567841,"volume_a":99450800,"open_bench":300.396516,"close_bench":301.567841},{"date":"2020-09-22","open_a":303.04349,"high_a":305.19246,"low_a":300.544009,"close_a":304.639069,"volume_a":63612100,"open_bench":303.04349,"close_bench":304.639069},{"date":"2020-09-23","open_a":305.192389,"high_a":305.469099,"low_a":297.076072,"close_a":297.574127,"volume_a":93112200,"open_bench":305.192389,"close_bench":297.574127},{"date":"2020-09-24","open_a":296.264535,"high_a":301.411014,"low_a":294.954842,"close_a":298.367401,"volume_a":76681300,"open_bench":296.264535,"close_bench":298.367401},{"date":"2020-09-25","open_a":297.51887,"high_a":303.975042,"low_a":296.651924,"close_a":303.191101,"volume_a":71069400,"open_bench":297.51887,"close_bench":303.191101},{"date":"2020-09-28","open_a":307.332347,"high_a":308.937159,"low_a":306.345468,"close_a":308.22699,"volume_a":64584600,"open_bench":307.332347,"close_bench":308.22699},{"date":"2020-09-29","open_a":308.023981,"high_a":308.761818,"low_a":305.856546,"close_a":306.548279,"volume_a":51304000,"open_bench":308.023981,"close_bench":306.548279},{"date":"2020-09-30","open_a":307.212353,"high_a":312.008378,"low_a":307.018675,"close_a":308.872528,"volume_a":104081100,"open_bench":307.212353,"close_bench":308.872528},{"date":"2020-10-01","open_a":311.454995,"high_a":312.42341,"low_a":308.98321,"close_a":310.855499,"volume_a":88698700,"open_bench":311.454995,"close_bench":310.855499},{"date":"2020-10-02","open_a":305.930293,"high_a":310.827758,"low_a":305.459906,"close_a":307.904022,"volume_a":89431100,"open_bench":305.930293,"close_bench":307.904022},{"date":"2020-10-05","open_a":309.951516,"high_a":313.54852,"low_a":309.905412,"close_a":313.364075,"volume_a":45713100,"open_bench":309.951516,"close_bench":313.364075},{"date":"2020-10-06","open_a":313.50254,"high_a":315.58697,"low_a":308.402164,"close_a":308.909424,"volume_a":90128900,"open_bench":313.50254,"close_bench":308.909424},{"date":"2020-10-07","open_a":311.851616,"high_a":315.088935,"low_a":311.823948,"close_a":314.28653,"volume_a":56999600,"open_bench":311.851616,"close_bench":314.28653},{"date":"2020-10-08","open_a":316.214036,"high_a":317.136346,"low_a":315.300929,"close_a":317.071777,"volume_a":45242500,"open_bench":316.214036,"close_bench":317.071777},{"date":"2020-10-09","open_a":318.713502,"high_a":320.364445,"low_a":318.09557,"close_a":319.90329,"volume_a":59528600,"open_bench":318.713502,"close_bench":319.90329},{"date":"2020-10-12","open_a":322.430447,"high_a":326.516275,"low_a":321.941624,"close_a":325.049805,"volume_a":80388500,"open_bench":322.430447,"close_bench":325.049805},{"date":"2020-10-13","open_a":324.911459,"high_a":325.0867,"low_a":321.969287,"close_a":322.928497,"volume_a":73255500,"open_bench":324.911459,"close_bench":322.928497},{"date":"2020-10-14","open_a":323.500336,"high_a":324.588655,"low_a":320.170809,"close_a":320.899414,"volume_a":57727900,"open_bench":323.500336,"close_bench":320.899414},{"date":"2020-10-15","open_a":317.007184,"high_a":320.982338,"low_a":316.472256,"close_a":320.502747,"volume_a":60357700,"open_bench":317.007184,"close_bench":320.502747},{"date":"2020-10-16","open_a":321.849324,"high_a":323.500267,"low_a":320.133841,"close_a":320.309082,"volume_a":89501900,"open_bench":321.849324,"close_bench":320.309082},{"date":"2020-10-19","open_a":321.563456,"high_a":322.190621,"low_a":314.544689,"close_a":315.439331,"volume_a":68425600,"open_bench":321.563456,"close_bench":315.439331},{"date":"2020-10-20","open_a":316.776684,"high_a":319.930997,"low_a":316.020411,"close_a":316.702911,"volume_a":60051900,"open_bench":316.776684,"close_bench":316.702911},{"date":"2020-10-21","open_a":316.656788,"high_a":318.815019,"low_a":315.799046,"close_a":316.103424,"volume_a":63575000,"open_bench":316.656788,"close_bench":316.103424},{"date":"2020-10-22","open_a":316.315535,"high_a":318.418401,"low_a":314.185,"close_a":317.837341,"volume_a":55399300,"open_bench":316.315535,"close_bench":317.837341},{"date":"2020-10-23","open_a":319.054814,"high_a":319.110151,"low_a":316.472357,"close_a":318.916473,"volume_a":49143900,"open_bench":319.054814,"close_bench":318.916473},{"date":"2020-10-26","open_a":315.549948,"high_a":316.333917,"low_a":309.5457,"close_a":313.022827,"volume_a":91473000,"open_bench":315.549948,"close_bench":313.022827},{"date":"2020-10-27","open_a":313.364183,"high_a":313.696201,"low_a":311.731676,"close_a":311.943817,"volume_a":65994100,"open_bench":313.364183,"close_bench":311.943817},{"date":"2020-10-28","open_a":306.299261,"high_a":311.971464,"low_a":300.793068,"close_a":301.281891,"volume_a":127094300,"open_bench":306.299261,"close_bench":301.281891},{"date":"2020-10-29","open_a":301.512434,"high_a":307.498218,"low_a":299.833823,"close_a":304.343933,"volume_a":90597700,"open_bench":301.512434,"close_bench":304.343933},{"date":"2020-10-30","open_a":302.776107,"high_a":304.076568,"low_a":297.537389,"close_a":301.171295,"volume_a":120287300,"open_bench":302.776107,"close_bench":301.171295},{"date":"2020-11-02","open_a":304.546875,"high_a":306.53904,"low_a":301.816816,"close_a":304.546875,"volume_a":86068300,"open_bench":304.546875,"close_bench":304.546875},{"date":"2020-11-03","open_a":307.765687,"high_a":311.971419,"low_a":304.629838,"close_a":309.923889,"volume_a":93294200,"open_bench":307.765687,"close_bench":309.923889},{"date":"2020-11-04","open_a":314.378712,"high_a":320.908685,"low_a":313.207388,"close_a":316.850525,"volume_a":126959700,"open_bench":314.378712,"close_bench":316.850525},{"date":"2020-11-05","open_a":322.107567,"high_a":324.828393,"low_a":321.757084,"close_a":323.029877,"volume_a":82039700,"open_bench":322.107567,"close_bench":323.029877},{"date":"2020-11-06","open_a":322.744035,"high_a":324.201301,"low_a":320.641169,"close_a":322.956177,"volume_a":74973000,"open_bench":322.744035,"close_bench":322.956177},{"date":"2020-11-09","open_a":335.693287,"high_a":336.071437,"low_a":326.553188,"close_a":327.014343,"volume_a":172304200,"open_bench":335.693287,"close_bench":327.014343},{"date":"2020-11-10","open_a":326.027381,"high_a":327.586087,"low_a":323.278915,"close_a":326.534668,"volume_a":85552000,"open_bench":326.027381,"close_bench":326.534668},{"date":"2020-11-11","open_a":328.711377,"high_a":329.78126,"low_a":327.475485,"close_a":328.960419,"volume_a":58649000,"open_bench":328.711377,"close_bench":328.960419},{"date":"2020-11-12","open_a":327.955097,"high_a":329.006544,"low_a":323.970738,"close_a":325.769226,"volume_a":67546200,"open_bench":327.955097,"close_bench":325.769226},{"date":"2020-11-13","open_a":327.669143,"high_a":331.017134,"low_a":327.152652,"close_a":330.279297,"volume_a":62892200,"open_bench":327.669143,"close_bench":330.279297},{"date":"2020-11-16","open_a":332.935538,"high_a":334.595685,"low_a":331.653514,"close_a":334.402008,"volume_a":74541100,"open_bench":332.935538,"close_bench":334.402008},{"date":"2020-11-17","open_a":332.00402,"high_a":333.802536,"low_a":330.500649,"close_a":332.603516,"volume_a":66111000,"open_bench":332.00402,"close_bench":332.603516},{"date":"2020-11-18","open_a":332.87104,"high_a":333.4152,"low_a":328.563838,"close_a":328.600739,"volume_a":70591300,"open_bench":332.87104,"close_bench":328.600739},{"date":"2020-11-19","open_a":327.973471,"high_a":330.353019,"low_a":326.63611,"close_a":329.9841,"volume_a":59940900,"open_bench":327.973471,"close_bench":329.9841},{"date":"2020-11-20","open_a":329.725882,"high_a":329.928792,"low_a":327.650684,"close_a":327.724457,"volume_a":70417300,"open_bench":329.725882,"close_bench":327.724457},{"date":"2020-11-23","open_a":329.522986,"high_a":330.943352,"low_a":327.300215,"close_a":329.688995,"volume_a":63230600,"open_bench":329.522986,"close_bench":329.688995},{"date":"2020-11-24","open_a":332.225303,"high_a":335.545624,"low_a":331.376793,"close_a":335.001465,"volume_a":62415900,"open_bench":332.225303,"close_bench":335.001465},{"date":"2020-11-25","open_a":334.918472,"high_a":334.94614,"low_a":333.396666,"close_a":334.484985,"volume_a":45330900,"open_bench":334.918472,"close_bench":334.484985},{"date":"2020-11-27","open_a":335.573403,"high_a":335.886985,"low_a":334.411283,"close_a":335.416626,"volume_a":28514100,"open_bench":335.573403,"close_bench":335.416626},{"date":"2020-11-30","open_a":334.641809,"high_a":334.909287,"low_a":331.266179,"close_a":333.931641,"volume_a":83872700,"open_bench":334.641809,"close_bench":333.931641},{"date":"2020-12-01","open_a":337.168992,"high_a":339.115053,"low_a":336.5787,"close_a":337.584015,"volume_a":74231400,"open_bench":337.168992,"close_bench":337.584015},{"date":"2020-12-02","open_a":336.477237,"high_a":338.450966,"low_a":335.905409,"close_a":338.294189,"volume_a":45927000,"open_bench":336.477237,"close_bench":338.294189},{"date":"2020-12-03","open_a":338.192703,"high_a":339.5854,"low_a":337.104383,"close_a":338.201935,"volume_a":62882000,"open_bench":338.192703,"close_bench":338.201935},{"date":"2020-12-04","open_a":338.783011,"high_a":341.116455,"low_a":338.690775,"close_a":341.116455,"volume_a":50749900,"open_bench":338.783011,"close_bench":341.116455},{"date":"2020-12-07","open_a":340.350928,"high_a":340.90432,"low_a":339.151936,"close_a":340.415497,"volume_a":48944300,"open_bench":340.350928,"close_bench":340.415497},{"date":"2020-12-08","open_a":339.151889,"high_a":341.974156,"low_a":339.105785,"close_a":341.41156,"volume_a":42458900,"open_bench":339.151889,"close_bench":341.41156},{"date":"2020-12-09","open_a":342.066458,"high_a":342.223235,"low_a":337.519475,"close_a":338.349548,"volume_a":74098300,"open_bench":342.066458,"close_bench":338.349548},{"date":"2020-12-10","open_a":336.984504,"high_a":339.281048,"low_a":336.117531,"close_a":338.238861,"volume_a":57735400,"open_bench":336.984504,"close_bench":338.238861},{"date":"2020-12-11","open_a":336.550965,"high_a":338.248012,"low_a":335.038391,"close_a":337.842194,"volume_a":57698600,"open_bench":336.550965,"close_bench":337.842194},{"date":"2020-12-14","open_a":340.000332,"high_a":341.070187,"low_a":336.154288,"close_a":336.329529,"volume_a":69216200,"open_bench":340.000332,"close_bench":336.329529},{"date":"2020-12-15","open_a":338.856695,"high_a":340.876556,"low_a":337.491694,"close_a":340.876556,"volume_a":63865300,"open_bench":338.856695,"close_bench":340.876556},{"date":"2020-12-16","open_a":341.088746,"high_a":342.324638,"low_a":340.21254,"close_a":341.41156,"volume_a":58420500,"open_bench":341.088746,"close_bench":341.41156},{"date":"2020-12-17","open_a":343.044119,"high_a":343.52371,"low_a":342.223249,"close_a":343.320801,"volume_a":64119500,"open_bench":343.044119,"close_bench":343.320801},{"date":"2020-12-18","open_a":343.607901,"high_a":343.774618,"low_a":339.949235,"close_a":341.949921,"volume_a":136542300,"open_bench":343.607901,"close_bench":341.949921},{"date":"2020-12-21","open_a":338.050471,"high_a":350.545464,"low_a":335.327318,"close_a":340.727295,"volume_a":96386700,"open_bench":338.050471,"close_bench":340.727295},{"date":"2020-12-22","open_a":341.05141,"high_a":341.162554,"low_a":339.03221,"close_a":340.152954,"volume_a":47949000,"open_bench":341.05141,"close_bench":340.152954},{"date":"2020-12-23","open_a":341.116333,"high_a":342.357494,"low_a":340.134519,"close_a":340.45871,"volume_a":46201400,"open_bench":341.116333,"close_bench":340.45871},{"date":"2020-12-24","open_a":340.93117,"high_a":341.811111,"low_a":340.347661,"close_a":341.783325,"volume_a":26457900,"open_bench":340.93117,"close_bench":341.783325},{"date":"2020-12-28","open_a":344.321116,"high_a":345.108427,"low_a":343.70055,"close_a":344.719421,"volume_a":39000400,"open_bench":344.321116,"close_bench":344.719421},{"date":"2020-12-29","open_a":346.238441,"high_a":346.41443,"low_a":343.478231,"close_a":344.061768,"volume_a":53680500,"open_bench":346.238441,"close_bench":344.061768},{"date":"2020-12-30","open_a":344.876894,"high_a":345.580847,"low_a":344.163698,"close_a":344.552704,"volume_a":49455300,"open_bench":344.876894,"close_bench":344.552704},{"date":"2020-12-31","open_a":344.358231,"high_a":347.025812,"low_a":343.848809,"close_a":346.303345,"volume_a":78520700,"open_bench":344.358231,"close_bench":346.303345},{"date":"2021-01-04","open_a":347.62777,"high_a":347.757457,"low_a":337.911503,"close_a":341.588684,"volume_a":110210800,"open_bench":347.62777,"close_bench":341.588684},{"date":"2021-01-05","open_a":340.949542,"high_a":345.024999,"low_a":340.903213,"close_a":343.941284,"volume_a":66426200,"open_bench":340.949542,"close_bench":343.941284},{"date":"2021-01-06","open_a":342.440824,"high_a":349.17462,"low_a":341.894345,"close_a":345.997589,"volume_a":107997700,"open_bench":342.440824,"close_bench":345.997589},{"date":"2021-01-07","open_a":348.359489,"high_a":351.879196,"low_a":348.183501,"close_a":351.138214,"volume_a":68766800,"open_bench":348.359489,"close_bench":351.138214},{"date":"2021-01-08","open_a":352.518352,"high_a":353.351964,"low_a":349.285777,"close_a":353.138947,"volume_a":71677200,"open_bench":352.518352,"close_bench":353.138947},{"date":"2021-01-11","open_a":349.980414,"high_a":352.509035,"low_a":349.859998,"close_a":350.758453,"volume_a":51034700,"open_bench":349.980414,"close_bench":350.758453},{"date":"2021-01-12","open_a":350.943753,"high_a":351.842181,"low_a":348.600335,"close_a":350.832581,"volume_a":52547700,"open_bench":350.943753,"close_bench":350.832581},{"date":"2021-01-13","open_a":350.758564,"high_a":352.768493,"low_a":349.980524,"close_a":351.777435,"volume_a":45303600,"open_bench":350.758564,"close_bench":351.777435},{"date":"2021-01-14","open_a":352.518279,"high_a":353.018457,"low_a":350.211946,"close_a":350.54538,"volume_a":49989100,"open_bench":352.518279,"close_bench":350.54538},{"date":"2021-01-15","open_a":348.933831,"high_a":349.730386,"low_a":346.136591,"close_a":347.989075,"volume_a":107160000,"open_bench":348.933831,"close_bench":347.989075},{"date":"2021-01-19","open_a":350.434181,"high_a":351.258549,"low_a":348.96146,"close_a":350.721313,"volume_a":51233300,"open_bench":350.434181,"close_bench":350.721313},{"date":"2021-01-20","open_a":353.000002,"high_a":356.408594,"low_a":352.610997,"close_a":355.574982,"volume_a":61836100,"open_bench":353.000002,"close_bench":355.574982},{"date":"2021-01-21","open_a":356.130639,"high_a":356.55673,"low_a":354.982108,"close_a":355.899078,"volume_a":47840100,"open_bench":356.130639,"close_bench":355.899078},{"date":"2021-01-22","open_a":354.055898,"high_a":355.797237,"low_a":353.676136,"close_a":354.639435,"volume_a":52860500,"open_bench":354.055898,"close_bench":354.639435},{"date":"2021-01-25","open_a":355.371191,"high_a":356.390034,"low_a":350.545451,"close_a":356.038086,"volume_a":70402000,"open_bench":355.371191,"close_bench":356.038086},{"date":"2021-01-26","open_a":356.982837,"high_a":357.390386,"low_a":355.25077,"close_a":355.48233,"volume_a":42665300,"open_bench":356.982837,"close_bench":355.48233},{"date":"2021-01-27","open_a":352.175621,"high_a":352.268251,"low_a":344.571185,"close_a":346.794159,"volume_a":123351100,"open_bench":352.175621,"close_bench":346.794159},{"date":"2021-01-28","open_a":348.600358,"high_a":353.759531,"low_a":348.165051,"close_a":349.776703,"volume_a":94198100,"open_bench":348.600358,"close_bench":349.776703},{"date":"2021-01-29","open_a":347.924194,"high_a":348.887493,"low_a":341.10704,"close_a":342.774292,"volume_a":126765100,"open_bench":347.924194,"close_bench":342.774292},{"date":"2021-02-01","open_a":346.155104,"high_a":349.508095,"low_a":343.06146,"close_a":348.47998,"volume_a":75817600,"open_bench":346.155104,"close_bench":348.47998},{"date":"2021-02-02","open_a":351.647708,"high_a":354.954399,"low_a":348.563336,"close_a":353.407562,"volume_a":64450700,"open_bench":351.647708,"close_bench":353.407562},{"date":"2021-02-03","open_a":354.231843,"high_a":355.398916,"low_a":352.416417,"close_a":353.685364,"volume_a":52427100,"open_bench":354.231843,"close_bench":353.685364},{"date":"2021-02-04","open_a":354.713551,"high_a":357.751623,"low_a":353.796581,"close_a":357.705322,"volume_a":47142600,"open_bench":354.713551,"close_bench":357.705322},{"date":"2021-02-05","open_a":359.567037,"high_a":359.817112,"low_a":357.658981,"close_a":359.113159,"volume_a":48669800,"open_bench":359.567037,"close_bench":359.113159},{"date":"2021-02-08","open_a":360.558136,"high_a":361.752996,"low_a":359.70601,"close_a":361.706696,"volume_a":38365200,"open_bench":360.558136,"close_bench":361.706696},{"date":"2021-02-09","open_a":360.873073,"high_a":362.05869,"low_a":360.465553,"close_a":361.465881,"volume_a":35551100,"open_bench":360.873073,"close_bench":361.465881},{"date":"2021-02-10","open_a":363.197891,"high_a":363.346093,"low_a":358.918658,"close_a":361.30835,"volume_a":59154400,"open_bench":363.197891,"close_bench":361.30835},{"date":"2021-02-11","open_a":362.382844,"high_a":362.799664,"low_a":359.47446,"close_a":361.891937,"volume_a":42913300,"open_bench":362.382844,"close_bench":361.891937},{"date":"2021-02-12","open_a":361.095343,"high_a":363.920369,"low_a":361.021228,"close_a":363.679565,"volume_a":50593300,"open_bench":361.095343,"close_bench":363.679565},{"date":"2021-02-16","open_a":364.902189,"high_a":365.09672,"low_a":362.651428,"close_a":363.364624,"volume_a":50972400,"open_bench":364.902189,"close_bench":363.364624},{"date":"2021-02-17","open_a":361.623331,"high_a":363.698104,"low_a":360.613703,"close_a":363.448029,"volume_a":52290600,"open_bench":361.623331,"close_bench":363.448029},{"date":"2021-02-18","open_a":360.854466,"high_a":362.642105,"low_a":359.140913,"close_a":361.901123,"volume_a":59712800,"open_bench":360.854466,"close_bench":361.901123},{"date":"2021-02-19","open_a":363.151626,"high_a":363.438759,"low_a":360.817479,"close_a":361.262085,"volume_a":83241000,"open_bench":363.151626,"close_bench":361.262085},{"date":"2021-02-22","open_a":358.511093,"high_a":360.882269,"low_a":358.214689,"close_a":358.483307,"volume_a":67414200,"open_bench":358.511093,"close_bench":358.483307},{"date":"2021-02-23","open_a":356.288148,"high_a":360.261732,"low_a":352.157117,"close_a":358.918671,"volume_a":107284100,"open_bench":356.288148,"close_bench":358.918671},{"date":"2021-02-24","open_a":357.834991,"high_a":363.29984,"low_a":356.853177,"close_a":362.873749,"volume_a":72433900,"open_bench":357.834991,"close_bench":362.873749},{"date":"2021-02-25","open_a":361.614054,"high_a":362.975631,"low_a":352.694341,"close_a":354.130005,"volume_a":146670500,"open_bench":361.614054,"close_bench":354.130005},{"date":"2021-02-26","open_a":356.00096,"high_a":357.140219,"low_a":350.332366,"close_a":352.305237,"volume_a":152701600,"open_bench":356.00096,"close_bench":352.305237},{"date":"2021-03-01","open_a":357.14955,"high_a":362.086434,"low_a":352.499826,"close_a":360.845245,"volume_a":105348800,"open_bench":357.14955,"close_bench":360.845245},{"date":"2021-03-02","open_a":361.067521,"high_a":361.299081,"low_a":357.529271,"close_a":358.029449,"volume_a":79595300,"open_bench":361.067521,"close_bench":358.029449},{"date":"2021-03-03","open_a":357.334812,"high_a":358.298083,"low_a":353.185239,"close_a":353.28714,"volume_a":119940200,"open_bench":357.334812,"close_bench":353.28714},{"date":"2021-03-04","open_a":353.101886,"high_a":355.676837,"low_a":344.450791,"close_a":348.915283,"volume_a":183433000,"open_bench":353.101886,"close_bench":348.915283},{"date":"2021-03-05","open_a":352.397908,"high_a":356.380764,"low_a":345.154719,"close_a":355.334106,"volume_a":152039600,"open_bench":352.397908,"close_bench":355.334106},{"date":"2021-03-08","open_a":356.288186,"high_a":359.085426,"low_a":353.287172,"close_a":353.565033,"volume_a":123149200,"open_bench":356.288186,"close_bench":353.565033},{"date":"2021-03-09","open_a":357.390422,"high_a":361.150961,"low_a":356.890243,"close_a":358.613068,"volume_a":113633600,"open_bench":357.390422,"close_bench":358.613068},{"date":"2021-03-10","open_a":360.947147,"high_a":362.531012,"low_a":359.539269,"close_a":360.845245,"volume_a":109899400,"open_bench":360.947147,"close_bench":360.845245},{"date":"2021-03-11","open_a":363.299773,"high_a":366.467504,"low_a":362.845895,"close_a":364.503876,"volume_a":86245000,"open_bench":363.299773,"close_bench":364.503876},{"date":"2021-03-12","open_a":363.151661,"high_a":365.133804,"low_a":362.345835,"close_a":364.994873,"volume_a":64653600,"open_bench":363.151661,"close_bench":364.994873},{"date":"2021-03-15","open_a":365.24491,"high_a":367.430855,"low_a":363.114565,"close_a":367.171509,"volume_a":73427200,"open_bench":365.24491,"close_bench":367.171509},{"date":"2021-03-16","open_a":367.782848,"high_a":368.486773,"low_a":365.939608,"close_a":366.708405,"volume_a":73722500,"open_bench":367.782848,"close_bench":366.708405},{"date":"2021-03-17","open_a":365.430212,"high_a":368.755417,"low_a":364.290924,"close_a":367.958862,"volume_a":97959300,"open_bench":365.430212,"close_bench":367.958862},{"date":"2021-03-18","open_a":365.383828,"high_a":367.4586,"low_a":361.928936,"close_a":362.605103,"volume_a":115349100,"open_bench":365.383828,"close_bench":362.605103},{"date":"2021-03-19","open_a":362.305878,"high_a":363.876356,"low_a":359.768946,"close_a":361.934174,"volume_a":113624500,"open_bench":362.305878,"close_bench":361.934174},{"date":"2021-03-22","open_a":362.445185,"high_a":366.199465,"low_a":362.389431,"close_a":364.824127,"volume_a":73778600,"open_bench":362.445185,"close_bench":364.824127},{"date":"2021-03-23","open_a":364.192347,"high_a":365.632712,"low_a":361.172201,"close_a":361.952789,"volume_a":90686600,"open_bench":364.192347,"close_bench":361.952789},{"date":"2021-03-24","open_a":363.346712,"high_a":364.972944,"low_a":360.066371,"close_a":360.112823,"volume_a":97588600,"open_bench":363.346712,"close_bench":360.112823},{"date":"2021-03-25","open_a":358.681675,"high_a":362.928442,"low_a":356.748767,"close_a":362.13858,"volume_a":116128600,"open_bench":358.681675,"close_bench":362.13858},{"date":"2021-03-26","open_a":363.281569,"high_a":368.374008,"low_a":362.686848,"close_a":367.974426,"volume_a":114409100,"open_bench":363.281569,"close_bench":367.974426},{"date":"2021-03-29","open_a":366.506231,"high_a":368.690034,"low_a":365.028686,"close_a":367.788635,"volume_a":108107600,"open_bench":366.506231,"close_bench":367.788635},{"date":"2021-03-30","open_a":366.524824,"high_a":367.481976,"low_a":365.223815,"close_a":366.812897,"volume_a":76262200,"open_bench":366.524824,"close_bench":366.812897},{"date":"2021-03-31","open_a":367.37974,"high_a":369.851616,"low_a":367.351862,"close_a":368.299713,"volume_a":112734200,"open_bench":367.37974,"close_bench":368.299713},{"date":"2021-04-01","open_a":370.223317,"high_a":372.332791,"low_a":370.018876,"close_a":372.277008,"volume_a":99682900,"open_bench":370.223317,"close_bench":372.277008},{"date":"2021-04-05","open_a":374.925498,"high_a":378.159388,"low_a":374.851169,"close_a":377.620392,"volume_a":91684800,"open_bench":374.925498,"close_bench":377.620392},{"date":"2021-04-06","open_a":377.062844,"high_a":378.438153,"low_a":376.72829,"close_a":377.397369,"volume_a":62021000,"open_bench":377.062844,"close_bench":377.397369},{"date":"2021-04-07","open_a":377.230053,"high_a":378.177903,"low_a":376.774717,"close_a":377.834076,"volume_a":55836300,"open_bench":377.230053,"close_bench":377.834076},{"date":"2021-04-08","open_a":379.079294,"high_a":379.683318,"low_a":378.150019,"close_a":379.627563,"volume_a":57863100,"open_bench":379.079294,"close_bench":379.627563},{"date":"2021-04-09","open_a":379.506842,"high_a":382.554864,"low_a":379.386031,"close_a":382.387573,"volume_a":61104600,"open_bench":379.506842,"close_bench":382.387573},{"date":"2021-04-12","open_a":381.792842,"high_a":382.796447,"low_a":381.188819,"close_a":382.526978,"volume_a":56704900,"open_bench":381.792842,"close_bench":382.526978},{"date":"2021-04-13","open_a":382.424751,"high_a":384.283302,"low_a":382.043745,"close_a":383.660675,"volume_a":56551000,"open_bench":382.424751,"close_bench":383.660675},{"date":"2021-04-14","open_a":383.632687,"high_a":384.682773,"low_a":381.811316,"close_a":382.350311,"volume_a":61659900,"open_bench":383.632687,"close_bench":382.350311},{"date":"2021-04-15","open_a":384.478402,"high_a":386.727261,"low_a":384.43195,"close_a":386.457764,"volume_a":60229800,"open_bench":384.478402,"close_bench":386.457764},{"date":"2021-04-16","open_a":387.740088,"high_a":388.353413,"low_a":386.3276,"close_a":387.74939,"volume_a":82037300,"open_bench":387.740088,"close_bench":387.74939},{"date":"2021-04-19","open_a":386.820269,"high_a":387.266303,"low_a":384.524957,"close_a":385.844513,"volume_a":78498500,"open_bench":386.820269,"close_bench":385.844513},{"date":"2021-04-20","open_a":384.63637,"high_a":385.732908,"low_a":381.551169,"close_a":383.01944,"volume_a":81851800,"open_bench":384.63637,"close_bench":383.01944},{"date":"2021-04-21","open_a":382.406152,"high_a":386.848088,"low_a":382.266738,"close_a":386.643646,"volume_a":66793000,"open_bench":382.406152,"close_bench":386.643646},{"date":"2021-04-22","open_a":386.476367,"high_a":387.303407,"low_a":382.053006,"close_a":383.112366,"volume_a":97582800,"open_bench":386.476367,"close_bench":383.112366},{"date":"2021-04-23","open_a":383.669883,"high_a":388.669389,"low_a":383.595554,"close_a":387.266174,"volume_a":73209200,"open_bench":383.669883,"close_bench":387.266174},{"date":"2021-04-26","open_a":387.916685,"high_a":388.641519,"low_a":387.331237,"close_a":388.074646,"volume_a":52182400,"open_bench":387.916685,"close_bench":388.074646},{"date":"2021-04-27","open_a":388.372095,"high_a":388.567263,"low_a":386.857372,"close_a":387.991089,"volume_a":51303100,"open_bench":388.372095,"close_bench":387.991089},{"date":"2021-04-28","open_a":388.260553,"high_a":389.375695,"low_a":387.414909,"close_a":387.879547,"volume_a":51238900,"open_bench":388.260553,"close_bench":387.879547},{"date":"2021-04-29","open_a":390.593061,"high_a":390.964766,"low_a":386.987468,"close_a":390.35144,"volume_a":78544300,"open_bench":390.593061,"close_bench":390.35144},{"date":"2021-04-30","open_a":388.093237,"high_a":388.938881,"low_a":386.894464,"close_a":387.78656,"volume_a":85527000,"open_bench":388.093237,"close_bench":387.78656},{"date":"2021-05-03","open_a":389.765946,"high_a":390.146952,"low_a":388.130441,"close_a":388.622955,"volume_a":68128300,"open_bench":389.765946,"close_bench":388.622955},{"date":"2021-05-04","open_a":386.643649,"high_a":387.136164,"low_a":382.554843,"close_a":386.225464,"volume_a":101591200,"open_bench":386.643649,"close_bench":386.225464},{"date":"2021-05-05","open_a":387.860976,"high_a":388.093295,"low_a":385.788682,"close_a":386.346252,"volume_a":60162200,"open_bench":387.860976,"close_bench":386.346252},{"date":"2021-05-06","open_a":386.420616,"high_a":389.561572,"low_a":384.42268,"close_a":389.431488,"volume_a":74321400,"open_bench":386.420616,"close_bench":389.431488},{"date":"2021-05-07","open_a":390.193481,"high_a":392.916251,"low_a":389.5151,"close_a":392.265747,"volume_a":67733800,"open_bench":390.193481,"close_bench":392.265747},{"date":"2021-05-10","open_a":392.618841,"high_a":392.841858,"low_a":388.260537,"close_a":388.381348,"volume_a":81852400,"open_bench":392.618841,"close_bench":388.381348},{"date":"2021-05-11","open_a":383.883649,"high_a":385.90016,"low_a":381.058644,"close_a":384.915131,"volume_a":116888000,"open_bench":383.883649,"close_bench":384.915131},{"date":"2021-05-12","open_a":382.146,"high_a":383.409802,"low_a":375.427328,"close_a":376.73761,"volume_a":134811000,"open_bench":382.146,"close_bench":376.73761},{"date":"2021-05-13","open_a":378.280247,"high_a":383.186822,"low_a":378.233766,"close_a":381.263214,"volume_a":106394000,"open_bench":378.280247,"close_bench":381.263214},{"date":"2021-05-14","open_a":383.98593,"high_a":387.963228,"low_a":383.958053,"close_a":387.117584,"volume_a":82201600,"open_bench":383.98593,"close_bench":387.117584},{"date":"2021-05-17","open_a":386.011695,"high_a":386.94097,"low_a":384.125239,"close_a":386.132477,"volume_a":65129200,"open_bench":386.011695,"close_bench":386.132477},{"date":"2021-05-18","open_a":386.392653,"high_a":386.634274,"low_a":382.647675,"close_a":382.805664,"volume_a":59810200,"open_bench":386.392653,"close_bench":382.805664},{"date":"2021-05-19","open_a":378.140805,"high_a":381.978689,"low_a":376.663232,"close_a":381.802124,"volume_a":106467100,"open_bench":378.140805,"close_bench":381.802124},{"date":"2021-05-20","open_a":382.675717,"high_a":387.164135,"low_a":382.554935,"close_a":385.909607,"volume_a":78022200,"open_bench":382.675717,"close_bench":385.909607},{"date":"2021-05-21","open_a":387.387031,"high_a":388.622983,"low_a":385.138201,"close_a":385.593536,"volume_a":76578700,"open_bench":387.387031,"close_bench":385.593536},{"date":"2021-05-24","open_a":387.823824,"high_a":390.593075,"low_a":387.582203,"close_a":389.524414,"volume_a":51376700,"open_bench":387.823824,"close_bench":389.524414},{"date":"2021-05-25","open_a":390.602308,"high_a":390.955437,"low_a":388.08398,"close_a":388.660126,"volume_a":57451400,"open_bench":390.602308,"close_bench":388.660126},{"date":"2021-05-26","open_a":389.245622,"high_a":389.933276,"low_a":388.214139,"close_a":389.431488,"volume_a":43088600,"open_bench":389.245622,"close_bench":389.431488},{"date":"2021-05-27","open_a":390.453662,"high_a":390.964752,"low_a":389.357095,"close_a":389.635895,"volume_a":56707700,"open_bench":390.453662,"close_bench":389.635895},{"date":"2021-05-28","open_a":391.197136,"high_a":391.457332,"low_a":390.100597,"close_a":390.332916,"volume_a":58520200,"open_bench":391.197136,"close_bench":390.332916},{"date":"2021-06-01","open_a":392.683968,"high_a":392.823354,"low_a":389.552314,"close_a":389.989075,"volume_a":54216600,"open_bench":392.683968,"close_bench":389.989075},{"date":"2021-06-02","open_a":390.639474,"high_a":391.438665,"low_a":389.635869,"close_a":390.602295,"volume_a":49097100,"open_bench":390.639474,"close_bench":390.602295},{"date":"2021-06-03","open_a":388.297701,"high_a":390.286335,"low_a":386.838732,"close_a":389.152618,"volume_a":58138800,"open_bench":388.297701,"close_bench":389.152618},{"date":"2021-06-04","open_a":390.992688,"high_a":393.009228,"low_a":389.217768,"close_a":392.711853,"volume_a":55938800,"open_bench":390.992688,"close_bench":392.711853},{"date":"2021-06-07","open_a":392.702454,"high_a":392.879019,"low_a":391.401474,"close_a":392.33075,"volume_a":51555000,"open_bench":392.702454,"close_bench":392.33075},{"date":"2021-06-08","open_a":393.185684,"high_a":393.278617,"low_a":390.593026,"close_a":392.414398,"volume_a":47134300,"open_bench":393.185684,"close_bench":392.414398},{"date":"2021-06-09","open_a":393.2508,"high_a":393.325158,"low_a":391.605993,"close_a":391.82901,"volume_a":48436300,"open_bench":393.2508,"close_bench":391.82901},{"date":"2021-06-10","open_a":393.046337,"high_a":394.598239,"low_a":391.736055,"close_a":393.65036,"volume_a":51020100,"open_bench":393.046337,"close_bench":393.65036},{"date":"2021-06-11","open_a":394.198697,"high_a":394.412412,"low_a":392.916292,"close_a":394.300903,"volume_a":45570800,"open_bench":394.198697,"close_bench":394.300903},{"date":"2021-06-14","open_a":394.41231,"high_a":395.285831,"low_a":393.176386,"close_a":395.183624,"volume_a":42358500,"open_bench":394.41231,"close_bench":395.183624},{"date":"2021-06-15","open_a":395.332322,"high_a":395.369473,"low_a":393.58528,"close_a":394.458801,"volume_a":51508500,"open_bench":395.332322,"close_bench":394.458801},{"date":"2021-06-16","open_a":394.598262,"high_a":394.821279,"low_a":390.221382,"close_a":392.25647,"volume_a":80386100,"open_bench":394.598262,"close_bench":392.25647},{"date":"2021-06-17","open_a":391.847571,"high_a":393.102071,"low_a":389.663769,"close_a":392.126343,"volume_a":90949700,"open_bench":391.847571,"close_bench":392.126343},{"date":"2021-06-18","open_a":388.859566,"high_a":389.54947,"low_a":386.631346,"close_a":386.836456,"volume_a":118676300,"open_bench":388.859566,"close_bench":386.836456},{"date":"2021-06-21","open_a":388.58916,"high_a":392.560834,"low_a":387.77805,"close_a":392.374359,"volume_a":72822000,"open_bench":388.58916,"close_bench":392.374359},{"date":"2021-06-22","open_a":392.365093,"high_a":395.301882,"low_a":391.647191,"close_a":394.472107,"volume_a":57700300,"open_bench":392.365093,"close_bench":394.472107},{"date":"2021-06-23","open_a":394.546645,"high_a":395.348423,"low_a":393.912677,"close_a":393.996582,"volume_a":49445400,"open_bench":394.546645,"close_bench":393.996582},{"date":"2021-06-24","open_a":396.131553,"high_a":396.746856,"low_a":395.87981,"close_a":396.327332,"volume_a":45110300,"open_bench":396.131553,"close_bench":396.327332},{"date":"2021-06-25","open_a":397.073293,"high_a":398.182751,"low_a":396.746977,"close_a":397.735229,"volume_a":58129500,"open_bench":397.073293,"close_bench":397.735229},{"date":"2021-06-28","open_a":398.257304,"high_a":398.704797,"low_a":397.063941,"close_a":398.536987,"volume_a":53159600,"open_bench":398.257304,"close_bench":398.536987},{"date":"2021-06-29","open_a":398.919122,"high_a":399.55309,"low_a":398.219886,"close_a":398.751312,"volume_a":35970500,"open_bench":398.919122,"close_bench":398.751312},{"date":"2021-06-30","open_a":398.294562,"high_a":399.758305,"low_a":398.266594,"close_a":399.087036,"volume_a":64827900,"open_bench":398.294562,"close_bench":399.087036},{"date":"2021-07-01","open_a":399.842251,"high_a":401.455168,"low_a":399.776982,"close_a":401.296661,"volume_a":53441000,"open_bench":399.842251,"close_bench":401.296661},{"date":"2021-07-02","open_a":402.452717,"high_a":404.718237,"low_a":401.380531,"close_a":404.363953,"volume_a":57697700,"open_bench":402.452717,"close_bench":404.363953},{"date":"2021-07-06","open_a":404.419885,"high_a":404.634328,"low_a":400.905065,"close_a":403.627411,"volume_a":68710400,"open_bench":404.419885,"close_bench":403.627411},{"date":"2021-07-07","open_a":404.307992,"high_a":405.333545,"low_a":402.303519,"close_a":405.053833,"volume_a":63549500,"open_bench":404.307992,"close_bench":405.053833},{"date":"2021-07-08","open_a":399.75834,"high_a":402.508683,"low_a":398.583613,"close_a":401.75351,"volume_a":97595200,"open_bench":399.75834,"close_bench":401.75351},{"date":"2021-07-09","open_a":403.254469,"high_a":406.340432,"low_a":401.557648,"close_a":406.042084,"volume_a":76238600,"open_bench":403.254469,"close_bench":406.042084},{"date":"2021-07-12","open_a":405.958175,"high_a":407.748234,"low_a":405.529318,"close_a":407.49649,"volume_a":52889600,"open_bench":405.958175,"close_bench":407.49649},{"date":"2021-07-13","open_a":406.713391,"high_a":408.205102,"low_a":405.846344,"close_a":406.107391,"volume_a":52911300,"open_bench":406.713391,"close_bench":406.107391},{"date":"2021-07-14","open_a":407.794868,"high_a":408.279691,"low_a":405.473412,"close_a":406.713379,"volume_a":64130400,"open_bench":407.794868,"close_bench":406.713379},{"date":"2021-07-15","open_a":405.380155,"high_a":406.051424,"low_a":403.431619,"close_a":405.324219,"volume_a":55126400,"open_bench":405.380155,"close_bench":405.324219},{"date":"2021-07-16","open_a":406.498915,"high_a":406.54552,"low_a":401.753432,"close_a":402.144989,"volume_a":75874700,"open_bench":406.498915,"close_bench":402.144989},{"date":"2021-07-19","open_a":397.343603,"high_a":402.210292,"low_a":393.40923,"close_a":396.206177,"volume_a":147987000,"open_bench":397.343603,"close_bench":396.206177},{"date":"2021-07-20","open_a":396.868079,"high_a":403.151906,"low_a":396.075605,"close_a":401.883942,"volume_a":99608200,"open_bench":396.868079,"close_bench":401.883942},{"date":"2021-07-21","open_a":403.077377,"high_a":405.277657,"low_a":401.83741,"close_a":405.137787,"volume_a":64724400,"open_bench":403.077377,"close_bench":405.137787},{"date":"2021-07-22","open_a":405.314877,"high_a":406.228556,"low_a":404.335957,"close_a":405.986145,"volume_a":47878500,"open_bench":405.314877,"close_bench":405.986145},{"date":"2021-07-23","open_a":407.906747,"high_a":410.498584,"low_a":407.226175,"close_a":410.162964,"volume_a":63766600,"open_bench":407.906747,"close_bench":410.162964},{"date":"2021-07-26","open_a":409.575609,"high_a":411.179193,"low_a":409.529005,"close_a":411.169861,"volume_a":43719200,"open_bench":409.575609,"close_bench":411.169861},{"date":"2021-07-27","open_a":410.134916,"high_a":410.162884,"low_a":406.480226,"close_a":409.295837,"volume_a":67397100,"open_bench":410.134916,"close_bench":409.295837},{"date":"2021-07-28","open_a":409.920587,"high_a":410.498618,"low_a":407.711003,"close_a":409.128113,"volume_a":52472400,"open_bench":409.920587,"close_bench":409.128113},{"date":"2021-07-29","open_a":410.051111,"high_a":411.897078,"low_a":410.041779,"close_a":410.824921,"volume_a":47435300,"open_bench":410.051111,"close_bench":410.824921},{"date":"2021-07-30","open_a":408.270317,"high_a":410.27479,"low_a":408.139779,"close_a":408.829712,"volume_a":68951200,"open_bench":408.270317,"close_bench":408.829712},{"date":"2021-08-02","open_a":410.535914,"high_a":411.085977,"low_a":407.617761,"close_a":407.972046,"volume_a":58783300,"open_bench":410.535914,"close_bench":407.972046},{"date":"2021-08-03","open_a":408.764418,"high_a":411.412191,"low_a":406.582803,"close_a":411.290985,"volume_a":58053900,"open_bench":408.764418,"close_bench":411.290985},{"date":"2021-08-04","open_a":410.013755,"high_a":411.263054,"low_a":409.034835,"close_a":409.267914,"volume_a":46732200,"open_bench":410.013755,"close_bench":409.267914},{"date":"2021-08-05","open_a":410.423998,"high_a":411.943677,"low_a":410.107014,"close_a":411.859772,"volume_a":38969700,"open_bench":410.423998,"close_bench":411.859772},{"date":"2021-08-06","open_a":412.176725,"high_a":412.959867,"low_a":411.897014,"close_a":412.540314,"volume_a":46930000,"open_bench":412.176725,"close_bench":412.540314},{"date":"2021-08-09","open_a":412.512333,"high_a":412.829317,"low_a":411.440176,"close_a":412.204681,"volume_a":41222600,"open_bench":412.512333,"close_bench":412.204681},{"date":"2021-08-10","open_a":412.65223,"high_a":413.426068,"low_a":411.971658,"close_a":412.717499,"volume_a":43339300,"open_bench":412.65223,"close_bench":412.717499},{"date":"2021-08-11","open_a":413.780282,"high_a":413.836218,"low_a":412.661492,"close_a":413.742981,"volume_a":44034300,"open_bench":413.780282,"close_bench":413.742981},{"date":"2021-08-12","open_a":413.593861,"high_a":415.122872,"low_a":412.698845,"close_a":414.983002,"volume_a":38909400,"open_bench":413.593861,"close_bench":414.983002},{"date":"2021-08-13","open_a":415.43051,"high_a":415.756826,"low_a":414.945716,"close_a":415.73819,"volume_a":39470300,"open_bench":415.43051,"close_bench":415.73819},{"date":"2021-08-16","open_a":414.442341,"high_a":416.847703,"low_a":412.894693,"close_a":416.717194,"volume_a":73740000,"open_bench":414.442341,"close_bench":416.717194},{"date":"2021-08-17","open_a":414.17192,"high_a":414.843188,"low_a":411.011384,"close_a":413.985474,"volume_a":92673900,"open_bench":414.17192,"close_bench":413.985474},{"date":"2021-08-18","open_a":412.978528,"high_a":414.535508,"low_a":409.211994,"close_a":409.454376,"volume_a":89351900,"open_bench":412.978528,"close_bench":409.454376},{"date":"2021-08-19","open_a":406.741399,"high_a":411.281801,"low_a":406.601557,"close_a":410.088409,"volume_a":92812200,"open_bench":406.741399,"close_bench":410.088409},{"date":"2021-08-20","open_a":410.433316,"high_a":413.677757,"low_a":409.948494,"close_a":413.35144,"volume_a":72008700,"open_bench":410.433316,"close_bench":413.35144},{"date":"2021-08-23","open_a":415.029619,"high_a":417.891835,"low_a":413.426035,"close_a":416.987488,"volume_a":54973000,"open_bench":415.029619,"close_bench":416.987488},{"date":"2021-08-24","open_a":417.649445,"high_a":418.180871,"low_a":417.136682,"close_a":417.649445,"volume_a":38744700,"open_bench":417.649445,"close_bench":417.649445},{"date":"2021-08-25","open_a":417.835883,"high_a":419.03855,"low_a":417.462934,"close_a":418.525787,"volume_a":40529700,"open_bench":417.835883,"close_bench":418.525787},{"date":"2021-08-26","open_a":418.246033,"high_a":418.479112,"low_a":415.961878,"close_a":416.055115,"volume_a":57829600,"open_bench":418.246033,"close_bench":416.055115},{"date":"2021-08-27","open_a":416.856963,"high_a":420.148037,"low_a":416.801027,"close_a":419.775116,"volume_a":77235100,"open_bench":416.856963,"close_bench":419.775116},{"date":"2021-08-30","open_a":420.446458,"high_a":422.404327,"low_a":420.204047,"close_a":421.621185,"volume_a":48357400,"open_bench":420.446458,"close_bench":421.621185},{"date":"2021-08-31","open_a":421.527826,"high_a":421.863446,"low_a":420.399732,"close_a":420.996399,"volume_a":59300200,"open_bench":421.527826,"close_bench":420.996399},{"date":"2021-09-01","open_a":421.928753,"high_a":422.441516,"low_a":420.987105,"close_a":421.220184,"volume_a":48721400,"open_bench":421.928753,"close_bench":421.220184},{"date":"2021-09-02","open_a":422.637258,"high_a":423.31783,"low_a":421.32269,"close_a":422.516052,"volume_a":42501000,"open_bench":422.637258,"close_bench":422.516052},{"date":"2021-09-03","open_a":421.387989,"high_a":422.926304,"low_a":420.987072,"close_a":422.413513,"volume_a":47170500,"open_bench":421.387989,"close_bench":422.413513},{"date":"2021-09-07","open_a":422.068623,"high_a":422.161861,"low_a":420.23196,"close_a":420.903229,"volume_a":51671500,"open_bench":422.068623,"close_bench":420.903229},{"date":"2021-09-08","open_a":420.371775,"high_a":421.09898,"low_a":418.479147,"close_a":420.390411,"volume_a":56181900,"open_bench":420.371775,"close_bench":420.390411},{"date":"2021-09-09","open_a":420.194648,"high_a":421.938074,"low_a":418.348653,"close_a":418.591064,"volume_a":57970400,"open_bench":420.194648,"close_bench":418.591064},{"date":"2021-09-10","open_a":420.511684,"high_a":420.931209,"low_a":415.169505,"close_a":415.29071,"volume_a":89948200,"open_bench":420.511684,"close_bench":415.29071},{"date":"2021-09-13","open_a":418.274112,"high_a":418.535159,"low_a":414.050695,"close_a":416.353516,"volume_a":83738600,"open_bench":418.274112,"close_bench":416.353516},{"date":"2021-09-14","open_a":417.789258,"high_a":417.994368,"low_a":413.220917,"close_a":414.106628,"volume_a":78197100,"open_bench":417.789258,"close_bench":414.106628},{"date":"2021-09-15","open_a":414.526103,"high_a":418.059586,"low_a":413.425977,"close_a":417.56546,"volume_a":78792200,"open_bench":414.526103,"close_bench":417.56546},{"date":"2021-09-16","open_a":417.043376,"high_a":418.012963,"low_a":413.966717,"close_a":416.903534,"volume_a":77786700,"open_bench":417.043376,"close_bench":416.903534},{"date":"2021-09-17","open_a":416.134843,"high_a":416.555712,"low_a":412.48714,"close_a":412.84256,"volume_a":118425000,"open_bench":416.134843,"close_bench":412.84256},{"date":"2021-09-20","open_a":406.74436,"high_a":408.315662,"low_a":401.113821,"close_a":405.95871,"volume_a":166445500,"open_bench":406.74436,"close_bench":405.95871},{"date":"2021-09-21","open_a":408.287659,"high_a":409.578381,"low_a":405.05152,"close_a":405.575287,"volume_a":92526100,"open_bench":408.287659,"close_bench":405.575287},{"date":"2021-09-22","open_a":407.838629,"high_a":411.561144,"low_a":405.687445,"close_a":409.531525,"volume_a":102350100,"open_bench":407.838629,"close_bench":409.531525},{"date":"2021-09-23","open_a":411.39284,"high_a":416.106773,"low_a":411.159015,"close_a":414.507385,"volume_a":76396000,"open_bench":411.39284,"close_bench":414.507385},{"date":"2021-09-24","open_a":412.879926,"high_a":415.900964,"low_a":412.664796,"close_a":415.190125,"volume_a":62094800,"open_bench":412.879926,"close_bench":415.190125},{"date":"2021-09-27","open_a":414.161305,"high_a":415.321071,"low_a":413.310176,"close_a":414.002319,"volume_a":61371100,"open_bench":414.161305,"close_bench":414.002319},{"date":"2021-09-28","open_a":411.243181,"high_a":411.570543,"low_a":404.929889,"close_a":405.659424,"volume_a":130436300,"open_bench":411.243181,"close_bench":405.659424},{"date":"2021-09-29","open_a":407.034247,"high_a":408.764562,"low_a":405.780945,"close_a":406.342133,"volume_a":82329200,"open_bench":407.034247,"close_bench":406.342133},{"date":"2021-09-30","open_a":407.810559,"high_a":408.512036,"low_a":401.038978,"close_a":401.375702,"volume_a":140506000,"open_bench":407.810559,"close_bench":401.375702},{"date":"2021-10-01","open_a":403.096625,"high_a":407.81989,"low_a":399.589241,"close_a":406.145691,"volume_a":129240100,"open_bench":403.096625,"close_bench":406.145691},{"date":"2021-10-04","open_a":404.985986,"high_a":405.883869,"low_a":398.775564,"close_a":400.908081,"volume_a":128570000,"open_bench":404.985986,"close_bench":400.908081},{"date":"2021-10-05","open_a":402.404488,"high_a":407.314826,"low_a":401.609504,"close_a":405.079468,"volume_a":90682500,"open_bench":402.404488,"close_bench":405.079468},{"date":"2021-10-06","open_a":401.497304,"high_a":406.968829,"low_a":399.879249,"close_a":406.763062,"volume_a":113032200,"open_bench":401.497304,"close_bench":406.763062},{"date":"2021-10-07","open_a":410.027233,"high_a":413.104358,"low_a":409.849524,"close_a":410.279755,"volume_a":72437500,"open_bench":410.027233,"close_bench":410.279755},{"date":"2021-10-08","open_a":411.046739,"high_a":411.430216,"low_a":408.904888,"close_a":409.531525,"volume_a":74557400,"open_bench":411.046739,"close_bench":409.531525},{"date":"2021-10-11","open_a":408.876818,"high_a":411.776261,"low_a":406.501142,"close_a":406.56662,"volume_a":65233300,"open_bench":408.876818,"close_bench":406.56662},{"date":"2021-10-12","open_a":407.483244,"high_a":407.885417,"low_a":404.780206,"close_a":405.565857,"volume_a":71181200,"open_bench":407.483244,"close_bench":405.565857},{"date":"2021-10-13","open_a":406.585309,"high_a":407.838611,"low_a":403.620416,"close_a":407.024902,"volume_a":72974000,"open_bench":406.585309,"close_bench":407.024902},{"date":"2021-10-14","open_a":410.672561,"high_a":414.020959,"low_a":410.204909,"close_a":413.871307,"volume_a":70236800,"open_bench":410.672561,"close_bench":413.871307},{"date":"2021-10-15","open_a":415.975751,"high_a":417.388067,"low_a":415.358448,"close_a":417.023285,"volume_a":66260200,"open_bench":415.975751,"close_bench":417.023285},{"date":"2021-10-18","open_a":415.246229,"high_a":418.594599,"low_a":414.591506,"close_a":418.257904,"volume_a":62213200,"open_bench":415.246229,"close_bench":418.257904},{"date":"2021-10-19","open_a":419.875959,"high_a":421.55013,"low_a":419.26799,"close_a":421.48468,"volume_a":46996800,"open_bench":419.875959,"close_bench":421.48468},{"date":"2021-10-20","open_a":421.943042,"high_a":423.439532,"low_a":421.83081,"close_a":423.140228,"volume_a":49571600,"open_bench":421.943042,"close_bench":423.140228},{"date":"2021-10-21","open_a":422.541578,"high_a":424.468299,"low_a":422.111347,"close_a":424.243835,"volume_a":41305400,"open_bench":422.541578,"close_bench":424.243835},{"date":"2021-10-22","open_a":423.813592,"high_a":425.253966,"low_a":421.868147,"close_a":423.80423,"volume_a":58845100,"open_bench":423.813592,"close_bench":423.80423},{"date":"2021-10-25","open_a":424.889203,"high_a":426.404388,"low_a":423.121496,"close_a":426.077026,"volume_a":45214500,"open_bench":424.889203,"close_bench":426.077026},{"date":"2021-10-26","open_a":427.620244,"high_a":428.826763,"low_a":426.086334,"close_a":426.460449,"volume_a":56075100,"open_bench":427.620244,"close_bench":426.460449},{"date":"2021-10-27","open_a":426.918846,"high_a":427.582903,"low_a":424.496388,"close_a":424.571228,"volume_a":72438000,"open_bench":426.918846,"close_bench":424.571228},{"date":"2021-10-28","open_a":425.992836,"high_a":428.742628,"low_a":425.983503,"close_a":428.667816,"volume_a":51437900,"open_bench":425.992836,"close_bench":428.667816},{"date":"2021-10-29","open_a":426.376301,"high_a":429.82757,"low_a":426.08636,"close_a":429.537628,"volume_a":70162400,"open_bench":426.376301,"close_bench":429.537628},{"date":"2021-11-01","open_a":430.519679,"high_a":430.893823,"low_a":428.555567,"close_a":430.27652,"volume_a":48433600,"open_bench":430.519679,"close_bench":430.27652},{"date":"2021-11-02","open_a":430.444873,"high_a":432.32484,"low_a":430.313917,"close_a":432.016174,"volume_a":48908400,"open_bench":430.444873,"close_bench":432.016174},{"date":"2021-11-03","open_a":431.45497,"high_a":435.05589,"low_a":431.015377,"close_a":434.653717,"volume_a":52509800,"open_bench":431.45497,"close_bench":434.653717},{"date":"2021-11-04","open_a":435.252412,"high_a":436.786322,"low_a":434.906355,"close_a":436.702148,"volume_a":52847100,"open_bench":435.252412,"close_bench":436.702148},{"date":"2021-11-05","open_a":438.918701,"high_a":440.200061,"low_a":436.711401,"close_a":438.217224,"volume_a":66390600,"open_bench":438.918701,"close_bench":438.217224},{"date":"2021-11-08","open_a":439.31151,"high_a":439.807219,"low_a":437.908556,"close_a":438.591309,"volume_a":50405200,"open_bench":439.31151,"close_bench":438.591309},{"date":"2021-11-09","open_a":438.956152,"high_a":439.189977,"low_a":435.738709,"close_a":437.141663,"volume_a":51149100,"open_bench":438.956152,"close_bench":437.141663},{"date":"2021-11-10","open_a":435.458124,"high_a":437.141685,"low_a":432.147174,"close_a":433.624939,"volume_a":69429700,"open_bench":435.458124,"close_bench":433.624939},{"date":"2021-11-11","open_a":435.111944,"high_a":435.186784,"low_a":433.74641,"close_a":433.765106,"volume_a":34848500,"open_bench":435.111944,"close_bench":433.765106},{"date":"2021-11-12","open_a":435.027802,"high_a":437.590521,"low_a":434.083138,"close_a":437.038696,"volume_a":53466700,"open_bench":435.027802,"close_bench":437.038696},{"date":"2021-11-15","open_a":438.320152,"high_a":438.479137,"low_a":436.06607,"close_a":437.188416,"volume_a":46980500,"open_bench":438.320152,"close_bench":437.188416},{"date":"2021-11-16","open_a":436.926533,"high_a":440.05044,"low_a":436.851721,"close_a":438.918732,"volume_a":48857500,"open_bench":436.926533,"close_bench":438.918732},{"date":"2021-11-17","open_a":438.656855,"high_a":438.834565,"low_a":437.235205,"close_a":437.852509,"volume_a":47858300,"open_bench":438.656855,"close_bench":437.852509},{"date":"2021-11-18","open_a":438.881282,"high_a":439.601483,"low_a":436.168911,"close_a":439.3396,"volume_a":50625600,"open_bench":438.881282,"close_bench":439.3396},{"date":"2021-11-19","open_a":439.227345,"high_a":440.471314,"low_a":438.189173,"close_a":438.553955,"volume_a":57315600,"open_bench":439.227345,"close_bench":438.553955},{"date":"2021-11-22","open_a":440.424577,"high_a":442.903123,"low_a":437.113599,"close_a":437.319366,"volume_a":72762000,"open_bench":440.424577,"close_bench":437.319366},{"date":"2021-11-23","open_a":436.992017,"high_a":438.75039,"low_a":434.401239,"close_a":437.899261,"volume_a":73206500,"open_bench":436.992017,"close_bench":437.899261},{"date":"2021-11-24","open_a":435.907033,"high_a":439.189953,"low_a":435.093324,"close_a":439.068359,"volume_a":61858800,"open_bench":435.907033,"close_bench":439.068359},{"date":"2021-11-26","open_a":432.427721,"high_a":433.886791,"low_a":428.153383,"close_a":429.275757,"volume_a":112669600,"open_bench":432.427721,"close_bench":429.275757},{"date":"2021-11-29","open_a":434.045795,"high_a":436.374689,"low_a":431.857191,"close_a":434.541504,"volume_a":86268800,"open_bench":434.045795,"close_bench":434.541504},{"date":"2021-11-30","open_a":432.109715,"high_a":434.008377,"low_a":425.843177,"close_a":426.086365,"volume_a":148559600,"open_bench":432.109715,"close_bench":426.086365},{"date":"2021-12-01","open_a":431.773074,"high_a":434.60704,"low_a":421.157385,"close_a":421.35379,"volume_a":131939200,"open_bench":431.773074,"close_bench":421.35379},{"date":"2021-12-02","open_a":421.568891,"high_a":429.36931,"low_a":421.176051,"close_a":427.807343,"volume_a":127637800,"open_bench":421.568891,"close_bench":427.807343},{"date":"2021-12-03","open_a":429.462766,"high_a":430.519634,"low_a":419.875918,"close_a":424.084778,"volume_a":137331600,"open_bench":429.462766,"close_bench":424.084778},{"date":"2021-12-06","open_a":426.619544,"high_a":430.978057,"low_a":424.21581,"close_a":429.107452,"volume_a":98977500,"open_bench":426.619544,"close_bench":429.107452},{"date":"2021-12-07","open_a":434.363753,"high_a":438.544555,"low_a":428.976402,"close_a":437.983368,"volume_a":95484700,"open_bench":434.363753,"close_bench":437.983368},{"date":"2021-12-08","open_a":438.376201,"high_a":439.592083,"low_a":436.627162,"close_a":439.143127,"volume_a":72238800,"open_bench":438.376201,"close_bench":439.143127},{"date":"2021-12-09","open_a":437.861847,"high_a":439.246105,"low_a":435.981909,"close_a":436.178314,"volume_a":61272600,"open_bench":437.861847,"close_bench":436.178314},{"date":"2021-12-10","open_a":438.872045,"high_a":440.433984,"low_a":436.328021,"close_a":440.284332,"volume_a":77159800,"open_bench":438.872045,"close_bench":440.284332},{"date":"2021-12-13","open_a":439.769854,"high_a":440.115911,"low_a":436.103456,"close_a":436.384064,"volume_a":87724700,"open_bench":439.769854,"close_bench":436.384064},{"date":"2021-12-14","open_a":433.129192,"high_a":435.607738,"low_a":430.472937,"close_a":433.381714,"volume_a":97264100,"open_bench":433.129192,"close_bench":433.381714},{"date":"2021-12-15","open_a":433.437916,"high_a":440.396541,"low_a":430.931283,"close_a":440.153381,"volume_a":116899300,"open_bench":433.437916,"close_bench":440.153381},{"date":"2021-12-16","open_a":441.995866,"high_a":442.276446,"low_a":434.728548,"close_a":436.27182,"volume_a":116568600,"open_bench":441.995866,"close_bench":436.27182},{"date":"2021-12-17","open_a":433.205423,"high_a":436.199522,"low_a":429.929759,"close_a":431.628601,"volume_a":135511600,"open_bench":433.205423,"close_bench":431.628601},{"date":"2021-12-20","open_a":426.569585,"high_a":427.43307,"low_a":423.434704,"close_a":427.038879,"volume_a":107134800,"open_bench":426.569585,"close_bench":427.038879},{"date":"2021-12-21","open_a":430.445951,"high_a":434.763463,"low_a":428.28721,"close_a":434.622681,"volume_a":69806300,"open_bench":430.445951,"close_bench":434.622681},{"date":"2021-12-22","open_a":434.369337,"high_a":439.08104,"low_a":434.172213,"close_a":438.968414,"volume_a":58890200,"open_bench":434.369337,"close_bench":438.968414},{"date":"2021-12-23","open_a":439.963282,"high_a":443.192029,"low_a":439.860051,"close_a":441.699677,"volume_a":56439700,"open_bench":439.963282,"close_bench":441.699677},{"date":"2021-12-27","open_a":443.069983,"high_a":447.997571,"low_a":443.023065,"close_a":447.950653,"volume_a":56808600,"open_bench":443.069983,"close_bench":447.950653},{"date":"2021-12-28","open_a":448.382431,"high_a":449.405489,"low_a":446.824371,"close_a":447.584625,"volume_a":47274600,"open_bench":448.382431,"close_bench":447.584625},{"date":"2021-12-29","open_a":447.687932,"high_a":449.170889,"low_a":446.693031,"close_a":448.157227,"volume_a":54503000,"open_bench":447.687932,"close_bench":448.157227},{"date":"2021-12-30","open_a":448.579535,"high_a":449.583831,"low_a":446.458344,"close_a":446.918243,"volume_a":55329000,"open_bench":448.579535,"close_bench":446.918243},{"date":"2021-12-31","open_a":446.430193,"high_a":447.575243,"low_a":445.519761,"close_a":445.791931,"volume_a":65237400,"open_bench":446.430193,"close_bench":445.791931},{"date":"2022-01-03","open_a":447.049634,"high_a":448.504463,"low_a":444.750109,"close_a":448.373047,"volume_a":72668200,"open_bench":447.049634,"close_bench":448.373047},{"date":"2022-01-04","open_a":449.790294,"high_a":450.503631,"low_a":446.373819,"close_a":448.222839,"volume_a":71178700,"open_bench":449.790294,"close_bench":448.222839},{"date":"2022-01-05","open_a":447.856864,"high_a":448.626513,"low_a":439.522194,"close_a":439.616058,"volume_a":104538900,"open_bench":447.856864,"close_bench":439.616058},{"date":"2022-01-06","open_a":439.156177,"high_a":441.906234,"low_a":436.847228,"close_a":439.203094,"volume_a":86858900,"open_bench":439.156177,"close_bench":439.203094},{"date":"2022-01-07","open_a":439.212525,"high_a":440.385761,"low_a":436.115166,"close_a":437.466736,"volume_a":85111600,"open_bench":439.212525,"close_bench":437.466736},{"date":"2022-01-10","open_a":434.28481,"high_a":437.138098,"low_a":428.559416,"close_a":436.922241,"volume_a":119362000,"open_bench":434.28481,"close_bench":436.922241},{"date":"2022-01-11","open_a":436.659477,"high_a":440.995751,"low_a":433.674744,"close_a":440.901886,"volume_a":74303100,"open_bench":436.659477,"close_bench":440.901886},{"date":"2022-01-12","open_a":442.628966,"high_a":444.140108,"low_a":440.141712,"close_a":442.093964,"volume_a":67605400,"open_bench":442.628966,"close_bench":442.093964},{"date":"2022-01-13","open_a":443.192063,"high_a":443.839691,"low_a":434.979414,"close_a":436.002472,"volume_a":91173100,"open_bench":443.192063,"close_bench":436.002472},{"date":"2022-01-14","open_a":432.86754,"high_a":436.528029,"low_a":431.656753,"close_a":436.180756,"volume_a":95890900,"open_bench":432.86754,"close_bench":436.180756},{"date":"2022-01-18","open_a":431.506558,"high_a":431.713049,"low_a":427.348619,"close_a":428.456146,"volume_a":109709100,"open_bench":431.506558,"close_bench":428.456146},{"date":"2022-01-19","open_a":429.995523,"high_a":431.384616,"low_a":423.735126,"close_a":424.007324,"volume_a":109357600,"open_bench":429.995523,"close_bench":424.007324},{"date":"2022-01-20","open_a":425.884511,"high_a":430.568058,"low_a":417.202568,"close_a":419.314392,"volume_a":122379700,"open_bench":425.884511,"close_bench":419.314392},{"date":"2022-01-21","open_a":418.197496,"high_a":420.543967,"low_a":411.054851,"close_a":411.083008,"volume_a":202271200,"open_bench":418.197496,"close_bench":411.083008},{"date":"2022-01-24","open_a":405.498332,"high_a":413.335551,"low_a":394.920451,"close_a":412.828705,"volume_a":251783900,"open_bench":405.498332,"close_bench":412.828705},{"date":"2022-01-25","open_a":406.4651,"high_a":412.716102,"low_a":400.918039,"close_a":407.788513,"volume_a":167997300,"open_bench":406.4651,"close_bench":407.788513},{"date":"2022-01-26","open_a":413.654708,"high_a":416.770829,"low_a":402.523034,"close_a":406.765472,"volume_a":186391100,"open_bench":413.654708,"close_bench":406.765472},{"date":"2022-01-27","open_a":411.345714,"high_a":414.4712,"low_a":403.076753,"close_a":404.756805,"volume_a":149878300,"open_bench":411.345714,"close_bench":404.756805},{"date":"2022-01-28","open_a":406.10844,"high_a":414.856092,"low_a":401.546914,"close_a":414.809174,"volume_a":164457400,"open_bench":406.10844,"close_bench":414.809174},{"date":"2022-01-31","open_a":414.14276,"high_a":422.627608,"low_a":412.800585,"close_a":422.280334,"volume_a":152251400,"open_bench":414.14276,"close_bench":422.280334},{"date":"2022-02-01","open_a":423.002992,"high_a":425.771839,"low_a":419.492681,"close_a":425.133606,"volume_a":123155400,"open_bench":423.002992,"close_bench":425.133606},{"date":"2022-02-02","open_a":427.526973,"high_a":429.986069,"low_a":425.22742,"close_a":429.263367,"volume_a":117361000,"open_bench":427.526973,"close_bench":429.263367},{"date":"2022-02-03","open_a":423.256449,"high_a":425.152387,"low_a":418.338227,"close_a":419.173584,"volume_a":118024400,"open_bench":423.256449,"close_bench":419.173584},{"date":"2022-02-04","open_a":418.938965,"high_a":424.974082,"low_a":416.573703,"close_a":421.144653,"volume_a":118454400,"open_bench":418.938965,"close_bench":421.144653},{"date":"2022-02-07","open_a":421.904823,"high_a":423.293915,"low_a":418.469586,"close_a":419.792999,"volume_a":84472900,"open_bench":421.904823,"close_bench":419.792999},{"date":"2022-02-08","open_a":419.295621,"high_a":424.166897,"low_a":417.878343,"close_a":423.24707,"volume_a":81012000,"open_bench":419.295621,"close_bench":423.24707},{"date":"2022-02-09","open_a":427.264271,"high_a":429.76092,"low_a":427.067175,"close_a":429.441803,"volume_a":92589900,"open_bench":427.264271,"close_bench":429.441803},{"date":"2022-02-10","open_a":423.62247,"high_a":429.601273,"low_a":419.736729,"close_a":421.726532,"volume_a":140103700,"open_bench":423.62247,"close_bench":421.726532},{"date":"2022-02-11","open_a":421.810991,"high_a":423.875868,"low_a":411.98397,"close_a":413.410614,"volume_a":153214600,"open_bench":421.810991,"close_bench":413.410614},{"date":"2022-02-14","open_a":412.903896,"high_a":414.480718,"low_a":408.605144,"close_a":412.059143,"volume_a":123006300,"open_bench":412.903896,"close_bench":412.059143},{"date":"2022-02-15","open_a":416.479865,"high_a":418.873254,"low_a":415.963624,"close_a":418.704315,"volume_a":88482700,"open_bench":416.479865,"close_bench":418.704315},{"date":"2022-02-16","open_a":416.66754,"high_a":420.543915,"low_a":414.799759,"close_a":419.173584,"volume_a":84863600,"open_bench":416.66754,"close_bench":419.173584},{"date":"2022-02-17","open_a":416.00116,"high_a":419.145437,"low_a":409.61877,"close_a":410.219452,"volume_a":102259100,"open_bench":416.00116,"close_bench":410.219452},{"date":"2022-02-18","open_a":410.472833,"high_a":411.721172,"low_a":405.301231,"close_a":407.563232,"volume_a":132642900,"open_bench":410.472833,"close_bench":407.563232},{"date":"2022-02-22","open_a":405.366955,"high_a":408.755245,"low_a":399.707239,"close_a":403.189423,"volume_a":124391800,"open_bench":405.366955,"close_bench":403.189423},{"date":"2022-02-23","open_a":406.089658,"high_a":406.652816,"low_a":395.474225,"close_a":396.037384,"volume_a":132578000,"open_bench":406.089658,"close_bench":396.037384},{"date":"2022-02-24","open_a":385.778599,"high_a":402.429177,"low_a":385.42196,"close_a":401.997406,"volume_a":213942900,"open_bench":385.778599,"close_bench":401.997406},{"date":"2022-02-25","open_a":403.226972,"high_a":410.951565,"low_a":401.584442,"close_a":410.867096,"volume_a":121804500,"open_bench":403.226972,"close_bench":410.867096},{"date":"2022-02-28","open_a":405.498345,"high_a":411.289448,"low_a":404.250035,"close_a":409.815857,"volume_a":145615000,"open_bench":405.498345,"close_bench":409.815857},{"date":"2022-03-01","open_a":408.323473,"high_a":410.322671,"low_a":400.880446,"close_a":403.574219,"volume_a":137785900,"open_bench":408.323473,"close_bench":403.574219},{"date":"2022-03-02","open_a":405.817478,"high_a":412.716109,"low_a":405.066619,"close_a":410.998505,"volume_a":117726500,"open_bench":405.817478,"close_bench":410.998505},{"date":"2022-03-03","open_a":413.420052,"high_a":414.020734,"low_a":407.159655,"close_a":408.952362,"volume_a":105501700,"open_bench":413.420052,"close_bench":408.952362},{"date":"2022-03-04","open_a":405.235571,"high_a":406.75608,"low_a":401.603239,"close_a":405.629791,"volume_a":113978200,"open_bench":405.235571,"close_bench":405.629791},{"date":"2022-03-07","open_a":405.047869,"high_a":405.75181,"low_a":393.606472,"close_a":393.67218,"volume_a":137896600,"open_bench":405.047869,"close_bench":393.67218},{"date":"2022-03-08","open_a":393.850447,"high_a":400.974329,"low_a":389.626799,"close_a":390.687408,"volume_a":164772700,"open_bench":393.850447,"close_bench":390.687408},{"date":"2022-03-09","open_a":399.031463,"high_a":403.13309,"low_a":396.853931,"close_a":401.162048,"volume_a":116990800,"open_bench":399.031463,"close_bench":401.162048},{"date":"2022-03-10","open_a":396.572344,"high_a":400.242228,"low_a":394.620093,"close_a":399.350586,"volume_a":93972700,"open_bench":396.572344,"close_bench":399.350586},{"date":"2022-03-11","open_a":401.828514,"high_a":402.438591,"low_a":393.766042,"close_a":394.272888,"volume_a":95636300,"open_bench":401.828514,"close_bench":394.272888},{"date":"2022-03-14","open_a":395.042542,"high_a":398.47775,"low_a":390.255734,"close_a":391.391418,"volume_a":95729200,"open_bench":395.042542,"close_bench":391.391418},{"date":"2022-03-15","open_a":393.991151,"high_a":400.626976,"low_a":392.72408,"close_a":399.998138,"volume_a":106219100,"open_bench":393.991151,"close_bench":399.998138},{"date":"2022-03-16","open_a":403.489766,"high_a":408.924172,"low_a":398.712326,"close_a":408.867859,"volume_a":144954800,"open_bench":403.489766,"close_bench":408.867859},{"date":"2022-03-17","open_a":406.962563,"high_a":413.983215,"low_a":406.587134,"close_a":413.983215,"volume_a":102676900,"open_bench":406.962563,"close_bench":413.983215},{"date":"2022-03-18","open_a":412.378859,"high_a":418.837564,"low_a":411.644487,"close_a":418.517456,"volume_a":106345500,"open_bench":412.378859,"close_bench":418.517456},{"date":"2022-03-21","open_a":418.347958,"high_a":420.343942,"low_a":414.902049,"close_a":418.39505,"volume_a":88349800,"open_bench":418.347958,"close_bench":418.39505},{"date":"2022-03-22","open_a":419.779072,"high_a":424.222974,"low_a":419.779072,"close_a":423.290894,"volume_a":74650400,"open_bench":419.779072,"close_bench":423.290894},{"date":"2022-03-23","open_a":420.767662,"high_a":422.255226,"low_a":417.754837,"close_a":417.839569,"volume_a":79426100,"open_bench":420.767662,"close_bench":417.839569},{"date":"2022-03-24","open_a":419.854412,"high_a":424.147669,"low_a":418.743444,"close_a":424.138245,"volume_a":64736900,"open_bench":419.854412,"close_bench":424.138245},{"date":"2022-03-25","open_a":424.769003,"high_a":426.482547,"low_a":422.198686,"close_a":426.209503,"volume_a":77101300,"open_bench":424.769003,"close_bench":426.209503},{"date":"2022-03-28","open_a":425.616384,"high_a":429.24118,"low_a":423.733375,"close_a":429.24118,"volume_a":68529800,"open_bench":425.616384,"close_bench":429.24118},{"date":"2022-03-29","open_a":433.110769,"high_a":435.04087,"low_a":430.436901,"close_a":434.55127,"volume_a":86581500,"open_bench":433.110769,"close_bench":434.55127},{"date":"2022-03-30","open_a":433.412064,"high_a":434.221772,"low_a":429.768447,"close_a":431.868011,"volume_a":79666900,"open_bench":433.412064,"close_bench":431.868011},{"date":"2022-03-31","open_a":431.105379,"high_a":431.924483,"low_a":424.769045,"close_a":425.220978,"volume_a":121699900,"open_bench":431.105379,"close_bench":425.220978},{"date":"2022-04-01","open_a":426.793289,"high_a":426.934509,"low_a":422.867232,"close_a":426.426117,"volume_a":89048800,"open_bench":426.793289,"close_bench":426.426117},{"date":"2022-04-04","open_a":426.623828,"high_a":430.182712,"low_a":425.804723,"close_a":430.079132,"volume_a":59601000,"open_bench":426.623828,"close_bench":430.079132},{"date":"2022-04-05","open_a":428.59148,"high_a":431.048792,"low_a":423.507364,"close_a":424.646576,"volume_a":74214500,"open_bench":428.59148,"close_bench":424.646576},{"date":"2022-04-06","open_a":420.748832,"high_a":422.66948,"low_a":417.528875,"close_a":420.400452,"volume_a":106898000,"open_bench":420.748832,"close_bench":420.400452},{"date":"2022-04-07","open_a":419.524883,"high_a":424.326561,"low_a":417.585387,"close_a":422.51886,"volume_a":78097200,"open_bench":419.524883,"close_bench":422.51886},{"date":"2022-04-08","open_a":421.765603,"high_a":424.270008,"low_a":419.854351,"close_a":421.389008,"volume_a":79272700,"open_bench":421.765603,"close_bench":421.389008},{"date":"2022-04-11","open_a":418.13143,"high_a":418.969383,"low_a":413.687558,"close_a":414.186554,"volume_a":89770500,"open_bench":418.13143,"close_bench":414.186554},{"date":"2022-04-12","open_a":417.16167,"high_a":419.675499,"low_a":411.107805,"close_a":412.651886,"volume_a":84363600,"open_bench":417.16167,"close_bench":412.651886},{"date":"2022-04-13","open_a":412.407094,"high_a":418.131427,"low_a":412.228206,"close_a":417.378235,"volume_a":74070400,"open_bench":412.407094,"close_bench":417.378235},{"date":"2022-04-14","open_a":417.604135,"high_a":418.715132,"low_a":412.077511,"close_a":412.181091,"volume_a":97869500,"open_bench":417.604135,"close_bench":412.181091},{"date":"2022-04-18","open_a":411.258468,"high_a":414.026493,"low_a":410.128651,"close_a":412.350616,"volume_a":66002500,"open_bench":411.258468,"close_bench":412.350616},{"date":"2022-04-19","open_a":412.247057,"high_a":419.722604,"low_a":412.077594,"close_a":419.00708,"volume_a":77821000,"open_bench":412.247057,"close_bench":419.00708},{"date":"2022-04-20","open_a":420.777126,"high_a":421.389098,"low_a":417.538349,"close_a":418.696381,"volume_a":65224400,"open_bench":420.777126,"close_bench":418.696381},{"date":"2022-04-21","open_a":422.302339,"high_a":423.686351,"low_a":411.531528,"close_a":412.435364,"volume_a":85417300,"open_bench":422.302339,"close_bench":412.435364},{"date":"2022-04-22","open_a":411.352584,"high_a":412.454128,"low_a":400.55353,"close_a":401.118439,"volume_a":132471800,"open_bench":411.352584,"close_bench":401.118439},{"date":"2022-04-25","open_a":398.887123,"high_a":403.613464,"low_a":394.339641,"close_a":403.444,"volume_a":119647700,"open_bench":398.887123,"close_bench":403.444},{"date":"2022-04-26","open_a":400.920767,"high_a":401.118504,"low_a":391.731705,"close_a":391.759949,"volume_a":103996300,"open_bench":400.920767,"close_bench":391.759949},{"date":"2022-04-27","open_a":392.833176,"high_a":398.180941,"low_a":390.73364,"close_a":392.86142,"volume_a":122030000,"open_bench":392.833176,"close_bench":392.86142},{"date":"2022-04-28","open_a":397.587819,"high_a":404.507881,"low_a":393.172162,"close_a":402.784912,"volume_a":105449100,"open_bench":397.587819,"close_bench":402.784912},{"date":"2022-04-29","open_a":398.811812,"high_a":400.958441,"low_a":387.155984,"close_a":387.89978,"volume_a":145491100,"open_bench":398.811812,"close_bench":387.89978},{"date":"2022-05-02","open_a":387.965652,"high_a":391.590449,"low_a":381.32803,"close_a":390.23468,"volume_a":158312500,"open_bench":387.965652,"close_bench":390.23468},{"date":"2022-05-03","open_a":390.733673,"high_a":394.424353,"low_a":389.180168,"close_a":392.023529,"volume_a":100028200,"open_bench":390.733673,"close_bench":392.023529},{"date":"2022-05-04","open_a":392.682561,"high_a":404.5267,"low_a":389.509697,"close_a":403.961792,"volume_a":144247900,"open_bench":392.682561,"close_bench":403.961792},{"date":"2022-05-05","open_a":399.715597,"high_a":400.139286,"low_a":385.489483,"close_a":389.603851,"volume_a":172929100,"open_bench":399.715597,"close_bench":389.603851},{"date":"2022-05-06","open_a":387.05243,"high_a":390.535978,"low_a":381.996556,"close_a":387.278381,"volume_a":151770800,"open_bench":387.05243,"close_bench":387.278381},{"date":"2022-05-09","open_a":381.403402,"high_a":382.636771,"low_a":373.306459,"close_a":374.878784,"volume_a":155586100,"open_bench":381.403402,"close_bench":374.878784},{"date":"2022-05-10","open_a":380.829051,"high_a":382.32604,"low_a":371.724721,"close_a":375.744934,"volume_a":132497200,"open_bench":380.829051,"close_bench":375.744934},{"date":"2022-05-11","open_a":374.784597,"high_a":380.405378,"low_a":369.031991,"close_a":369.775787,"volume_a":142361000,"open_bench":374.784597,"close_bench":369.775787},{"date":"2022-05-12","open_a":366.593471,"high_a":372.647336,"low_a":362.620323,"close_a":369.38974,"volume_a":125090800,"open_bench":366.593471,"close_bench":369.38974},{"date":"2022-05-13","open_a":373.504124,"high_a":379.595657,"low_a":372.468464,"close_a":378.221069,"volume_a":104174400,"open_bench":373.504124,"close_bench":378.221069},{"date":"2022-05-16","open_a":376.58285,"high_a":380.339442,"low_a":374.342065,"close_a":376.686401,"volume_a":78622400,"open_bench":376.58285,"close_bench":376.686401},{"date":"2022-05-17","open_a":382.749697,"high_a":384.670374,"low_a":379.030744,"close_a":384.434998,"volume_a":83029700,"open_bench":382.749697,"close_bench":384.434998},{"date":"2022-05-18","open_a":379.896959,"high_a":380.179399,"low_a":367.704467,"close_a":368.937836,"volume_a":117674500,"open_bench":379.896959,"close_bench":368.937836},{"date":"2022-05-19","open_a":365.887332,"high_a":371.084453,"low_a":364.465652,"close_a":366.678192,"volume_a":98510700,"open_bench":365.887332,"close_bench":366.678192},{"date":"2022-05-20","open_a":370.246559,"high_a":373.805444,"low_a":358.280047,"close_a":366.838318,"volume_a":131432200,"open_bench":370.246559,"close_bench":366.838318},{"date":"2022-05-23","open_a":369.851096,"high_a":374.46449,"low_a":367.544428,"close_a":373.701874,"volume_a":76414900,"open_bench":369.851096,"close_bench":373.701874},{"date":"2022-05-24","open_a":369.596965,"high_a":372.035458,"low_a":364.324535,"close_a":370.849182,"volume_a":91448800,"open_bench":369.596965,"close_bench":370.849182},{"date":"2022-05-25","open_a":369.36151,"high_a":376.083864,"low_a":368.966094,"close_a":374.125519,"volume_a":91472900,"open_bench":369.36151,"close_bench":374.125519},{"date":"2022-05-26","open_a":375.349501,"high_a":383.229887,"low_a":375.142369,"close_a":381.601074,"volume_a":82168300,"open_bench":375.349501,"close_bench":381.601074},{"date":"2022-05-27","open_a":384.048993,"high_a":391.082031,"low_a":383.851286,"close_a":390.969055,"volume_a":84768700,"open_bench":384.048993,"close_bench":390.969055},{"date":"2022-05-31","open_a":389.359088,"high_a":392.098869,"low_a":386.045003,"close_a":388.77536,"volume_a":95937000,"open_bench":389.359088,"close_bench":388.77536},{"date":"2022-06-01","open_a":390.884347,"high_a":391.891735,"low_a":383.126332,"close_a":385.630737,"volume_a":86585800,"open_bench":390.884347,"close_bench":385.630737},{"date":"2022-06-02","open_a":385.470607,"high_a":393.021459,"low_a":383.229823,"close_a":392.974396,"volume_a":79609600,"open_bench":385.470607,"close_bench":392.974396},{"date":"2022-06-03","open_a":388.276331,"high_a":389.820411,"low_a":385.555398,"close_a":386.525146,"volume_a":71874300,"open_bench":388.276331,"close_bench":386.525146},{"date":"2022-06-06","open_a":390.517145,"high_a":392.240086,"low_a":386.534572,"close_a":387.702057,"volume_a":57508900,"open_bench":390.517145,"close_bench":387.702057},{"date":"2022-06-07","open_a":384.227882,"high_a":391.872892,"low_a":383.766525,"close_a":391.420959,"volume_a":59272400,"open_bench":384.227882,"close_bench":391.420959},{"date":"2022-06-08","open_a":389.716782,"high_a":391.496238,"low_a":386.374454,"close_a":387.165314,"volume_a":64350000,"open_bench":389.716782,"close_bench":387.165314},{"date":"2022-06-09","open_a":385.395367,"high_a":387.654972,"low_a":377.957489,"close_a":377.957489,"volume_a":86289800,"open_bench":385.395367,"close_bench":377.957489},{"date":"2022-06-10","open_a":371.781117,"high_a":372.628465,"low_a":366.951197,"close_a":366.99826,"volume_a":132893900,"open_bench":371.781117,"close_bench":366.99826},{"date":"2022-06-13","open_a":357.630388,"high_a":359.475728,"low_a":351.463518,"close_a":353.064087,"volume_a":170004900,"open_bench":357.630388,"close_bench":353.064087},{"date":"2022-06-14","open_a":354.805876,"high_a":355.832112,"low_a":348.91205,"close_a":352.000183,"volume_a":104011800,"open_bench":354.805876,"close_bench":352.000183},{"date":"2022-06-15","open_a":355.286039,"high_a":361.443485,"low_a":350.352566,"close_a":357.018433,"volume_a":125666800,"open_bench":355.286039,"close_bench":357.018433},{"date":"2022-06-16","open_a":348.836767,"high_a":349.241607,"low_a":342.782872,"close_a":345.202545,"volume_a":134473300,"open_bench":348.836767,"close_bench":345.202545},{"date":"2022-06-17","open_a":345.615698,"high_a":349.275053,"low_a":342.457494,"close_a":345.946625,"volume_a":111113900,"open_bench":345.615698,"close_bench":345.946625},{"date":"2022-06-21","open_a":351.648547,"high_a":356.035983,"low_a":351.572885,"close_a":354.655457,"volume_a":76811900,"open_bench":351.648547,"close_bench":354.655457},{"date":"2022-06-22","open_a":350.447629,"high_a":358.106762,"low_a":350.031575,"close_a":354.012451,"volume_a":90059400,"open_bench":350.447629,"close_bench":354.012451},{"date":"2022-06-23","open_a":356.13991,"high_a":358.210684,"low_a":352.594019,"close_a":357.482605,"volume_a":79292100,"open_bench":356.13991,"close_bench":357.482605},{"date":"2022-06-24","open_a":360.640838,"high_a":368.857854,"low_a":360.612472,"close_a":368.848389,"volume_a":98050300,"open_bench":360.640838,"close_bench":368.848389},{"date":"2022-06-27","open_a":369.765581,"high_a":370.058705,"low_a":366.352083,"close_a":367.439484,"volume_a":66009600,"open_bench":369.765581,"close_bench":367.439484},{"date":"2022-06-28","open_a":368.990228,"high_a":371.760745,"low_a":359.818177,"close_a":359.931641,"volume_a":86548900,"open_bench":368.990228,"close_bench":359.931641},{"date":"2022-06-29","open_a":360.480183,"high_a":361.463557,"low_a":357.82313,"close_a":359.638611,"volume_a":65676000,"open_bench":360.480183,"close_bench":359.638611},{"date":"2022-06-30","open_a":355.76173,"high_a":359.941168,"low_a":352.282035,"close_a":356.716766,"volume_a":112508300,"open_bench":355.76173,"close_bench":356.716766},{"date":"2022-07-01","open_a":356.064206,"high_a":360.924455,"low_a":353.454421,"close_a":360.489471,"volume_a":74839700,"open_bench":356.064206,"close_bench":360.489471},{"date":"2022-07-05","open_a":355.421349,"high_a":361.18934,"low_a":352.603536,"close_a":361.17041,"volume_a":81438000,"open_bench":355.421349,"close_bench":361.17041},{"date":"2022-07-06","open_a":361.312202,"high_a":364.867559,"low_a":358.938838,"close_a":362.390167,"volume_a":70426200,"open_bench":361.312202,"close_bench":362.390167},{"date":"2022-07-07","open_a":364.158423,"high_a":368.612056,"low_a":362.40911,"close_a":367.81778,"volume_a":64525900,"open_bench":364.158423,"close_bench":367.81778},{"date":"2022-07-08","open_a":366.191405,"high_a":369.378005,"low_a":364.669049,"close_a":367.515228,"volume_a":72397800,"open_bench":366.191405,"close_bench":367.515228},{"date":"2022-07-11","open_a":364.848654,"high_a":365.813126,"low_a":362.626556,"close_a":363.316833,"volume_a":58366900,"open_bench":364.848654,"close_bench":363.316833},{"date":"2022-07-12","open_a":362.768447,"high_a":365.14184,"low_a":358.362081,"close_a":360.101929,"volume_a":62219200,"open_bench":362.768447,"close_bench":360.101929},{"date":"2022-07-13","open_a":354.683762,"high_a":361.132564,"low_a":354.267708,"close_a":358.210724,"volume_a":84224600,"open_bench":354.683762,"close_bench":358.210724},{"date":"2022-07-14","open_a":353.274877,"high_a":358.418787,"low_a":350.844781,"close_a":357.340851,"volume_a":89704800,"open_bench":353.274877,"close_bench":357.340851},{"date":"2022-07-15","open_a":361.728287,"high_a":364.281341,"low_a":359.827708,"close_a":364.167877,"volume_a":79060400,"open_bench":361.728287,"close_bench":364.167877},{"date":"2022-07-18","open_a":367.240935,"high_a":367.912282,"low_a":359.941124,"close_a":361.160919,"volume_a":63203600,"open_bench":367.240935,"close_bench":361.160919},{"date":"2022-07-19","open_a":365.066072,"high_a":371.486507,"low_a":364.413654,"close_a":370.919159,"volume_a":78506000,"open_bench":365.066072,"close_bench":370.919159},{"date":"2022-07-20","open_a":371.10834,"high_a":374.692063,"low_a":369.746715,"close_a":373.283142,"volume_a":71843800,"open_bench":371.10834,"close_bench":373.283142},{"date":"2022-07-21","open_a":372.706291,"high_a":377.131557,"low_a":370.313997,"close_a":377.08429,"volume_a":64903900,"open_bench":372.706291,"close_bench":377.08429},{"date":"2022-07-22","open_a":377.207339,"high_a":378.39874,"low_a":371.373151,"close_a":373.585785,"volume_a":72197300,"open_bench":377.207339,"close_bench":373.585785},{"date":"2022-07-25","open_a":374.209838,"high_a":374.890651,"low_a":371.808079,"close_a":374.039642,"volume_a":53631500,"open_bench":374.209838,"close_bench":374.039642},{"date":"2022-07-26","open_a":372.403798,"high_a":372.611825,"low_a":368.725541,"close_a":369.61438,"volume_a":52946400,"open_bench":372.403798,"close_bench":369.61438},{"date":"2022-07-27","open_a":372.895476,"high_a":380.951762,"low_a":372.602351,"close_a":379.211914,"volume_a":82342100,"open_bench":372.895476,"close_bench":379.211914},{"date":"2022-07-28","open_a":380.015628,"high_a":384.658358,"low_a":376.479172,"close_a":383.968109,"volume_a":73966600,"open_bench":380.015628,"close_bench":383.968109},{"date":"2022-07-29","open_a":385.395975,"high_a":390.549351,"low_a":384.630065,"close_a":389.565948,"volume_a":87003700,"open_bench":385.395975,"close_bench":389.565948},{"date":"2022-08-01","open_a":386.880472,"high_a":390.908614,"low_a":386.171293,"close_a":388.412292,"volume_a":69997500,"open_bench":386.880472,"close_bench":388.412292},{"date":"2022-08-02","open_a":386.852096,"high_a":390.520917,"low_a":384.677293,"close_a":385.849792,"volume_a":63435400,"open_bench":386.852096,"close_bench":385.849792},{"date":"2022-08-03","open_a":387.967885,"high_a":393.055063,"low_a":387.684225,"close_a":391.892029,"volume_a":67820600,"open_bench":387.967885,"close_bench":391.892029},{"date":"2022-08-04","open_a":391.816325,"high_a":392.497137,"low_a":389.991379,"close_a":391.627228,"volume_a":45656600,"open_bench":391.816325,"close_bench":391.627228},{"date":"2022-08-05","open_a":387.362708,"high_a":391.608314,"low_a":387.305976,"close_a":390.965332,"volume_a":56814900,"open_bench":387.362708,"close_bench":390.965332},{"date":"2022-08-08","open_a":392.648414,"high_a":394.889413,"low_a":389.414548,"close_a":390.511414,"volume_a":53886100,"open_bench":392.648414,"close_bench":390.511414},{"date":"2022-08-09","open_a":389.783427,"high_a":390.284579,"low_a":387.892285,"close_a":388.960785,"volume_a":44931800,"open_bench":389.783427,"close_bench":388.960785},{"date":"2022-08-10","open_a":395.98633,"high_a":397.272321,"low_a":394.038455,"close_a":397.130463,"volume_a":68665700,"open_bench":395.98633,"close_bench":397.130463},{"date":"2022-08-11","open_a":399.967176,"high_a":401.820517,"low_a":396.392918,"close_a":397.130463,"volume_a":59489700,"open_bench":399.967176,"close_bench":397.130463},{"date":"2022-08-12","open_a":399.059402,"high_a":403.957454,"low_a":398.113831,"close_a":403.853455,"volume_a":61694500,"open_bench":399.059402,"close_bench":403.853455},{"date":"2022-08-15","open_a":401.650257,"high_a":406.037721,"low_a":401.593525,"close_a":405.517639,"volume_a":54048300,"open_bench":401.650257,"close_bench":405.517639},{"date":"2022-08-16","open_a":404.449174,"high_a":408.231459,"low_a":403.645433,"close_a":406.311951,"volume_a":59289000,"open_bench":404.449174,"close_bench":406.311951},{"date":"2022-08-17","open_a":402.728265,"high_a":406.122862,"low_a":401.432837,"close_a":403.427979,"volume_a":63563400,"open_bench":402.728265,"close_bench":403.427979},{"date":"2022-08-18","open_a":403.626498,"high_a":405.281247,"low_a":402.340535,"close_a":404.600464,"volume_a":49023200,"open_bench":403.626498,"close_bench":404.600464},{"date":"2022-08-19","open_a":401.84881,"high_a":402.113568,"low_a":398.293453,"close_a":399.163391,"volume_a":68016900,"open_bench":401.84881,"close_bench":399.163391},{"date":"2022-08-22","open_a":394.350464,"high_a":394.520689,"low_a":389.953563,"close_a":390.851868,"volume_a":77695600,"open_bench":394.350464,"close_bench":390.851868},{"date":"2022-08-23","open_a":390.426364,"high_a":392.809222,"low_a":389.357864,"close_a":389.906311,"volume_a":49105200,"open_bench":390.426364,"close_bench":389.906311},{"date":"2022-08-24","open_a":389.679362,"high_a":392.516076,"low_a":388.998579,"close_a":391.15448,"volume_a":49177800,"open_bench":389.679362,"close_bench":391.15448},{"date":"2022-08-25","open_a":392.638967,"high_a":396.723842,"low_a":391.551566,"close_a":396.676575,"volume_a":50942300,"open_bench":392.638967,"close_bench":396.676575},{"date":"2022-08-26","open_a":396.563109,"high_a":397.102063,"low_a":383.192719,"close_a":383.249451,"volume_a":103087000,"open_bench":396.563109,"close_bench":383.249451},{"date":"2022-08-29","open_a":380.308804,"high_a":383.750669,"low_a":379.363233,"close_a":380.715393,"volume_a":65370800,"open_bench":380.308804,"close_bench":380.715393},{"date":"2022-08-30","open_a":381.868924,"high_a":382.105317,"low_a":374.446184,"close_a":376.535889,"volume_a":85652400,"open_bench":381.868924,"close_bench":376.535889},{"date":"2022-08-31","open_a":378.162208,"high_a":379.400904,"low_a":373.538381,"close_a":373.670746,"volume_a":76029700,"open_bench":378.162208,"close_bench":373.670746},{"date":"2022-09-01","open_a":371.505488,"high_a":375.183745,"low_a":368.810604,"close_a":374.843353,"volume_a":78740100,"open_bench":371.505488,"close_bench":374.843353},{"date":"2022-09-02","open_a":378.493301,"high_a":379.703632,"low_a":369.084855,"close_a":370.8909,"volume_a":99632100,"open_bench":378.493301,"close_bench":370.8909},{"date":"2022-09-06","open_a":371.732485,"high_a":372.668592,"low_a":367.278852,"close_a":369.491486,"volume_a":76637400,"open_bench":371.732485,"close_bench":369.491486},{"date":"2022-09-07","open_a":369.179349,"high_a":376.895213,"low_a":368.961886,"close_a":376.129303,"volume_a":70964200,"open_bench":369.179349,"close_bench":376.129303},{"date":"2022-09-08","open_a":373.869438,"high_a":379.041685,"low_a":372.668544,"close_a":378.58783,"volume_a":80821700,"open_bench":373.869438,"close_bench":378.58783},{"date":"2022-09-09","open_a":380.819319,"high_a":385.329711,"low_a":380.55456,"close_a":384.469238,"volume_a":76706900,"open_bench":380.819319,"close_bench":384.469238},{"date":"2022-09-12","open_a":386.530576,"high_a":389.320023,"low_a":386.227987,"close_a":388.601379,"volume_a":69256300,"open_bench":386.530576,"close_bench":388.601379},{"date":"2022-09-13","open_a":379.95889,"high_a":381.159784,"low_a":370.588304,"close_a":371.704071,"volume_a":122947100,"open_bench":379.95889,"close_bench":371.704071},{"date":"2022-09-14","open_a":372.999508,"high_a":374.635356,"low_a":369.831838,"close_a":373.122437,"volume_a":85023700,"open_bench":372.999508,"close_bench":373.122437},{"date":"2022-09-15","open_a":371.571557,"high_a":374.40827,"low_a":367.619077,"close_a":368.886139,"volume_a":87633800,"open_bench":371.571557,"close_bench":368.886139},{"date":"2022-09-16","open_a":364.723872,"high_a":366.727209,"low_a":362.796449,"close_a":366.072083,"volume_a":103084800,"open_bench":364.723872,"close_bench":366.072083},{"date":"2022-09-19","open_a":362.938865,"high_a":368.910919,"low_a":362.862892,"close_a":368.910919,"volume_a":73278500,"open_bench":362.938865,"close_bench":368.910919},{"date":"2022-09-20","open_a":365.597395,"high_a":366.603816,"low_a":361.93251,"close_a":364.676422,"volume_a":77274900,"open_bench":365.597395,"close_bench":364.676422},{"date":"2022-09-21","open_a":366.594226,"high_a":369.632495,"low_a":358.305499,"close_a":358.315002,"volume_a":106746600,"open_bench":366.594226,"close_bench":358.315002},{"date":"2022-09-22","open_a":357.545938,"high_a":359.179002,"low_a":354.564663,"close_a":355.305237,"volume_a":89472600,"open_bench":357.545938,"close_bench":355.305237},{"date":"2022-09-23","open_a":351.849278,"high_a":351.887264,"low_a":344.927767,"close_a":349.352234,"volume_a":122346900,"open_bench":351.849278,"close_bench":349.352234},{"date":"2022-09-26","open_a":347.890011,"high_a":351.497931,"low_a":344.680847,"close_a":345.896149,"volume_a":92581200,"open_bench":347.890011,"close_bench":345.896149},{"date":"2022-09-27","open_a":349.418612,"high_a":351.67832,"low_a":342.63001,"close_a":345.013153,"volume_a":108294100,"open_bench":349.418612,"close_bench":345.013153},{"date":"2022-09-28","open_a":345.962612,"high_a":353.482284,"low_a":344.272582,"close_a":351.801758,"volume_a":110802200,"open_bench":345.962612,"close_bench":351.801758},{"date":"2022-09-29","open_a":348.269804,"high_a":348.554629,"low_a":341.519189,"close_a":344.453003,"volume_a":112952300,"open_bench":348.269804,"close_bench":344.453003},{"date":"2022-09-30","open_a":343.513037,"high_a":347.415315,"low_a":338.993648,"close_a":339.126556,"volume_a":153711200,"open_bench":343.513037,"close_bench":339.126556},{"date":"2022-10-03","open_a":342.829437,"high_a":349.921872,"low_a":341.053959,"close_a":348.079926,"volume_a":89756500,"open_bench":342.829437,"close_bench":348.079926},{"date":"2022-10-04","open_a":353.577248,"high_a":358.894205,"low_a":348.041935,"close_a":358.865723,"volume_a":103602800,"open_bench":353.577248,"close_bench":358.865723},{"date":"2022-10-05","open_a":354.517153,"high_a":360.280325,"low_a":352.20048,"close_a":358.030121,"volume_a":88065700,"open_bench":354.517153,"close_bench":358.030121},{"date":"2022-10-06","open_a":356.634489,"high_a":359.577807,"low_a":353.843087,"close_a":354.336823,"volume_a":82333500,"open_bench":356.634489,"close_bench":354.336823},{"date":"2022-10-07","open_a":350.320632,"high_a":354.422287,"low_a":342.696504,"close_a":344.453003,"volume_a":107789500,"open_bench":350.320632,"close_bench":344.453003},{"date":"2022-10-10","open_a":345.563855,"high_a":345.801219,"low_a":339.591801,"close_a":341.822998,"volume_a":76042800,"open_bench":345.563855,"close_bench":341.822998},{"date":"2022-10-11","open_a":340.1329,"high_a":344.6808,"low_a":337.730779,"close_a":339.658173,"volume_a":92482800,"open_bench":340.1329,"close_bench":339.658173},{"date":"2022-10-12","open_a":340.06648,"high_a":341.633076,"low_a":338.290974,"close_a":338.537842,"volume_a":76991800,"open_bench":340.06648,"close_bench":338.537842},{"date":"2022-10-13","open_a":331.559375,"high_a":348.934429,"low_a":330.514968,"close_a":347.47226,"volume_a":147254500,"open_bench":331.559375,"close_bench":347.47226},{"date":"2022-10-14","open_a":349.921747,"high_a":351.545336,"low_a":338.917563,"close_a":339.553711,"volume_a":123737000,"open_bench":349.921747,"close_bench":339.553711},{"date":"2022-10-17","open_a":345.611329,"high_a":349.380669,"low_a":339.221483,"close_a":348.279297,"volume_a":93168200,"open_bench":345.611329,"close_bench":348.279297},{"date":"2022-10-18","open_a":356.169251,"high_a":356.473084,"low_a":348.94388,"close_a":352.371429,"volume_a":97162900,"open_bench":356.169251,"close_bench":352.371429},{"date":"2022-10-19","open_a":350.339553,"high_a":353.05501,"low_a":347.073423,"close_a":349.874329,"volume_a":79746900,"open_bench":350.339553,"close_bench":349.874329},{"date":"2022-10-20","open_a":349.428121,"high_a":353.833608,"low_a":346.18097,"close_a":346.940552,"volume_a":88283100,"open_bench":349.428121,"close_bench":346.940552},{"date":"2022-10-21","open_a":346.665184,"high_a":355.855907,"low_a":345.165058,"close_a":355.371704,"volume_a":131038400,"open_bench":346.665184,"close_bench":355.371704},{"date":"2022-10-24","open_a":356.890886,"high_a":360.8501,"low_a":354.251372,"close_a":359.720245,"volume_a":85436900,"open_bench":356.890886,"close_bench":359.720245},{"date":"2022-10-25","open_a":359.64422,"high_a":365.777694,"low_a":359.53029,"close_a":365.464386,"volume_a":78846300,"open_bench":359.64422,"close_bench":365.464386},{"date":"2022-10-26","open_a":362.331192,"high_a":367.989938,"low_a":362.074849,"close_a":362.710968,"volume_a":104087300,"open_bench":362.331192,"close_bench":362.710968},{"date":"2022-10-27","open_a":363.707954,"high_a":365.540396,"low_a":360.15697,"close_a":360.774139,"volume_a":81971800,"open_bench":363.707954,"close_bench":360.774139},{"date":"2022-10-28","open_a":360.669725,"high_a":369.831967,"low_a":360.489326,"close_a":369.357239,"volume_a":100302000,"open_bench":360.669725,"close_bench":369.357239},{"date":"2022-10-31","open_a":366.907594,"high_a":368.768519,"low_a":365.787244,"close_a":366.689209,"volume_a":96631300,"open_bench":366.907594,"close_bench":366.689209},{"date":"2022-11-01","open_a":370.42059,"high_a":370.657953,"low_a":363.916814,"close_a":365.084625,"volume_a":85407600,"open_bench":370.42059,"close_bench":365.084625},{"date":"2022-11-02","open_a":364.495976,"high_a":368.986911,"low_a":355.817968,"close_a":355.922394,"volume_a":126990400,"open_bench":364.495976,"close_bench":355.922394},{"date":"2022-11-03","open_a":352.694218,"high_a":355.286242,"low_a":350.149685,"close_a":352.257477,"volume_a":87100100,"open_bench":352.694218,"close_bench":352.257477},{"date":"2022-11-04","open_a":357.944685,"high_a":359.720162,"low_a":351.298498,"close_a":357.327545,"volume_a":103505200,"open_bench":357.944685,"close_bench":357.327545},{"date":"2022-11-07","open_a":358.618866,"high_a":361.334324,"low_a":356.54906,"close_a":360.745667,"volume_a":68286900,"open_bench":358.618866,"close_bench":360.745667},{"date":"2022-11-08","open_a":361.846957,"high_a":365.654282,"low_a":358.628318,"close_a":362.691986,"volume_a":84641100,"open_bench":361.846957,"close_bench":362.691986},{"date":"2022-11-09","open_a":360.726587,"high_a":361.875449,"low_a":354.726022,"close_a":355.219757,"volume_a":78495500,"open_bench":360.726587,"close_bench":355.219757},{"date":"2022-11-10","open_a":368.436202,"high_a":375.072916,"low_a":366.14804,"close_a":374.740601,"volume_a":141455800,"open_bench":368.436202,"close_bench":374.740601},{"date":"2022-11-11","open_a":375.59507,"high_a":379.165032,"low_a":373.715138,"close_a":378.367493,"volume_a":93839900,"open_bench":375.59507,"close_bench":378.367493},{"date":"2022-11-14","open_a":376.611034,"high_a":379.953107,"low_a":374.873515,"close_a":375.148865,"volume_a":71903500,"open_bench":376.611034,"close_bench":375.148865},{"date":"2022-11-15","open_a":380.874066,"high_a":381.975438,"low_a":374.550689,"close_a":378.348511,"volume_a":93194500,"open_bench":380.874066,"close_bench":378.348511},{"date":"2022-11-16","open_a":376.724921,"high_a":377.674377,"low_a":374.835514,"close_a":375.462158,"volume_a":68508500,"open_bench":376.724921,"close_bench":375.462158},{"date":"2022-11-17","open_a":370.724383,"high_a":374.987458,"low_a":370.42058,"close_a":374.313324,"volume_a":74496300,"open_bench":370.724383,"close_bench":374.313324},{"date":"2022-11-18","open_a":377.636378,"high_a":377.702847,"low_a":373.173955,"close_a":376.012817,"volume_a":92922500,"open_bench":377.636378,"close_bench":376.012817},{"date":"2022-11-21","open_a":374.69309,"high_a":375.81344,"low_a":372.813157,"close_a":374.645599,"volume_a":51243200,"open_bench":374.69309,"close_bench":374.645599},{"date":"2022-11-22","open_a":376.582547,"high_a":379.848676,"low_a":375.177342,"close_a":379.687256,"volume_a":60429000,"open_bench":376.582547,"close_bench":379.687256},{"date":"2022-11-23","open_a":379.354933,"high_a":382.564098,"low_a":379.127073,"close_a":382.079895,"volume_a":68261600,"open_bench":379.354933,"close_bench":382.079895},{"date":"2022-11-25","open_a":381.519657,"high_a":382.545085,"low_a":381.244336,"close_a":381.994385,"volume_a":30545400,"open_bench":381.519657,"close_bench":381.994385},{"date":"2022-11-28","open_a":378.918187,"high_a":380.551252,"low_a":375.139344,"close_a":375.898926,"volume_a":67881600,"open_bench":378.918187,"close_bench":375.898926},{"date":"2022-11-29","open_a":376.031797,"high_a":377.218616,"low_a":373.420794,"close_a":375.253265,"volume_a":52310000,"open_bench":376.031797,"close_bench":375.253265},{"date":"2022-11-30","open_a":375.50014,"high_a":387.074005,"low_a":373.591755,"close_a":387.074005,"volume_a":144566700,"open_bench":375.50014,"close_bench":387.074005},{"date":"2022-12-01","open_a":388.108913,"high_a":389.276753,"low_a":384.292112,"close_a":386.789185,"volume_a":76398200,"open_bench":388.108913,"close_bench":386.789185},{"date":"2022-12-02","open_a":381.91846,"high_a":387.244891,"low_a":381.814034,"close_a":386.342926,"volume_a":85342700,"open_bench":381.91846,"close_bench":386.342926},{"date":"2022-12-05","open_a":383.532647,"high_a":384.463095,"low_a":378.044794,"close_a":379.393005,"volume_a":77289800,"open_bench":383.532647,"close_bench":379.393005},{"date":"2022-12-06","open_a":379.231523,"high_a":379.772691,"low_a":371.844761,"close_a":373.924042,"volume_a":77972200,"open_bench":379.231523,"close_bench":373.924042},{"date":"2022-12-07","open_a":373.079052,"high_a":375.642593,"low_a":372.158079,"close_a":373.287933,"volume_a":65927900,"open_bench":373.079052,"close_bench":373.287933},{"date":"2022-12-08","open_a":375.167841,"high_a":377.275605,"low_a":373.392335,"close_a":376.212219,"volume_a":60737900,"open_bench":375.167841,"close_bench":376.212219},{"date":"2022-12-09","open_a":374.977924,"high_a":377.522458,"low_a":373.278391,"close_a":373.401825,"volume_a":81447700,"open_bench":374.977924,"close_bench":373.401825},{"date":"2022-12-12","open_a":374.189949,"high_a":378.785339,"low_a":373.525347,"close_a":378.785339,"volume_a":75405800,"open_bench":374.189949,"close_bench":378.785339},{"date":"2022-12-13","open_a":389.485657,"high_a":389.742,"low_a":378.899234,"close_a":381.652649,"volume_a":123782500,"open_bench":389.485657,"close_bench":381.652649},{"date":"2022-12-14","open_a":381.310813,"high_a":385.004209,"low_a":376.27871,"close_a":379.212524,"volume_a":108111300,"open_bench":381.310813,"close_bench":379.212524},{"date":"2022-12-15","open_a":374.370312,"high_a":375.272306,"low_a":368.284327,"close_a":369.936371,"volume_a":117705900,"open_bench":374.370312,"close_bench":369.936371},{"date":"2022-12-16","open_a":367.390547,"high_a":368.725882,"low_a":363.441767,"close_a":365.568756,"volume_a":119858000,"open_bench":367.390547,"close_bench":365.568756},{"date":"2022-12-19","open_a":365.759639,"high_a":366.09348,"low_a":360.809334,"close_a":362.468964,"volume_a":79878100,"open_bench":365.759639,"close_bench":362.468964},{"date":"2022-12-20","open_a":361.715409,"high_a":364.576855,"low_a":360.399139,"close_a":362.964905,"volume_a":74427200,"open_bench":361.715409,"close_bench":362.964905},{"date":"2022-12-21","open_a":365.54974,"high_a":369.517616,"low_a":365.015606,"close_a":368.39212,"volume_a":78167400,"open_bench":365.54974,"close_bench":368.39212},{"date":"2022-12-22","open_a":365.358974,"high_a":368.373034,"low_a":357.461383,"close_a":363.136597,"volume_a":100120900,"open_bench":365.358974,"close_bench":363.136597},{"date":"2022-12-23","open_a":362.116077,"high_a":365.368591,"low_a":360.5709,"close_a":365.225525,"volume_a":59857300,"open_bench":362.116077,"close_bench":365.225525},{"date":"2022-12-27","open_a":365.111065,"high_a":365.454425,"low_a":362.116071,"close_a":363.785248,"volume_a":51638200,"open_bench":365.111065,"close_bench":363.785248},{"date":"2022-12-28","open_a":363.718431,"high_a":365.683318,"low_a":359.035223,"close_a":359.26413,"volume_a":70911500,"open_bench":363.718431,"close_bench":359.26413},{"date":"2022-12-29","open_a":362.096953,"high_a":366.598963,"low_a":361.572337,"close_a":365.730988,"volume_a":66970900,"open_bench":362.096953,"close_bench":365.730988},{"date":"2022-12-30","open_a":363.060239,"high_a":364.910614,"low_a":360.952287,"close_a":364.767548,"volume_a":84022200,"open_bench":363.060239,"close_bench":364.767548},{"date":"2023-01-03","open_a":366.618028,"high_a":368.582885,"low_a":360.380067,"close_a":363.231995,"volume_a":74850700,"open_bench":366.618028,"close_bench":363.231995},{"date":"2023-01-04","open_a":365.482995,"high_a":368.058308,"low_a":362.449869,"close_a":366.036224,"volume_a":85934100,"open_bench":365.482995,"close_bench":366.036224},{"date":"2023-01-05","open_a":364.090415,"high_a":364.204868,"low_a":361.267129,"close_a":361.85849,"volume_a":76970500,"open_bench":364.090415,"close_bench":361.85849},{"date":"2023-01-06","open_a":364.939276,"high_a":371.272623,"low_a":361.887084,"close_a":370.156647,"volume_a":104189600,"open_bench":364.939276,"close_bench":370.156647},{"date":"2023-01-09","open_a":372.340863,"high_a":375.517084,"low_a":369.765579,"close_a":369.946777,"volume_a":73978100,"open_bench":372.340863,"close_bench":369.946777},{"date":"2023-01-10","open_a":369.365006,"high_a":372.607973,"low_a":368.430257,"close_a":372.541199,"volume_a":65358100,"open_bench":369.365006,"close_bench":372.541199},{"date":"2023-01-11","open_a":374.11505,"high_a":377.329404,"low_a":373.304301,"close_a":377.253082,"volume_a":68881100,"open_bench":374.11505,"close_bench":377.253082},{"date":"2023-01-12","open_a":378.349971,"high_a":380.085893,"low_a":374.296255,"close_a":378.626556,"volume_a":90157700,"open_bench":378.349971,"close_bench":378.626556},{"date":"2023-01-13","open_a":375.440805,"high_a":380.667724,"low_a":375.173738,"close_a":380.095428,"volume_a":63903900,"open_bench":375.440805,"close_bench":380.095428},{"date":"2023-01-17","open_a":380.076308,"high_a":381.745485,"low_a":378.721878,"close_a":379.399078,"volume_a":62677300,"open_bench":380.076308,"close_bench":379.399078},{"date":"2023-01-18","open_a":380.581828,"high_a":381.640549,"low_a":373.208826,"close_a":373.409119,"volume_a":99632300,"open_bench":380.581828,"close_bench":373.409119},{"date":"2023-01-19","open_a":371.377546,"high_a":373.018109,"low_a":369.374557,"close_a":370.690826,"volume_a":86958900,"open_bench":371.377546,"close_bench":370.690826},{"date":"2023-01-20","open_a":372.083384,"high_a":377.749049,"low_a":370.44282,"close_a":377.596436,"volume_a":91806400,"open_bench":372.083384,"close_bench":377.596436},{"date":"2023-01-23","open_a":378.397653,"high_a":384.053772,"low_a":377.443838,"close_a":382.127075,"volume_a":84178800,"open_bench":378.397653,"close_bench":382.127075},{"date":"2023-01-24","open_a":380.457906,"high_a":382.623057,"low_a":379.275184,"close_a":381.716949,"volume_a":59524900,"open_bench":380.457906,"close_bench":381.716949},{"date":"2023-01-25","open_a":377.663234,"high_a":382.193857,"low_a":375.383601,"close_a":381.860016,"volume_a":84800300,"open_bench":377.663234,"close_bench":381.860016},{"date":"2023-01-26","open_a":384.511678,"high_a":386.219016,"low_a":381.554844,"close_a":386.056854,"volume_a":72287400,"open_bench":384.511678,"close_bench":386.056854},{"date":"2023-01-27","open_a":385.017121,"high_a":389.30929,"low_a":384.80728,"close_a":386.943817,"volume_a":68346200,"open_bench":385.017121,"close_bench":386.943817},{"date":"2023-01-30","open_a":384.196883,"high_a":386.419289,"low_a":381.793278,"close_a":382.088959,"volume_a":74202000,"open_bench":384.196883,"close_bench":382.088959},{"date":"2023-01-31","open_a":382.604021,"high_a":387.754619,"low_a":382.260632,"close_a":387.70694,"volume_a":86811800,"open_bench":382.604021,"close_bench":387.70694},{"date":"2023-02-01","open_a":386.495567,"high_a":394.564867,"low_a":383.767669,"close_a":391.827393,"volume_a":101459200,"open_bench":386.495567,"close_bench":391.827393},{"date":"2023-02-02","open_a":395.699942,"high_a":398.990617,"low_a":393.811405,"close_a":397.531281,"volume_a":101654500,"open_bench":395.699942,"close_bench":397.531281},{"date":"2023-02-03","open_a":392.580877,"high_a":397.712409,"low_a":392.103969,"close_a":393.305786,"volume_a":94736800,"open_bench":392.580877,"close_bench":393.305786},{"date":"2023-02-06","open_a":390.864029,"high_a":392.294752,"low_a":389.252079,"close_a":390.902161,"volume_a":60295300,"open_bench":390.864029,"close_bench":390.902161},{"date":"2023-02-07","open_a":389.986498,"high_a":397.254566,"low_a":388.746549,"close_a":396.014618,"volume_a":90990700,"open_bench":389.986498,"close_bench":396.014618},{"date":"2023-02-08","open_a":394.049799,"high_a":395.385135,"low_a":390.997578,"close_a":391.684326,"volume_a":76227500,"open_bench":394.049799,"close_bench":391.684326},{"date":"2023-02-09","open_a":395.270632,"high_a":395.423246,"low_a":387.067814,"close_a":388.288696,"volume_a":78694900,"open_bench":395.270632,"close_bench":388.288696},{"date":"2023-02-10","open_a":387.115515,"high_a":389.576375,"low_a":386.304795,"close_a":389.194855,"volume_a":70769700,"open_bench":387.115515,"close_bench":389.194855},{"date":"2023-02-13","open_a":389.843353,"high_a":393.897067,"low_a":389.385511,"close_a":393.763519,"volume_a":64913500,"open_bench":389.843353,"close_bench":393.763519},{"date":"2023-02-14","open_a":392.247093,"high_a":395.881128,"low_a":389.643196,"close_a":393.582458,"volume_a":88389300,"open_bench":392.247093,"close_bench":393.582458},{"date":"2023-02-15","open_a":391.39818,"high_a":394.936827,"low_a":390.558818,"close_a":394.860535,"volume_a":61555700,"open_bench":391.39818,"close_bench":394.860535},{"date":"2023-02-16","open_a":389.910192,"high_a":393.839906,"low_a":389.290218,"close_a":389.423737,"volume_a":76431500,"open_bench":389.910192,"close_bench":389.423737},{"date":"2023-02-17","open_a":387.306307,"high_a":388.689351,"low_a":385.389129,"close_a":388.450897,"volume_a":89257800,"open_bench":387.306307,"close_bench":388.450897},{"date":"2023-02-21","open_a":384.444882,"high_a":385.494085,"low_a":380.400714,"close_a":380.658234,"volume_a":82655900,"open_bench":384.444882,"close_bench":380.658234},{"date":"2023-02-22","open_a":381.068357,"high_a":382.604015,"low_a":378.683818,"close_a":380.133636,"volume_a":83742300,"open_bench":381.068357,"close_bench":380.133636},{"date":"2023-02-23","open_a":383.014068,"high_a":383.624523,"low_a":377.949311,"close_a":382.15564,"volume_a":96242400,"open_bench":383.014068,"close_bench":382.15564},{"date":"2023-02-24","open_a":377.157679,"high_a":378.903148,"low_a":375.459889,"close_a":378.073334,"volume_a":108194400,"open_bench":377.157679,"close_bench":378.073334},{"date":"2023-02-27","open_a":381.402142,"high_a":382.756573,"low_a":378.426243,"close_a":379.360992,"volume_a":80444700,"open_bench":381.402142,"close_bench":379.360992},{"date":"2023-02-28","open_a":378.884095,"high_a":380.839405,"low_a":377.853958,"close_a":377.958893,"volume_a":96438600,"open_bench":378.884095,"close_bench":377.958893},{"date":"2023-03-01","open_a":377.148163,"high_a":378.369046,"low_a":375.211919,"close_a":376.509094,"volume_a":99706800,"open_bench":377.148163,"close_bench":376.509094},{"date":"2023-03-02","open_a":374.544179,"high_a":380.276618,"low_a":374.210338,"close_a":379.437256,"volume_a":85127800,"open_bench":374.544179,"close_bench":379.437256},{"date":"2023-03-03","open_a":381.249601,"high_a":385.770706,"low_a":380.601013,"close_a":385.522705,"volume_a":90120000,"open_bench":381.249601,"close_bench":385.522705},{"date":"2023-03-06","open_a":386.342903,"high_a":388.632084,"low_a":385.350956,"close_a":385.789703,"volume_a":72795900,"open_bench":386.342903,"close_bench":385.789703},{"date":"2023-03-07","open_a":385.741994,"high_a":385.980448,"low_a":379.26558,"close_a":379.876007,"volume_a":108310600,"open_bench":385.741994,"close_bench":379.876007},{"date":"2023-03-08","open_a":379.990481,"high_a":381.249495,"low_a":378.273596,"close_a":380.496002,"volume_a":74746600,"open_bench":379.990481,"close_bench":380.496002},{"date":"2023-03-09","open_a":381.278249,"high_a":382.937908,"low_a":372.493615,"close_a":373.476044,"volume_a":111945300,"open_bench":381.278249,"close_bench":373.476044},{"date":"2023-03-10","open_a":372.932284,"high_a":375.002076,"low_a":366.570351,"close_a":368.086914,"volume_a":189253000,"open_bench":372.932284,"close_bench":368.086914},{"date":"2023-03-13","open_a":364.176223,"high_a":372.359975,"low_a":363.069794,"close_a":367.562256,"volume_a":157790000,"open_bench":364.176223,"close_bench":367.562256},{"date":"2023-03-14","open_a":372.464889,"high_a":375.278656,"low_a":369.174214,"close_a":373.638092,"volume_a":149752400,"open_bench":372.464889,"close_bench":373.638092},{"date":"2023-03-15","open_a":368.06785,"high_a":371.501563,"low_a":365.988511,"close_a":371.30127,"volume_a":172996900,"open_bench":368.06785,"close_bench":371.30127},{"date":"2023-03-16","open_a":368.954902,"high_a":378.159215,"low_a":368.449381,"close_a":377.815826,"volume_a":143254200,"open_bench":368.954902,"close_bench":377.815826},{"date":"2023-03-17","open_a":376.490694,"high_a":377.620485,"low_a":372.019363,"close_a":373.398102,"volume_a":140553400,"open_bench":376.490694,"close_bench":373.398102},{"date":"2023-03-20","open_a":374.173664,"high_a":377.400315,"low_a":373.47474,"close_a":376.988586,"volume_a":93055800,"open_bench":374.173664,"close_bench":376.988586},{"date":"2023-03-21","open_a":380.339665,"high_a":382.417357,"low_a":378.750286,"close_a":381.938629,"volume_a":91524200,"open_bench":380.339665,"close_bench":381.938629},{"date":"2023-03-22","open_a":381.766329,"high_a":385.366343,"low_a":375.389671,"close_a":375.427948,"volume_a":111746600,"open_bench":381.766329,"close_bench":375.427948},{"date":"2023-03-23","open_a":378.28117,"high_a":382.302495,"low_a":373.742839,"close_a":376.442871,"volume_a":119351300,"open_bench":378.28117,"close_bench":376.442871},{"date":"2023-03-24","open_a":375.169401,"high_a":378.999223,"low_a":372.833206,"close_a":378.913055,"volume_a":107682400,"open_bench":375.169401,"close_bench":378.913055},{"date":"2023-03-27","open_a":381.18227,"high_a":381.948252,"low_a":378.731186,"close_a":379.621613,"volume_a":74010400,"open_bench":381.18227,"close_bench":379.621613},{"date":"2023-03-28","open_a":378.932191,"high_a":379.62156,"low_a":376.940696,"close_a":378.76944,"volume_a":62871700,"open_bench":378.932191,"close_bench":378.76944},{"date":"2023-03-29","open_a":382.915241,"high_a":384.514205,"low_a":381.718422,"close_a":384.274841,"volume_a":77497900,"open_bench":382.915241,"close_bench":384.274841},{"date":"2023-03-30","open_a":386.898234,"high_a":387.147182,"low_a":384.667375,"close_a":386.524841,"volume_a":69840000,"open_bench":386.898234,"close_bench":386.524841},{"date":"2023-03-31","open_a":387.444033,"high_a":392.269618,"low_a":387.338697,"close_a":391.972809,"volume_a":112062600,"open_bench":387.444033,"close_bench":391.972809},{"date":"2023-04-03","open_a":391.455707,"high_a":393.868485,"low_a":391.063147,"close_a":393.46637,"volume_a":67391100,"open_bench":391.455707,"close_bench":393.46637},{"date":"2023-04-04","open_a":394.107863,"high_a":394.395117,"low_a":389.914203,"close_a":391.283386,"volume_a":66601500,"open_bench":394.107863,"close_bench":391.283386},{"date":"2023-04-05","open_a":390.555781,"high_a":391.312179,"low_a":388.612147,"close_a":390.258972,"volume_a":65200200,"open_bench":390.555781,"close_bench":390.258972},{"date":"2023-04-06","open_a":389.464225,"high_a":392.058951,"low_a":388.420602,"close_a":391.781281,"volume_a":63743300,"open_bench":389.464225,"close_bench":391.781281},{"date":"2023-04-10","open_a":389.311013,"high_a":392.259993,"low_a":388.698257,"close_a":392.18338,"volume_a":63681000,"open_bench":389.311013,"close_bench":392.18338},{"date":"2023-04-11","open_a":392.805761,"high_a":393.686604,"low_a":391.522774,"close_a":392.288727,"volume_a":59297900,"open_bench":392.805761,"close_bench":392.288727},{"date":"2023-04-12","open_a":394.347246,"high_a":394.6345,"low_a":390.105724,"close_a":390.689758,"volume_a":86420400,"open_bench":394.347246,"close_bench":390.689758},{"date":"2023-04-13","open_a":391.771658,"high_a":396.233404,"low_a":390.632283,"close_a":395.87915,"volume_a":85814800,"open_bench":391.771658,"close_bench":395.87915},{"date":"2023-04-14","open_a":395.247225,"high_a":397.430222,"low_a":392.614222,"close_a":394.912109,"volume_a":78161500,"open_bench":395.247225,"close_bench":394.912109},{"date":"2023-04-17","open_a":394.825981,"high_a":396.348332,"low_a":393.600439,"close_a":396.329193,"volume_a":66436400,"open_bench":394.825981,"close_bench":396.329193},{"date":"2023-04-18","open_a":397.899448,"high_a":398.033506,"low_a":395.218583,"close_a":396.587738,"volume_a":63560000,"open_bench":397.899448,"close_bench":396.587738},{"date":"2023-04-19","open_a":394.682394,"high_a":397.420703,"low_a":394.624949,"close_a":396.520721,"volume_a":55227300,"open_bench":394.682394,"close_bench":396.520721},{"date":"2023-04-20","open_a":393.715273,"high_a":396.099357,"low_a":392.815262,"close_a":394.356781,"volume_a":75840400,"open_bench":393.715273,"close_bench":394.356781},{"date":"2023-04-21","open_a":394.653655,"high_a":395.122799,"low_a":392.719605,"close_a":394.663239,"volume_a":73457400,"open_bench":394.653655,"close_bench":394.663239},{"date":"2023-04-24","open_a":394.462135,"high_a":395.496203,"low_a":393.131287,"close_a":395.074921,"volume_a":64332100,"open_bench":394.462135,"close_bench":395.074921},{"date":"2023-04-25","open_a":393.112139,"high_a":393.66748,"low_a":388.746144,"close_a":388.803589,"volume_a":97766700,"open_bench":393.112139,"close_bench":388.803589},{"date":"2023-04-26","open_a":389.416318,"high_a":390.488664,"low_a":386.601396,"close_a":387.156708,"volume_a":80447000,"open_bench":389.416318,"close_bench":387.156708},{"date":"2023-04-27","open_a":389.68445,"high_a":395.132375,"low_a":389.435502,"close_a":394.864288,"volume_a":92968400,"open_bench":389.68445,"close_bench":394.864288},{"date":"2023-04-28","open_a":393.983362,"high_a":398.24405,"low_a":393.925917,"close_a":398.234467,"volume_a":89433100,"open_bench":393.983362,"close_bench":398.234467},{"date":"2023-05-01","open_a":397.794091,"high_a":399.852615,"low_a":397.602588,"close_a":397.832397,"volume_a":62122300,"open_bench":397.794091,"close_bench":397.832397},{"date":"2023-05-02","open_a":397.123878,"high_a":397.171768,"low_a":390.469578,"close_a":393.361084,"volume_a":103998500,"open_bench":397.123878,"close_bench":393.361084},{"date":"2023-05-03","open_a":393.85894,"high_a":396.262163,"low_a":390.421677,"close_a":390.661041,"volume_a":91531800,"open_bench":393.85894,"close_bench":390.661041},{"date":"2023-05-04","open_a":389.617452,"high_a":389.942983,"low_a":386.563165,"close_a":387.894043,"volume_a":94901900,"open_bench":389.617452,"close_bench":387.894043},{"date":"2023-05-05","open_a":391.513185,"high_a":396.118544,"low_a":391.254682,"close_a":395.074921,"volume_a":87891800,"open_bench":391.513185,"close_bench":395.074921},{"date":"2023-05-08","open_a":395.400462,"high_a":395.658965,"low_a":393.78236,"close_a":395.180237,"volume_a":50046800,"open_bench":395.400462,"close_bench":395.180237},{"date":"2023-05-09","open_a":393.638738,"high_a":394.557887,"low_a":393.217455,"close_a":393.447235,"volume_a":49220100,"open_bench":393.638738,"close_bench":393.447235},{"date":"2023-05-10","open_a":396.271762,"high_a":396.903686,"low_a":391.474899,"close_a":395.285583,"volume_a":96142900,"open_bench":396.271762,"close_bench":395.285583},{"date":"2023-05-11","open_a":394.423887,"high_a":394.883447,"low_a":392.528114,"close_a":394.596222,"volume_a":70157100,"open_bench":394.423887,"close_bench":394.596222},{"date":"2023-05-12","open_a":395.831415,"high_a":396.042056,"low_a":391.666476,"close_a":394.079254,"volume_a":70481500,"open_bench":395.831415,"close_bench":394.079254},{"date":"2023-05-15","open_a":394.682353,"high_a":395.840866,"low_a":392.777026,"close_a":395.438751,"volume_a":54289400,"open_bench":394.682353,"close_bench":395.438751},{"date":"2023-05-16","open_a":394.337663,"high_a":395.256841,"low_a":392.786589,"close_a":392.796173,"volume_a":57705500,"open_bench":394.337663,"close_bench":392.796173},{"date":"2023-05-17","open_a":394.806854,"high_a":398.167504,"low_a":393.169613,"close_a":397.564331,"volume_a":87287000,"open_bench":394.806854,"close_bench":397.564331},{"date":"2023-05-18","open_a":397.248305,"high_a":401.815387,"low_a":397.028109,"close_a":401.394104,"volume_a":97177200,"open_bench":397.248305,"close_bench":401.394104},{"date":"2023-05-19","open_a":402.294163,"high_a":402.820752,"low_a":399.594131,"close_a":400.810089,"volume_a":103793300,"open_bench":402.294163,"close_bench":400.810089},{"date":"2023-05-22","open_a":400.829257,"high_a":402.504805,"low_a":399.594131,"close_a":400.97287,"volume_a":60745400,"open_bench":400.829257,"close_bench":400.97287},{"date":"2023-05-23","open_a":399.335592,"high_a":400.905833,"low_a":396.080249,"close_a":396.472809,"volume_a":86383500,"open_bench":399.335592,"close_bench":396.472809},{"date":"2023-05-24","open_a":394.873866,"high_a":395.256842,"low_a":392.44192,"close_a":393.600433,"volume_a":89213700,"open_bench":394.873866,"close_bench":393.600433},{"date":"2023-05-25","open_a":397.09514,"high_a":398.45474,"low_a":394.864281,"close_a":397.008972,"volume_a":90961600,"open_bench":397.09514,"close_bench":397.008972},{"date":"2023-05-26","open_a":397.660043,"high_a":402.868604,"low_a":397.58346,"close_a":402.150513,"volume_a":93830000,"open_bench":397.660043,"close_bench":402.150513},{"date":"2023-05-30","open_a":404.074979,"high_a":404.601568,"low_a":400.924942,"close_a":402.30368,"volume_a":72216000,"open_bench":404.074979,"close_bench":402.30368},{"date":"2023-05-31","open_a":400.484575,"high_a":401.384586,"low_a":398.512218,"close_a":400.072876,"volume_a":110811800,"open_bench":400.484575,"close_bench":400.072876},{"date":"2023-06-01","open_a":400.302551,"high_a":404.927078,"low_a":399.057871,"close_a":403.873871,"volume_a":88865000,"open_bench":400.302551,"close_bench":403.873871},{"date":"2023-06-02","open_a":406.439996,"high_a":410.4996,"low_a":405.913407,"close_a":409.714508,"volume_a":91426200,"open_bench":406.439996,"close_bench":409.714508},{"date":"2023-06-05","open_a":410.059081,"high_a":411.342068,"low_a":408.230338,"close_a":408.929291,"volume_a":65460200,"open_bench":410.059081,"close_bench":408.929291},{"date":"2023-06-06","open_a":408.517607,"high_a":410.346322,"low_a":407.866515,"close_a":409.819733,"volume_a":64022200,"open_bench":408.517607,"close_bench":409.819733},{"date":"2023-06-07","open_a":410.212315,"high_a":411.342106,"low_a":407.981427,"close_a":408.40271,"volume_a":85373300,"open_bench":410.212315,"close_bench":408.40271},{"date":"2023-06-08","open_a":408.469763,"high_a":411.322991,"low_a":407.70381,"close_a":410.872986,"volume_a":61952800,"open_bench":408.469763,"close_bench":410.872986},{"date":"2023-06-09","open_a":411.667613,"high_a":413.611247,"low_a":410.62399,"close_a":411.610168,"volume_a":85742800,"open_bench":411.667613,"close_bench":411.610168},{"date":"2023-06-12","open_a":412.586791,"high_a":415.420851,"low_a":411.868699,"close_a":415.344238,"volume_a":76104300,"open_bench":412.586791,"close_bench":415.344238},{"date":"2023-06-13","open_a":416.799624,"high_a":418.72409,"low_a":416.138977,"close_a":418.082611,"volume_a":95899700,"open_bench":416.799624,"close_bench":418.082611},{"date":"2023-06-14","open_a":418.417693,"high_a":420.380466,"low_a":415.143182,"close_a":418.580444,"volume_a":100612100,"open_bench":418.417693,"close_bench":418.580444},{"date":"2023-06-15","open_a":417.766662,"high_a":425.014609,"low_a":417.670939,"close_a":423.769928,"volume_a":110303100,"open_bench":417.766662,"close_bench":423.769928},{"date":"2023-06-16","open_a":425.747679,"high_a":426.314672,"low_a":421.85559,"close_a":422.326477,"volume_a":114165800,"open_bench":425.747679,"close_bench":422.326477},{"date":"2023-06-20","open_a":420.394838,"high_a":421.278952,"low_a":418.069175,"close_a":420.135345,"volume_a":76160400,"open_bench":420.394838,"close_bench":420.135345},{"date":"2023-06-21","open_a":419.155041,"high_a":419.952668,"low_a":417.396372,"close_a":417.982605,"volume_a":76982300,"open_bench":419.155041,"close_bench":417.982605},{"date":"2023-06-22","open_a":417.031266,"high_a":419.597152,"low_a":416.694906,"close_a":419.491455,"volume_a":70637200,"open_bench":417.031266,"close_bench":419.491455},{"date":"2023-06-23","open_a":416.051017,"high_a":418.097977,"low_a":415.608959,"close_a":416.320099,"volume_a":92074500,"open_bench":416.051017,"close_bench":416.320099},{"date":"2023-06-26","open_a":415.753097,"high_a":417.665502,"low_a":414.378857,"close_a":414.61911,"volume_a":72823600,"open_bench":415.753097,"close_bench":414.61911},{"date":"2023-06-27","open_a":415.493599,"high_a":419.779705,"low_a":415.041923,"close_a":419.164673,"volume_a":72813700,"open_bench":415.493599,"close_bench":419.164673},{"date":"2023-06-28","open_a":418.088317,"high_a":420.38515,"low_a":417.473284,"close_a":419.376099,"volume_a":75636000,"open_bench":418.088317,"close_bench":419.376099},{"date":"2023-06-29","open_a":418.962913,"high_a":421.192468,"low_a":418.559304,"close_a":421.029083,"volume_a":67882300,"open_bench":418.962913,"close_bench":421.029083},{"date":"2023-06-30","open_a":424.229239,"high_a":426.977719,"low_a":423.912088,"close_a":425.997498,"volume_a":104964000,"open_bench":424.229239,"close_bench":425.997498},{"date":"2023-07-03","open_a":425.651595,"high_a":426.766343,"low_a":425.372893,"close_a":426.487671,"volume_a":32793400,"open_bench":425.651595,"close_bench":426.487671},{"date":"2023-07-05","open_a":424.680896,"high_a":426.583711,"low_a":424.671277,"close_a":425.853333,"volume_a":58418400,"open_bench":424.680896,"close_bench":425.853333},{"date":"2023-07-06","open_a":422.287982,"high_a":422.941463,"low_a":420.019978,"close_a":422.518616,"volume_a":80658300,"open_bench":422.287982,"close_bench":422.518616},{"date":"2023-07-07","open_a":421.528802,"high_a":425.38247,"low_a":421.211651,"close_a":421.451904,"volume_a":86134200,"open_bench":421.528802,"close_bench":421.451904},{"date":"2023-07-10","open_a":421.096307,"high_a":422.691591,"low_a":420.529313,"close_a":422.518616,"volume_a":62443500,"open_bench":421.096307,"close_bench":422.518616},{"date":"2023-07-11","open_a":423.277797,"high_a":425.699537,"low_a":422.307166,"close_a":425.209412,"volume_a":64463800,"open_bench":423.277797,"close_bench":425.209412},{"date":"2023-07-12","open_a":428.986275,"high_a":430.033775,"low_a":427.563967,"close_a":428.630676,"volume_a":91924500,"open_bench":428.986275,"close_bench":428.630676},{"date":"2023-07-13","open_a":430.4374,"high_a":432.820721,"low_a":430.004962,"close_a":432.032684,"volume_a":72425200,"open_bench":430.4374,"close_bench":432.032684},{"date":"2023-07-14","open_a":432.916776,"high_a":433.762442,"low_a":431.004342,"close_a":431.76355,"volume_a":69815800,"open_bench":432.916776,"close_bench":431.76355},{"date":"2023-07-17","open_a":431.619434,"high_a":434.310256,"low_a":431.571365,"close_a":433.262756,"volume_a":52680200,"open_bench":431.619434,"close_bench":433.262756},{"date":"2023-07-18","open_a":432.93607,"high_a":437.12607,"low_a":432.503603,"close_a":436.482208,"volume_a":80668200,"open_bench":432.93607,"close_bench":436.482208},{"date":"2023-07-19","open_a":437.270256,"high_a":438.634877,"low_a":436.405321,"close_a":437.45285,"volume_a":65891700,"open_bench":437.270256,"close_bench":437.45285},{"date":"2023-07-20","open_a":436.46291,"high_a":437.356645,"low_a":433.839336,"close_a":434.550476,"volume_a":70591600,"open_bench":436.46291,"close_bench":434.550476},{"date":"2023-07-21","open_a":436.261077,"high_a":436.46291,"low_a":434.540886,"close_a":434.550476,"volume_a":71275600,"open_bench":436.261077,"close_bench":434.550476},{"date":"2023-07-24","open_a":435.694134,"high_a":437.299038,"low_a":434.665844,"close_a":436.491791,"volume_a":54023400,"open_bench":435.694134,"close_bench":436.491791},{"date":"2023-07-25","open_a":436.222683,"high_a":438.932715,"low_a":436.174614,"close_a":437.683411,"volume_a":55191200,"open_bench":436.222683,"close_bench":437.683411},{"date":"2023-07-26","open_a":436.75121,"high_a":439.172951,"low_a":435.703711,"close_a":437.750671,"volume_a":71052900,"open_bench":436.75121,"close_bench":437.750671},{"date":"2023-07-27","open_a":441.123766,"high_a":441.527404,"low_a":433.945004,"close_a":434.848358,"volume_a":92194400,"open_bench":441.123766,"close_bench":434.848358},{"date":"2023-07-28","open_a":438.106282,"high_a":439.9322,"low_a":434.848437,"close_a":439.105743,"volume_a":80011800,"open_bench":438.106282,"close_bench":439.105743},{"date":"2023-07-31","open_a":439.576644,"high_a":440.297403,"low_a":438.269652,"close_a":439.941833,"volume_a":62040400,"open_bench":439.576644,"close_bench":439.941833},{"date":"2023-08-01","open_a":438.481119,"high_a":439.422922,"low_a":437.731531,"close_a":438.682953,"volume_a":55291500,"open_bench":438.481119,"close_bench":438.682953},{"date":"2023-08-02","open_a":435.578767,"high_a":435.83823,"low_a":431.830825,"close_a":432.580414,"volume_a":93933400,"open_bench":435.578767,"close_bench":432.580414},{"date":"2023-08-03","open_a":430.5719,"high_a":433.214684,"low_a":429.928009,"close_a":431.340698,"volume_a":64276100,"open_bench":430.5719,"close_bench":431.340698},{"date":"2023-08-04","open_a":433.147423,"high_a":435.242423,"low_a":428.870907,"close_a":429.389862,"volume_a":100128900,"open_bench":433.147423,"close_bench":429.389862},{"date":"2023-08-07","open_a":431.215823,"high_a":433.291613,"low_a":430.523893,"close_a":433.137848,"volume_a":58357500,"open_bench":431.215823,"close_bench":433.137848},{"date":"2023-08-08","open_a":430.61032,"high_a":433.128197,"low_a":427.909879,"close_a":431.254211,"volume_a":71361300,"open_bench":430.61032,"close_bench":431.254211},{"date":"2023-08-09","open_a":431.523273,"high_a":431.686658,"low_a":427.611947,"close_a":428.371155,"volume_a":78789600,"open_bench":431.523273,"close_bench":428.371155},{"date":"2023-08-10","open_a":430.716039,"high_a":434.089201,"low_a":427.362116,"close_a":428.524933,"volume_a":93005500,"open_bench":430.716039,"close_bench":428.524933},{"date":"2023-08-11","open_a":426.660562,"high_a":429.284135,"low_a":426.064739,"close_a":428.275055,"volume_a":68690900,"open_bench":426.660562,"close_bench":428.275055},{"date":"2023-08-14","open_a":427.362195,"high_a":430.639221,"low_a":427.054664,"close_a":430.639221,"volume_a":47867400,"open_bench":427.362195,"close_bench":430.639221},{"date":"2023-08-15","open_a":428.870908,"high_a":429.226507,"low_a":425.055688,"close_a":425.622711,"volume_a":75707500,"open_bench":428.870908,"close_bench":425.622711},{"date":"2023-08-16","open_a":425.209452,"high_a":426.862394,"low_a":422.393694,"close_a":422.49942,"volume_a":80107200,"open_bench":425.209452,"close_bench":422.49942},{"date":"2023-08-17","open_a":423.960185,"high_a":424.219648,"low_a":418.761105,"close_a":419.28006,"volume_a":95711300,"open_bench":423.960185,"close_bench":419.28006},{"date":"2023-08-18","open_a":416.473839,"high_a":420.510102,"low_a":416.127888,"close_a":419.481812,"volume_a":98852000,"open_bench":416.473839,"close_bench":419.481812},{"date":"2023-08-21","open_a":420.4909,"high_a":422.951089,"low_a":418.347862,"close_a":422.211121,"volume_a":68719000,"open_bench":420.4909,"close_bench":422.211121},{"date":"2023-08-22","open_a":423.97934,"high_a":423.97934,"low_a":420.5101,"close_a":421.067474,"volume_a":64994200,"open_bench":423.97934,"close_bench":421.067474},{"date":"2023-08-23","open_a":422.124638,"high_a":426.372325,"low_a":421.980492,"close_a":425.757263,"volume_a":68441000,"open_bench":422.124638,"close_bench":425.757263},{"date":"2023-08-24","open_a":427.352512,"high_a":427.861847,"low_a":419.82777,"close_a":419.856628,"volume_a":88517300,"open_bench":427.352512,"close_bench":419.856628},{"date":"2023-08-25","open_a":421.576784,"high_a":424.094631,"low_a":418.040266,"close_a":422.816498,"volume_a":102325100,"open_bench":421.576784,"close_bench":422.816498},{"date":"2023-08-28","open_a":424.998058,"high_a":426.112835,"low_a":422.81657,"close_a":425.497803,"volume_a":61595400,"open_bench":424.998058,"close_bench":425.497803},{"date":"2023-08-29","open_a":425.392055,"high_a":431.926956,"low_a":425.209461,"close_a":431.648254,"volume_a":83081900,"open_bench":425.392055,"close_bench":431.648254},{"date":"2023-08-30","open_a":431.984599,"high_a":434.060389,"low_a":431.283049,"close_a":433.426117,"volume_a":69053900,"open_bench":431.984599,"close_bench":433.426117},{"date":"2023-08-31","open_a":434.041144,"high_a":435.175131,"low_a":432.609245,"close_a":432.79184,"volume_a":66084600,"open_bench":434.041144,"close_bench":432.79184},{"date":"2023-09-01","open_a":435.501875,"high_a":435.982381,"low_a":432.147922,"close_a":433.59906,"volume_a":58944100,"open_bench":435.501875,"close_bench":433.59906},{"date":"2023-09-05","open_a":433.157056,"high_a":433.474177,"low_a":431.65788,"close_a":431.725128,"volume_a":55166200,"open_bench":433.157056,"close_bench":431.725128},{"date":"2023-09-06","open_a":430.917937,"high_a":431.023664,"low_a":426.506894,"close_a":428.822937,"volume_a":70758500,"open_bench":430.917937,"close_bench":428.822937},{"date":"2023-09-07","open_a":425.834105,"high_a":428.178978,"low_a":425.488155,"close_a":427.506287,"volume_a":70355400,"open_bench":425.834105,"close_bench":427.506287},{"date":"2023-09-08","open_a":427.554354,"high_a":429.678183,"low_a":427.198784,"close_a":428.150177,"volume_a":61659700,"open_bench":427.554354,"close_bench":428.150177},{"date":"2023-09-11","open_a":430.764139,"high_a":431.273475,"low_a":429.063158,"close_a":430.965973,"volume_a":60180100,"open_bench":430.764139,"close_bench":430.965973},{"date":"2023-09-12","open_a":429.52443,"high_a":431.042816,"low_a":428.025253,"close_a":428.601837,"volume_a":67565400,"open_bench":429.52443,"close_bench":428.601837},{"date":"2023-09-13","open_a":428.822861,"high_a":430.25476,"low_a":427.727293,"close_a":429.101562,"volume_a":60199300,"open_bench":428.822861,"close_bench":429.101562},{"date":"2023-09-14","open_a":431.561738,"high_a":433.493352,"low_a":430.264365,"close_a":432.801422,"volume_a":83430800,"open_bench":431.561738,"close_bench":432.801422},{"date":"2023-09-15","open_a":431.222728,"high_a":431.550622,"low_a":427.152951,"close_a":427.586914,"volume_a":111848900,"open_bench":431.222728,"close_bench":427.586914},{"date":"2023-09-18","open_a":427.278307,"high_a":429.129972,"low_a":426.80576,"close_a":427.837677,"volume_a":55752200,"open_bench":427.278307,"close_bench":427.837677},{"date":"2023-09-19","open_a":426.921448,"high_a":427.509748,"low_a":424.278996,"close_a":426.950378,"volume_a":66514600,"open_bench":426.921448,"close_bench":426.950378},{"date":"2023-09-20","open_a":428.204133,"high_a":428.618819,"low_a":422.822753,"close_a":423.025299,"volume_a":82562600,"open_bench":428.204133,"close_bench":423.025299},{"date":"2023-09-21","open_a":420.189956,"high_a":420.450334,"low_a":415.879078,"close_a":416.033386,"volume_a":104095800,"open_bench":420.189956,"close_bench":416.033386},{"date":"2023-09-22","open_a":417.055696,"high_a":418.646954,"low_a":414.683246,"close_a":415.097961,"volume_a":100829700,"open_bench":417.055696,"close_bench":415.097961},{"date":"2023-09-25","open_a":413.892408,"high_a":416.88203,"low_a":413.458415,"close_a":416.843475,"volume_a":70874500,"open_bench":413.892408,"close_bench":416.843475},{"date":"2023-09-26","open_a":413.815266,"high_a":414.51929,"low_a":409.890142,"close_a":410.719543,"volume_a":96168400,"open_bench":413.815266,"close_bench":410.719543},{"date":"2023-09-27","open_a":411.88647,"high_a":412.44584,"low_a":407.257352,"close_a":410.883484,"volume_a":104705800,"open_bench":411.88647,"close_bench":410.883484},{"date":"2023-09-28","open_a":410.333833,"high_a":414.934021,"low_a":409.745533,"close_a":413.265594,"volume_a":92258300,"open_bench":410.333833,"close_bench":413.265594},{"date":"2023-09-29","open_a":416.30342,"high_a":416.477005,"low_a":410.748455,"close_a":412.262573,"volume_a":115111300,"open_bench":416.30342,"close_bench":412.262573},{"date":"2023-10-02","open_a":411.433193,"high_a":413.34272,"low_a":409.350081,"close_a":412.098633,"volume_a":83798600,"open_bench":411.433193,"close_bench":412.098633},{"date":"2023-10-03","open_a":409.928752,"high_a":412.156518,"low_a":405.222465,"close_a":406.582275,"volume_a":103760600,"open_bench":409.928752,"close_bench":406.582275},{"date":"2023-10-04","open_a":407.045171,"high_a":410.285548,"low_a":405.588915,"close_a":409.542969,"volume_a":87453000,"open_bench":407.045171,"close_bench":409.542969},{"date":"2023-10-05","open_a":409.253611,"high_a":410.227666,"low_a":406.177196,"close_a":409.388641,"volume_a":70142700,"open_bench":409.253611,"close_bench":409.388641},{"date":"2023-10-06","open_a":406.948737,"high_a":415.782663,"low_a":405.627511,"close_a":414.249268,"volume_a":113273300,"open_bench":406.948737,"close_bench":414.249268},{"date":"2023-10-09","open_a":412.359012,"high_a":417.470361,"low_a":411.809326,"close_a":416.901367,"volume_a":80374400,"open_bench":412.359012,"close_bench":416.901367},{"date":"2023-10-10","open_a":417.528209,"high_a":421.655849,"low_a":417.132801,"close_a":419.071259,"volume_a":78607300,"open_bench":417.528209,"close_bench":419.071259},{"date":"2023-10-11","open_a":420.132147,"high_a":421.038658,"low_a":417.759697,"close_a":420.787933,"volume_a":62451700,"open_bench":420.132147,"close_bench":420.787933},{"date":"2023-10-12","open_a":421.395517,"high_a":421.771618,"low_a":415.879136,"close_a":418.222626,"volume_a":81154200,"open_bench":421.395517,"close_bench":418.222626},{"date":"2023-10-13","open_a":419.717419,"high_a":420.913298,"low_a":414.577169,"close_a":416.139496,"volume_a":95143100,"open_bench":419.717419,"close_bench":416.139496},{"date":"2023-10-16","open_a":418.376879,"high_a":421.578701,"low_a":418.135779,"close_a":420.517853,"volume_a":75433200,"open_bench":418.376879,"close_bench":420.517853},{"date":"2023-10-17","open_a":417.402874,"high_a":422.543153,"low_a":417.055703,"close_a":420.498596,"volume_a":75324700,"open_bench":417.402874,"close_bench":420.498596},{"date":"2023-10-18","open_a":418.733747,"high_a":419.688496,"low_a":413.815291,"close_a":414.895416,"volume_a":93559800,"open_bench":418.733747,"close_bench":414.895416},{"date":"2023-10-19","open_a":415.609055,"high_a":417.412482,"low_a":410.574875,"close_a":411.249939,"volume_a":121323000,"open_bench":415.609055,"close_bench":411.249939},{"date":"2023-10-20","open_a":410.815997,"high_a":411.35606,"low_a":406.090403,"close_a":406.196503,"volume_a":123919900,"open_bench":410.815997,"close_bench":406.196503},{"date":"2023-10-23","open_a":404.672745,"high_a":409.340477,"low_a":402.92718,"close_a":405.492493,"volume_a":92035100,"open_bench":404.672745,"close_bench":405.492493},{"date":"2023-10-24","open_a":407.604558,"high_a":409.697324,"low_a":405.762546,"close_a":408.549683,"volume_a":78564200,"open_bench":407.604558,"close_bench":408.549683},{"date":"2023-10-25","open_a":406.871566,"high_a":406.900497,"low_a":402.174903,"close_a":402.686035,"volume_a":94223200,"open_bench":406.871566,"close_bench":402.686035},{"date":"2023-10-26","open_a":401.625205,"high_a":402.473854,"low_a":396.947849,"close_a":397.864014,"volume_a":115156800,"open_bench":401.625205,"close_bench":397.864014},{"date":"2023-10-27","open_a":399.445699,"high_a":399.841108,"low_a":394.642966,"close_a":396.060638,"volume_a":107367700,"open_bench":399.445699,"close_bench":396.060638},{"date":"2023-10-30","open_a":398.838133,"high_a":401.847063,"low_a":397.545838,"close_a":400.795868,"volume_a":86562700,"open_bench":398.838133,"close_bench":400.795868},{"date":"2023-10-31","open_a":401.364847,"high_a":403.631197,"low_a":399.464973,"close_a":403.312958,"volume_a":79665200,"open_bench":401.364847,"close_bench":403.312958},{"date":"2023-11-01","open_a":404.277374,"high_a":408.424291,"low_a":403.746935,"close_a":407.614197,"volume_a":98068100,"open_bench":404.277374,"close_bench":407.614197},{"date":"2023-11-02","open_a":411.394589,"high_a":415.58012,"low_a":411.375312,"close_a":415.425812,"volume_a":94938900,"open_bench":411.394589,"close_bench":415.425812},{"date":"2023-11-03","open_a":417.721161,"high_a":420.759022,"low_a":417.595784,"close_a":419.215973,"volume_a":100167800,"open_bench":417.721161,"close_bench":419.215973},{"date":"2023-11-06","open_a":419.968159,"high_a":420.623945,"low_a":418.241871,"close_a":420.180328,"volume_a":67831700,"open_bench":419.968159,"close_bench":420.180328},{"date":"2023-11-07","open_a":420.180341,"high_a":422.012699,"low_a":419.042354,"close_a":421.37619,"volume_a":64256100,"open_bench":420.180341,"close_bench":421.37619},{"date":"2023-11-08","open_a":421.974093,"high_a":422.494878,"low_a":419.389502,"close_a":421.684784,"volume_a":61746000,"open_bench":421.974093,"close_bench":421.684784},{"date":"2023-11-09","open_a":422.82275,"high_a":422.861335,"low_a":417.971809,"close_a":418.396149,"volume_a":83174400,"open_bench":422.82275,"close_bench":418.396149},{"date":"2023-11-10","open_a":420.459954,"high_a":425.233726,"low_a":418.386466,"close_a":424.92511,"volume_a":89462200,"open_bench":420.459954,"close_bench":424.92511},{"date":"2023-11-13","open_a":423.594263,"high_a":425.619483,"low_a":422.8131,"close_a":424.520081,"volume_a":52236100,"open_bench":423.594263,"close_bench":424.520081},{"date":"2023-11-14","open_a":430.431922,"high_a":434.038776,"low_a":430.210099,"close_a":432.756134,"volume_a":97176900,"open_bench":430.431922,"close_bench":432.756134},{"date":"2023-11-15","open_a":434.086988,"high_a":435.311797,"low_a":432.823624,"close_a":433.672302,"volume_a":77327600,"open_bench":434.086988,"close_bench":433.672302},{"date":"2023-11-16","open_a":433.228673,"high_a":434.520968,"low_a":432.167825,"close_a":434.202728,"volume_a":66665800,"open_bench":433.228673,"close_bench":434.202728},{"date":"2023-11-17","open_a":434.212359,"high_a":435.350376,"low_a":433.296195,"close_a":434.742798,"volume_a":83133200,"open_bench":434.212359,"close_bench":434.742798},{"date":"2023-11-20","open_a":434.492034,"high_a":438.928289,"low_a":434.48238,"close_a":438.089264,"volume_a":69936200,"open_bench":434.492034,"close_bench":438.089264},{"date":"2023-11-21","open_a":437.047668,"high_a":437.963861,"low_a":435.871096,"close_a":437.13446,"volume_a":49244600,"open_bench":437.047668,"close_bench":437.13446},{"date":"2023-11-22","open_a":438.78365,"high_a":440.133807,"low_a":437.732455,"close_a":438.822205,"volume_a":59394900,"open_bench":438.78365,"close_bench":438.822205},{"date":"2023-11-24","open_a":438.8704,"high_a":439.285086,"low_a":438.542507,"close_a":439.092194,"volume_a":29737400,"open_bench":438.8704,"close_bench":439.092194},{"date":"2023-11-27","open_a":438.465385,"high_a":439.275479,"low_a":437.915668,"close_a":438.301453,"volume_a":50506000,"open_bench":438.465385,"close_bench":438.301453},{"date":"2023-11-28","open_a":437.915665,"high_a":440.027708,"low_a":437.356325,"close_a":438.735413,"volume_a":62115000,"open_bench":437.915665,"close_bench":438.735413},{"date":"2023-11-29","open_a":440.876347,"high_a":442.00471,"low_a":438.031379,"close_a":438.426758,"volume_a":63146000,"open_bench":440.876347,"close_bench":438.426758},{"date":"2023-11-30","open_a":439.265904,"high_a":440.500337,"low_a":437.202069,"close_a":440.153137,"volume_a":79752700,"open_bench":439.265904,"close_bench":440.153137},{"date":"2023-12-01","open_a":439.545483,"high_a":443.287367,"low_a":438.957212,"close_a":442.756958,"volume_a":89183400,"open_bench":439.545483,"close_bench":442.756958},{"date":"2023-12-04","open_a":439.381606,"high_a":442.77629,"low_a":438.166449,"close_a":440.4328,"volume_a":72430900,"open_bench":439.381606,"close_bench":440.4328},{"date":"2023-12-05","open_a":439.053683,"high_a":441.300726,"low_a":438.677552,"close_a":440.345978,"volume_a":69793500,"open_bench":439.053683,"close_bench":440.345978},{"date":"2023-12-06","open_a":442.47735,"high_a":442.506281,"low_a":438.13754,"close_a":438.571533,"volume_a":69124700,"open_bench":442.47735,"close_bench":438.571533},{"date":"2023-12-07","open_a":440.644921,"high_a":442.564071,"low_a":440.046996,"close_a":441.917938,"volume_a":66995400,"open_bench":440.644921,"close_bench":441.917938},{"date":"2023-12-08","open_a":441.175389,"high_a":444.34828,"low_a":440.934289,"close_a":443.817871,"volume_a":83194400,"open_bench":441.175389,"close_bench":443.817871},{"date":"2023-12-11","open_a":443.325955,"high_a":445.717682,"low_a":443.113785,"close_a":445.544067,"volume_a":65002200,"open_bench":443.325955,"close_bench":445.544067},{"date":"2023-12-12","open_a":445.196905,"high_a":447.675426,"low_a":444.203572,"close_a":447.578979,"volume_a":68327600,"open_bench":445.196905,"close_bench":447.578979},{"date":"2023-12-13","open_a":447.955064,"high_a":454.001883,"low_a":447.59824,"close_a":453.751129,"volume_a":93278000,"open_bench":447.955064,"close_bench":453.751129},{"date":"2023-12-14","open_a":455.679914,"high_a":456.866139,"low_a":452.545608,"close_a":455.207367,"volume_a":119026000,"open_bench":455.679914,"close_bench":455.207367},{"date":"2023-12-15","open_a":454.61285,"high_a":455.784529,"low_a":452.618129,"close_a":454.457916,"volume_a":141553700,"open_bench":454.61285,"close_bench":454.457916},{"date":"2023-12-18","open_a":456.055693,"high_a":457.992318,"low_a":455.000236,"close_a":457.014313,"volume_a":70375300,"open_bench":456.055693,"close_bench":457.014313},{"date":"2023-12-19","open_a":457.556506,"high_a":459.870786,"low_a":457.479054,"close_a":459.793304,"volume_a":55761800,"open_bench":457.556506,"close_bench":459.793304},{"date":"2023-12-20","open_a":458.941206,"high_a":460.819734,"low_a":452.995785,"close_a":453.421844,"volume_a":102921000,"open_bench":458.941206,"close_bench":453.421844},{"date":"2023-12-21","open_a":456.394548,"high_a":457.992287,"low_a":453.983461,"close_a":457.721161,"volume_a":86667500,"open_bench":456.394548,"close_bench":457.721161},{"date":"2023-12-22","open_a":458.84439,"high_a":460.316243,"low_a":456.752861,"close_a":458.641052,"volume_a":67160400,"open_bench":458.84439,"close_bench":458.641052},{"date":"2023-12-26","open_a":459.047717,"high_a":461.47816,"low_a":458.970235,"close_a":460.577637,"volume_a":55387000,"open_bench":459.047717,"close_bench":460.577637},{"date":"2023-12-27","open_a":460.374269,"high_a":461.555611,"low_a":459.841709,"close_a":461.41037,"volume_a":68000300,"open_bench":460.374269,"close_bench":461.41037},{"date":"2023-12-28","open_a":461.768698,"high_a":462.417451,"low_a":461.16835,"close_a":461.584717,"volume_a":77158100,"open_bench":461.768698,"close_bench":461.584717},{"date":"2023-12-29","open_a":461.390984,"high_a":461.91388,"low_a":458.302066,"close_a":460.248383,"volume_a":122283100,"open_bench":461.390984,"close_bench":460.248383},{"date":"2024-01-02","open_a":457.198205,"high_a":458.660366,"low_a":455.581111,"close_a":457.672668,"volume_a":123623700,"open_bench":457.198205,"close_bench":457.672668},{"date":"2024-01-03","open_a":455.523045,"high_a":456.258971,"low_a":453.334679,"close_a":453.935028,"volume_a":103585900,"open_bench":455.523045,"close_bench":453.935028},{"date":"2024-01-04","open_a":453.460568,"high_a":456.036282,"low_a":452.250178,"close_a":452.4729,"volume_a":84232200,"open_bench":453.460568,"close_bench":452.4729},{"date":"2024-01-05","open_a":452.676255,"high_a":455.532787,"low_a":451.649846,"close_a":453.092651,"volume_a":86118900,"open_bench":452.676255,"close_bench":453.092651},{"date":"2024-01-08","open_a":453.586385,"high_a":459.706123,"low_a":453.460499,"close_a":459.560883,"volume_a":74879100,"open_bench":453.586385,"close_bench":459.560883},{"date":"2024-01-09","open_a":456.917422,"high_a":459.880455,"low_a":456.413911,"close_a":458.863739,"volume_a":65931400,"open_bench":456.917422,"close_bench":458.863739},{"date":"2024-01-10","open_a":459.134889,"high_a":462.320644,"low_a":458.85407,"close_a":461.458832,"volume_a":67310600,"open_bench":459.134889,"close_bench":461.458832},{"date":"2024-01-11","open_a":462.456191,"high_a":462.969395,"low_a":457.2951,"close_a":461.255493,"volume_a":77940700,"open_bench":462.456191,"close_bench":461.255493},{"date":"2024-01-12","open_a":462.698288,"high_a":463.434215,"low_a":460.171008,"close_a":461.575043,"volume_a":58026400,"open_bench":462.698288,"close_bench":461.575043},{"date":"2024-01-16","open_a":460.199992,"high_a":461.507189,"low_a":458.069693,"close_a":459.880432,"volume_a":85014900,"open_bench":460.199992,"close_bench":459.880432},{"date":"2024-01-17","open_a":456.86905,"high_a":457.808314,"low_a":454.980829,"close_a":457.324158,"volume_a":68843900,"open_bench":456.86905,"close_bench":457.324158},{"date":"2024-01-18","open_a":458.989627,"high_a":461.942967,"low_a":457.450014,"close_a":461.391022,"volume_a":91856200,"open_bench":458.989627,"close_bench":461.391022},{"date":"2024-01-19","open_a":462.514291,"high_a":467.423641,"low_a":461.439479,"close_a":467.142822,"volume_a":110834500,"open_bench":462.514291,"close_bench":467.142822},{"date":"2024-01-22","open_a":468.672807,"high_a":469.844456,"low_a":467.481772,"close_a":468.130554,"volume_a":75844900,"open_bench":468.672807,"close_bench":468.130554},{"date":"2024-01-23","open_a":468.672716,"high_a":469.737836,"low_a":467.588212,"close_a":469.495758,"volume_a":49945300,"open_bench":468.672716,"close_bench":469.495758},{"date":"2024-01-24","open_a":472.352332,"high_a":473.281903,"low_a":469.515185,"close_a":470.009033,"volume_a":81765000,"open_bench":472.352332,"close_bench":470.009033},{"date":"2024-01-25","open_a":472.129616,"high_a":472.836495,"low_a":470.00904,"close_a":472.565369,"volume_a":72525000,"open_bench":472.129616,"close_bench":472.565369},{"date":"2024-01-26","open_a":472.139346,"high_a":473.620863,"low_a":471.12263,"close_a":471.965057,"volume_a":76641600,"open_bench":472.139346,"close_bench":471.965057},{"date":"2024-01-29","open_a":472.274803,"high_a":475.847877,"low_a":471.732551,"close_a":475.702606,"volume_a":61322800,"open_bench":472.274803,"close_bench":475.702606},{"date":"2024-01-30","open_a":475.015188,"high_a":476.041596,"low_a":474.579435,"close_a":475.334747,"volume_a":58618400,"open_bench":475.015188,"close_bench":475.334747},{"date":"2024-01-31","open_a":473.136685,"high_a":473.5821,"low_a":467.559198,"close_a":467.578583,"volume_a":126011100,"open_bench":473.136685,"close_bench":467.578583},{"date":"2024-02-01","open_a":469.27314,"high_a":473.727382,"low_a":468.469424,"close_a":473.698334,"volume_a":91891600,"open_bench":469.27314,"close_bench":473.698334},{"date":"2024-02-02","open_a":474.134041,"high_a":480.331233,"low_a":473.795126,"close_a":478.68512,"volume_a":99228200,"open_bench":474.134041,"close_bench":478.68512},{"date":"2024-02-05","open_a":478.055721,"high_a":478.714166,"low_a":474.695677,"close_a":476.942139,"volume_a":75757100,"open_bench":478.055721,"close_bench":476.942139},{"date":"2024-02-06","open_a":477.881368,"high_a":478.656035,"low_a":476.457948,"close_a":478.326813,"volume_a":55918600,"open_bench":477.881368,"close_bench":478.326813},{"date":"2024-02-07","open_a":480.563642,"high_a":482.732651,"low_a":479.663089,"close_a":482.316284,"volume_a":70556500,"open_bench":480.563642,"close_bench":482.316284},{"date":"2024-02-08","open_a":482.316267,"high_a":482.906923,"low_a":481.502888,"close_a":482.529297,"volume_a":52343600,"open_bench":482.316267,"close_bench":482.529297},{"date":"2024-02-09","open_a":483.032822,"high_a":485.753777,"low_a":482.693907,"close_a":485.318054,"volume_a":63979400,"open_bench":483.032822,"close_bench":485.318054},{"date":"2024-02-12","open_a":485.288962,"high_a":487.545116,"low_a":484.388409,"close_a":485.10498,"volume_a":56502300,"open_bench":485.288962,"close_bench":485.10498},{"date":"2024-02-13","open_a":478.859397,"high_a":481.338274,"low_a":475.17013,"close_a":478.423645,"volume_a":113099200,"open_bench":478.859397,"close_bench":478.423645},{"date":"2024-02-14","open_a":481.047798,"high_a":483.255549,"low_a":478.733518,"close_a":482.771393,"volume_a":68387800,"open_bench":481.047798,"close_bench":482.771393},{"date":"2024-02-15","open_a":483.468576,"high_a":486.286368,"low_a":482.994083,"close_a":486.102386,"volume_a":61683000,"open_bench":483.468576,"close_bench":486.102386},{"date":"2024-02-16","open_a":485.802216,"high_a":486.935125,"low_a":482.945683,"close_a":483.68161,"volume_a":75532900,"open_bench":485.802216,"close_bench":483.68161},{"date":"2024-02-20","open_a":481.948309,"high_a":482.616447,"low_a":478.781939,"close_a":481.018738,"volume_a":71736700,"open_bench":481.948309,"close_bench":481.018738},{"date":"2024-02-21","open_a":479.72118,"high_a":481.609371,"low_a":477.920104,"close_a":481.454437,"volume_a":59293400,"open_bench":479.72118,"close_bench":481.454437},{"date":"2024-02-22","open_a":488.038966,"high_a":492.376985,"low_a":487.080317,"close_a":491.418365,"volume_a":76402500,"open_bench":488.038966,"close_bench":491.418365},{"date":"2024-02-23","open_a":493.13228,"high_a":493.965044,"low_a":491.03106,"close_a":491.757294,"volume_a":61321800,"open_bench":493.13228,"close_bench":491.757294},{"date":"2024-02-26","open_a":492.193036,"high_a":492.628789,"low_a":489.830352,"close_a":489.956238,"volume_a":50386700,"open_bench":492.193036,"close_bench":489.956238},{"date":"2024-02-27","open_a":490.643762,"high_a":491.089178,"low_a":488.755542,"close_a":490.866455,"volume_a":48854500,"open_bench":490.643762,"close_bench":490.866455},{"date":"2024-02-28","open_a":489.31719,"high_a":490.798707,"low_a":488.958919,"close_a":490.217743,"volume_a":56506600,"open_bench":489.31719,"close_bench":490.217743},{"date":"2024-02-29","open_a":491.970287,"high_a":493.587351,"low_a":489.336477,"close_a":491.97995,"volume_a":83924800,"open_bench":491.970287,"close_bench":491.97995},{"date":"2024-03-01","open_a":492.851483,"high_a":497.024876,"low_a":492.444778,"close_a":496.598816,"volume_a":76844800,"open_bench":492.851483,"close_bench":496.598816},{"date":"2024-03-04","open_a":495.80488,"high_a":497.906101,"low_a":495.775802,"close_a":496.066284,"volume_a":49799300,"open_bench":495.80488,"close_bench":496.066284},{"date":"2024-03-05","open_a":494.071522,"high_a":494.516967,"low_a":488.910432,"close_a":491.10849,"volume_a":72855600,"open_bench":494.071522,"close_bench":491.10849},{"date":"2024-03-06","open_a":494.371684,"high_a":495.843537,"low_a":492.309204,"close_a":493.597046,"volume_a":68382400,"open_bench":494.371684,"close_bench":493.597046},{"date":"2024-03-07","open_a":496.87964,"high_a":499.542498,"low_a":493.655144,"close_a":498.496704,"volume_a":58652100,"open_bench":496.87964,"close_bench":498.496704},{"date":"2024-03-08","open_a":499.126207,"high_a":501.798699,"low_a":494.933399,"close_a":495.5047,"volume_a":86532500,"open_bench":499.126207,"close_bench":495.5047},{"date":"2024-03-11","open_a":494.303945,"high_a":495.659576,"low_a":492.386676,"close_a":495.078583,"volume_a":62557200,"open_bench":494.303945,"close_bench":495.078583},{"date":"2024-03-12","open_a":497.179862,"high_a":500.985322,"low_a":494.671908,"close_a":500.404358,"volume_a":73114400,"open_bench":497.179862,"close_bench":500.404358},{"date":"2024-03-13","open_a":500.723824,"high_a":500.898113,"low_a":498.186851,"close_a":499.619934,"volume_a":55104100,"open_bench":500.723824,"close_bench":499.619934},{"date":"2024-03-14","open_a":500.588244,"high_a":500.743207,"low_a":495.601472,"close_a":498.632294,"volume_a":110171800,"open_bench":500.588244,"close_bench":498.632294},{"date":"2024-03-15","open_a":495.577542,"high_a":497.02483,"low_a":493.547485,"close_a":495.208435,"volume_a":107646300,"open_bench":495.577542,"close_bench":495.208435},{"date":"2024-03-18","open_a":499.25887,"high_a":500.696406,"low_a":497.743612,"close_a":498.15155,"volume_a":88893300,"open_bench":499.25887,"close_bench":498.15155},{"date":"2024-03-19","open_a":497.461901,"high_a":501.201461,"low_a":496.461412,"close_a":500.9198,"volume_a":60755300,"open_bench":497.461901,"close_bench":500.9198},{"date":"2024-03-20","open_a":500.978096,"high_a":505.688978,"low_a":500.307882,"close_a":505.552979,"volume_a":69594600,"open_bench":500.978096,"close_bench":505.552979},{"date":"2024-03-21","open_a":508.379568,"high_a":509.07889,"low_a":506.941973,"close_a":507.223694,"volume_a":60256100,"open_bench":508.379568,"close_bench":507.223694},{"date":"2024-03-22","open_a":507.136269,"high_a":507.621929,"low_a":506.028949,"close_a":506.262115,"volume_a":79070800,"open_bench":507.136269,"close_bench":506.262115},{"date":"2024-03-25","open_a":504.892421,"high_a":506.009463,"low_a":504.707867,"close_a":504.863312,"volume_a":48512100,"open_bench":504.892421,"close_bench":504.863312},{"date":"2024-03-26","open_a":506.281457,"high_a":506.621455,"low_a":503.532662,"close_a":503.930878,"volume_a":65463700,"open_bench":506.281457,"close_bench":503.930878},{"date":"2024-03-27","open_a":506.747803,"high_a":508.204784,"low_a":504.59144,"close_a":508.165894,"volume_a":82999800,"open_bench":506.747803,"close_bench":508.165894},{"date":"2024-03-28","open_a":508.204694,"high_a":509.564508,"low_a":507.787033,"close_a":508.068695,"volume_a":96294900,"open_bench":508.204694,"close_bench":508.068695},{"date":"2024-04-01","open_a":508.806902,"high_a":509.341116,"low_a":506.028879,"close_a":507.184753,"volume_a":62477500,"open_bench":508.806902,"close_bench":507.184753},{"date":"2024-04-02","open_a":503.377194,"high_a":504.095962,"low_a":501.66766,"close_a":503.960022,"volume_a":74230300,"open_bench":503.377194,"close_bench":503.960022},{"date":"2024-04-03","open_a":502.872168,"high_a":506.009574,"low_a":502.823614,"close_a":504.513702,"volume_a":59155800,"open_bench":502.872168,"close_bench":504.513702},{"date":"2024-04-04","open_a":508.505812,"high_a":508.845751,"low_a":498.054392,"close_a":498.355499,"volume_a":96858100,"open_bench":508.505812,"close_bench":498.355499},{"date":"2024-04-05","open_a":499.705653,"high_a":505.514132,"low_a":499.268547,"close_a":503.561768,"volume_a":74546500,"open_bench":499.705653,"close_bench":503.561768},{"date":"2024-04-08","open_a":504.261226,"high_a":505.261656,"low_a":503.037352,"close_a":503.843506,"volume_a":48401800,"open_bench":504.261226,"close_bench":503.843506},{"date":"2024-04-09","open_a":505.572329,"high_a":505.815159,"low_a":499.598684,"close_a":504.426178,"volume_a":68049200,"open_bench":505.572329,"close_bench":504.426178},{"date":"2024-04-10","open_a":498.753737,"high_a":501.35687,"low_a":497.403647,"close_a":499.375397,"volume_a":82652800,"open_bench":498.753737,"close_bench":499.375397},{"date":"2024-04-11","open_a":500.890694,"high_a":504.581701,"low_a":497.393963,"close_a":503.144165,"volume_a":70099000,"open_bench":500.890694,"close_bench":503.144165},{"date":"2024-04-12","open_a":499.618196,"high_a":501.026623,"low_a":494.479901,"close_a":496.199158,"volume_a":92561100,"open_bench":499.618196,"close_bench":496.199158},{"date":"2024-04-15","open_a":500.356425,"high_a":500.521533,"low_a":489.137653,"close_a":489.982727,"volume_a":92101400,"open_bench":500.356425,"close_bench":489.982727},{"date":"2024-04-16","open_a":490.458646,"high_a":491.973904,"low_a":487.80693,"close_a":489.089081,"volume_a":73484000,"open_bench":490.458646,"close_bench":489.089081},{"date":"2024-04-17","open_a":491.536875,"high_a":491.702013,"low_a":484.805629,"close_a":486.194611,"volume_a":75910300,"open_bench":491.536875,"close_bench":486.194611},{"date":"2024-04-18","open_a":487.583562,"high_a":489.671896,"low_a":484.261632,"close_a":485.194092,"volume_a":74548100,"open_bench":487.583562,"close_bench":485.194092},{"date":"2024-04-19","open_a":485.11645,"high_a":486.107186,"low_a":479.696463,"close_a":480.959198,"volume_a":102212600,"open_bench":485.11645,"close_bench":480.959198},{"date":"2024-04-22","open_a":483.552556,"high_a":487.972083,"low_a":481.221393,"close_a":485.388367,"volume_a":67961000,"open_bench":483.552556,"close_bench":485.388367},{"date":"2024-04-23","open_a":487.389309,"high_a":491.575699,"low_a":485.203837,"close_a":491.148315,"volume_a":64633600,"open_bench":487.389309,"close_bench":491.148315},{"date":"2024-04-24","open_a":492.032235,"high_a":492.819003,"low_a":488.700612,"close_a":490.915222,"volume_a":55928100,"open_bench":492.032235,"close_bench":490.915222},{"date":"2024-04-25","open_a":484.863872,"high_a":489.807891,"low_a":483.222338,"close_a":489.050262,"volume_a":69122400,"open_bench":484.863872,"close_bench":489.050262},{"date":"2024-04-26","open_a":491.828246,"high_a":495.257006,"low_a":491.196893,"close_a":493.683472,"volume_a":64306100,"open_bench":491.828246,"close_bench":493.683472},{"date":"2024-04-29","open_a":495.460962,"high_a":496.102037,"low_a":492.702415,"close_a":495.431824,"volume_a":46415400,"open_bench":495.460962,"close_bench":495.431824},{"date":"2024-04-30","open_a":493.974834,"high_a":494.946155,"low_a":487.583557,"close_a":487.583557,"volume_a":77483600,"open_bench":493.974834,"close_bench":487.583557},{"date":"2024-05-01","open_a":487.000703,"high_a":493.615394,"low_a":485.534,"close_a":486.000244,"volume_a":80242800,"open_bench":487.000703,"close_bench":486.000244},{"date":"2024-05-02","open_a":489.691345,"high_a":491.381463,"low_a":485.223264,"close_a":490.546112,"volume_a":62550200,"open_bench":489.691345,"close_bench":490.546112},{"date":"2024-05-03","open_a":496.50028,"high_a":497.8504,"low_a":493.97484,"close_a":496.626556,"volume_a":72756700,"open_bench":496.50028,"close_bench":496.626556},{"date":"2024-05-06","open_a":499.016026,"high_a":501.793989,"low_a":498.57892,"close_a":501.755157,"volume_a":47264700,"open_bench":499.016026,"close_bench":501.755157},{"date":"2024-05-07","open_a":502.716745,"high_a":503.697789,"low_a":501.638594,"close_a":502.308807,"volume_a":52561300,"open_bench":502.716745,"close_bench":502.308807},{"date":"2024-05-08","open_a":500.482719,"high_a":502.891575,"low_a":500.366165,"close_a":502.357361,"volume_a":42047200,"open_bench":500.482719,"close_bench":502.357361},{"date":"2024-05-09","open_a":502.541929,"high_a":505.290783,"low_a":501.89116,"close_a":505.251892,"volume_a":43643700,"open_bench":502.541929,"close_bench":505.251892},{"date":"2024-05-10","open_a":506.84474,"high_a":507.650953,"low_a":504.688437,"close_a":505.902588,"volume_a":52233200,"open_bench":506.84474,"close_bench":505.902588},{"date":"2024-05-13","open_a":507.573284,"high_a":507.680115,"low_a":504.834153,"close_a":505.970581,"volume_a":36716400,"open_bench":507.573284,"close_bench":505.970581},{"date":"2024-05-14","open_a":506.164889,"high_a":508.806912,"low_a":505.630675,"close_a":508.292084,"volume_a":57535900,"open_bench":506.164889,"close_bench":508.292084},{"date":"2024-05-15","open_a":510.749515,"high_a":514.877627,"low_a":510.118132,"close_a":514.586243,"volume_a":59504900,"open_bench":510.749515,"close_bench":514.586243},{"date":"2024-05-16","open_a":514.683401,"high_a":516.276381,"low_a":513.381805,"close_a":513.527527,"volume_a":50244800,"open_bench":514.683401,"close_bench":513.527527},{"date":"2024-05-17","open_a":513.644149,"high_a":514.333808,"low_a":512.19689,"close_a":514.265808,"volume_a":59187600,"open_bench":513.644149,"close_bench":514.265808},{"date":"2024-05-20","open_a":514.382217,"high_a":516.315135,"low_a":513.993665,"close_a":514.858154,"volume_a":37764200,"open_bench":514.382217,"close_bench":514.858154},{"date":"2024-05-21","open_a":514.100606,"high_a":516.276355,"low_a":513.896608,"close_a":516.120911,"volume_a":33437000,"open_bench":514.100606,"close_bench":516.120911},{"date":"2024-05-22","open_a":515.431439,"high_a":516.140484,"low_a":512.468863,"close_a":514.634949,"volume_a":48390000,"open_bench":515.431439,"close_bench":514.634949},{"date":"2024-05-23","open_a":517.675099,"high_a":517.78193,"low_a":509.671367,"close_a":510.875854,"volume_a":57211200,"open_bench":517.675099,"close_bench":510.875854},{"date":"2024-05-24","open_a":512.711495,"high_a":515.062133,"low_a":511.769342,"close_a":514.25592,"volume_a":41291100,"open_bench":512.711495,"close_bench":514.25592},{"date":"2024-05-28","open_a":515.062307,"high_a":515.295415,"low_a":511.9929,"close_a":514.615479,"volume_a":36269600,"open_bench":515.062307,"close_bench":514.615479},{"date":"2024-05-29","open_a":510.603933,"high_a":512.187191,"low_a":510.302826,"close_a":511.011871,"volume_a":45190300,"open_bench":510.603933,"close_bench":511.011871},{"date":"2024-05-30","open_a":509.477112,"high_a":510.137603,"low_a":506.378597,"close_a":507.621857,"volume_a":46377600,"open_bench":509.477112,"close_bench":507.621857},{"date":"2024-05-31","open_a":508.573861,"high_a":512.371699,"low_a":503.493813,"close_a":512.245422,"volume_a":90785800,"open_bench":508.573861,"close_bench":512.245422},{"date":"2024-06-03","open_a":513.848067,"high_a":514.129729,"low_a":507.612146,"close_a":512.663025,"volume_a":46835700,"open_bench":513.848067,"close_bench":512.663025},{"date":"2024-06-04","open_a":511.361625,"high_a":513.97448,"low_a":509.904644,"close_a":513.236267,"volume_a":34632700,"open_bench":511.361625,"close_bench":513.236267},{"date":"2024-06-05","open_a":515.547884,"high_a":519.355444,"low_a":513.566351,"close_a":519.335999,"volume_a":47610400,"open_bench":515.547884,"close_bench":519.335999},{"date":"2024-06-06","open_a":519.637185,"high_a":520.064568,"low_a":517.403159,"close_a":519.326355,"volume_a":30808500,"open_bench":519.637185,"close_bench":519.326355},{"date":"2024-06-07","open_a":518.354887,"high_a":521.492293,"low_a":517.267013,"close_a":518.694885,"volume_a":43224500,"open_bench":518.354887,"close_bench":518.694885},{"date":"2024-06-10","open_a":517.888751,"high_a":520.61816,"low_a":517.29626,"close_a":520.297607,"volume_a":35686100,"open_bench":517.888751,"close_bench":520.297607},{"date":"2024-06-11","open_a":518.753312,"high_a":521.608997,"low_a":516.791225,"close_a":521.55072,"volume_a":36383400,"open_bench":518.753312,"close_bench":521.55072},{"date":"2024-06-12","open_a":526.096443,"high_a":528.515022,"low_a":524.80457,"close_a":525.834167,"volume_a":63251300,"open_bench":526.096443,"close_bench":525.834167},{"date":"2024-06-13","open_a":527.572881,"high_a":527.747711,"low_a":524.114981,"close_a":526.892944,"volume_a":44760900,"open_bench":527.572881,"close_bench":526.892944},{"date":"2024-06-14","open_a":525.367968,"high_a":527.24261,"low_a":524.367479,"close_a":527.213501,"volume_a":40089900,"open_bench":525.367968,"close_bench":527.213501},{"date":"2024-06-17","open_a":526.533556,"high_a":532.798587,"low_a":526.077005,"close_a":531.409546,"volume_a":55839500,"open_bench":526.533556,"close_bench":531.409546},{"date":"2024-06-18","open_a":531.467771,"high_a":532.88592,"low_a":531.05011,"close_a":532.759644,"volume_a":41376400,"open_bench":531.467771,"close_bench":532.759644},{"date":"2024-06-20","open_a":533.682464,"high_a":534.342955,"low_a":529.544628,"close_a":531.312439,"volume_a":70328200,"open_bench":533.682464,"close_bench":531.312439},{"date":"2024-06-21","open_a":530.492922,"high_a":531.71099,"low_a":529.14817,"close_a":530.600098,"volume_a":64338600,"open_bench":530.492922,"close_bench":530.600098},{"date":"2024-06-24","open_a":530.424714,"high_a":532.977779,"low_a":528.758375,"close_a":528.875305,"volume_a":45528700,"open_bench":530.424714,"close_bench":528.875305},{"date":"2024-06-25","open_a":530.093359,"high_a":531.27247,"low_a":528.582966,"close_a":530.911926,"volume_a":37936200,"open_bench":530.093359,"close_bench":530.911926},{"date":"2024-06-26","open_a":529.801071,"high_a":532.285918,"low_a":529.157957,"close_a":531.574585,"volume_a":38550600,"open_bench":529.801071,"close_bench":531.574585},{"date":"2024-06-27","open_a":531.438143,"high_a":532.987552,"low_a":530.697549,"close_a":532.412598,"volume_a":35041500,"open_bench":531.438143,"close_bench":532.412598},{"date":"2024-06-28","open_a":533.182342,"high_a":536.222693,"low_a":529.079927,"close_a":530.317444,"volume_a":76144500,"open_bench":533.182342,"close_bench":530.317444},{"date":"2024-07-01","open_a":531.691506,"high_a":531.935119,"low_a":528.660967,"close_a":531.408936,"volume_a":40297800,"open_bench":531.691506,"close_bench":531.408936},{"date":"2024-07-02","open_a":529.810818,"high_a":534.985168,"low_a":529.762108,"close_a":534.985168,"volume_a":40434800,"open_bench":529.810818,"close_bench":534.985168},{"date":"2024-07-03","open_a":534.673362,"high_a":537.733163,"low_a":534.634405,"close_a":537.37262,"volume_a":32789900,"open_bench":534.673362,"close_bench":537.37262},{"date":"2024-07-05","open_a":537.674635,"high_a":540.870814,"low_a":537.041216,"close_a":540.471313,"volume_a":41488400,"open_bench":537.674635,"close_bench":540.471313},{"date":"2024-07-08","open_a":541.250918,"high_a":542.040224,"low_a":540.03285,"close_a":541.095032,"volume_a":36110500,"open_bench":541.250918,"close_bench":541.095032},{"date":"2024-07-09","open_a":542.049978,"high_a":542.946459,"low_a":541.328891,"close_a":541.621216,"volume_a":27289700,"open_bench":542.049978,"close_bench":541.621216},{"date":"2024-07-10","open_a":542.839282,"high_a":547.321748,"low_a":542.546958,"close_a":546.980713,"volume_a":38701200,"open_bench":542.839282,"close_bench":546.980713},{"date":"2024-07-11","open_a":547.097658,"high_a":547.964937,"low_a":541.630984,"close_a":542.264343,"volume_a":53054200,"open_bench":547.097658,"close_bench":542.264343},{"date":"2024-07-12","open_a":543.384994,"high_a":549.270677,"low_a":542.917275,"close_a":545.684692,"volume_a":53084400,"open_bench":543.384994,"close_bench":545.684692},{"date":"2024-07-15","open_a":547.672469,"high_a":550.410682,"low_a":545.333755,"close_a":547.185242,"volume_a":40584300,"open_bench":547.672469,"close_bench":547.185242},{"date":"2024-07-16","open_a":548.491082,"high_a":550.722561,"low_a":547.740733,"close_a":550.430237,"volume_a":36475300,"open_bench":548.491082,"close_bench":550.430237},{"date":"2024-07-17","open_a":544.525056,"high_a":546.191394,"low_a":542.390999,"close_a":542.712585,"volume_a":57119000,"open_bench":544.525056,"close_bench":542.712585},{"date":"2024-07-18","open_a":544.242402,"high_a":545.22661,"low_a":536.368795,"close_a":538.541809,"volume_a":56270400,"open_bench":544.242402,"close_bench":538.541809},{"date":"2024-07-19","open_a":538.30807,"high_a":539.925697,"low_a":533.913271,"close_a":534.965698,"volume_a":65509100,"open_bench":538.30807,"close_bench":534.965698},{"date":"2024-07-22","open_a":538.873267,"high_a":541.085297,"low_a":536.943866,"close_a":540.48114,"volume_a":43346700,"open_bench":538.873267,"close_bench":540.48114},{"date":"2024-07-23","open_a":540.373897,"high_a":542.517709,"low_a":539.146135,"close_a":539.633362,"volume_a":34439600,"open_bench":540.373897,"close_bench":539.633362},{"date":"2024-07-24","open_a":534.838961,"high_a":535.141039,"low_a":526.48788,"close_a":527.40387,"volume_a":74515300,"open_bench":534.838961,"close_bench":527.40387},{"date":"2024-07-25","open_a":527.520843,"high_a":533.474804,"low_a":523.720507,"close_a":524.655945,"volume_a":61158300,"open_bench":527.520843,"close_bench":524.655945},{"date":"2024-07-26","open_a":528.427065,"high_a":533.211609,"low_a":527.657208,"close_a":530.53186,"volume_a":53763800,"open_bench":528.427065,"close_bench":530.53186},{"date":"2024-07-29","open_a":532.071572,"high_a":533.075229,"low_a":528.855825,"close_a":530.84375,"volume_a":39515800,"open_bench":532.071572,"close_bench":530.84375},{"date":"2024-07-30","open_a":532.305298,"high_a":533.357725,"low_a":524.763033,"close_a":528.154114,"volume_a":46853600,"open_bench":532.305298,"close_bench":528.154114},{"date":"2024-07-31","open_a":534.955929,"high_a":539.360481,"low_a":533.591728,"close_a":536.739197,"volume_a":65663400,"open_bench":534.955929,"close_bench":536.739197},{"date":"2024-08-01","open_a":538.454208,"high_a":540.695441,"low_a":525.649865,"close_a":529.138428,"volume_a":76428700,"open_bench":538.454208,"close_bench":529.138428},{"date":"2024-08-02","open_a":522.063976,"high_a":523.27229,"low_a":515.096602,"close_a":519.286804,"volume_a":82789100,"open_bench":522.063976,"close_bench":519.286804},{"date":"2024-08-05","open_a":498.569759,"high_a":510.204744,"low_a":497.234732,"close_a":504.163116,"volume_a":146267400,"open_bench":498.569759,"close_bench":504.163116},{"date":"2024-08-06","open_a":505.956198,"high_a":516.217231,"low_a":504.640708,"close_a":508.811401,"volume_a":84826300,"open_bench":505.956198,"close_bench":508.811401},{"date":"2024-08-07","open_a":514.969887,"high_a":518.01024,"low_a":504.816089,"close_a":505.410492,"volume_a":70698300,"open_bench":514.969887,"close_bench":505.410492},{"date":"2024-08-08","open_a":510.526306,"high_a":517.717783,"low_a":508.509238,"close_a":517.094177,"volume_a":63276600,"open_bench":510.526306,"close_bench":517.094177},{"date":"2024-08-09","open_a":516.275571,"high_a":520.855518,"low_a":515.057504,"close_a":519.374329,"volume_a":45619600,"open_bench":516.275571,"close_bench":519.374329},{"date":"2024-08-12","open_a":520.563267,"high_a":522.044397,"low_a":517.386537,"close_a":519.647278,"volume_a":42542100,"open_bench":520.563267,"close_bench":519.647278},{"date":"2024-08-13","open_a":522.823983,"high_a":528.427095,"low_a":522.58037,"close_a":528.193176,"volume_a":52333100,"open_bench":522.823983,"close_bench":528.193176},{"date":"2024-08-14","open_a":528.982525,"high_a":531.038669,"low_a":526.322284,"close_a":529.859558,"volume_a":42446900,"open_bench":528.982525,"close_bench":529.859558},{"date":"2024-08-15","open_a":535.462598,"high_a":539.223977,"low_a":534.858441,"close_a":538.941406,"volume_a":60846800,"open_bench":535.462598,"close_bench":538.941406},{"date":"2024-08-16","open_a":537.333654,"high_a":540.841725,"low_a":537.177767,"close_a":540.149841,"volume_a":44430700,"open_bench":537.333654,"close_bench":540.149841},{"date":"2024-08-19","open_a":540.55899,"high_a":545.314331,"low_a":539.711219,"close_a":545.314331,"volume_a":39121800,"open_bench":540.55899,"close_bench":545.314331},{"date":"2024-08-20","open_a":544.86619,"high_a":546.51302,"low_a":543.092676,"close_a":544.427673,"volume_a":33732300,"open_bench":544.86619,"close_bench":544.427673},{"date":"2024-08-21","open_a":545.470322,"high_a":547.750511,"low_a":540.559034,"close_a":546.298584,"volume_a":41514600,"open_bench":545.470322,"close_bench":546.298584},{"date":"2024-08-22","open_a":548.189053,"high_a":548.79321,"low_a":540.802672,"close_a":542.010986,"volume_a":56121500,"open_bench":548.189053,"close_bench":542.010986},{"date":"2024-08-23","open_a":545.236462,"high_a":548.705517,"low_a":543.053634,"close_a":547.77002,"volume_a":50639400,"open_bench":545.236462,"close_bench":547.77002},{"date":"2024-08-26","open_a":548.793071,"high_a":549.504403,"low_a":544.768571,"close_a":546.464111,"volume_a":35788600,"open_bench":548.793071,"close_bench":546.464111},{"date":"2024-08-27","open_a":545.197411,"high_a":547.701766,"low_a":544.057316,"close_a":547.214539,"volume_a":32693900,"open_bench":545.197411,"close_bench":547.214539},{"date":"2024-08-28","open_a":546.873476,"high_a":547.302238,"low_a":540.86105,"close_a":544.037781,"volume_a":41066000,"open_bench":546.873476,"close_bench":544.037781},{"date":"2024-08-29","open_a":545.9965,"high_a":549.280406,"low_a":542.946453,"close_a":544.086548,"volume_a":38715200,"open_bench":545.9965,"close_bench":544.086548},{"date":"2024-08-30","open_a":546.44476,"high_a":549.787131,"low_a":542.907486,"close_a":549.280396,"volume_a":62700100,"open_bench":546.44476,"close_bench":549.280396},{"date":"2024-09-03","open_a":546.152434,"high_a":546.483775,"low_a":535.472452,"close_a":537.976807,"volume_a":60600100,"open_bench":546.152434,"close_bench":537.976807},{"date":"2024-09-04","open_a":536.144709,"high_a":540.266631,"low_a":535.423622,"close_a":536.875549,"volume_a":47224900,"open_bench":536.144709,"close_bench":536.875549},{"date":"2024-09-05","open_a":536.817093,"high_a":539.652729,"low_a":533.123874,"close_a":535.569763,"volume_a":44264300,"open_bench":536.817093,"close_bench":535.569763},{"date":"2024-09-06","open_a":535.891442,"high_a":537.50901,"low_a":525.659671,"close_a":526.556152,"volume_a":68493800,"open_bench":535.891442,"close_bench":526.556152},{"date":"2024-09-09","open_a":530.736549,"high_a":533.718376,"low_a":528.816843,"close_a":532.451538,"volume_a":40445800,"open_bench":530.736549,"close_bench":532.451538},{"date":"2024-09-10","open_a":534.351744,"high_a":535.121601,"low_a":529.49898,"close_a":534.770752,"volume_a":36394600,"open_bench":534.351744,"close_bench":534.770752},{"date":"2024-09-11","open_a":534.683047,"high_a":541.172886,"low_a":526.166327,"close_a":540.256897,"volume_a":75248600,"open_bench":534.683047,"close_bench":540.256897},{"date":"2024-09-12","open_a":540.831888,"high_a":545.109757,"low_a":538.619858,"close_a":544.807678,"volume_a":51819600,"open_bench":540.831888,"close_bench":544.807678},{"date":"2024-09-13","open_a":545.411782,"high_a":548.646977,"low_a":545.158415,"close_a":547.653015,"volume_a":39310500,"open_bench":545.411782,"close_bench":547.653015},{"date":"2024-09-16","open_a":547.389979,"high_a":548.724976,"low_a":545.597016,"close_a":548.461914,"volume_a":36656100,"open_bench":547.389979,"close_bench":548.461914},{"date":"2024-09-17","open_a":550.664024,"high_a":552.106257,"low_a":546.464129,"close_a":548.685913,"volume_a":49321000,"open_bench":550.664024,"close_bench":548.685913},{"date":"2024-09-18","open_a":549.338906,"high_a":554.162467,"low_a":546.50327,"close_a":547.058716,"volume_a":59044900,"open_bench":549.338906,"close_bench":547.058716},{"date":"2024-09-19","open_a":556.423122,"high_a":558.245347,"low_a":553.567978,"close_a":556.39386,"volume_a":75315500,"open_bench":556.423122,"close_bench":556.39386},{"date":"2024-09-20","open_a":555.031342,"high_a":556.468155,"low_a":552.421526,"close_a":555.432068,"volume_a":77503100,"open_bench":555.031342,"close_bench":555.432068},{"date":"2024-09-23","open_a":556.497616,"high_a":557.465275,"low_a":555.285536,"close_a":556.820129,"volume_a":44116900,"open_bench":556.497616,"close_bench":556.820129},{"date":"2024-09-24","open_a":557.611758,"high_a":558.471913,"low_a":554.796717,"close_a":558.413269,"volume_a":46805700,"open_bench":557.611758,"close_bench":558.413269},{"date":"2024-09-25","open_a":558.256925,"high_a":558.990007,"low_a":556.077186,"close_a":557.181702,"volume_a":38428600,"open_bench":558.256925,"close_bench":557.181702},{"date":"2024-09-26","open_a":561.423846,"high_a":561.746419,"low_a":557.044919,"close_a":559.390747,"volume_a":48336000,"open_bench":561.423846,"close_bench":559.390747},{"date":"2024-09-27","open_a":560.456141,"high_a":561.267375,"low_a":557.553103,"close_a":558.579407,"volume_a":42100900,"open_bench":560.456141,"close_bench":558.579407},{"date":"2024-09-30","open_a":557.553184,"high_a":561.423881,"low_a":555.266,"close_a":560.817871,"volume_a":63557400,"open_bench":557.553184,"close_bench":560.817871},{"date":"2024-10-01","open_a":560.465969,"high_a":561.111056,"low_a":553.232866,"close_a":555.793762,"volume_a":72668800,"open_bench":560.465969,"close_bench":555.793762},{"date":"2024-10-02","open_a":554.904357,"high_a":557.044961,"low_a":552.519393,"close_a":556.028381,"volume_a":38097800,"open_bench":554.904357,"close_bench":556.028381},{"date":"2024-10-03","open_a":554.562256,"high_a":556.947221,"low_a":552.734442,"close_a":555.011902,"volume_a":40846500,"open_bench":554.562256,"close_bench":555.011902},{"date":"2024-10-04","open_a":559.439565,"high_a":560.426792,"low_a":555.285432,"close_a":560.055359,"volume_a":42939100,"open_bench":559.439565,"close_bench":560.055359},{"date":"2024-10-07","open_a":558.413361,"high_a":559.058507,"low_a":553.848718,"close_a":554.99231,"volume_a":49964700,"open_bench":558.413361,"close_bench":554.99231},{"date":"2024-10-08","open_a":557.55312,"high_a":560.837375,"low_a":556.683241,"close_a":560.241089,"volume_a":37398700,"open_bench":557.55312,"close_bench":560.241089},{"date":"2024-10-09","open_a":560.231257,"high_a":564.678671,"low_a":559.635031,"close_a":564.121521,"volume_a":37912200,"open_bench":560.231257,"close_bench":564.121521},{"date":"2024-10-10","open_a":562.782412,"high_a":564.551582,"low_a":561.531256,"close_a":563.134277,"volume_a":44138100,"open_bench":562.782412,"close_bench":563.134277},{"date":"2024-10-11","open_a":563.056067,"high_a":567.239552,"low_a":562.919211,"close_a":566.50647,"volume_a":42268000,"open_bench":563.056067,"close_bench":566.50647},{"date":"2024-10-14","open_a":568.109539,"high_a":572.068231,"low_a":567.630601,"close_a":571.139648,"volume_a":36217200,"open_bench":568.109539,"close_bench":571.139648},{"date":"2024-10-15","open_a":571.403587,"high_a":571.706592,"low_a":565.490008,"close_a":566.702087,"volume_a":54203600,"open_bench":571.403587,"close_bench":566.702087},{"date":"2024-10-16","open_a":566.702045,"high_a":569.683235,"low_a":565.900534,"close_a":569.165161,"volume_a":30725400,"open_bench":566.702045,"close_bench":569.165161},{"date":"2024-10-17","open_a":572.693746,"high_a":572.89903,"low_a":569.028334,"close_a":569.21405,"volume_a":34393700,"open_bench":572.693746,"close_bench":569.21405},{"date":"2024-10-18","open_a":570.895275,"high_a":572.185507,"low_a":569.438894,"close_a":571.403564,"volume_a":37416800,"open_bench":570.895275,"close_bench":571.403564},{"date":"2024-10-21","open_a":570.680219,"high_a":571.657662,"low_a":567.503528,"close_a":570.46521,"volume_a":36439000,"open_bench":570.680219,"close_bench":570.46521},{"date":"2024-10-22","open_a":567.943355,"high_a":571.315546,"low_a":567.288485,"close_a":570.16217,"volume_a":34183800,"open_bench":567.943355,"close_bench":570.16217},{"date":"2024-10-23","open_a":568.14859,"high_a":568.588451,"low_a":561.462852,"close_a":564.952332,"volume_a":49314600,"open_bench":568.14859,"close_bench":564.952332},{"date":"2024-10-24","open_a":566.897554,"high_a":566.975766,"low_a":563.564498,"close_a":566.174255,"volume_a":34979900,"open_bench":566.897554,"close_bench":566.174255},{"date":"2024-10-25","open_a":568.392953,"high_a":571.276422,"low_a":565.04033,"close_a":565.978638,"volume_a":47268200,"open_bench":568.392953,"close_bench":565.978638},{"date":"2024-10-28","open_a":569.438797,"high_a":569.565869,"low_a":567.425267,"close_a":567.728271,"volume_a":30174700,"open_bench":569.438797,"close_bench":567.728271},{"date":"2024-10-29","open_a":566.770483,"high_a":569.761457,"low_a":565.38253,"close_a":568.647217,"volume_a":42899700,"open_bench":566.770483,"close_bench":568.647217},{"date":"2024-10-30","open_a":568.177976,"high_a":570.162215,"low_a":566.22309,"close_a":566.92688,"volume_a":41435800,"open_bench":568.177976,"close_bench":566.92688},{"date":"2024-10-31","open_a":562.577183,"high_a":562.645612,"low_a":555.617793,"close_a":555.813293,"volume_a":60182500,"open_bench":562.577183,"close_bench":555.813293},{"date":"2024-11-01","open_a":558.432892,"high_a":562.567458,"low_a":557.74867,"close_a":558.15918,"volume_a":45667500,"open_bench":558.432892,"close_bench":558.15918},{"date":"2024-11-04","open_a":558.296002,"high_a":559.586234,"low_a":555.080235,"close_a":556.956909,"volume_a":38217000,"open_bench":558.296002,"close_bench":556.956909},{"date":"2024-11-05","open_a":557.865945,"high_a":563.730605,"low_a":557.650936,"close_a":563.691528,"volume_a":39478300,"open_bench":557.865945,"close_bench":563.691528},{"date":"2024-11-06","open_a":575.909546,"high_a":578.577947,"low_a":572.18549,"close_a":577.708008,"volume_a":68182000,"open_bench":575.909546,"close_bench":577.708008},{"date":"2024-11-07","open_a":579.701966,"high_a":583.191445,"low_a":579.623754,"close_a":582.174866,"volume_a":47233200,"open_bench":579.701966,"close_bench":582.174866},{"date":"2024-11-08","open_a":582.722323,"high_a":586.114082,"low_a":582.722323,"close_a":584.696777,"volume_a":46444900,"open_bench":582.722323,"close_bench":584.696777},{"date":"2024-11-11","open_a":586.28021,"high_a":586.632075,"low_a":583.533597,"close_a":585.253906,"volume_a":37586800,"open_bench":586.28021,"close_bench":585.253906},{"date":"2024-11-12","open_a":585.17567,"high_a":585.771896,"low_a":580.962892,"close_a":583.435852,"volume_a":43006100,"open_bench":585.17567,"close_bench":583.435852},{"date":"2024-11-13","open_a":583.895232,"high_a":585.713262,"low_a":581.53962,"close_a":583.719299,"volume_a":47388600,"open_bench":583.895232,"close_bench":583.719299},{"date":"2024-11-14","open_a":583.846362,"high_a":584.325299,"low_a":579.281719,"close_a":579.965881,"volume_a":38904100,"open_bench":583.846362,"close_bench":579.965881},{"date":"2024-11-15","open_a":576.417836,"high_a":576.887049,"low_a":570.690032,"close_a":572.537415,"volume_a":75988800,"open_bench":576.417836,"close_bench":572.537415},{"date":"2024-11-18","open_a":572.996661,"high_a":576.192919,"low_a":572.136566,"close_a":574.883179,"volume_a":37001700,"open_bench":572.996661,"close_bench":574.883179},{"date":"2024-11-19","open_a":571.520806,"high_a":577.707979,"low_a":570.856152,"close_a":576.98468,"volume_a":49412000,"open_bench":571.520806,"close_bench":576.98468},{"date":"2024-11-20","open_a":577.062948,"high_a":577.463674,"low_a":571.44265,"close_a":577.180237,"volume_a":50032600,"open_bench":577.062948,"close_bench":577.180237},{"date":"2024-11-21","open_a":580.014817,"high_a":581.695991,"low_a":574.199018,"close_a":580.278687,"volume_a":46750300,"open_bench":580.014817,"close_bench":580.278687},{"date":"2024-11-22","open_a":580.268965,"high_a":582.702849,"low_a":579.770519,"close_a":582.077271,"volume_a":38226400,"open_bench":580.268965,"close_bench":582.077271},{"date":"2024-11-25","open_a":585.996799,"high_a":587.30654,"low_a":581.774237,"close_a":584.051697,"volume_a":42441400,"open_bench":585.996799,"close_bench":584.051697},{"date":"2024-11-26","open_a":585.292952,"high_a":587.765912,"low_a":584.579437,"close_a":587.101257,"volume_a":45621300,"open_bench":585.292952,"close_bench":587.101257},{"date":"2024-11-27","open_a":586.915564,"high_a":587.296721,"low_a":583.807302,"close_a":585.322327,"volume_a":34000200,"open_bench":586.915564,"close_bench":585.322327},{"date":"2024-11-29","open_a":586.13361,"high_a":589.740378,"low_a":585.859957,"close_a":588.958435,"volume_a":30177400,"open_bench":586.13361,"close_bench":588.958435},{"date":"2024-12-02","open_a":589.368892,"high_a":590.688476,"low_a":588.880171,"close_a":590.014038,"volume_a":31746000,"open_bench":589.368892,"close_bench":590.014038},{"date":"2024-12-03","open_a":589.779429,"high_a":590.532019,"low_a":588.753125,"close_a":590.287659,"volume_a":26906600,"open_bench":589.779429,"close_bench":590.287659},{"date":"2024-12-04","open_a":591.968946,"high_a":594.197486,"low_a":591.304292,"close_a":593.953125,"volume_a":42787600,"open_bench":591.968946,"close_bench":593.953125},{"date":"2024-12-05","open_a":593.95309,"high_a":594.754601,"low_a":592.623782,"close_a":592.975647,"volume_a":28762200,"open_bench":593.95309,"close_bench":592.975647},{"date":"2024-12-06","open_a":593.738082,"high_a":595.331319,"low_a":593.327573,"close_a":594.099731,"volume_a":31241500,"open_bench":593.738082,"close_bench":594.099731},{"date":"2024-12-09","open_a":593.982519,"high_a":594.148668,"low_a":590.453963,"close_a":591.040405,"volume_a":34742700,"open_bench":593.982519,"close_bench":591.040405},{"date":"2024-12-10","open_a":591.714795,"high_a":592.135088,"low_a":588.547889,"close_a":589.202759,"volume_a":37234500,"open_bench":591.714795,"close_bench":589.202759},{"date":"2024-12-11","open_a":592.115532,"high_a":594.705721,"low_a":591.841819,"close_a":593.757629,"volume_a":28677700,"open_bench":592.115532,"close_bench":593.757629},{"date":"2024-12-12","open_a":592.897489,"high_a":593.464363,"low_a":590.698242,"close_a":590.698242,"volume_a":31543800,"open_bench":592.897489,"close_bench":590.698242},{"date":"2024-12-13","open_a":592.721597,"high_a":593.435111,"low_a":589.212549,"close_a":590.580994,"volume_a":35904700,"open_bench":592.721597,"close_bench":590.580994},{"date":"2024-12-16","open_a":592.330624,"high_a":594.070502,"low_a":591.558466,"close_a":593.102783,"volume_a":43695200,"open_bench":592.330624,"close_bench":593.102783},{"date":"2024-12-17","open_a":590.561337,"high_a":591.519212,"low_a":589.290673,"close_a":590.659058,"volume_a":55773500,"open_bench":590.561337,"close_bench":590.659058},{"date":"2024-12-18","open_a":590.356242,"high_a":592.731423,"low_a":572.674325,"close_a":573.055542,"volume_a":108248700,"open_bench":590.356242,"close_bench":573.055542},{"date":"2024-12-19","open_a":578.020755,"high_a":579.623776,"low_a":572.635034,"close_a":572.879395,"volume_a":85919500,"open_bench":578.020755,"close_bench":572.879395},{"date":"2024-12-20","open_a":570.561035,"high_a":584.271663,"low_a":569.717559,"close_a":579.760315,"volume_a":125716700,"open_bench":570.561035,"close_bench":579.760315},{"date":"2024-12-23","open_a":579.505283,"high_a":583.830288,"low_a":576.337475,"close_a":583.232056,"volume_a":57635800,"open_bench":579.505283,"close_bench":583.232056},{"date":"2024-12-24","open_a":584.575691,"high_a":589.753989,"low_a":583.997032,"close_a":589.714722,"volume_a":33160100,"open_bench":584.575691,"close_bench":589.714722},{"date":"2024-12-26","open_a":587.949392,"high_a":590.871957,"low_a":586.556768,"close_a":589.753967,"volume_a":41219100,"open_bench":587.949392,"close_bench":589.753967},{"date":"2024-12-27","open_a":586.027122,"high_a":586.262548,"low_a":579.377784,"close_a":583.545898,"volume_a":64969300,"open_bench":586.027122,"close_bench":583.545898},{"date":"2024-12-30","open_a":576.563059,"high_a":580.338857,"low_a":573.150068,"close_a":576.886658,"volume_a":56578800,"open_bench":576.563059,"close_bench":576.886658},{"date":"2024-12-31","open_a":578.544251,"high_a":579.260227,"low_a":573.160035,"close_a":574.788086,"volume_a":57052700,"open_bench":578.544251,"close_bench":574.788086},{"date":"2025-01-02","open_a":578.034214,"high_a":579.74068,"low_a":569.315484,"close_a":573.375732,"volume_a":50204000,"open_bench":578.034214,"close_bench":573.375732},{"date":"2025-01-03","open_a":576.210038,"high_a":581.182301,"low_a":575.131196,"close_a":580.544861,"volume_a":37888500,"open_bench":576.210038,"close_bench":580.544861},{"date":"2025-01-06","open_a":584.7816,"high_a":588.145506,"low_a":582.163,"close_a":583.889099,"volume_a":47679400,"open_bench":584.7816,"close_bench":583.889099},{"date":"2025-01-07","open_a":585.909439,"high_a":586.233098,"low_a":575.474486,"close_a":577.288818,"volume_a":60393100,"open_bench":585.909439,"close_bench":577.288818},{"date":"2025-01-08","open_a":577.357506,"high_a":579.201288,"low_a":573.92494,"close_a":578.132263,"volume_a":47304700,"open_bench":577.357506,"close_bench":578.132263},{"date":"2025-01-10","open_a":574.591767,"high_a":574.660426,"low_a":567.402979,"close_a":569.305603,"volume_a":73105000,"open_bench":574.591767,"close_bench":569.305603},{"date":"2025-01-13","open_a":564.67658,"high_a":570.541343,"low_a":564.264629,"close_a":570.188293,"volume_a":47910100,"open_bench":564.67658,"close_bench":570.188293},{"date":"2025-01-14","open_a":573.101074,"high_a":573.728757,"low_a":567.20686,"close_a":570.9729,"volume_a":48420600,"open_bench":573.101074,"close_bench":570.9729},{"date":"2025-01-15","open_a":578.956079,"high_a":582.496511,"low_a":577.847846,"close_a":581.358887,"volume_a":56900200,"open_bench":578.956079,"close_bench":581.358887},{"date":"2025-01-16","open_a":582.722068,"high_a":582.898593,"low_a":579.544503,"close_a":580.240845,"volume_a":43319700,"open_bench":582.722068,"close_bench":580.240845},{"date":"2025-01-17","open_a":585.458357,"high_a":587.81208,"low_a":584.134331,"close_a":586.066406,"volume_a":58070600,"open_bench":585.458357,"close_bench":586.066406},{"date":"2025-01-21","open_a":589.09682,"high_a":591.440786,"low_a":587.135354,"close_a":591.430969,"volume_a":42532900,"open_bench":589.09682,"close_bench":591.430969},{"date":"2025-01-22","open_a":594.245737,"high_a":596.109154,"low_a":593.696529,"close_a":594.755737,"volume_a":48196000,"open_bench":594.245737,"close_bench":594.755737},{"date":"2025-01-23","open_a":594.127985,"high_a":598.001892,"low_a":593.853411,"close_a":598.001892,"volume_a":41152100,"open_bench":594.127985,"close_bench":598.001892},{"date":"2025-01-24","open_a":598.060739,"high_a":599.012082,"low_a":595.108724,"close_a":596.256165,"volume_a":34604700,"open_bench":598.060739,"close_bench":596.256165},{"date":"2025-01-27","open_a":583.34976,"high_a":588.135741,"low_a":583.183052,"close_a":587.821899,"volume_a":70361100,"open_bench":583.34976,"close_bench":587.821899},{"date":"2025-01-28","open_a":589.047798,"high_a":593.70628,"low_a":585.742733,"close_a":592.872681,"volume_a":44433300,"open_bench":589.047798,"close_bench":592.872681},{"date":"2025-01-29","open_a":592.088017,"high_a":592.490151,"low_a":587.674719,"close_a":590.214844,"volume_a":37177400,"open_bench":592.088017,"close_bench":590.214844},{"date":"2025-01-30","open_a":592.323481,"high_a":594.91257,"low_a":589.145856,"close_a":593.382629,"volume_a":39281300,"open_bench":592.323481,"close_bench":593.382629},{"date":"2025-01-31","open_a":595.795226,"high_a":598.20785,"low_a":589.469487,"close_a":590.22467,"volume_a":66566900,"open_bench":595.795226,"close_bench":590.22467},{"date":"2025-02-03","open_a":581.251033,"high_a":588.724214,"low_a":579.113042,"close_a":586.252808,"volume_a":65857200,"open_bench":581.251033,"close_bench":586.252808},{"date":"2025-02-04","open_a":586.311579,"high_a":590.695426,"low_a":585.772188,"close_a":590.185486,"volume_a":33457800,"open_bench":586.311579,"close_bench":590.185486},{"date":"2025-02-05","open_a":589.067389,"high_a":592.725503,"low_a":587.047081,"close_a":592.578369,"volume_a":30653100,"open_bench":589.067389,"close_bench":592.578369},{"date":"2025-02-06","open_a":594.314403,"high_a":594.765562,"low_a":591.019154,"close_a":594.638062,"volume_a":35771500,"open_bench":594.314403,"close_bench":594.638062},{"date":"2025-02-07","open_a":595.197027,"high_a":596.413126,"low_a":588.488787,"close_a":589.194946,"volume_a":50788500,"open_bench":595.197027,"close_bench":589.194946},{"date":"2025-02-10","open_a":592.392141,"high_a":593.833789,"low_a":591.126957,"close_a":593.196289,"volume_a":26048700,"open_bench":592.392141,"close_bench":593.196289},{"date":"2025-02-11","open_a":590.940628,"high_a":594.186852,"low_a":590.822945,"close_a":593.647461,"volume_a":30056700,"open_bench":590.940628,"close_bench":593.647461},{"date":"2025-02-12","open_a":587.655163,"high_a":592.90206,"low_a":586.978455,"close_a":591.734985,"volume_a":45076100,"open_bench":587.655163,"close_bench":591.734985},{"date":"2025-02-13","open_a":592.833452,"high_a":598.188275,"low_a":591.578145,"close_a":597.9823,"volume_a":40921300,"open_bench":592.833452,"close_bench":597.9823},{"date":"2025-02-14","open_a":598.188186,"high_a":599.217944,"low_a":597.334953,"close_a":597.95282,"volume_a":26910400,"open_bench":598.188186,"close_bench":597.95282},{"date":"2025-02-18","open_a":599.110141,"high_a":599.708374,"low_a":596.658309,"close_a":599.708374,"volume_a":26749000,"open_bench":599.110141,"close_bench":599.708374},{"date":"2025-02-19","open_a":598.325541,"high_a":601.414813,"low_a":597.81554,"close_a":601.120605,"volume_a":31011100,"open_bench":598.325541,"close_bench":601.120605},{"date":"2025-02-20","open_a":599.757497,"high_a":599.894814,"low_a":595.324624,"close_a":598.619873,"volume_a":36554000,"open_bench":599.757497,"close_bench":598.619873},{"date":"2025-02-21","open_a":598.404044,"high_a":598.541361,"low_a":587.920006,"close_a":588.380981,"volume_a":76519800,"open_bench":598.404044,"close_bench":588.380981},{"date":"2025-02-24","open_a":590.420752,"high_a":591.411302,"low_a":584.997271,"close_a":585.70343,"volume_a":50737200,"open_bench":590.420752,"close_bench":585.70343},{"date":"2025-02-25","open_a":585.644615,"high_a":586.370348,"low_a":578.200827,"close_a":582.790649,"volume_a":58266500,"open_bench":585.644615,"close_bench":582.790649},{"date":"2025-02-26","open_a":584.448194,"high_a":588.027893,"low_a":580.456604,"close_a":583.084961,"volume_a":43321600,"open_bench":584.448194,"close_bench":583.084961},{"date":"2025-02-27","open_a":585.35047,"high_a":586.49797,"low_a":573.385575,"close_a":573.777832,"volume_a":74196700,"open_bench":585.35047,"close_bench":573.777832},{"date":"2025-02-28","open_a":574.27802,"high_a":583.261508,"low_a":571.218138,"close_a":582.731934,"volume_a":88744100,"open_bench":574.27802,"close_bench":582.731934},{"date":"2025-03-03","open_a":584.69333,"high_a":585.831013,"low_a":568.727029,"close_a":572.522461,"volume_a":74249200,"open_bench":584.69333,"close_bench":572.522461},{"date":"2025-03-04","open_a":568.54073,"high_a":574.111286,"low_a":561.224441,"close_a":565.745605,"volume_a":109648200,"open_bench":568.54073,"close_bench":565.745605},{"date":"2025-03-05","open_a":565.578847,"high_a":573.611052,"low_a":562.038416,"close_a":571.826111,"volume_a":71230500,"open_bench":565.578847,"close_bench":571.826111},{"date":"2025-03-06","open_a":564.392188,"high_a":568.991828,"low_a":559.135473,"close_a":561.675598,"volume_a":80094900,"open_bench":564.392188,"close_bench":561.675598},{"date":"2025-03-07","open_a":559.900492,"high_a":566.265439,"low_a":554.732009,"close_a":564.82373,"volume_a":81158800,"open_bench":559.900492,"close_bench":564.82373},{"date":"2025-03-10","open_a":556.654121,"high_a":558.566502,"low_a":544.885328,"close_a":549.779175,"volume_a":99326600,"open_bench":556.654121,"close_bench":549.779175},{"date":"2025-03-11","open_a":548.622037,"high_a":553.153018,"low_a":541.384223,"close_a":545.209045,"volume_a":88102100,"open_bench":548.622037,"close_bench":545.209045},{"date":"2025-03-12","open_a":551.338641,"high_a":552.260532,"low_a":543.022044,"close_a":548.102234,"volume_a":69588200,"open_bench":551.338641,"close_bench":548.102234},{"date":"2025-03-13","open_a":547.729504,"high_a":548.337554,"low_a":539.08925,"close_a":540.795715,"volume_a":74079400,"open_bench":547.729504,"close_bench":540.795715},{"date":"2025-03-14","open_a":545.395386,"high_a":552.966675,"low_a":540.864404,"close_a":551.966309,"volume_a":62660300,"open_bench":545.395386,"close_bench":551.966309},{"date":"2025-03-17","open_a":551.946737,"high_a":558.733453,"low_a":551.515212,"close_a":556.222778,"volume_a":49008700,"open_bench":551.946737,"close_bench":556.222778},{"date":"2025-03-18","open_a":553.918016,"high_a":554.133809,"low_a":548.288618,"close_a":550.210876,"volume_a":66041400,"open_bench":553.918016,"close_bench":550.210876},{"date":"2025-03-19","open_a":551.985864,"high_a":559.949409,"low_a":550.808972,"close_a":556.203003,"volume_a":66556000,"open_bench":551.985864,"close_bench":556.203003},{"date":"2025-03-20","open_a":552.476369,"high_a":559.576867,"low_a":551.760394,"close_a":554.594727,"volume_a":62958200,"open_bench":552.476369,"close_bench":554.594727},{"date":"2025-03-21","open_a":550.154386,"high_a":555.672834,"low_a":548.924782,"close_a":554.777649,"volume_a":83763000,"open_bench":550.154386,"close_bench":554.777649},{"date":"2025-03-24","open_a":561.48632,"high_a":565.765378,"low_a":560.896134,"close_a":564.71283,"volume_a":58766800,"open_bench":561.48632,"close_bench":564.71283},{"date":"2025-03-25","open_a":565.912951,"high_a":567.004825,"low_a":564.329235,"close_a":566.070374,"volume_a":38355700,"open_bench":565.912951,"close_bench":566.070374},{"date":"2025-03-26","open_a":565.804724,"high_a":566.926137,"low_a":557.935259,"close_a":559.312439,"volume_a":51848300,"open_bench":565.804724,"close_bench":559.312439},{"date":"2025-03-27","open_a":557.925371,"high_a":561.584703,"low_a":555.72193,"close_a":557.827026,"volume_a":42164200,"open_bench":557.925371,"close_bench":557.827026},{"date":"2025-03-28","open_a":556.302268,"high_a":557.030184,"low_a":546.012922,"close_a":546.593262,"volume_a":71662700,"open_bench":556.302268,"close_bench":546.593262},{"date":"2025-03-31","open_a":540.858564,"high_a":551.561042,"low_a":537.94684,"close_a":550.262573,"volume_a":95328200,"open_bench":540.858564,"close_bench":550.262573},{"date":"2025-04-01","open_a":548.354248,"high_a":553.75466,"low_a":544.645743,"close_a":551.816772,"volume_a":54609600,"open_bench":548.354248,"close_bench":551.816772},{"date":"2025-04-02","open_a":545.993327,"high_a":558.161483,"low_a":545.757253,"close_a":555.308838,"volume_a":76014500,"open_bench":545.993327,"close_bench":555.308838},{"date":"2025-04-03","open_a":536.21556,"high_a":539.028879,"low_a":527.94281,"close_a":527.94281,"volume_a":125986000,"open_bench":536.21556,"close_bench":527.94281},{"date":"2025-04-04","open_a":515.125349,"high_a":517.289464,"low_a":496.819019,"close_a":497.035431,"volume_a":217965100,"open_bench":515.125349,"close_bench":497.035431},{"date":"2025-04-07","open_a":481.207996,"high_a":514.633533,"low_a":473.938563,"close_a":496.150146,"volume_a":256611400,"open_bench":481.207996,"close_bench":496.150146},{"date":"2025-04-08","open_a":513.344914,"high_a":516.414001,"low_a":481.17849,"close_a":488.379059,"volume_a":165816600,"open_bench":513.344914,"close_bench":488.379059},{"date":"2025-04-09","open_a":485.388642,"high_a":539.668274,"low_a":485.004991,"close_a":539.668274,"volume_a":241867300,"open_bench":485.388642,"close_bench":539.668274},{"date":"2025-04-10","open_a":523.486691,"high_a":524.795007,"low_a":501.009553,"close_a":516.020569,"volume_a":162331200,"open_bench":523.486691,"close_bench":516.020569},{"date":"2025-04-11","open_a":514.476193,"high_a":527.677206,"low_a":511.584162,"close_a":525.227844,"volume_a":97866300,"open_bench":514.476193,"close_bench":525.227844},{"date":"2025-04-14","open_a":535.172854,"high_a":535.399142,"low_a":525.14912,"close_a":530.323303,"volume_a":68034000,"open_bench":535.172854,"close_bench":530.323303},{"date":"2025-04-15","open_a":530.864276,"high_a":534.366185,"low_a":528.050956,"close_a":528.837891,"volume_a":56892900,"open_bench":530.864276,"close_bench":528.837891},{"date":"2025-04-16","open_a":523.004678,"high_a":529.113372,"low_a":511.800512,"close_a":517.082886,"volume_a":83484800,"open_bench":523.004678,"close_bench":517.082886},{"date":"2025-04-17","open_a":519.03065,"high_a":522.503021,"low_a":515.361471,"close_a":517.820679,"volume_a":79868100,"open_bench":519.03065,"close_bench":517.820679},{"date":"2025-04-21","open_a":512.6563,"high_a":513.187527,"low_a":500.163541,"close_a":505.495117,"volume_a":69368100,"open_bench":512.6563,"close_bench":505.495117},{"date":"2025-04-22","open_a":511.652999,"high_a":520.663511,"low_a":510.718488,"close_a":518.646973,"volume_a":75948100,"open_bench":511.652999,"close_bench":518.646973},{"date":"2025-04-23","open_a":531.611917,"high_a":536.530333,"low_a":525.168804,"close_a":526.683655,"volume_a":90590700,"open_bench":531.611917,"close_bench":526.683655},{"date":"2025-04-24","open_a":527.962423,"high_a":538.497691,"low_a":526.713186,"close_a":537.769775,"volume_a":64150400,"open_bench":527.962423,"close_bench":537.769775},{"date":"2025-04-25","open_a":537.730448,"high_a":542.058618,"low_a":534.818724,"close_a":541.655334,"volume_a":61119600,"open_bench":537.730448,"close_bench":541.655334},{"date":"2025-04-28","open_a":542.393105,"high_a":544.517834,"low_a":536.127048,"close_a":541.861877,"volume_a":47613800,"open_bench":542.393105,"close_bench":541.861877},{"date":"2025-04-29","open_a":539.953569,"high_a":546.386897,"low_a":539.599458,"close_a":545.27533,"volume_a":47775100,"open_bench":539.953569,"close_bench":545.27533},{"date":"2025-04-30","open_a":538.635456,"high_a":547.439433,"low_a":532.684184,"close_a":545.491699,"volume_a":93101500,"open_bench":538.635456,"close_bench":545.491699},{"date":"2025-05-01","open_a":551.226505,"high_a":554.866144,"low_a":548.757451,"close_a":549.357483,"volume_a":63186100,"open_bench":551.226505,"close_bench":549.357483},{"date":"2025-05-02","open_a":555.515362,"high_a":559.10583,"low_a":553.203731,"close_a":557.512268,"volume_a":60717300,"open_bench":555.515362,"close_bench":557.512268},{"date":"2025-05-05","open_a":553.390704,"high_a":557.404149,"low_a":552.534904,"close_a":554.315369,"volume_a":38659200,"open_bench":553.390704,"close_bench":554.315369},{"date":"2025-05-06","open_a":548.826329,"high_a":554.157875,"low_a":547.872186,"close_a":549.682129,"volume_a":48264700,"open_bench":548.826329,"close_bench":549.682129},{"date":"2025-05-07","open_a":551.010152,"high_a":554.620253,"low_a":546.967169,"close_a":551.993835,"volume_a":55588000,"open_bench":551.010152,"close_bench":551.993835},{"date":"2025-05-08","open_a":556.017083,"high_a":561.004364,"low_a":552.534866,"close_a":555.840027,"volume_a":65130800,"open_bench":556.017083,"close_bench":555.840027},{"date":"2025-05-09","open_a":557.23675,"high_a":558.240126,"low_a":553.577478,"close_a":555.131714,"volume_a":37603400,"open_bench":557.23675,"close_bench":555.131714},{"date":"2025-05-12","open_a":571.982199,"high_a":573.487263,"low_a":567.62449,"close_a":573.477417,"volume_a":78993600,"open_bench":571.982199,"close_bench":573.477417},{"date":"2025-05-13","open_a":573.890623,"high_a":579.46815,"low_a":573.329976,"close_a":577.264709,"volume_a":67947200,"open_bench":573.890623,"close_bench":577.264709},{"date":"2025-05-14","open_a":578.218823,"high_a":579.369715,"low_a":575.985843,"close_a":578.002441,"volume_a":66283500,"open_bench":578.218823,"close_bench":578.002441},{"date":"2025-05-15","open_a":576.00549,"high_a":581.327189,"low_a":575.552974,"close_a":580.825562,"volume_a":71268100,"open_bench":576.00549,"close_bench":580.825562},{"date":"2025-05-16","open_a":581.6027,"high_a":584.799671,"low_a":579.664873,"close_a":584.504578,"volume_a":76052100,"open_bench":581.6027,"close_bench":584.504578},{"date":"2025-05-19","open_a":578.503999,"high_a":585.822604,"low_a":578.503999,"close_a":585.14386,"volume_a":68168500,"open_bench":578.503999,"close_bench":585.14386},{"date":"2025-05-20","open_a":583.412709,"high_a":584.357007,"low_a":579.979604,"close_a":583.176575,"volume_a":60614500,"open_bench":583.412709,"close_bench":583.176575},{"date":"2025-05-21","open_a":578.838517,"high_a":582.91098,"low_a":572.326539,"close_a":573.349548,"volume_a":95197700,"open_bench":578.838517,"close_bench":573.349548},{"date":"2025-05-22","open_a":573.15283,"high_a":577.048237,"low_a":571.923226,"close_a":573.575867,"volume_a":70860400,"open_bench":573.15283,"close_bench":573.575867},{"date":"2025-05-23","open_a":566.581833,"high_a":572.316723,"low_a":566.208029,"close_a":569.660767,"volume_a":76029000,"open_bench":566.581833,"close_bench":569.660767},{"date":"2025-05-27","open_a":576.507267,"high_a":581.661757,"low_a":568.991912,"close_a":581.504395,"volume_a":72588500,"open_bench":576.507267,"close_bench":581.504395},{"date":"2025-05-28","open_a":581.907599,"high_a":583.097877,"low_a":577.41216,"close_a":578.140076,"volume_a":68445500,"open_bench":581.907599,"close_bench":578.140076},{"date":"2025-05-29","open_a":583.383137,"high_a":583.520867,"low_a":576.507201,"close_a":580.422241,"volume_a":69973300,"open_bench":583.383137,"close_bench":580.422241},{"date":"2025-05-30","open_a":579.320494,"high_a":581.484609,"low_a":573.723335,"close_a":579.77301,"volume_a":90601200,"open_bench":579.320494,"close_bench":579.77301},{"date":"2025-06-02","open_a":578.169696,"high_a":583.117592,"low_a":575.513739,"close_a":583.03894,"volume_a":61630500,"open_bench":578.169696,"close_bench":583.03894},{"date":"2025-06-03","open_a":582.674957,"high_a":587.337606,"low_a":582.192902,"close_a":586.36377,"volume_a":63606200,"open_bench":582.674957,"close_bench":586.36377},{"date":"2025-06-04","open_a":587.219582,"high_a":588.193419,"low_a":585.773537,"close_a":586.20636,"volume_a":57314200,"open_bench":587.219582,"close_bench":586.20636},{"date":"2025-06-05","open_a":587.878515,"high_a":589.226156,"low_a":581.405864,"close_a":583.37323,"volume_a":92278700,"open_bench":587.878515,"close_bench":583.37323},{"date":"2025-06-06","open_a":588.891744,"high_a":591.02638,"low_a":587.121126,"close_a":589.363953,"volume_a":66588700,"open_bench":588.891744,"close_bench":589.363953},{"date":"2025-06-09","open_a":589.934406,"high_a":591.43947,"low_a":588.724495,"close_a":589.895081,"volume_a":53016400,"open_bench":589.934406,"close_bench":589.895081},{"date":"2025-06-10","open_a":590.426245,"high_a":593.623215,"low_a":589.314738,"close_a":593.239624,"volume_a":66247000,"open_bench":590.426245,"close_bench":593.239624},{"date":"2025-06-11","open_a":594.331509,"high_a":595.187308,"low_a":589.491804,"close_a":591.547668,"volume_a":73658200,"open_bench":594.331509,"close_bench":591.547668},{"date":"2025-06-12","open_a":590.219716,"high_a":593.898682,"low_a":589.737721,"close_a":593.898682,"volume_a":64129000,"open_bench":590.219716,"close_bench":593.898682},{"date":"2025-06-13","open_a":588.734436,"high_a":592.029751,"low_a":585.763693,"close_a":587.258911,"volume_a":89506000,"open_bench":588.734436,"close_bench":587.258911},{"date":"2025-06-16","open_a":590.603486,"high_a":594.587391,"low_a":590.42637,"close_a":592.846252,"volume_a":79984100,"open_bench":590.603486,"close_bench":592.846252},{"date":"2025-06-17","open_a":590.416537,"high_a":591.931388,"low_a":587.022818,"close_a":587.780273,"volume_a":82209400,"open_bench":590.416537,"close_bench":587.780273},{"date":"2025-06-18","open_a":588.675334,"high_a":591.409941,"low_a":586.737447,"close_a":587.69165,"volume_a":76605000,"open_bench":588.675334,"close_bench":587.69165},{"date":"2025-06-20","open_a":590.356462,"high_a":591.421998,"low_a":584.91046,"close_a":586.311462,"volume_a":94051400,"open_bench":590.356462,"close_bench":586.311462},{"date":"2025-06-23","open_a":587.061195,"high_a":592.487447,"low_a":583.953469,"close_a":592.102722,"volume_a":87426000,"open_bench":587.061195,"close_bench":592.102722},{"date":"2025-06-24","open_a":596.22676,"high_a":599.699521,"low_a":595.319053,"close_a":598.643921,"volume_a":67735300,"open_bench":596.22676,"close_bench":598.643921},{"date":"2025-06-25","open_a":599.758695,"high_a":600.44932,"low_a":597.420478,"close_a":598.979309,"volume_a":62114800,"open_bench":599.758695,"close_bench":598.979309},{"date":"2025-06-26","open_a":600.824201,"high_a":604.099691,"low_a":600.212519,"close_a":603.665588,"volume_a":78548400,"open_bench":600.824201,"close_bench":603.665588},{"date":"2025-06-27","open_a":604.662046,"high_a":608.124991,"low_a":602.639546,"close_a":606.664795,"volume_a":86258400,"open_bench":604.662046,"close_bench":606.664795},{"date":"2025-06-30","open_a":609.1017,"high_a":610.916994,"low_a":606.79305,"close_a":609.565369,"volume_a":92502500,"open_bench":609.1017,"close_bench":609.565369},{"date":"2025-07-01","open_a":608.095423,"high_a":610.532335,"low_a":607.26672,"close_a":609.368164,"volume_a":70030100,"open_bench":608.095423,"close_bench":609.368164},{"date":"2025-07-02","open_a":608.963575,"high_a":612.169996,"low_a":608.342017,"close_a":612.130554,"volume_a":66510400,"open_bench":608.963575,"close_bench":612.130554},{"date":"2025-07-03","open_a":614.103693,"high_a":617.882354,"low_a":614.083942,"close_a":616.954956,"volume_a":51065800,"open_bench":614.103693,"close_bench":616.954956},{"date":"2025-07-07","open_a":615.001479,"high_a":615.662539,"low_a":609.585103,"close_a":612.357422,"volume_a":74814500,"open_bench":615.001479,"close_bench":612.357422},{"date":"2025-07-08","open_a":613.01844,"high_a":613.768259,"low_a":611.213022,"close_a":612.022034,"volume_a":59024600,"open_bench":613.01844,"close_bench":612.022034},{"date":"2025-07-09","open_a":614.419458,"high_a":616.343262,"low_a":612.584352,"close_a":615.692139,"volume_a":66113300,"open_bench":614.419458,"close_bench":615.692139},{"date":"2025-07-10","open_a":615.830316,"high_a":618.464498,"low_a":614.65627,"close_a":617.428589,"volume_a":57529000,"open_bench":615.830316,"close_bench":617.428589},{"date":"2025-07-11","open_a":614.389791,"high_a":616.481359,"low_a":613.196054,"close_a":615.257996,"volume_a":63670200,"open_bench":614.389791,"close_bench":615.257996},{"date":"2025-07-14","open_a":614.804168,"high_a":616.777351,"low_a":613.462419,"close_a":616.432068,"volume_a":51898500,"open_bench":614.804168,"close_bench":616.432068},{"date":"2025-07-15","open_a":619.105778,"high_a":619.441186,"low_a":613.718968,"close_a":613.797913,"volume_a":74317300,"open_bench":619.105778,"close_bench":613.797913},{"date":"2025-07-16","open_a":615.376431,"high_a":616.353147,"low_a":609.762724,"close_a":615.849976,"volume_a":88987500,"open_bench":615.376431,"close_bench":615.849976},{"date":"2025-07-17","open_a":616.027567,"high_a":619.973932,"low_a":615.810486,"close_a":619.618713,"volume_a":68885700,"open_bench":616.027567,"close_bench":619.618713},{"date":"2025-07-18","open_a":620.861886,"high_a":621.02959,"low_a":618.060001,"close_a":619.164978,"volume_a":65621600,"open_bench":620.861886,"close_bench":619.164978},{"date":"2025-07-21","open_a":620.338989,"high_a":623.071806,"low_a":619.914762,"close_a":620.338989,"volume_a":63375000,"open_bench":620.338989,"close_bench":620.338989},{"date":"2025-07-22","open_a":620.664568,"high_a":621.286125,"low_a":617.793613,"close_a":620.427795,"volume_a":60046300,"open_bench":620.664568,"close_bench":620.427795},{"date":"2025-07-23","open_a":623.081627,"high_a":625.705994,"low_a":621.286024,"close_a":625.705994,"volume_a":70511000,"open_bench":623.081627,"close_bench":625.705994},{"date":"2025-07-24","open_a":626.090848,"high_a":627.620113,"low_a":625.489042,"close_a":625.913269,"volume_a":71307100,"open_bench":626.090848,"close_bench":625.913269},{"date":"2025-07-25","open_a":626.574253,"high_a":629.030856,"low_a":626.327605,"close_a":628.557251,"volume_a":56865400,"open_bench":626.574253,"close_bench":628.557251},{"date":"2025-07-28","open_a":628.932091,"high_a":629.484579,"low_a":627.018101,"close_a":628.399353,"volume_a":54917100,"open_bench":628.932091,"close_bench":628.399353},{"date":"2025-07-29","open_a":629.790476,"high_a":630.106193,"low_a":625.834296,"close_a":626.741943,"volume_a":60556300,"open_bench":629.790476,"close_bench":626.741943},{"date":"2025-07-30","open_a":627.393082,"high_a":629.129493,"low_a":623.071808,"close_a":625.952698,"volume_a":80418900,"open_bench":627.393082,"close_bench":625.952698},{"date":"2025-07-31","open_a":630.885601,"high_a":631.270326,"low_a":622.312121,"close_a":623.604553,"volume_a":103385200,"open_bench":630.885601,"close_bench":623.604553},{"date":"2025-08-01","open_a":617.902027,"high_a":617.941529,"low_a":610.986014,"close_a":613.383423,"volume_a":140103600,"open_bench":617.902027,"close_bench":613.383423},{"date":"2025-08-04","open_a":617.280475,"high_a":622.756044,"low_a":617.191715,"close_a":622.706726,"volume_a":73218000,"open_bench":617.280475,"close_bench":622.706726},{"date":"2025-08-05","open_a":623.318468,"high_a":624.127481,"low_a":618.63216,"close_a":619.549683,"volume_a":68051400,"open_bench":623.318468,"close_bench":619.549683},{"date":"2025-08-06","open_a":620.615201,"high_a":624.946351,"low_a":619.707554,"close_a":624.295227,"volume_a":64357500,"open_bench":620.615201,"close_bench":624.295227},{"date":"2025-08-07","open_a":627.708828,"high_a":628.438896,"low_a":620.674428,"close_a":623.772339,"volume_a":74205800,"open_bench":627.708828,"close_bench":623.772339},{"date":"2025-08-08","open_a":625.55807,"high_a":629.09996,"low_a":625.242354,"close_a":628.63623,"volume_a":64051600,"open_bench":625.55807,"close_bench":628.63623},{"date":"2025-08-11","open_a":628.912394,"high_a":630.382405,"low_a":626.149891,"close_a":627.393005,"volume_a":58742300,"open_bench":628.912394,"close_bench":627.393005},{"date":"2025-08-12","open_a":629.731301,"high_a":634.230155,"low_a":628.251414,"close_a":634.072327,"volume_a":64730800,"open_bench":629.731301,"close_bench":634.072327},{"date":"2025-08-13","open_a":636.262489,"high_a":637.525354,"low_a":634.06241,"close_a":636.242798,"volume_a":60092800,"open_bench":636.262489,"close_bench":636.242798},{"date":"2025-08-14","open_a":634.170992,"high_a":636.963062,"low_a":633.727074,"close_a":636.302063,"volume_a":59327500,"open_bench":634.170992,"close_bench":636.302063},{"date":"2025-08-15","open_a":637.328051,"high_a":637.426747,"low_a":633.904609,"close_a":634.812256,"volume_a":68592500,"open_bench":637.328051,"close_bench":634.812256},{"date":"2025-08-18","open_a":634.240031,"high_a":635.364759,"low_a":633.569156,"close_a":634.674133,"volume_a":43804900,"open_bench":634.240031,"close_bench":634.674133},{"date":"2025-08-19","open_a":634.496572,"high_a":635.473287,"low_a":629.918774,"close_a":631.230957,"volume_a":69750700,"open_bench":634.496572,"close_bench":631.230957},{"date":"2025-08-20","open_a":630.826452,"high_a":631.082915,"low_a":624.462927,"close_a":629.553711,"volume_a":88890300,"open_bench":630.826452,"close_bench":629.553711},{"date":"2025-08-21","open_a":627.748329,"high_a":629.41561,"low_a":625.311417,"close_a":627.028076,"volume_a":54805800,"open_bench":627.748329,"close_bench":627.028076},{"date":"2025-08-22","open_a":629.208414,"high_a":637.831211,"low_a":628.705243,"close_a":636.657166,"volume_a":84083200,"open_bench":629.208414,"close_bench":636.657166},{"date":"2025-08-25","open_a":635.40418,"high_a":636.637419,"low_a":633.736838,"close_a":633.855225,"volume_a":51274300,"open_bench":635.40418,"close_bench":633.855225},{"date":"2025-08-26","open_a":633.588884,"high_a":636.854498,"low_a":632.967327,"close_a":636.509155,"volume_a":51581600,"open_bench":633.588884,"close_bench":636.509155},{"date":"2025-08-27","open_a":635.927097,"high_a":638.689541,"low_a":635.779084,"close_a":637.959473,"volume_a":48341100,"open_bench":635.927097,"close_bench":637.959473},{"date":"2025-08-28","open_a":638.561284,"high_a":640.771239,"low_a":636.686797,"close_a":640.21875,"volume_a":61519500,"open_bench":638.561284,"close_bench":640.21875},{"date":"2025-08-29","open_a":638.78823,"high_a":639.153324,"low_a":634.516333,"close_a":636.400696,"volume_a":74522200,"open_bench":638.78823,"close_bench":636.400696},{"date":"2025-09-02","open_a":628.951877,"high_a":631.901775,"low_a":626.406454,"close_a":631.684753,"volume_a":81983500,"open_bench":628.951877,"close_bench":631.684753},{"date":"2025-09-03","open_a":634.052556,"high_a":635.571945,"low_a":631.872227,"close_a":635.108215,"volume_a":70820900,"open_bench":634.052556,"close_bench":635.108215},{"date":"2025-09-04","open_a":635.779086,"high_a":640.445703,"low_a":634.881314,"close_a":640.416077,"volume_a":65219200,"open_bench":635.779086,"close_bench":640.416077},{"date":"2025-09-05","open_a":642.744416,"high_a":643.464669,"low_a":634.703734,"close_a":638.561279,"volume_a":85178900,"open_bench":642.744416,"close_bench":638.561279},{"date":"2025-09-08","open_a":639.92286,"high_a":641.126533,"low_a":638.551484,"close_a":640.130066,"volume_a":63133100,"open_bench":639.92286,"close_bench":640.130066},{"date":"2025-09-09","open_a":640.268114,"high_a":642.132786,"low_a":638.54158,"close_a":641.609924,"volume_a":66133900,"open_bench":640.268114,"close_bench":641.609924},{"date":"2025-09-10","open_a":644.855789,"high_a":645.773312,"low_a":641.905891,"close_a":643.464722,"volume_a":78034500,"open_bench":644.855789,"close_bench":643.464722},{"date":"2025-09-11","open_a":645.408199,"high_a":649.502577,"low_a":644.826144,"close_a":648.811951,"volume_a":69934400,"open_bench":645.408199,"close_bench":648.811951},{"date":"2025-09-12","open_a":648.782364,"high_a":650.272127,"low_a":648.091799,"close_a":648.59491,"volume_a":72780100,"open_bench":648.782364,"close_bench":648.59491},{"date":"2025-09-15","open_a":650.795044,"high_a":652.176235,"low_a":650.499078,"close_a":652.047974,"volume_a":63772400,"open_bench":650.795044,"close_bench":652.047974},{"date":"2025-09-16","open_a":652.600468,"high_a":652.906369,"low_a":650.370822,"close_a":651.150208,"volume_a":61169000,"open_bench":652.600468,"close_bench":651.150208},{"date":"2025-09-17","open_a":651.160135,"high_a":652.847168,"low_a":645.526677,"close_a":650.341248,"volume_a":101952200,"open_bench":651.160135,"close_bench":650.341248},{"date":"2025-09-18","open_a":653.014849,"high_a":655.974622,"low_a":651.416576,"close_a":653.379883,"volume_a":90459200,"open_bench":653.014849,"close_bench":653.379883},{"date":"2025-09-19","open_a":655.260655,"high_a":657.456931,"low_a":653.321554,"close_a":656.616028,"volume_a":97945600,"open_bench":655.260655,"close_bench":656.616028},{"date":"2025-09-22","open_a":655.132105,"high_a":660.167744,"low_a":655.102396,"close_a":659.722595,"volume_a":69452200,"open_bench":655.132105,"close_bench":659.722595},{"date":"2025-09-23","open_a":659.603711,"high_a":660.217149,"low_a":654.914313,"close_a":656.131226,"volume_a":81708900,"open_bench":659.603711,"close_bench":656.131226},{"date":"2025-09-24","open_a":657.417338,"high_a":657.516247,"low_a":652.628972,"close_a":654.043701,"volume_a":68082200,"open_bench":657.417338,"close_bench":654.043701},{"date":"2025-09-25","open_a":650.917495,"high_a":652.371776,"low_a":647.425143,"close_a":651.026306,"volume_a":89622100,"open_bench":650.917495,"close_bench":651.026306},{"date":"2025-09-26","open_a":652.470701,"high_a":655.30016,"low_a":650.858094,"close_a":654.756042,"volume_a":69179200,"open_bench":652.470701,"close_bench":654.756042},{"date":"2025-09-29","open_a":657.268926,"high_a":658.17915,"low_a":654.79561,"close_a":656.596191,"volume_a":73499000,"open_bench":657.268926,"close_bench":656.596191},{"date":"2025-09-30","open_a":655.854147,"high_a":659.534473,"low_a":654.548229,"close_a":659.069458,"volume_a":86288000,"open_bench":655.854147,"close_bench":659.069458},{"date":"2025-10-01","open_a":656.091635,"high_a":662.225471,"low_a":655.982823,"close_a":661.315308,"volume_a":72545400,"open_bench":656.091635,"close_bench":661.315308},{"date":"2025-10-02","open_a":663.293939,"high_a":663.412653,"low_a":659.663128,"close_a":662.077026,"volume_a":56896000,"open_bench":663.293939,"close_bench":662.077026},{"date":"2025-10-03","open_a":662.838843,"high_a":665.500134,"low_a":661.028359,"close_a":662.0672,"volume_a":70494400,"open_bench":662.838843,"close_bench":662.0672},{"date":"2025-10-06","open_a":664.451431,"high_a":665.331946,"low_a":662.314513,"close_a":664.441528,"volume_a":54623300,"open_bench":664.451431,"close_bench":664.441528},{"date":"2025-10-07","open_a":665.361629,"high_a":665.806838,"low_a":660.543614,"close_a":661.978149,"volume_a":72020100,"open_bench":665.361629,"close_bench":661.978149},{"date":"2025-10-08","open_a":663.096078,"high_a":666.024506,"low_a":662.27492,"close_a":665.925537,"volume_a":60702200,"open_bench":663.096078,"close_bench":665.925537},{"date":"2025-10-09","open_a":666.341158,"high_a":666.746756,"low_a":662.06726,"close_a":663.996399,"volume_a":66501900,"open_bench":666.341158,"close_bench":663.996399},{"date":"2025-10-10","open_a":664.956002,"high_a":666.756583,"low_a":645.871916,"close_a":646.049988,"volume_a":159422600,"open_bench":664.956002,"close_bench":646.049988},{"date":"2025-10-13","open_a":653.598568,"high_a":658.030732,"low_a":652.727956,"close_a":655.963013,"volume_a":79560500,"open_bench":653.598568,"close_bench":655.963013},{"date":"2025-10-14","open_a":650.155692,"high_a":658.723294,"low_a":646.198386,"close_a":655.161682,"volume_a":88779600,"open_bench":650.155692,"close_bench":655.161682},{"date":"2025-10-15","open_a":659.702725,"high_a":663.076302,"low_a":651.896925,"close_a":658.070312,"volume_a":81702600,"open_bench":659.702725,"close_bench":658.070312},{"date":"2025-10-16","open_a":659.702715,"high_a":661.572556,"low_a":650.096332,"close_a":653.588684,"volume_a":110563300,"open_bench":659.702715,"close_bench":653.588684},{"date":"2025-10-17","open_a":652.460824,"high_a":658.654017,"low_a":651.115354,"close_a":657.298645,"volume_a":96500900,"open_bench":652.460824,"close_bench":657.298645},{"date":"2025-10-20","open_a":660.197387,"high_a":665.035209,"low_a":660.147933,"close_a":664.134888,"volume_a":60493400,"open_bench":660.197387,"close_bench":664.134888},{"date":"2025-10-21","open_a":664.273362,"high_a":665.806806,"low_a":662.828924,"close_a":664.124939,"volume_a":56249000,"open_bench":664.273362,"close_bench":664.124939},{"date":"2025-10-22","open_a":664.827425,"high_a":664.827425,"low_a":656.220272,"close_a":660.672241,"volume_a":80564000,"open_bench":664.827425,"close_bench":660.672241},{"date":"2025-10-23","open_a":660.988803,"high_a":665.529838,"low_a":660.672211,"close_a":664.589966,"volume_a":65604500,"open_bench":660.988803,"close_bench":664.589966},{"date":"2025-10-24","open_a":669.239755,"high_a":671.22825,"low_a":668.438403,"close_a":670.021301,"volume_a":74356500,"open_bench":669.239755,"close_bench":670.021301},{"date":"2025-10-27","open_a":675.442867,"high_a":678.222872,"low_a":674.839393,"close_a":677.926086,"volume_a":63339800,"open_bench":675.442867,"close_bench":677.926086},{"date":"2025-10-28","open_a":679.716721,"high_a":681.556853,"low_a":677.520445,"close_a":679.726624,"volume_a":61738100,"open_bench":679.716721,"close_bench":679.726624},{"date":"2025-10-29","open_a":681.368861,"high_a":682.338442,"low_a":675.581326,"close_a":680.053101,"volume_a":85657100,"open_bench":681.368861,"close_bench":680.053101},{"date":"2025-10-30","open_a":676.60048,"high_a":678.618685,"low_a":672.573914,"close_a":672.573914,"volume_a":76335800,"open_bench":676.60048,"close_bench":672.573914},{"date":"2025-10-31","open_a":677.728203,"high_a":677.767815,"low_a":671.990121,"close_a":674.780029,"volume_a":87164100,"open_bench":677.728203,"close_bench":674.780029},{"date":"2025-11-03","open_a":678.351474,"high_a":678.480091,"low_a":672.682653,"close_a":676.046387,"volume_a":57315000,"open_bench":678.351474,"close_bench":676.046387},{"date":"2025-11-04","open_a":668.893485,"high_a":672.702428,"low_a":667.379847,"close_a":668.032776,"volume_a":78427000,"open_bench":668.893485,"close_bench":668.032776},{"date":"2025-11-05","open_a":667.775554,"high_a":673.592799,"low_a":666.974202,"close_a":670.347839,"volume_a":74402400,"open_bench":667.775554,"close_bench":670.347839},{"date":"2025-11-06","open_a":669.249681,"high_a":670.150002,"low_a":661.582401,"close_a":663.155457,"volume_a":85035300,"open_bench":669.249681,"close_bench":663.155457},{"date":"2025-11-07","open_a":660.781013,"high_a":663.917221,"low_a":654.152574,"close_a":663.80835,"volume_a":100592400,"open_bench":660.781013,"close_bench":663.80835},{"date":"2025-11-10","open_a":670.011443,"high_a":674.898718,"low_a":667.82507,"close_a":674.166626,"volume_a":75842900,"open_bench":670.011443,"close_bench":674.166626},{"date":"2025-11-11","open_a":672.692527,"high_a":676.273884,"low_a":671.485518,"close_a":675.709961,"volume_a":58953400,"open_bench":672.692527,"close_bench":675.709961},{"date":"2025-11-12","open_a":677.480861,"high_a":677.64909,"low_a":673.681881,"close_a":676.085938,"volume_a":62312500,"open_bench":677.480861,"close_bench":676.085938},{"date":"2025-11-13","open_a":673.236729,"high_a":673.592872,"low_a":663.363269,"close_a":664.867004,"volume_a":103457800,"open_bench":673.236729,"close_bench":664.867004},{"date":"2025-11-14","open_a":658.278041,"high_a":668.448286,"low_a":656.190577,"close_a":664.758118,"volume_a":96846700,"open_bench":658.278041,"close_bench":664.758118},{"date":"2025-11-17","open_a":662.551956,"high_a":666.519165,"low_a":655.102299,"close_a":658.564941,"volume_a":90456100,"open_bench":662.551956,"close_bench":658.564941},{"date":"2025-11-18","open_a":655.033066,"high_a":658.020852,"low_a":648.859679,"close_a":653.034668,"volume_a":114467500,"open_bench":655.033066,"close_bench":653.034668},{"date":"2025-11-19","open_a":653.727143,"high_a":660.217122,"low_a":651.718782,"close_a":655.557373,"volume_a":94703000,"open_bench":653.727143,"close_bench":655.557373},{"date":"2025-11-20","open_a":665.727665,"high_a":668.349404,"low_a":644.932063,"close_a":645.565247,"volume_a":165293500,"open_bench":665.727665,"close_bench":645.565247},{"date":"2025-11-21","open_a":648.058289,"high_a":657.456891,"low_a":643.903106,"close_a":651.99585,"volume_a":123956200,"open_bench":648.058289,"close_bench":651.99585},{"date":"2025-11-24","open_a":655.616775,"high_a":662.908106,"low_a":654.52854,"close_a":661.592285,"volume_a":80437900,"open_bench":655.616775,"close_bench":661.592285},{"date":"2025-11-25","open_a":661.493314,"high_a":668.992425,"low_a":657.387586,"close_a":667.815125,"volume_a":81077100,"open_bench":661.493314,"close_bench":667.815125},{"date":"2025-11-26","open_a":670.397308,"high_a":674.423874,"low_a":669.496987,"close_a":672.425415,"volume_a":71879600,"open_bench":670.397308,"close_bench":672.425415},{"date":"2025-11-28","open_a":673.592861,"high_a":676.372866,"low_a":673.236718,"close_a":676.095886,"volume_a":49212000,"open_bench":673.592861,"close_bench":676.095886},{"date":"2025-12-01","open_a":671.564717,"high_a":675.700094,"low_a":671.495457,"close_a":673.009155,"volume_a":61201200,"open_bench":671.564717,"close_bench":673.009155},{"date":"2025-12-02","open_a":674.641468,"high_a":676.521212,"low_a":672.079146,"close_a":674.255676,"volume_a":62953800,"open_bench":674.641468,"close_bench":674.255676},{"date":"2025-12-03","open_a":673.305944,"high_a":677.599587,"low_a":672.435332,"close_a":676.590515,"volume_a":57238500,"open_bench":673.305944,"close_bench":676.590515},{"date":"2025-12-04","open_a":677.985405,"high_a":678.054665,"low_a":674.06771,"close_a":677.085144,"volume_a":61970300,"open_bench":677.985405,"close_bench":677.085144},{"date":"2025-12-05","open_a":678.153594,"high_a":681.04247,"low_a":677.273139,"close_a":678.371277,"volume_a":79241000,"open_bench":678.153594,"close_bench":678.371277},{"date":"2025-12-08","open_a":679.261741,"high_a":679.311196,"low_a":674.295303,"close_a":676.333313,"volume_a":55231500,"open_bench":679.261741,"close_bench":676.333313},{"date":"2025-12-09","open_a":675.858445,"high_a":678.074526,"low_a":675.304424,"close_a":675.749573,"volume_a":58310100,"open_bench":675.858445,"close_bench":675.749573},{"date":"2025-12-10","open_a":675.274666,"high_a":681.616222,"low_a":674.038008,"close_a":680.231201,"volume_a":85671300,"open_bench":675.274666,"close_bench":680.231201},{"date":"2025-12-11","open_a":677.827193,"high_a":681.893311,"low_a":674.888862,"close_a":681.814148,"volume_a":86173700,"open_bench":677.827193,"close_bench":681.814148},{"date":"2025-12-12","open_a":680.824833,"high_a":681.527277,"low_a":671.920894,"close_a":674.483276,"volume_a":113160300,"open_bench":680.824833,"close_bench":674.483276},{"date":"2025-12-15","open_a":678.420708,"high_a":678.440513,"low_a":671.999989,"close_a":673.464172,"volume_a":90811000,"open_bench":678.420708,"close_bench":673.464172},{"date":"2025-12-16","open_a":671.980228,"high_a":673.810518,"low_a":667.77559,"close_a":671.624084,"volume_a":122030600,"open_bench":671.980228,"close_bench":671.624084},{"date":"2025-12-17","open_a":672.633198,"high_a":673.177315,"low_a":664.035948,"close_a":664.233826,"volume_a":110625200,"open_bench":672.633198,"close_bench":664.233826},{"date":"2025-12-18","open_a":670.367639,"high_a":673.474138,"low_a":667.696505,"close_a":669.249695,"volume_a":108650100,"open_bench":670.367639,"close_bench":669.249695},{"date":"2025-12-19","open_a":671.346369,"high_a":675.811494,"low_a":671.227243,"close_a":675.315369,"volume_a":103599500,"open_bench":671.346369,"close_bench":675.315369},{"date":"2025-12-22","open_a":678.639344,"high_a":680.048322,"low_a":675.315331,"close_a":679.522461,"volume_a":69556700,"open_bench":678.639344,"close_bench":679.522461},{"date":"2025-12-23","open_a":678.619507,"high_a":682.866365,"low_a":678.569906,"close_a":682.628235,"volume_a":64840000,"open_bench":678.619507,"close_bench":682.628235},{"date":"2025-12-24","open_a":682.61832,"high_a":685.476005,"low_a":682.469458,"close_a":685.02948,"volume_a":39445600,"open_bench":682.61832,"close_bench":685.02948},{"date":"2025-12-26","open_a":685.287481,"high_a":686.299535,"low_a":683.928104,"close_a":684.960022,"volume_a":41613300,"open_bench":685.287481,"close_bench":684.960022},{"date":"2025-12-29","open_a":682.211448,"high_a":683.858617,"low_a":680.75287,"close_a":682.519043,"volume_a":62559500,"open_bench":682.211448,"close_bench":682.519043},{"date":"2025-12-30","open_a":682.122139,"high_a":683.223522,"low_a":681.258887,"close_a":681.685547,"volume_a":47160700,"open_bench":682.122139,"close_bench":681.685547},{"date":"2025-12-31","open_a":681.814586,"high_a":682.032851,"low_a":676.426676,"close_a":676.63501,"volume_a":74144800,"open_bench":681.814586,"close_bench":676.63501}]')
df = pd.DataFrame(DF_RECORDS)
csv_bytes = df.to_csv(index=False).encode('utf-8')
slice_sha256 = hashlib.sha256(csv_bytes).hexdigest()
assert slice_sha256 == EXPECTED_SNAPSHOT['aligned_slice_sha256'], (
    f'Data slice SHA-256 mismatch: {slice_sha256} != {EXPECTED_SNAPSHOT["aligned_slice_sha256"]}'
)
assert df['date'].is_monotonic_increasing, 'Dates must be strictly monotonic increasing!'
assert not df.isna().any().any(), 'Aligned DataFrame contains unexpected NaN values!'

n_bars = len(df)
split_idx = max(25, min(n_bars - 25, int(round(n_bars * spec.execution.train_split_ratio))))
split_date = str(df['date'].iloc[split_idx])
assert split_date == EXPECTED_SNAPSHOT['split_date'], f'Split date mismatch: {split_date}'

print(f'Verified {n_bars} aligned trading days ({df["date"].iloc[0]} to {df["date"].iloc[-1]})')
print(f'Formation Split [0:{split_idx}]: {df["date"].iloc[0]} to {df["date"].iloc[split_idx - 1]} ({split_idx} days)')
print(f'Out-of-Sample Split [{split_idx}:{n_bars}]: {split_date} to {df["date"].iloc[-1]} ({n_bars - split_idx} days)')
display(df.head(5))

Verified 2011 aligned trading days (2018-01-02 to 2025-12-31)
Formation Split [0:1006]: 2018-01-02 to 2021-12-29 (1006 days)
Out-of-Sample Split [1006:2011]: 2021-12-30 to 2025-12-31 (1005 days)


,date,open_a,high_a,low_a,close_a,volume_a,open_bench,close_bench
0,2018-01-02,234.555390,235.404849,234.170067,235.369812,86655700,234.555390,235.369812
1,2018-01-03,235.536252,237.007498,235.536252,236.858612,90070400,235.536252,236.858612
2,2018-01-04,237.497832,238.338525,236.919847,237.856857,80636400,237.497832,237.856857
3,2018-01-05,238.645069,239.564574,238.154663,239.441986,83524000,238.645069,239.441986
4,2018-01-08,239.345582,240.037416,239.056604,239.879791,57319200,239.345582,239.879791


## Cell 4 — Strictly Causal Signal Construction & Prefix-Invariance Proof
Computes the signal series at bar $t$ using only prices $[0..t]$. In `ols_log` pairs mode, OLS parameters $(\hat{\alpha}_t, \hat{\beta}_t)$ are estimated on expanding windows $[0..t]$ inside the Formation split and frozen at `split_idx` for Out-of-Sample evaluation. We prove zero look-ahead bias by mutating future prices both inside Formation ($m = \lfloor k/2 \rfloor$) and at the split boundary ($m = k$) and asserting that all prior signals $[0..m)$ are 100% invariant.

In [4]:
# Cell 4: Compute Causal Signals & Prove Prefix Invariance
precomputed = compute_causal_signals(df, spec, split_idx)
signal_values, indicators, pairs_diag, ols_alpha, ols_beta = precomputed

# Prefix-Invariance Test 1: Mutate prices in second half of Formation [m_form : split_idx)
m_form = max(35, split_idx // 2)
df_mut_form = df.copy(deep=True)
df_mut_form.loc[m_form:, 'close_a'] *= np.linspace(1.0, 1.45, n_bars - m_form)
sig_mut_form, _, _, _, _ = compute_causal_signals(df_mut_form, spec, split_idx)
mask_form = np.isfinite(signal_values[:m_form])
assert np.allclose(signal_values[:m_form][mask_form], sig_mut_form[:m_form][mask_form], atol=1e-12), (
    'Look-ahead violation detected inside Formation window!'
)

# Prefix-Invariance Test 2: Mutate Out-of-Sample prices [split_idx : n_bars)
df_mut_oos = df.copy(deep=True)
df_mut_oos.loc[split_idx:, 'close_a'] *= 1.75
sig_mut_oos, _, _, _, _ = compute_causal_signals(df_mut_oos, spec, split_idx)
mask_oos = np.isfinite(signal_values[:split_idx])
assert np.allclose(signal_values[:split_idx][mask_oos], sig_mut_oos[:split_idx][mask_oos], atol=1e-12), (
    'Look-ahead violation detected across split boundary!'
)
print(f'Causal prefix-invariance verified at m_form={m_form} ({df["date"].iloc[m_form]}) and split_idx={split_idx} ({split_date}).')

Causal prefix-invariance verified at m_form=503 (2020-01-02) and split_idx=1006 (2021-12-30).


## Cell 5 — Walk-Forward Trade Simulation & Exact Compounding Reconciliation
Executes signals triggered at the close of bar $t$ on bar $t+1$ (`next_open` or `next_close`) with unit gross portfolio exposure ($|w_A| + |w_B| = 1.0$) and explicit commission + slippage fees. Verifies that compounded per-trade net returns reconcile with terminal portfolio net equity.

In [5]:
# Cell 5: Run Bar-by-Bar Walk-Forward Simulation & Verify Trade Ledger
sim = simulate_strategy_core(df, spec, split_idx, precomputed_signals=precomputed)
trades = sim['trades']
equity_net = sim['equity_net']
equity_gross = sim['equity_gross']
daily_net_ret = sim['daily_net_ret']
daily_gross_ret = sim['daily_gross_ret']
daily_bench_ret = sim['daily_bench_ret']
daily_pos = sim['daily_position']

# Reconcile compounded daily returns vs terminal equity
initial_cap = spec.execution.initial_capital
assert math.isclose(initial_cap * float(np.prod(1.0 + daily_net_ret)), float(equity_net[-1]), rel_tol=1e-9)

# Reconcile compounded trade net returns vs terminal equity
if trades:
    compounded_trades_eq = initial_cap * float(np.prod([1.0 + t.net_return_pct / 100.0 for t in trades]))
    assert math.isclose(compounded_trades_eq, float(equity_net[-1]), rel_tol=1e-3), (
        f'Trade ledger vs daily equity mismatch: {compounded_trades_eq:.2f} vs {equity_net[-1]:.2f}'
    )
    trades_df = pd.DataFrame([t.model_dump() for t in trades])
    print(f'Executed {len(trades)} trades. Terminal Net Equity: ${equity_net[-1]:,.2f} (Compounded Trade Ledger: ${compounded_trades_eq:,.2f})')
    display(trades_df[['trade_id', 'split', 'direction', 'signal_date', 'entry_date', 'exit_date', 'holding_days', 'leg_weight_a', 'leg_weight_b', 'gross_return_pct', 'cost_pct', 'net_return_pct', 'exit_reason', 'cumulative_equity_after']].head(15))
else:
    print('Zero trades triggered across sample.')

Executed 76 trades. Terminal Net Equity: $94,266.95 (Compounded Trade Ledger: $94,266.56)


,trade_id,split,direction,signal_date,entry_date,exit_date,holding_days,leg_weight_a,leg_weight_b,gross_return_pct,cost_pct,net_return_pct,exit_reason,cumulative_equity_after
0,1,in_sample,long,2018-02-02,2018-02-05,2018-02-06,1,1.0,0.0,-4.9406,0.2,-5.1406,holding_period (1d),94859.42
1,2,in_sample,long,2018-02-08,2018-02-09,2018-02-12,1,1.0,0.0,1.1618,0.2,0.9618,holding_period (1d),95771.77
2,3,in_sample,long,2018-03-22,2018-03-23,2018-03-26,1,1.0,0.0,-0.7722,0.2,-0.9722,holding_period (1d),94840.64
3,4,in_sample,long,2018-04-02,2018-04-03,2018-04-04,1,1.0,0.0,-0.8189,0.2,-1.0189,holding_period (1d),93874.30
4,5,in_sample,long,2018-04-06,2018-04-09,2018-04-10,1,1.0,0.0,1.1095,0.2,0.9095,holding_period (1d),94728.11
5,6,in_sample,long,2018-10-10,2018-10-11,2018-10-12,1,1.0,0.0,-0.1119,0.2,-0.3119,holding_period (1d),94432.70
6,7,in_sample,long,2018-10-24,2018-10-25,2018-10-26,1,1.0,0.0,-0.5460,0.2,-0.7460,holding_period (1d),93728.19
7,8,in_sample,long,2018-12-04,2018-12-06,2018-12-07,1,1.0,0.0,1.3312,0.2,1.1312,holding_period (1d),94788.47
8,9,in_sample,long,2018-12-07,2018-12-10,2018-12-11,1,1.0,0.0,1.6289,0.2,1.4289,holding_period (1d),96142.89
9,10,in_sample,long,2018-12-21,2018-12-24,2018-12-26,1,1.0,0.0,-1.2843,0.2,-1.4843,holding_period (1d),94715.86


## Cell 6 — Running High-Water Mark ($E_0$ Included) & Maximum Drawdown Verification
Verifies the exact drawdown equations:
$$E_0 = C_0, \quad H_t = \max\!\left(E_0, \max_{1 \le s \le t} E_s\right), \quad D_t = 1 - \frac{E_t}{H_t}, \quad \mathrm{MDD} = \max_{1 \le t \le T} D_t$$
where $D_t \in [0, 1]$ is reported as a positive loss percentage in tables and $U_t = -D_t \le 0\%$ is plotted on underwater charts.

In [6]:
# Cell 6: Verify Running High-Water Mark Including Initial Capital E_0
running_peak = np.maximum(initial_cap, np.maximum.accumulate(equity_net))
drawdown_loss_pct = ((running_peak - equity_net) / running_peak) * 100.0
underwater_series_pct = -drawdown_loss_pct
max_dd_full = float(np.max(drawdown_loss_pct))

print(f'Full-Sample Max Drawdown (positive loss magnitude): {max_dd_full:.2f}%')
print(f'Underwater Chart Minimum (signed trough): {float(np.min(underwater_series_pct)):.2f}%')
if sim['circuit_breaker_tripped']:
    print(f'Portfolio Circuit Breaker Triggered on: {sim["circuit_breaker_date"]} (Limit: {spec.risk_controls.max_drawdown_circuit_breaker_pct}%)')

Full-Sample Max Drawdown (positive loss magnitude): 14.64%
Underwater Chart Minimum (signed trough): -14.64%


## Cell 7 — Split Performance & Risk Metrics Table
Computes `PeriodMetrics` across In-Sample Formation, Held-Out Out-of-Sample, and Full Sample, explicitly flagging zero-exposure / prior-halt splits.

In [7]:
# Cell 7: Compute PeriodMetrics for Full Sample, In-Sample, and Out-of-Sample
dates = sim['dates']
is_trades = [t for t in trades if t.split == 'in_sample']
oos_trades = [t for t in trades if t.split == 'out_of_sample']
cb_tripped = bool(sim['circuit_breaker_tripped'])
cb_date = sim['circuit_breaker_date']
cb_in_is = bool(cb_tripped and cb_date is not None and cb_date < split_date)
cb_in_oos = bool(cb_tripped and cb_date is not None and cb_date >= split_date)

full_metrics = _compute_period_metrics('Full Sample', dates, daily_gross_ret, daily_net_ret, daily_bench_ret, daily_pos, trades, cb_tripped, cb_date)
is_metrics = _compute_period_metrics('In-Sample Formation', dates[:split_idx], daily_gross_ret[:split_idx], daily_net_ret[:split_idx], daily_bench_ret[:split_idx], daily_pos[:split_idx], is_trades, cb_in_is, cb_date if cb_in_is else None, halted_prior_to_period=False)
oos_metrics = _compute_period_metrics('Out-of-Sample (Held-Out)', dates[split_idx:], daily_gross_ret[split_idx:], daily_net_ret[split_idx:], daily_bench_ret[split_idx:], daily_pos[split_idx:], oos_trades, cb_in_oos, cb_date if cb_tripped else None, halted_prior_to_period=cb_in_is)

def _fmt_sr(m):
    if m.is_zero_exposure:
        return 'N/A (0 trades — Halted in Formation)' if m.halted_prior_to_period else 'N/A (0 trades)'
    return f'{m.sharpe_ratio:.3f} (PSR {m.probabilistic_sharpe_psr_pct:.1f}%)'

period_summary_df = pd.DataFrame([
    {
        'Period': m.period_label,
        'Date Range': f'{m.start_date} -> {m.end_date}',
        'Days': m.trading_days,
        'Trades (N)': m.trade_count,
        'Exposure (%)': f'{m.exposure_pct:.1f}%',
        'Net Return (%)': f'{m.total_net_return_pct:+.2f}%',
        'Benchmark (%)': f'{m.benchmark_return_pct:+.2f}%',
        'Excess (pp)': f'{m.excess_return_pct:+.2f} pp',
        'Max DD (%)': f'{m.max_drawdown_pct:.2f}%',
        'Sharpe / PSR': _fmt_sr(m),
        'Circuit Breaker': m.circuit_breaker_date or ('Halted Prior' if m.halted_prior_to_period else 'Active'),
    }
    for m in (oos_metrics, is_metrics, full_metrics)
])
display(period_summary_df)

,Period,Date Range,Days,Trades (N),Exposure (%),Net Return (%),Benchmark (%),Excess (pp),Max DD (%),Sharpe / PSR,Circuit Breaker
0,Out-of-Sample (Held-Out),2021-12-30 -> 2025-12-31,1005,34,3.4%,-2.45%,+50.98%,-53.43 pp,9.23%,-0.103 (PSR 41.8%),Active
1,In-Sample Formation,2018-01-02 -> 2021-12-29,1006,42,4.2%,-3.37%,+90.41%,-93.77 pp,14.64%,-0.050 (PSR 46.0%),Active
2,Full Sample,2018-01-02 -> 2025-12-31,2011,76,3.8%,-5.73%,+187.48%,-193.21 pp,14.64%,-0.066 (PSR 42.6%),Active


## Cell 8 — Econometric & Non-IID Statistical Inference
Computes Lo (2002) / Mertens (2002) non-normality-adjusted Sharpe standard errors, Newey-West (1987) HAC factor regression against the benchmark, Politis & Romano (1994) Stationary Bootstrap confidence intervals, and (for pairs) Engle-Granger / ADF cointegration and discrete Ornstein-Uhlenbeck half-life $t_{1/2} = -\ln(2)/\ln(1 + \hat{\lambda})$.

In [8]:
# Cell 8: Econometric Diagnostics (Newey-West HAC, Stationary Bootstrap, Pairs Cointegration)
factor_reg = newey_west_factor_regression(daily_net_ret, daily_bench_ret)
boot_cis = stationary_bootstrap_cis(daily_net_ret, daily_bench_ret, [t.net_return_pct for t in trades], replications=1000, seed=BOOTSTRAP_SEED)

print(f'Newey-West HAC Annualized Alpha: {factor_reg.annualized_alpha_pct:+.2f}% (t_HAC = {factor_reg.alpha_t_stat_hac:.2f}, p = {factor_reg.alpha_p_value_hac:.4f}), Beta = {factor_reg.beta:.3f}')
boot_df = pd.DataFrame([b.model_dump() for b in boot_cis])
display(boot_df[['metric_name', 'point_estimate', 'ci_lower_95', 'ci_upper_95', 'bootstrap_std_error', 'block_length', 'replications']])

if pairs_diag is not None:
    print('\nPairs Cointegration & Co-Movement Diagnostics (Formation Window):')
    pairs_df = pd.DataFrame([{
        'Daily Return Corr (r)': round(pairs_diag.return_correlation, 4),
        'Log-Level Corr': round(pairs_diag.level_correlation, 4),
        'OLS Beta': round(pairs_diag.ols_hedge_ratio_beta, 4),
        'OLS Alpha': round(pairs_diag.ols_alpha, 4),
        'ADF Stat': round(pairs_diag.adf_statistic, 3),
        'ADF p-value': round(pairs_diag.adf_p_value, 4),
        'OU Lambda': round(pairs_diag.ou_lambda, 5),
        'Half-Life (days)': pairs_diag.half_life_days,
        'Hurst H': round(pairs_diag.hurst_exponent, 3),
    }])
    display(pairs_df)

Newey-West HAC Annualized Alpha: -2.86% (t_HAC = -1.44, p = 0.1495), Beta = 0.157


,metric_name,point_estimate,ci_lower_95,ci_upper_95,bootstrap_std_error,block_length,replications
0,Annualized Return (Mean),-0.470,-4.110,2.910,1.760,13,1000
1,Annualized Sharpe Ratio,-0.066,-0.578,0.423,0.255,13,1000
2,Annualized Excess Return vs. Benchmark,-15.610,-25.660,-5.250,5.200,13,1000
3,Mean Net Return per Trade,-0.057,-0.505,0.371,0.224,1,1000


## Cell 9 — Standard Literature Threshold Bracket & 25-Trial Parameter Surface (DSR)
Evaluates Conservative, Baseline, and Aggressive literature profiles alongside the $5 \times 5$ parameter sensitivity grid ($K=25$ trials) and Bailey & López de Prado (2014) Deflated Sharpe Ratio.

In [9]:
# Cell 9: Compute Standard Bracket Profiles & 5x5 Parameter Sensitivity Surface
bracket_profiles = compute_standard_bracket_profiles(df, spec, split_idx)
param_surface = compute_parameter_sensitivity_surface(df, spec, split_idx) if 'parameter_sensitivity_grid' in spec.selected_diagnostics else None

bracket_df = pd.DataFrame([
    {
        'Profile': bp.profile_label,
        'Entry': bp.entry_threshold,
        'Exit': bp.exit_threshold,
        'IS Net (%)': bp.in_sample_metrics.total_net_return_pct,
        'OOS Net (%)': bp.out_of_sample_metrics.total_net_return_pct,
        'Full Net (%)': bp.full_sample_metrics.total_net_return_pct,
        'Full Sharpe': bp.full_sample_metrics.sharpe_ratio,
        'Max DD (%)': bp.full_sample_metrics.max_drawdown_pct,
    }
    for bp in bracket_profiles
])
display(bracket_df)
if param_surface is not None:
    print(f'25-Cell Grid Expected Max SR under Null: {param_surface.expected_max_sr_under_null:.3f} | Selected Spec DSR: {param_surface.selected_spec_deflated_sharpe_psr_pct:.1f}%')

,Profile,Entry,Exit,IS Net (%),OOS Net (%),Full Net (%),Full Sharpe,Max DD (%)
0,Aggressive (1.30% Shock),1.3,0.0,-6.71,-20.57,-25.90,-0.429,25.96
1,Baseline (2.00% Shock),2.0,0.0,-3.37,-2.45,-5.73,-0.066,14.64
2,Conservative Tail (3.00% Shock),3.0,0.0,-5.59,-4.07,-9.44,-0.169,16.72


25-Cell Grid Expected Max SR under Null: 0.535 | Selected Spec DSR: 4.4%


## Cell 10 — Five Adversarial Stress Tests & Non-Overlapping Monte Carlo Null
Runs all 5 adversarial stress tests: (1) Right-Tail Outlier Trade Removal, (2) $t+2$ Execution Latency & $2\times/3\times$ Slippage Shock, (3) Realized Volatility & Bull/Bear Macro Regime Split, (4) 500-Run Matched Non-Overlapping Monte Carlo Random-Entry Benchmark, and (5) Out-of-Sample Cointegration Structural Break / Parameter Cliff-Edge.

In [10]:
# Cell 10: Execute Adversarial Stress Test Suite
def _stress_sim_cb(d_frame, s_spec, s_idx, cost_override, delay_bars):
    return simulate_strategy_core(
        d_frame, s_spec, s_idx,
        override_cost_bps_per_leg=cost_override,
        entry_delay_bars=delay_bars,
        precomputed_signals=precomputed,
    )

stress_suite = run_adversarial_stress_suite(
    df=df, spec=spec, split_idx=split_idx, base_sim=sim,
    full_metrics=full_metrics, oos_metrics=oos_metrics,
    pairs_diag=pairs_diag, parameter_surface=param_surface,
    simulate_fn=_stress_sim_cb,
)
stress_df = pd.DataFrame([
    {
        'Test ID': t.test_id,
        'Title': t.title,
        'Outcome': t.outcome_badge,
        'Baseline': f'{t.baseline_label}: {t.baseline_value}',
        'Stressed': f'{t.stressed_label}: {t.stressed_value}',
    }
    for t in stress_suite.tests
])
display(stress_df)

,Test ID,Title,Outcome,Baseline,Stressed
0,outlier_removal,Right-Tail Outlier Concentration Test (Excl. T...,HIGH_VULNERABILITY,All Trades Compounded: -5.73% (N=76),Excl. Top 3 Trades: -15.07% (N=73)
1,execution_latency_shock,Execution Latency (t+2 Fill Delay) & Liquidity...,HIGH_VULNERABILITY,"Baseline (t+1, 10 bps): -5.73% (SR -0.07)",t+2 Delay + 2× Cost: -16.49% (SR -0.35)
2,regime_vulnerability,Realized Volatility Terciles & Bull/Bear Macro...,HIGH_VULNERABILITY,Positive Regimes: 1 / 5 Regimes,High-Vol vs Low-Vol SR: 0.03 vs 0.61
3,monte_carlo_random_entry,Monte Carlo Random-Entry Null Benchmark (500 N...,MODERATE_SENSITIVITY,Actual Signal Return: -5.73% (71th pct),Random-Entry Median: -10.99% (p=0.290±0.020)
4,parameter_cliff_edge,Parameter Neighborhood Cliff-Edge Perturbation...,MODERATE_SENSITIVITY,Selected Spec Sharpe: -0.07,Grid Median (Pos Cells): -0.14 (10/25 pos)


## Cell 11 — Publication Figures (Equity Curve, Underwater Drawdown & Causal Signal Trajectory)
Generates the primary visual diagnostics with explicit Out-of-Sample split and circuit-breaker annotations.

In [11]:
# Cell 11: Plot Equity Curve, Underwater Drawdown, and Causal Signal Trajectory
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

bench_equity = initial_cap * np.cumprod(1.0 + daily_bench_ret)
x_idx = np.arange(n_bars)
fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)

axes[0].plot(x_idx, equity_net, color='#059669', lw=1.8, label='Confirmed Rule (Net)')
axes[0].plot(x_idx, bench_equity, color='#78716c', lw=1.3, ls='--', label=f'Benchmark ({spec.benchmark_ticker})')
axes[0].axvline(split_idx, color='#d97706', ls=':', lw=1.2, label=f'OOS Split ({split_date})')
axes[0].set_ylabel('Portfolio Equity ($)')
axes[0].set_title(f'{spec.title} — Walk-Forward Equity, Underwater Drawdown & Causal Signal')
axes[0].legend(loc='upper left', fontsize=8)
axes[0].grid(True, alpha=0.25)

axes[1].fill_between(x_idx, underwater_series_pct, 0.0, color='#e11d48', alpha=0.22)
axes[1].plot(x_idx, underwater_series_pct, color='#e11d48', lw=1.2, label='Underwater Drawdown (%)')
axes[1].axvline(split_idx, color='#d97706', ls=':', lw=1.2)
axes[1].set_ylabel('Drawdown (%)')
axes[1].legend(loc='lower left', fontsize=8)
axes[1].grid(True, alpha=0.25)

axes[2].plot(x_idx, signal_values, color='#0284c7', lw=1.1, label=f'Causal Signal ({spec.signal_variant})')
axes[2].axvline(split_idx, color='#d97706', ls=':', lw=1.2)
axes[2].set_ylabel('Signal Value')
axes[2].set_xlabel('Trading Bar Index')
axes[2].legend(loc='upper left', fontsize=8)
axes[2].grid(True, alpha=0.25)

plt.tight_layout()
plt.close(fig)
print('Generated 3-panel Matplotlib diagnostic figure successfully.')

Generated 3-panel Matplotlib diagnostic figure successfully.


## Cell 12 — Empirical Hypothesis Scorecard, Multi-Question Breakdown & Formula Ledger
Synthesizes the 3-layer hypothesis comparison (separating return co-movement, cointegration stationarity, and trading rule P&L), the 5 scorecard dimensions, and the step-by-step substituted formula ledger.

In [12]:
# Cell 12: Build Empirical Hypothesis Scorecard & Formula Calculation Ledger
if len(trades) > 0 and full_metrics.total_gross_return_pct > 0:
    avg_gross_bps = float(np.mean([t.gross_return_pct for t in trades])) * 100.0
    break_even_bps = round(max(0.0, avg_gross_bps / 2.0), 1)
else:
    break_even_bps = 0.0

scorecard = build_hypothesis_scorecard_and_ledger(
    spec=spec,
    full_metrics=full_metrics,
    in_metrics=is_metrics,
    oos_metrics=oos_metrics,
    factor_regression=factor_reg,
    bootstrap_cis=boot_cis,
    break_even_cost_bps=break_even_bps,
    parameter_surface=param_surface,
    pairs_diag=pairs_diag,
    stress_suite=stress_suite,
)

print(f'Historical Verdict: {scorecard.historical_verdict_label} ({scorecard.supported_checks_count}/{scorecard.evaluated_checks_count} evaluated checks supported)')
print(scorecard.executive_summary)

if scorecard.hypothesis_comparison_breakdown:
    display(pd.DataFrame(scorecard.hypothesis_comparison_breakdown))

display(pd.DataFrame([d.model_dump() for d in scorecard.dimensions]))
display(pd.DataFrame([s.model_dump() for s in scorecard.calculation_steps]))

Historical Verdict: CONTRADICTED BY HISTORICAL DATA (0/5 evaluated checks supported)
Historical Audit Summary for 'I think the S&P 500 tends to recover the day after falling more than 2%.': Across 2011 trading days (2018-01-02 to 2025-12-31), the specified rule triggered 76 trades (In-Sample N = 42, Out-of-Sample N = 34), generating -5.73% net return (Sharpe -0.07, Max Drawdown 14.64%) versus +187.48% (Sharpe 0.78) for passive SPY Buy-and-Hold (-193.21 percentage points excess return). The trading rule satisfied 0 of 5 evaluated empirical checks (CONTRADICTED BY HISTORICAL DATA). Note: This scorecard summarizes historical sample evidence using explicit heuristic decision thresholds and does not constitute investment advice or a prediction of future performance.


,aspect,lens_title,question_answered,evidence,primary_metric,verdict,status,explanation,interpretation
0,1. Passive Benchmark Drift (SPY Buy-and-Hold),1. Passive Benchmark Drift (SPY Buy-and-Hold),How much did unconditional SPY drift return ov...,"Return: +187.48% (CAGR +14.15%, Sharpe 0.78, M...","Return: +187.48% (CAGR +14.15%, Sharpe 0.78, M...",PASSIVE BASELINE,MIXED,Unconditional buy-and-hold performance of SPY ...,Unconditional buy-and-hold performance of SPY ...
1,2. Post-Signal Event Expectancy (Net of Fees),2. Post-Signal Event Expectancy (Net of Fees),What was the mean net return per triggered SPY...,Mean Trade: -0.057% | Win Rate: 54.0% (N = 76 ...,Mean Trade: -0.057% | Win Rate: 54.0% (N = 76 ...,NEGATIVE / ZERO EXPECTANCY,CONTRADICTED,Average net return per triggered trade after 1...,Average net return per triggered trade after 1...
2,3. Active Strategy vs. Benchmark & Out-of-Samp...,3. Active Strategy vs. Benchmark & Out-of-Samp...,Did the active rule outperform SPY and persist...,"Net Return: -5.73% (Excess: -193.21 pp, IS N =...","Net Return: -5.73% (Excess: -193.21 pp, IS N =...",UNDERPERFORMED BENCHMARK,CONTRADICTED,Strategy achieved Sharpe -0.07 with 3.8% marke...,Strategy achieved Sharpe -0.07 with 3.8% marke...


,dimension_id,name,question,status,evidence_metric,explanation
0,post_signal_direction,1. Post-Signal Direction & Net Trade Expectancy,Did historical occurrences of the signal produ...,MIXED,Mean Net Trade: -0.057% | Win Rate: 54.0% (N =...,Across 76 historical signals from 2018-01-02 t...
1,statistical_significance,2. Statistical Significance vs. Noise (Lo 2002...,Is the observed return/Sharpe ratio statistica...,CONTRADICTED,Sharpe: -0.07 ± 0.35 (PSR 42.6%) | HAC Alpha t...,After adjusting for sample skewness (-2.89) an...
2,out_of_sample_persistence,3. Held-Out Out-of-Sample Persistence,Did the hypothesis continue to hold in the uns...,CONTRADICTED,"In-Sample SR: -0.05 (-3.4%, N=42) → Out-of-Sam...",In the held-out validation sample (2021-12-30 ...
3,transaction_cost_survival,4. Transaction Cost & Microstructure Friction ...,Do historical returns survive realistic commis...,CONTRADICTED,Break-Even Cost: 7.1 bps/notional (Configured:...,Deducting 10.0 bps per unit of portfolio notio...
4,stress_test_resilience,5. Adversarial Stress-Test & Null-Benchmark Re...,"Did the signal survive outlier removal, regime...",CONTRADICTED,0/5 Resilient | 2/5 Moderate | 3/5 High Vulner...,"Out of 5 adversarial stress tests in Step 5, t..."


,step_id,title,formula,substituted_inputs,output_value,interpretation
0,calc_net_return,"Step A — Compounded Net Trade Return, Unit Gro...","R_{net,i} = R_{gross,i} − 2·c, |w_A| + |w_B|...","Initial Capital E_0 = $100,000.00, Unit gross ...",Gross = +9.74% → Net = -5.73% (Mean Net/Trade ...,Every trade deducts round-trip commission + sl...
1,calc_lo_sharpe,Step B — Annualized Sharpe Ratio & Lo (2002) /...,"SR = √252 × (μ_d / σ_d), SE(SR_d) = √[(1 − γ...","T = 2011 days, Ann. Vol = 7.25%, Skewness γ₃ =...","Annualized SR = -0.066, Lo (2002) SE = 0.352, ...",Adjusts standard Sharpe inference for non-norm...
2,calc_dsr,Step C — Bailey & López de Prado (2014) Multip...,E[max SR₀] ≈ √V[{SR_k}] × [(1 − γ)·Φ⁻¹(1 − 1/K...,"K = 25 parameter grid trials, Null Hurdle E[ma...",Selected Spec DSR = 4.4% (Grid Best DSR = 41.3%),Penalizes the observed Sharpe ratio for the nu...
3,calc_newey_west,Step D — Newey-West (1987) HAC Benchmark Facto...,"R_strat,t = α + β · R_bench,t + ε_t, Var_HAC...","Benchmark = SPY, Bartlett kernel lags L = 7, S...","Annualized α = -2.86% (t_HAC = -1.44, p = 0.14...",Separates passive correlation with the benchma...
4,calc_stationary_bootstrap,Step E — Politis & Romano (1994) Stationary Bo...,"L_block ~ Geometric(p = 1 / b_opt), CI_95% =...","Metric = Annualized Return (Mean), Mean block ...","Point Estimate = -0.47%, 95% CI = [-4.11%, 2.9...",Preserves volatility clustering and serial dep...


## Cell 13 — Automated End-to-End Reconciliation Assertions
Programmatically asserts that every metric, trade count, drawdown, econometric statistic, and verdict recomputed in this notebook matches the API `AuditReport` payload.

In [13]:
# Cell 13: Assert Exact Reconciliation Between Notebook Execution and AuditReport
assert len(trades) == EXPECTED_SNAPSHOT['trade_count'], f'Trade count mismatch: {len(trades)} != {EXPECTED_SNAPSHOT["trade_count"]}'

for split_key, comp_metrics in [
    ('full_sample', full_metrics),
    ('in_sample', is_metrics),
    ('out_of_sample', oos_metrics),
]:
    exp_m = EXPECTED_SNAPSHOT[split_key]
    for field_name in (
        'trading_days', 'trade_count', 'exposure_pct', 'total_gross_return_pct',
        'total_net_return_pct', 'cagr_net_pct', 'benchmark_return_pct',
        'max_drawdown_pct', 'benchmark_max_drawdown_pct', 'sharpe_ratio',
        'probabilistic_sharpe_psr_pct', 'win_rate_pct', 'is_zero_exposure',
        'circuit_breaker_triggered', 'halted_prior_to_period'
    ):
        actual_val = getattr(comp_metrics, field_name)
        expected_val = exp_m[field_name]
        if isinstance(actual_val, float):
            assert math.isclose(actual_val, float(expected_val), abs_tol=1e-6), (
                f'Mismatch in {split_key}.{field_name}: {actual_val} != {expected_val}'
            )
        else:
            assert actual_val == expected_val, (
                f'Mismatch in {split_key}.{field_name}: {actual_val} != {expected_val}'
            )

for exp_b, act_b in zip(EXPECTED_SNAPSHOT['bootstrap_cis'], boot_cis):
    assert exp_b['metric_name'] == act_b.metric_name, f'Bootstrap metric name mismatch: {act_b.metric_name}'
    assert math.isclose(float(exp_b['point_estimate']), act_b.point_estimate, abs_tol=1e-6), f'Bootstrap point estimate mismatch for {act_b.metric_name}'
    assert math.isclose(float(exp_b['ci_lower_95']), act_b.ci_lower_95, abs_tol=1e-6), f'Bootstrap CI lower mismatch for {act_b.metric_name}'
    assert math.isclose(float(exp_b['ci_upper_95']), act_b.ci_upper_95, abs_tol=1e-6), f'Bootstrap CI upper mismatch for {act_b.metric_name}'

for exp_st, act_st in zip(EXPECTED_SNAPSHOT['stress_test_outcomes'], stress_suite.tests):
    assert exp_st['test_id'] == act_st.test_id, f'Stress test ID mismatch: {act_st.test_id}'
    assert exp_st['outcome_badge'] == act_st.outcome_badge, f'Stress test outcome mismatch for {act_st.test_id}'
    assert exp_st['baseline_value'] == act_st.baseline_value, f'Stress test baseline mismatch for {act_st.test_id}'
    assert exp_st['stressed_value'] == act_st.stressed_value, f'Stress test stressed mismatch for {act_st.test_id}'

assert scorecard.historical_verdict_label == EXPECTED_SNAPSHOT['verdict_label'], 'Verdict label mismatch!'
assert scorecard.supported_checks_count == EXPECTED_SNAPSHOT['supported_checks_count'], 'Supported checks count mismatch!'
assert scorecard.evaluated_checks_count == EXPECTED_SNAPSHOT['evaluated_checks_count'], 'Evaluated checks count mismatch!'
for d in scorecard.dimensions:
    assert d.status == EXPECTED_SNAPSHOT['dimension_statuses'][d.dimension_id], f'Dimension {d.dimension_id} status mismatch!'

print('ALL AUTOMATED RECONCILIATION ASSERTIONS PASSED (100% match with API AuditReport).')

ALL AUTOMATED RECONCILIATION ASSERTIONS PASSED (100% match with API AuditReport).


## Cell 14 — Complete UI-to-Notebook-to-Engine Coverage Map
Maps every metric, table, chart, and badge in the 6-Step Alpha Auditor UI and PDF report to the exact production Python function and notebook cell that verifies it.

| UI Step & Component | Displayed Outputs | Production Function | Notebook Verification Cell |
| :--- | :--- | :--- | :--- |
| **Step 1 & 2: Intake & Spec Builder** | `StrategySpec`, User vs. Software Defaults, Ambiguities | `engine.hypothesis_parser.interpret_hypothesis` | Cell 2 |
| **Step 3: Literature & Rigor Checklist** | Curated Papers, Causal & Split Safeguards | `engine.literature.build_literature_review` | Cell 3 & Cell 4 |
| **Step 4: KPI Strip & Split Table** | OOS / IS / Full Net Return, Excess pp, Sharpe, PSR, MDD, N | `engine.backtester._compute_period_metrics` | Cell 6 & Cell 7 |
| **Step 4: Equity, Drawdown & Signal Charts** | `equity_net`, `equity_benchmark`, `drawdown_net_pct`, `signal_values` | `engine.backtester.simulate_strategy_core` & `compute_causal_signals` | Cell 4, Cell 5, Cell 6, Cell 11 |
| **Step 4: Econometrics & 25-Cell DSR Grid** | Newey-West HAC $\alpha/\beta$, Stationary Bootstrap CI, Pairs ADF/OU, 5×5 DSR | `engine.econometrics.*` & `compute_parameter_sensitivity_surface` | Cell 8 & Cell 9 |
| **Step 5: Adversarial Stress Suite** | Outlier Removal, $t+2$ Latency, Regimes, Non-Overlapping MC Null, Drift | `engine.stress_tests.run_adversarial_stress_suite` | Cell 10 |
| **Step 6: Verdict, Multi-Question Breakdown & Blotter** | Co-Movement vs. Cointegration vs. Rule P&L, 5 Dimensions, Formula Ledger, Trades | `engine.stress_tests.build_hypothesis_scorecard_and_ledger` | Cell 5, Cell 12, Cell 13 |